# [캐글 v60 = v58 + 진단패치 3종] (v58 = v57(LB 8.19) + 자산 씨앗·승격 렌즈) v56 골격(LB 2.5) + **서술 에이전트 v2** — 규칙함수 선해결 → 서술·렌즈·샌드박스 다중턴 → 2패스 재개방

**한 줄**: LB 2.5 를 낸 `56qwen3827bfp8rulekgmonarc262025winning` 의 골격(인터넷 OFF 휠하우스 vLLM · 규칙 스택 69 도구 선해결 · 더미 제출 즉시 저장 · easy-first · 1패스/2패스 · 11.4h 자동 스톱)은 그대로 두고, **LLM 경로만** 'K16 규칙 투표' 에서 **v2 서술 에이전트**(모델이 태스크마다 파이썬 샌드박스와 다중턴으로 서술 → 코드 → 등급 검증 → 서술 수정; 하네스가 0턴에 `task_report()` 실행) 로 바꿨다. 목표: 2.5 에서 조금이라도 오르는지 확인.

| 셀 | 내용 | 출처 |
|---|---|---|
| 1 | 환경: 유틸 경로 차단 · py3.12 확인 · 휠하우스 오프라인 vLLM 설치 · `_t0.json`(자동 스톱 기준) | 풀이전용판 셀1 검증 코드 (= v56 셀1 + py3.12/의존성 하드체크) |
| 1.5 | 규칙함수 엔진 내장 (order_engine 3322줄 + rule_stack_all 3002줄) | **v56 그대로** |
| 2 | 설정 · 데이터(rerun=test 240 / 리허설=train 4 + eval 1) · 더미 submission · **규칙 선해결(v56 그대로, eval 120 실측 11 채택/오답 0)** · LLM 대상 풀 | v56 골격 + v2 노브 |
| 3~4 | 샌드박스 소스: 격자 프리미티브 50 + 매크로(task_report/try_transforms/check/submit) + **서술함수 24**(관찰카드·게이트1·4축/silx 등급·단계·반례·렌즈증거·씨앗) + 워커 | v2 |
| 5 | 프롬프트: 서술 우선 프로토콜 + 헬퍼/매크로/배경/**렌즈 카탈로그** 부록, 첫 메시지/도구 메시지/**재개방 메시지** | v2 (+reopen) |
| 6~7 | 서브프로세스 샌드박스 · vLLM 엔진(messages 입력, 사고 2패스, 32768→16384 폴백) | v2 |
| 8 | **락스텝 루프: 1패스(5턴, 예산 적응) → 2패스(근접 문제 +4턴 재개방, 같은 대화 이어서 새 샌드박스) + 미착수 잔여** | v2 + v56 의 2패스 설계 |
| 9 | 요약 · 오프라인 채점(v56 과 같은 계산) · 매니페스트 | |

**실행 절차**: ① Add Input 3종 (**arc-prize-2026** / `driessmit1/arc3-vllm-h100-wheelhouse-v3` / Models `foysalemonshanto/qwen3-8-27b-fp8-repacked-v1` hf-fp8) ② Settings: Environment **Always use latest (py3.12)** · GPU **L4×4** · Internet **OFF** · 유틸 `pip_install_unsloth_flash_patch` **detach** ③ **Save & Run All** = 리허설 (train `fc754716 feca6190 ff28f65a fe45cba4` + eval `7b5033c1`, 예산 60분, 채점 출력) ④ 리허설 로그에서 라운드 시간·전쌍통과 제출 수 확인 후 **Submit to competition** = rerun (test 240, 11.4h).

**시간 산정 (v56 실측 기반)**: 적재 ~8분, K16×3.5K 토큰 생성 230s → 집계 ~245 tok/s. 에이전트 턴 = 사고 ≤3072 + 답 ≤1280 ≈ 4K 토큰, 동시 28 → 라운드 ≈ 6~8분. 240 − 규칙해결(~20) = 220 태스크 × 5턴 / 28 ≈ 40 라운드 ≈ 4~5h (조기 submit 시 더 짧음) → 2패스에 5h 이상 남는다. 라운드 시간이 예상보다 길면 `plan_turns` 가 새 태스크의 턴 상한을 3 까지 자동으로 줄인다. **★리허설(5문제 동시) 라운드 시간 × 2(28동시 환산) × 40 라운드가 11h 를 넘으면** 제출 전에 `THINK_BUDGET`(3072→2048) 또는 `MAX_TURNS_P1`(5→4) 을 낮춘다 (rerun 중에는 `plan_turns` 가 자동 대응).

**v56 대비 바뀐 것 / 안 바뀐 것**: 규칙 선해결·더미 제출·easy-first·2패스·마감 규약·데이터 탐색·모델 탐색 = 그대로. LLM 경로 = 서술 에이전트 (attempt_1 = 전쌍통과 submit, 없으면 train 통과 쌍수/정규화 점수 최고 후보; attempt_2 = 다른 후보). 강제 submit 은 하네스 최선 후보보다 train 통과 쌍수가 적으면 attempt_2 로 밀린다. ★미검증 2건(GPU): `MAX_MODEL_LEN=32768` 적재(실패 시 16384 자동), 27B 다중턴 처리량.


**v58 변경**: 셀4b 신설 — Add Input 의 자산 저장소(`arc2-desc-assets` Dataset / 채굴기 output 의 `assets/`)에서 씨앗·승격 렌즈를 읽어 워커 few-shot(SEEDS)과 렌즈 카탈로그에 더한다. 없으면 내장 폴백 씨앗만(= v57 과 같은 동작). 셀2 POOL 에 tid(자기 씨앗 제외), 셀5 승격 렌즈 한 줄, 셀4 매크로에 은행 변환 폴백 후보(train 쌍 ≥1 맞춘 은행 변환의 test 예측을 후보로 — 모델이 solve 를 못 내면 identity 대신 채택)와 서술 채우기. 규칙 선해결·2패스·노브는 v57 그대로.

---

## v60 변경점 (2026-09-20 진단패치 — 점수 기능 추가 아님, 계측·공정성 수정)

**셀10(서술규칙 후처리)은 포함하지 않는다.** v59 의 그 셀은 (a) `budget_per_rule=240` × 10 규칙 = 최대 ~40.5분이
공통 `DEADLINE` 을 공유하지 않아 11.4h 뒤 12h 한도를 넘길 위험, (b) 유효·상이한 `attempt_2` 를 무조건 교체,
(c) `_DESC_ORDER` 첫 매치 승 — 세 가지가 미해결이다. 베이스는 기능 상위집합인 v58.

| 패치 | 셀 | 내용 |
|---|---|---|
| P1 | 8 | `is_near` 에서 `or run.score == 1.0` 제거. 정답은 채점에만. 리허설과 제출의 탐색 경로 일치 |
| P2 | 8 | `pick_attempts` 를 테스트 입력별 중복 제거로. 묶음 비교가 입력 2개 태스크에서 슬롯을 낭비했다 |
| P3 | 7 | `disable_log_stats=False` + 1·2차 `finish_reason`·토큰 중앙값·프롬프트 최대 길이·실제 배치 크기 로깅 |

**P3 가 푸는 것**: 기존 로그의 `답 이어쓰기 0` 은 1차에서 사고를 끝낸 답만 세므로, 강제 `</think>` 후
2차 생성(`TOK_ANSWER`=1280)이 잘렸는지 알 수 없었다. 이제 `2차 finish_reason` 이 찍힌다.
사고 강제종료도 `'</think>' 없음` 분기가 아니라 실제 `finish_reason` 으로 구분된다.

### 먼저 할 것 — KV 판정 (코드 수정 0, ~8분)

이 모델은 `Qwen3_5ForConditionalGeneration` + GDN 선형어텐션 **하이브리드**다. GDN 상태는 요청당 상수(길이 무관),
풀어텐션 KV 만 길이 비례. 그래서 `GPU KV cache size: 166,208` 을 길이로 나눠 동시성을 구할 수 없다
(166,208/32,768 = 5.07 인데 로그는 18.91x 를 출력한다 — 두 줄은 단위가 다르다).

요청당 비용을 `C + L·k` 로 두면 L=32,768 에서 18.91. L=16,384 에서의 용량은 **18.91(상수지배) ~ 37.8(어텐션지배)**.
`CONCURRENCY=28` 이 그 구간 안에 있다.

환경변수로 한 번 적재하고 `Maximum concurrency` 한 줄만 읽으면 C/k 가 풀린다:

```
ARC2_KNOBS='{"MAX_MODEL_LEN": 16384}'
```

- ≈18.9 → 상수 지배. 28은 어떤 사고 예산에서도 안 들어간다. `CONCURRENCY` 를 내리는 것 외에 답이 없고
  prefix caching·fp8 KV·시퀀스 단축은 전부 무력 → 4조합 실험 불필요.
- ≈37.8 → 어텐션 지배. 28이 16K 길이에서 들어간다 → 그때 (prefix_caching × kv_cache_dtype) 4조합을 돌린다.

### 이 노트북으로 아직 하지 않은 것

`THINK_BUDGET`·`MAX_TURNS_P1`·`CONCURRENCY`·`enable_prefix_caching`·`kv_cache_dtype` 는 **v58 그대로 두었다.**
위 KV 판정과 P3 계측 결과를 보기 전에는 어느 방향이 맞는지 결정할 근거가 없다.


In [1]:
# [셀1] 환경 — ★인터넷 OFF 전제. vLLM 은 Add Input 휠 데이터셋에서 오프라인 설치. (풀이전용판 셀1 검증 코드 그대로 + 모드 스위치)
#   실측 교훈 누적판: 유틸경로 차단 / 하우스 격리(겹침 금지) / cp 호환 필터 /
#   모듈 캐시 청소 / --ignore-installed / 대체휠(blake3)은 필요할 때만.
import os, glob, subprocess, sys, time
import importlib.util as _iu

# ---- 모드 스위치 (v14 덕 노트북과 동일 규약): 대회 재실행이면 제출판, 아니면 오프라인 풀이판 ----
TRUE_SUBMISSION = bool(os.getenv('KAGGLE_IS_COMPETITION_RERUN'))   # v56 규약 그대로
RUN_VARIANT = 'arc2-desc-agent-v58-submit-assets'   # v57(8.19) + 채굴 자산 씨앗·승격 렌즈 주입
print(f'RUN_VARIANT={RUN_VARIANT} TRUE_SUBMISSION={TRUE_SUBMISSION}', flush=True)

# ---- (0) 유틸 스크립트 경로 원천 차단 — 반드시 다른 import 보다 먼저 ----
#   실측: pip_install_unsloth_flash_patch 의 cp311 잔재(numpy/torch/triton)가
#   py3.12 커널을 깨뜨림. detach 가 정석이나 안 떼도 무해하도록 차단.
_BAD = 'pip_install_unsloth_flash_patch'
sys.path[:] = [p for p in sys.path if _BAD not in p]
os.environ['PYTHONPATH'] = os.pathsep.join(
    p for p in os.environ.get('PYTHONPATH', '').split(os.pathsep)
    if p and _BAD not in p)
_purged0 = [k for k, m in list(sys.modules.items())
            if getattr(m, '__file__', None) and _BAD in str(m.__file__)]
for _k in _purged0:
    del sys.modules[_k]
print(f'유틸 경로 차단: {_BAD} (모듈 퇴출 {len(_purged0)}개) — detach 권장')

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('VLLM_WORKER_MULTIPROC_METHOD', 'spawn')
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'

# ---- (1) 존재 확인만 — ★import 금지 (실측: 미리 import 한 구버전 transformers
#      캐시가 새로 설치된 버전을 가려 vllm import 를 죽였다) ----
for pkg in ['transformers', 'accelerate', 'psutil']:
    if _iu.find_spec(pkg) is None:
        print(f'★{pkg} 없음 — 인터넷 OFF 이므로 설치 불가. 이미지 확인 필요.')

PYV = f'{sys.version_info.major}.{sys.version_info.minor}'
print(f'Python {PYV}')
# ★실측(08-27 v56 1차 런): 새 노트북이 원본 고정 환경(py3.11)으로 돌아 cp312 전용 휠
#   44개(aiohttp·msgspec·pydantic_core...)가 빠졌고, import vllm(지연)은 통과했다가
#   셀2 from vllm import LLM 이 msgspec 없음으로 사망. 환경이 틀리면 여기서 바로 멈춘다.
assert PYV == '3.12', (
    f'★환경이 py{PYV} — 이 노트북과 vLLM 휠하우스는 py3.12 전용. '
    '우측 Settings > Environment 를 "Always use latest environment" 로 바꾸고 재실행할 것.')


def _purge_hf_mods(with_torch=False):
    """새 설치가 캐시된 구버전에 가리지 않게 관련 모듈을 전부 내린다."""
    roots = {'transformers', 'tokenizers', 'huggingface_hub', 'safetensors',
             'vllm', 'pydantic', 'pydantic_core'}
    if with_torch:
        roots |= {'torch', 'triton'}
    n = 0
    for k in list(sys.modules):
        if k.split('.')[0] in roots:
            del sys.modules[k]; n += 1
    return n


# ---------------- vLLM: 사전설치 → 휠하우스(오프라인, 하우스 격리) ----------------
HAS_VLLM = False
try:
    import vllm
    HAS_VLLM = True
    print(f'★vLLM 사전설치됨: {vllm.__version__}')
except Exception:
    t0 = time.time()
    import re as _re, zipfile

    all_whl = glob.glob('/kaggle/input/**/*.whl', recursive=True)
    vllm_whl = [p for p in all_whl
                if os.path.basename(p).lower().startswith(('vllm-', 'vllm_'))]
    print(f'인풋 스캔: .whl {len(all_whl)}개 / vllm 휠 {len(vllm_whl)}개 '
          f'({time.time()-t0:.0f}s)')
    for p in vllm_whl[:5]:
        print('  vllm 휠:', os.path.basename(p))

    _py = f'cp{sys.version_info.major}{sys.version_info.minor}'
    _minor = sys.version_info.minor

    def _compat(p):
        b = os.path.basename(p)
        if 'py3-none-any' in b or 'py2.py3-none-any' in b:
            return True
        m = _re.search(r'-(cp\d{2,3})-(cp\d{2,3}|abi3)-', b)
        if not m:
            return True
        if m.group(2) == 'abi3':
            return int(m.group(1)[3:] or 0) <= _minor   # cp312-abi3 은 3.12+ 전용
        return m.group(1) == _py

    def _house_root(p):
        parts = p.split('/')
        if 'datasets' in parts:
            i = parts.index('datasets')
            return '/'.join(parts[:i + 3])
        return '/'.join(parts[:4])

    def _ver(p):
        m = _re.match(r'vllm-([0-9][^-]*)-', os.path.basename(p))
        return m.group(1) if m else None

    def _prio(p):   # ARC3 원조(driessmit1 = 이 Qwen3.8 repack 실증 조합) 우선
        return (0 if 'driessmit1' in p else 1, p)

    _name = lambda p: os.path.basename(p).split('-')[0].lower()

    # blake3 대체휠 (하우스에 호환 진짜 휠이 없을 때만 사용)
    SHIM_SRC = {'blake3': (
        'import hashlib\n'
        'class blake3:\n'
        '    name = "blake3"; digest_size = 32; block_size = 64\n'
        '    def __init__(self, data=b"", **kw):\n'
        '        self._h = hashlib.sha256()\n'
        '        if data: self._h.update(data)\n'
        '    def update(self, d):\n'
        '        self._h.update(d); return self\n'
        '    def digest(self, length=32):\n'
        '        d = self._h.digest()\n'
        '        while len(d) < length: d += hashlib.sha256(d).digest()\n'
        '        return d[:length]\n'
        '    def hexdigest(self, length=32):\n'
        '        return self.digest(length).hex()\n'
        '    def copy(self):\n'
        '        n = blake3(); n._h = self._h.copy(); return n\n'
        'OUT_LEN = 32; KEY_LEN = 32\n')}
    SHIM_DIR = '/kaggle/working/_shim_wheels'
    os.makedirs(SHIM_DIR, exist_ok=True)

    def build_stub_wheel(name, version='9.9.9'):
        fn = f'{SHIM_DIR}/{name}-{version}-py3-none-any.whl'
        if os.path.exists(fn):
            return fn
        di = f'{name}-{version}.dist-info'
        files = {f'{name}.py': SHIM_SRC[name],
                 f'{di}/METADATA': f'Metadata-Version: 2.1\nName: {name}\nVersion: {version}\n',
                 f'{di}/WHEEL': 'Wheel-Version: 1.0\nGenerator: shim\nRoot-Is-Purelib: true\nTag: py3-none-any\n'}
        files[f'{di}/RECORD'] = ''.join(f'{p},,\n' for p in files) + f'{di}/RECORD,,\n'
        with zipfile.ZipFile(fn, 'w', zipfile.ZIP_DEFLATED) as z:
            for p, c in files.items():
                z.writestr(p, c)
        print(f'  대체 휠 제작: {os.path.basename(fn)}')
        return fn

    def _try_import_vllm(tag):
        global HAS_VLLM
        n = _purge_hf_mods()
        _miss = [m for m in ('msgspec', 'aiohttp', 'cloudpickle', 'psutil')
                 if _iu.find_spec(m) is None]
        if _miss:
            print(f'  [{tag}] ★핵심 의존성 없음: {_miss} — vllm 지연임포트가 셀2에서 '
                  f'죽으므로 실패 처리 (환경 py3.12 인지, 휠하우스 호환 수 확인)')
            return False
        try:
            import vllm as _v
            HAS_VLLM = True
            print(f'★vLLM 준비 완료({tag}): {getattr(_v, "__version__", "?")} '
                  f'(캐시청소 {n}, {time.time()-t0:.0f}s)')
        except Exception as e:
            print(f'  [{tag}] import 실패: {repr(e)[:300]}')
        return HAS_VLLM

    # ---- 하우스 격리 설치 — ★실패한 하우스 위에 다른 하우스를 겹치지 않는다
    #      (실측: 0.19 설치 후 0.27 을 겹치자 transformers/torch 키메라 발생) ----
    installed_any = False
    for whl in sorted(vllm_whl, key=_prio):
        root, ver = _house_root(whl), _ver(whl)
        tag = f'{root.split("/")[-1]}:{ver}'
        house_all = sorted(p for p in all_whl if p.startswith(root + '/'))
        house_ok = [p for p in house_all if _compat(p)]
        skip_n = len(house_all) - len(house_ok)
        # 컴파일 짝이 이 파이썬에 없으면 순수휠도 금지 (pydantic/transformers 하드체크)
        _have = {_name(p) for p in house_ok}
        _pair = {'pydantic': 'pydantic_core', 'transformers': 'tokenizers'}
        _pure_skip = {k for k, v in _pair.items() if v not in _have}
        house_ok = [p for p in house_ok if _name(p) not in _pure_skip]
        shims = []
        for m in SHIM_SRC:
            if m not in _have:
                shims.append(build_stub_wheel(m))
        print(f'[하우스 {tag}] 호환 {len(house_ok)}/{len(house_all)} '
              f'(비호환 {skip_n} 제외'
              + (f', 순수휠 금지 {sorted(_pure_skip)}' if _pure_skip else '') + ')')

        links = []
        for d in sorted({os.path.dirname(p) for p in house_ok}) + [SHIM_DIR]:
            links += ['--find-links', d]
        spec = f'vllm=={ver}' if ver else 'vllm'

        # 1차: 정식 해석 (이 하우스만)
        print(f'  해석 설치: {spec} (수 분)...')
        r = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '--no-index',
             '--no-warn-conflicts', '--disable-pip-version-check', *links, spec],
            capture_output=True, text=True)
        if r.returncode == 0:
            installed_any = True
            if _try_import_vllm(tag):
                break
            print('  ★설치는 성공 — 다른 하우스를 겹치면 키메라가 되므로 '
                  '추가 하우스 시도 중단')
            break
        print('  해석 실패 — 마지막 8줄:')
        print('\n'.join((r.stdout + '\n' + r.stderr).splitlines()[-8:]))

        # 2차: 강제 (이 하우스 호환 휠 전량 --no-deps --ignore-installed)
        print(f'  강제 설치: 휠 {len(house_ok)}개 + 대체휠 {len(shims)}개 ...')
        r2 = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps',
             '--ignore-installed', '--no-warn-conflicts',
             '--disable-pip-version-check', *house_ok, *shims],
            capture_output=True, text=True)
        if r2.returncode != 0:
            print('  일괄 실패 → 개별 강등...')
            okc, bad = 0, []
            for p in house_ok + shims:
                r3 = subprocess.run(
                    [sys.executable, '-m', 'pip', 'install', '--no-index',
                     '--no-deps', '--ignore-installed', '--no-warn-conflicts',
                     '--disable-pip-version-check', p],
                    capture_output=True, text=True)
                okc += (r3.returncode == 0)
                if r3.returncode != 0:
                    bad.append(os.path.basename(p))
            print(f'  개별: 성공 {okc} / 실패 {len(bad)}'
                  + (f' — {bad[:5]}' if bad else ''))
        installed_any = True
        _try_import_vllm(tag + ':강제')
        break        # ★어떤 결과든 첫 하우스에서 종료 — 겹침 금지

if not HAS_VLLM:
    print('\n' + '=' * 70)
    print('★ vLLM 확보 실패 — 위 import 실패 메시지가 다음 수를 결정한다.')
    print('   (지금 바로 돌릴 수 있는 것 = AB_MODEL="a3b" 기준선 팔)')
    print('=' * 70)

import json as _json
_WORK = '/kaggle/working' if os.path.isdir('/kaggle') else os.path.abspath('./working')
os.makedirs(_WORK, exist_ok=True)
_json.dump({'t0': time.time()}, open(f'{_WORK}/_t0.json', 'w'))   # ★자동 스톱 기준 시각 (제출 11.4h / 리허설 50분)
if TRUE_SUBMISSION:
    assert HAS_VLLM, '★제출 런인데 vLLM 확보 실패 — 휠하우스 attach 를 확인 (제출 슬롯 낭비 방지 하드스톱)'
print('환경 준비 완료 / HAS_VLLM =', HAS_VLLM, flush=True)


RUN_VARIANT=arc2-desc-agent-v58-submit-assets TRUE_SUBMISSION=False
유틸 경로 차단: pip_install_unsloth_flash_patch (모듈 퇴출 51개) — detach 권장
Python 3.12
인풋 스캔: .whl 370개 / vllm 휠 2개 (1s)
  vllm 휠: vllm-0.19.0-cp38-abi3-manylinux_2_31_x86_64.whl
  vllm 휠: vllm-0.27.1-cp38-abi3-manylinux_2_28_x86_64.whl
[하우스 arc3-vllm-h100-wheelhouse-v3:0.19.0] 호환 174/174 (비호환 0 제외)
  해석 설치: vllm==0.19.0 (수 분)...
★vLLM 준비 완료(arc3-vllm-h100-wheelhouse-v3:0.19.0): 0.19.0 (캐시청소 0, 86s)
환경 준비 완료 / HAS_VLLM = True


In [2]:
# [셀1.5] ★규칙함수 엔진 내장 — order_engine.py(3322줄) + rule_stack_all.py(3002줄, 도구 69). CPU 전용, 수 초.
import os
_ORDER_ENGINE = r'''# ============================================================================
# [오더 엔진 v1] 수학풀이식 본체의 첫 구현.
#   단계(stage) = train 쌍에서 도출되는 grid->grid 변환 함수
#   오더 = 단계들의 순서 있는 합성 (색깔 먼저 / 크기 먼저 / ... 전부 시도)
#   채택 = LOO train-exact (나머지 쌍에서 도출한 파이프라인이 빠진 쌍을 정확 재현)
#   LLM 없음. GPU 없음. 순수 전수검사 + 검산.
# ============================================================================
import numpy as np
from itertools import permutations

# ----------------------------------------------------------- 단계 도출기들
def d_color(pairs):
    """전단사 색 매핑 (같은 크기 쌍에서 픽셀 전이가 함수적이고 일관될 때)"""
    mapping = {}
    for e in pairs:
        i = np.asarray(e['input']); o = np.asarray(e['output'])
        if i.shape != o.shape: return None
        for a, b in zip(i.flatten().tolist(), o.flatten().tolist()):
            if a in mapping and mapping[a] != b: return None
            mapping[a] = b
    if all(a == b for a, b in mapping.items()): return None   # identity -> 무의미
    m = dict(mapping)
    def fn(g):
        g = np.asarray(g)
        out = g.copy()
        for a, b in m.items(): out[g == a] = b
        return out
    return fn

_D4 = {
    'rot90':  lambda g: np.rot90(g, 1), 'rot180': lambda g: np.rot90(g, 2),
    'rot270': lambda g: np.rot90(g, 3), 'flipH': lambda g: g[:, ::-1],
    'flipV':  lambda g: g[::-1, :], 'transpose': lambda g: np.asarray(g).T,
    'anti':   lambda g: np.rot90(np.asarray(g).T, 2),
}
def d_d4(pairs):
    for name, t in _D4.items():
        if all(np.asarray(e['output']).shape == t(np.asarray(e['input'])).shape and
               np.array_equal(t(np.asarray(e['input'])), np.asarray(e['output']))
               for e in pairs):
            return t
    return None

def d_tile(pairs):
    """output = input의 k x m 타일 (각 타일 orig/flip)"""
    def pat(i, o):
        i = np.asarray(i); o = np.asarray(o)
        ih, iw = i.shape; oh, ow = o.shape
        if ih == 0 or iw == 0 or oh % ih or ow % iw: return None
        k, m = oh // ih, ow // iw
        if (k, m) == (1, 1): return None
        var = {'orig': i, 'flipH': i[:, ::-1], 'flipV': i[::-1, :], 'flipHV': i[::-1, ::-1]}
        rows = []
        for r in range(k):
            row = []
            for c in range(m):
                t = o[r*ih:(r+1)*ih, c*iw:(c+1)*iw]
                n = next((nm for nm, v in var.items() if np.array_equal(t, v)), None)
                if n is None: return None
                row.append(n)
            rows.append(tuple(row))
        return (k, m, tuple(rows))
    ps = [pat(e['input'], e['output']) for e in pairs]
    if any(p is None for p in ps) or len({str(p) for p in ps}) != 1: return None
    k, m, tiles = ps[0]
    def fn(g):
        g = np.asarray(g)
        var = {'orig': g, 'flipH': g[:, ::-1], 'flipV': g[::-1, :], 'flipHV': g[::-1, ::-1]}
        return np.block([[var[tiles[r][c]] for c in range(m)] for r in range(k)])
    return fn

def d_scale(pairs):
    """정수 확대 (모양 비율로 도출; 내용 일치는 파이프라인 말미 전체 검산이 보증)"""
    ks = []
    for e in pairs:
        i = np.asarray(e['input']); o = np.asarray(e['output'])
        ih, iw = i.shape; oh, ow = o.shape
        if ih == 0 or iw == 0 or oh % ih or ow % iw: return None
        k, m = oh // ih, ow // iw
        if (k, m) == (1, 1): return None
        ks.append((k, m))
    if len(set(ks)) != 1: return None
    k, m = ks[0]
    return lambda g: np.kron(np.asarray(g), np.ones((k, m), dtype=np.asarray(g).dtype))

def d_shrink(pairs):
    """정수 축소 (모양 비율로 도출 + 입력 블록 균일성 요구)"""
    ks = []
    for e in pairs:
        i = np.asarray(e['input']); o = np.asarray(e['output'])
        oh, ow = o.shape; ih, iw = i.shape
        if oh == 0 or ow == 0 or ih % oh or iw % ow: return None
        k, m = ih // oh, iw // ow
        if (k, m) == (1, 1): return None
        if not np.array_equal(np.kron(i[::k, ::m], np.ones((k, m), dtype=i.dtype)), i):
            return None                     # 입력이 k x m 균일 블록이어야 축소 의미
        ks.append((k, m))
    if len(set(ks)) != 1: return None
    k, m = ks[0]
    return lambda g: np.asarray(g)[::k, ::m]

# ----------------------------------------------------------- 복원 (대칭 수복)
def _sym_transforms(g, unk):
    """알려진 셀에서 무모순인 대칭 생성기 목록: (이름, 매핑함수)
    - 반사축은 중심에서 벗어날 수 있고 반정수(2p) 허용 (골프 코드 방식)
    - 병진 주기(행/열), 정방이면 전치/반전치"""
    H, W = g.shape
    known = ~unk
    out = []
    rows = np.arange(H)[:, None] * np.ones((1, W), int)
    cols = np.ones((H, 1), int) * np.arange(W)[None, :]
    def check(map_r, map_c, name):
        inb = (map_r >= 0) & (map_r < H) & (map_c >= 0) & (map_c < W)
        m = known & inb
        if not m.any(): return
        tr = map_r[m]; tc = map_c[m]
        src_v = g[m]; dst_known = known[tr, tc]
        both = dst_known
        if not both.any(): return
        if (src_v[both] == g[tr[both], tc[both]]).all() and both.sum() >= (H * W) // 4:
            out.append((name, map_r, map_c))
    for p2 in range(1, 2 * W - 2):           # 열 반사 (좌우 거울, 축 오프셋 허용)
        check(rows, p2 - cols, f'colmir{p2}')
    for p2 in range(1, 2 * H - 2):           # 행 반사
        check(p2 - rows, cols, f'rowmir{p2}')
    for pr in range(1, H):                    # 행 주기
        check(rows + pr, cols, f'rowper{pr}')
        check(rows - pr, cols, f'rowper-{pr}')
    for pc in range(1, W):                    # 열 주기
        check(rows, cols + pc, f'colper{pc}')
        check(rows, cols - pc, f'colper-{pc}')
    for dy in range(1, min(H, 13)):           # 2D 평행이동 (대각 주기, 벽지 패턴)
        for dx in range(1, min(W, 13)):
            check(rows + dy, cols + dx, f'per{dy},{dx}')
            check(rows - dy, cols - dx, f'per-{dy},-{dx}')
            check(rows + dy, cols - dx, f'per{dy},-{dx}')
            check(rows - dy, cols + dx, f'per-{dy},{dx}')
    if H == W:
        check(cols, rows, 'transpose')
        check(W - 1 - cols, H - 1 - rows, 'anti')
    return out

_RESTORE_MEMO = {}

def _restore_grid(grid, occ):
    """가림색 occ 셀을 대칭군 전파로 복원. 전부 못 채우면 None. (메모이즈)"""
    _k = (np.asarray(grid).astype(np.int8).tobytes(),
          np.asarray(grid).shape, int(occ))
    if _k in _RESTORE_MEMO:
        v = _RESTORE_MEMO[_k]
        return None if v is None else v.copy()
    if len(_RESTORE_MEMO) > 4000:
        _RESTORE_MEMO.clear()
    g = np.asarray(grid).copy()
    unk = (g == occ)
    if not unk.any():
        _RESTORE_MEMO[_k] = g.copy()
        return g
    trs = _sym_transforms(g, unk)
    if not trs:
        _RESTORE_MEMO[_k] = None
        return None
    for _ in range(12):
        changed = False
        unk = (g == occ)
        if not unk.any(): break
        for _name, mr, mc in trs:
            inb = (mr >= 0) & (mr < g.shape[0]) & (mc >= 0) & (mc < g.shape[1])
            m = unk & inb
            if not m.any(): continue
            src = g[mr[m], mc[m]]
            fill = src != occ
            if fill.any():
                idx = np.where(m)
                g[idx[0][fill], idx[1][fill]] = src[fill]
                changed = True
        if not changed: break
    _res = g if not (g == occ).any() else None
    _RESTORE_MEMO[_k] = None if _res is None else _res.copy()
    return _res

def _restore_unanimous(grid, occ, agree_th=0.80, max_off=7, require_complete=True):
    """[사용자 명세] 중간선(행/열, 중앙 +-max_off) 탐지 -> 사분면 대응
    (행반사/열반사/180회전 + 축교점 기준 90/270회전) -> "만장일치" 복원.
    - 셀 채움 조건: 대칭 짝들의 값(가림색 제외)이 존재하고 "모두 일치"
    - 하나라도 다르면 채우지 않음 (다수결 아님, 추측 배제)
    - occ 는 int 또는 가림색 집합(tuple/set) - 알지 못하는 색은 일치 검사에서 제외"""
    g = np.asarray(grid).copy()
    H, W = g.shape
    occs = set([occ]) if isinstance(occ, (int, np.integer)) else set(int(c) for c in occ)
    def is_unk(a):
        m = np.zeros(a.shape, bool)
        for c in occs: m |= (a == c)
        return m
    unk = is_unk(g)
    if not unk.any():
        return g
    def best_axis(L, axis):
        best = (None, 0.0)
        center = L - 1
        for p2 in range(max(1, center - max_off), min(2 * L - 2, center + max_off + 1)):
            tot = agr = 0
            for a in range(L):
                b = p2 - a
                if not (0 <= b < L) or a >= b: continue
                A = g[a] if axis == 'row' else g[:, a]
                B = g[b] if axis == 'row' else g[:, b]
                m = ~is_unk(A) & ~is_unk(B)
                if not m.any(): continue
                tot += int(m.sum()); agr += int((A[m] == B[m]).sum())
            if tot >= (H * W) // 8 and agr / max(tot, 1) >= agree_th:
                r = agr / tot
                if r > best[1]: best = (p2, r)
        return best[0]
    p2r = best_axis(H, 'row')
    p2c = best_axis(W, 'col')
    if p2r is None and p2c is None:
        return None
    rows = np.arange(H)[:, None] * np.ones((1, W), int)
    cols = np.ones((H, 1), int) * np.arange(W)[None, :]
    maps = []
    if p2r is not None: maps.append((p2r - rows, cols))
    if p2c is not None: maps.append((rows, p2c - cols))
    if p2r is not None and p2c is not None:
        maps.append((p2r - rows, p2c - cols))               # 180회전
        # 축교점 기준 90/270 회전 (사용자 명세: 사분면 90/180/270)
        # 2배 좌표: Y=2y, X=2x, 중심 (p2r, p2c)
        Y = 2 * rows; X = 2 * cols
        for sgn in (1, -1):                                  # 90 / 270
            Yn = p2r + sgn * (X - p2c)
            Xn = p2c - sgn * (Y - p2r)
            ok = (Yn % 2 == 0) & (Xn % 2 == 0)
            mr = np.where(ok, Yn // 2, -1)
            mc = np.where(ok, Xn // 2, -1)
            # 회전 대응도 알려진 셀 일치율 검증 후 채택
            inb = (mr >= 0) & (mr < H) & (mc >= 0) & (mc < W)
            m = ~unk & inb
            if m.any():
                tr = mr[m]; tc = mc[m]
                both = ~is_unk(g[tr, tc])
                if both.sum() >= (H * W) // 8 and \
                   float((g[m][both] == g[tr[both], tc[both]]).mean()) >= agree_th:
                    maps.append((mr, mc))
    for _ in range(12):
        unk = is_unk(g)
        if not unk.any(): break
        changed = False
        ys, xs = np.where(unk)
        for y, x in zip(ys.tolist(), xs.tolist()):
            vals = set()
            for mr, mc in maps:
                ty, tx = int(mr[y, x]), int(mc[y, x])
                if 0 <= ty < H and 0 <= tx < W and int(g[ty, tx]) not in occs:
                    vals.add(int(g[ty, tx]))
            if len(vals) == 1:                    # 만장일치일 때만 채움
                g[y, x] = vals.pop()
                changed = True
        if not changed: break
    if require_complete:
        return g if not is_unk(g).any() else None
    return g                                  # 부분 결과 허용 (문맥 폴백 연결용)

def _restore_context(grid, occ, r=3, max_unknown=60):
    """[최종 폴백] 미러 포함 문맥매칭 - 만장일치.
    가려진 셀의 이웃 문맥(반경 r, 기지 셀만)과 일치하는 위치를 그리드의
    D4 8방향 사본 전체에서 찾아, 후보 중심값이 "모두 일치"할 때만 채움.
    경계 보정: 요구 겹침 = max(6, 가용 기지 이웃의 80%)."""
    g = np.asarray(grid).copy()
    H, W = g.shape
    unk = (g == occ)
    if not unk.any(): return g
    if int(unk.sum()) > max_unknown: return None
    D4 = [lambda a: a, lambda a: a[:, ::-1], lambda a: a[::-1, :],
          lambda a: a[::-1, ::-1], lambda a: a.T, lambda a: a.T[:, ::-1],
          lambda a: a.T[::-1, :], lambda a: a.T[::-1, ::-1]]
    for _ in range(6):
        unk = (g == occ)
        if not unk.any(): break
        changed = False
        ys, xs = np.where(unk)
        for y, x in zip(ys.tolist(), xs.tolist()):
            nbr = []
            for dy in range(-r, r + 1):
                for dx in range(-r, r + 1):
                    if (dy, dx) == (0, 0): continue
                    yy, xx = y + dy, x + dx
                    if 0 <= yy < H and 0 <= xx < W and g[yy, xx] != occ:
                        nbr.append((dy, dx, int(g[yy, xx])))
            need = max(6, int(len(nbr) * 0.8))
            if len(nbr) < need: continue
            vals = set()
            for T in D4:
                tg = T(g)
                TH, TW = tg.shape
                pad = np.full((TH + 2 * r, TW + 2 * r), -1, dtype=tg.dtype)
                pad[r:r + TH, r:r + TW] = tg
                ok = (tg != occ)
                ov = np.zeros((TH, TW), dtype=int)
                for dy, dx, v in nbr:
                    sl = pad[r + dy: r + dy + TH, r + dx: r + dx + TW]
                    hit = (sl == v)
                    ok &= hit | (sl == -1) | (sl == occ)
                    ov += hit.astype(int)
                m = ok & (ov >= need)
                if T is D4[0]: m[y, x] = False
                if m.any():
                    vals |= set(int(v) for v in np.unique(tg[m]))
                if len(vals) > 1: break
            if len(vals) == 1:                    # 만장일치일 때만
                g[y, x] = vals.pop()
                changed = True
        if not changed: break
    return g if not (g == occ).any() else None

def _occ_bbox(grid, occ):
    ys, xs = np.where(np.asarray(grid) == occ)
    if len(ys) == 0: return None
    return (ys.min(), ys.max() + 1, xs.min(), xs.max() + 1)

def _occ_is_rects(g, occ):
    """가림 셀들이 꽉 찬 사각형(들)인가 (복원도형=사각형 힌트)"""
    g = np.asarray(g); mask = (g == occ)
    seen = np.zeros_like(mask)
    H, W = mask.shape
    for r in range(H):
        for c in range(W):
            if mask[r, c] and not seen[r, c]:
                r2 = r
                while r2 + 1 < H and mask[r2 + 1, c]: r2 += 1
                c2 = c
                while c2 + 1 < W and mask[r, c2 + 1]: c2 += 1
                if not mask[r:r2+1, c:c2+1].all(): return False
                seen[r:r2+1, c:c2+1] = True
    return True

def d_restore(pairs):
    """가림색/출력변형(full|patch) 도출 -> 복원 함수 "후보 목록".
    내용 일치는 요구하지 않음 (모양 + 복원가능성만; 정확성은 말미 검산/LOO 담당)"""
    cand = set(int(v) for v in np.unique(np.asarray(pairs[0]['input'])))
    for e in pairs[1:]:
        cand &= set(int(v) for v in np.unique(np.asarray(e['input'])))
    # 가림색 후보 선정 (출력 근거):
    #   같은 크기: 모든 쌍에서 "출력에서 소멸하는 색" / 크기 다름: "bbox == 출력 크기"
    #   사각형 형성 여부는 우선순위로만 사용 (하드 필터 아님)
    ins_ = [np.asarray(e['input']) for e in pairs]
    outs_ = [np.asarray(e['output']) for e in pairs]
    same_sz = all(i.shape == o.shape for i, o in zip(ins_, outs_))
    strong = set()
    for c in cand:
        if same_sz:
            if all((o == c).sum() == 0 and (i == c).sum() > 0
                   for i, o in zip(ins_, outs_)):
                strong.add(c)
        else:
            ok_bb = True
            for i, o in zip(ins_, outs_):
                ys, xs = np.where(i == c)
                if len(ys) == 0 or (int(ys.max()-ys.min()+1),
                                    int(xs.max()-xs.min()+1)) != o.shape:
                    ok_bb = False; break
            if ok_bb: strong.add(c)
    def prio(c):
        rect = all(_occ_is_rects(e['input'], c) for e in pairs)
        return (0 if c in strong else 1, 0 if rect else 1, 0 if c == 0 else 1, c)
    cand = sorted(cand, key=prio)
    if strong:
        cand = [c for c in cand if c in strong] + [c for c in cand if c not in strong][:2]
    fns = []
    for occ in cand:
        restos = []
        ok = True
        mode = 'strict'
        for e in pairs:
            i = np.asarray(e['input'])
            if not (i == occ).any(): ok = False; break
            r = _restore_grid(i, occ)
            if r is None:
                r = _restore_unanimous(i, occ)       # 폴백1: 사분면 만장일치
                mode = 'majority'
            if r is None:
                part = _restore_unanimous(i, occ, require_complete=False)
                base = part if part is not None else i
                r = _restore_context(base, occ)      # 폴백2: 부분채움 + 문맥
                mode = 'majority'
            if r is None: ok = False; break
            restos.append((i, r))
        if not ok: continue
        # 변형 판별: 출력 "모양"만 사용
        full_ok = all(np.asarray(e['output']).shape == i.shape
                      for (i, _r), e in zip(restos, pairs))
        patch_ok = True
        for (i, _r), e in zip(restos, pairs):
            bb = _occ_bbox(i, occ)
            if bb is None: patch_ok = False; break
            y0, y1, x0, x1 = bb
            if np.asarray(e['output']).shape != (y1 - y0, x1 - x0):
                patch_ok = False; break
        variant = 'full' if full_ok else ('patch' if patch_ok else None)
        if variant is None: continue
        def fn(g, _occ=occ, _v=variant, _m=mode):
            r = _restore_grid(np.asarray(g), _occ)
            if r is None:
                r = _restore_unanimous(np.asarray(g), _occ)   # 폴백1: 사분면 만장일치
            if r is None:
                part = _restore_unanimous(np.asarray(g), _occ, require_complete=False)
                base = part if part is not None else np.asarray(g)
                r = _restore_context(base, _occ)              # 폴백2: 부분채움 + 문맥
            if r is None: raise ValueError('restore fail')
            if _v == 'full': return r
            y0, y1, x0, x1 = _occ_bbox(g, _occ)
            return r[y0:y1, x0:x1]
        fns.append(fn)
    return fns if fns else None

# ----------------------------------------------------------- 패널 결합 (그리드)
def _find_sep(g):
    """단일 구분선(꽉 찬 행/열)으로 같은 크기 패널 2개 분할. (방향, 색, 패널A, 패널B)"""
    g = np.asarray(g); H, W = g.shape
    out = []
    for c in range(W):
        col = g[:, c]
        if len(set(col.tolist())) == 1 and c == W // 2 and W % 2 == 1:
            out.append(('v', int(col[0]), g[:, :c], g[:, c+1:]))
    for r in range(H):
        row = g[r, :]
        if len(set(row.tolist())) == 1 and r == H // 2 and H % 2 == 1:
            out.append(('h', int(row[0]), g[:r, :], g[r+1:, :]))
    return out

def d_panels(pairs):
    """구분선 분할 + 셀쌍 (a,b)->값 매핑 도출"""
    infos = []
    for e in pairs:
        cands = _find_sep(e['input'])
        o = np.asarray(e['output'])
        cands = [x for x in cands if x[2].shape == o.shape and x[3].shape == o.shape]
        if not cands: return None
        infos.append(cands)
    # 방향+구분색 조합이 모든 쌍에서 일치하는 것 선택
    keysets = [set((d, c) for d, c, _a, _b in cs) for cs in infos]
    common = set.intersection(*keysets)
    fns = []
    for key in sorted(common):
        mapping = {}
        ok = True
        for e, cs in zip(pairs, infos):
            d_, c_, A, B = next(x for x in cs if (x[0], x[1]) == key)
            o = np.asarray(e['output'])
            for a, b, v in zip(A.flatten().tolist(), B.flatten().tolist(),
                               o.flatten().tolist()):
                if (a, b) in mapping and mapping[(a, b)] != v: ok = False; break
                mapping[(a, b)] = v
            if not ok: break
        if not ok: continue
        def fn(g, _key=key, _m=dict(mapping)):
            cands = _find_sep(g)
            hit = next((x for x in cands if (x[0], x[1]) == _key), None)
            if hit is None: raise ValueError('sep not found')
            _d, _c, A, B = hit
            out = np.empty_like(A)
            fa = A.flatten(); fb = B.flatten(); fo = out.flatten()
            for i, (a, b) in enumerate(zip(fa.tolist(), fb.tolist())):
                if (a, b) not in _m: raise ValueError('unseen pair')
                fo[i] = _m[(a, b)]
            return fo.reshape(A.shape)
        fns.append(fn)
    return fns if fns else None

# ----------------------------------------------------------- 흐름 (낙하/뻗음)
#   사용자 CoT: 1.크기같음 2.한 색만 변함(나머지 고정) 3.그 색의 흐름 법칙
def _flow_fall(g, c, d):
    """색 c 픽셀이 방향 d로 장애물(0 아닌 셀)/벽까지 낙하"""
    g = np.asarray(g).copy(); H, W = g.shape
    dy, dx = {'down': (1,0), 'up': (-1,0), 'right': (0,1), 'left': (0,-1)}[d]
    ys, xs = np.where(g == c)
    order = sorted(zip(ys.tolist(), xs.tolist()),
                   key=lambda p: -(p[0]*dy + p[1]*dx))   # 진행방향 먼 것부터
    for y, x in order:
        g[y, x] = 0
        ny, nx = y, x
        while True:
            ty, tx = ny + dy, nx + dx
            if not (0 <= ty < H and 0 <= tx < W) or g[ty, tx] != 0: break
            ny, nx = ty, tx
        g[ny, nx] = c
    return g

def _flow_extend(g, c, d):
    """색 c 셀에서 방향 d로 장애물/벽까지 c 를 칠하며 뻗음 (원천 유지)"""
    g = np.asarray(g).copy(); H, W = g.shape
    dy, dx = {'down': (1,0), 'up': (-1,0), 'right': (0,1), 'left': (0,-1)}[d]
    ys, xs = np.where(g == c)
    for y, x in zip(ys.tolist(), xs.tolist()):
        ny, nx = y + dy, x + dx
        while 0 <= ny < H and 0 <= nx < W and g[ny, nx] == 0:
            g[ny, nx] = c
            ny, nx = ny + dy, nx + dx
    return g

def _fall_all(g, d):
    """모든 색이 방향 d로 낙하 (행/열 압축, 순서 보존)"""
    g = np.asarray(g).copy(); H, W = g.shape
    if d in ('down', 'up'):
        for x in range(W):
            col = [v for v in g[:, x].tolist() if v != 0]
            g[:, x] = 0
            if col:
                if d == 'down': g[H-len(col):, x] = col
                else: g[:len(col), x] = col
    else:
        for y in range(H):
            row = [v for v in g[y, :].tolist() if v != 0]
            g[y, :] = 0
            if row:
                if d == 'right': g[y, W-len(row):] = row
                else: g[y, :len(row)] = row
    return g

_FLOW_LAWS = [(m, d) for m in ('fall', 'extend')
              for d in ('down', 'up', 'left', 'right')]

def d_flow(pairs):
    ins = [np.asarray(e['input']) for e in pairs]
    outs = [np.asarray(e['output']) for e in pairs]
    if any(i.shape != o.shape for i, o in zip(ins, outs)): return None
    # 변하는 색 c 후보: 모든 변경 셀이 c 관련 (in==c 또는 out==c), 나머지 고정
    cands = set(int(v) for v in np.unique(ins[0]) if v != 0)
    for i, o in zip(ins, outs):
        ok = set()
        for c in cands:
            ch = (i != o)
            if not ch.any(): continue
            if bool((((i == c) | (o == c)) | ~ch).all()):
                ok.add(c)
        cands &= ok
    fns = []
    # 전색 동시 낙하 (색 무관 압축)
    for d in ('down', 'up', 'left', 'right'):
        if all(np.array_equal(_fall_all(i, d), o) for i, o in zip(ins, outs)):
            fns.append((lambda g, _d=d: _fall_all(g, _d)))
    for c in sorted(cands):
        for mode, d in _FLOW_LAWS:
            f = _flow_fall if mode == 'fall' else _flow_extend
            if all(np.array_equal(f(i, c, d), o) for i, o in zip(ins, outs)):
                fns.append((lambda g, _c=c, _f=f, _d=d: _f(g, _c, _d)))
    return fns if fns else None

def _rank_insert_part(rank):
    """part 를 최상위로 승격하되, 실검증 통과한 restore 는 밀어내지 않음."""
    if 'part' in rank: return
    pos = 1 if rank[:1] == ['restore'] else 0
    rank.insert(pos, 'part')

# ----------------------------------------------------------- 부분 (서브그리드 잘라내기)
#   사용자 CoT: 1.출력이 입력보다 작음 2.무엇을 잘라낼지 선택 규칙
#   선택기 메뉴 (전부 입력만으로 결정 - answer-key trap 없음):
#     A. 도형 선택: (지표 px/bbox) x (순위 최대/2번째/최소) x (렌더 raw/mask)
#     B. 고유 도형: 색이 고유한 도형 / 실루엣이 혼자 다른 도형
#     C. 색 bbox: 색 c 의 bbox 잘라내기 / 테두리 안쪽(inner) 잘라내기
#   도출기: 모든 train 쌍에서 출력 '크기'가 맞는 fn 만 반환.
#   정확 일치 fn 을 앞에 배치 -> 단일 part 오더가 먼저 성립,
#   크기만 맞는 fn 은 뒤에 -> part->color / part->d4 파이프라인용.
def _part_objs(g):
    """배경 감지 + 도형 추출. 도형 없으면 예외 (파이프라인이 잡음)."""
    g = np.asarray(g)
    bg = _detect_bg2(g)
    objs = _objs8(g, bg)
    if not objs: raise ValueError('no objects')
    return g, bg, objs

def _part_bbox(pos):
    ys = [y for y, _x in pos]; xs = [x for _y, x in pos]
    return min(ys), min(xs), max(ys), max(xs)

def _part_render(g, bg, pos, mode):
    r0, c0, r1, c1 = _part_bbox(pos)
    if mode == 'raw':
        return np.asarray(g)[r0:r1+1, c0:c1+1].copy()
    out = np.full((r1-r0+1, c1-c0+1), bg, dtype=np.asarray(g).dtype)
    for y, x in pos:
        out[y-r0, x-c0] = np.asarray(g)[y, x]
    return out

def _part_pick_rank(objs, metric, rank):
    """지표로 정렬해 rank 번째 도형 선택. 동률이면 모호 -> 포기(예외).
    (train 우연 통과 후 test 동률 오답 방지 - 1f85a75f 교훈)"""
    def m(t):
        pos = t[2]
        if metric == 'px': return len(pos)
        r0, c0, r1, c1 = _part_bbox(pos)
        return (r1-r0+1) * (c1-c0+1)
    srt = sorted(objs, key=lambda t: (-m(t),) + _part_bbox(t[2])[:2])
    if rank == -1:
        srt = srt[::-1]
        rank = 0
    if rank >= len(srt): raise ValueError('rank out of range')
    ms = [m(t) for t in srt]
    if (rank + 1 < len(ms) and ms[rank] == ms[rank+1]) or \
       (rank > 0 and ms[rank] == ms[rank-1]):
        raise ValueError('rank tie - ambiguous')
    return srt[rank]

def _part_pick_unique_color(objs):
    """다른 어떤 도형에도 없는 색을 가진 유일한 도형."""
    csets = [frozenset(c for _y, _x, c in t[1]) for t in objs]
    hits = []
    for i, cs in enumerate(csets):
        others = set().union(*(csets[:i] + csets[i+1:])) if len(csets) > 1 else set()
        if cs - others: hits.append(i)
    if len(hits) != 1: raise ValueError('unique color ambiguous')
    return objs[hits[0]]

def _part_pick_odd_shape(objs):
    """실루엣(D4 정규화)이 혼자 다른 도형. 나머지는 전부 동일해야 함."""
    if len(objs) < 3: raise ValueError('need >=3 objects')
    canons = [_d4_canon(t[0]) for t in objs]
    from collections import Counter as _C
    cnt = _C(canons)
    if len(cnt) != 2: raise ValueError('odd shape ambiguous')
    (a, na), (b, nb) = cnt.most_common()
    if nb != 1: raise ValueError('odd shape ambiguous')
    return objs[canons.index(b)]

def _part_color_bbox(g, c, inner):
    g = np.asarray(g)
    ys, xs = np.where(g == c)
    if len(ys) == 0: raise ValueError('color absent')
    r0, c0, r1, c1 = int(ys.min()), int(xs.min()), int(ys.max()), int(xs.max())
    if inner:
        r0 += 1; c0 += 1; r1 -= 1; c1 -= 1
        if r0 > r1 or c0 > c1: raise ValueError('empty inner')
    return g[r0:r1+1, c0:c1+1].copy()

def _part_obj_match(i, o, bg=None):
    """도형 대 도형 정합 (탐지 전용): 입력 최대(0)/2번째(1) 도형(동률 포함)이
    출력의 어떤 도형과 실루엣 정확 일치(D4 변환 포함)하는지.
    '제자리 무변화(id·같은 위치·같은 색)'는 규칙 참여가 아니므로 제외.
    단, 출력<입력(잘라내기)일 때는 무변화 등장도 참여로 침.
    반환: [(src, rank, tname, moved, recolored)]"""
    i = np.asarray(i); o = np.asarray(o)
    hits = []
    try:
        bg_i = _detect_bg2(i) if bg is None else bg     # 합의 배경 우선
        i2 = np.where(i == bg_i, 0, i) if bg_i != 0 else i
        bg_o = bg_i                                     # 같은 퍼즐 배경은 하나
        o2 = np.where(o == bg_o, 0, o) if bg_o != 0 else o
        E = DiagonalShapeExtractor()
        outs = E.extract_all_shapes(o2.tolist()) + E.extract_all_regions(o2.tolist())
    except Exception:
        return hits
    crop = o.size < i.size
    # 출력 도형: (픽셀수, 실루엣 상대좌표, 색집합, 위치) 사전 구축
    oo = []
    for d in outs:
        py, px = d['position']
        sil = frozenset((y - py, x - px) for y, x in d['pixels'])
        cols = frozenset(int(o2[y, x]) for y, x in d['pixels'])
        oo.append((len(d['pixels']), sil, cols, (py, px)))
    # [사용자 제안] 출력 raw 색깔별 도형 (배경 감지 무의존)
    for c in np.unique(o):
        ys, xs = np.where(o == c)
        if len(ys) == 0 or len(ys) == o.size: continue
        py, px = int(ys.min()), int(xs.min())
        sil = frozenset((int(y) - py, int(x) - px) for y, x in zip(ys, xs))
        oo.append((len(ys), sil, frozenset([int(c)]), (py, px)))
    oo = list({(n, s, cl, p) for n, s, cl, p in oo})
    # [사용자 제안] 입력 raw 색깔별 도형: 색 c 전체 = 도형, rank = 픽셀수 순위
    _cs = []
    _cnt = {int(c): int((i == c).sum()) for c in np.unique(i)
            if 0 < int((i == c).sum()) < i.size}
    _szs = sorted(set(_cnt.values()), reverse=True)
    for c, n in _cnt.items():
        _rk = _szs.index(n)
        if _rk > 0: continue                       # [사용자 제안] 색깔별 '최대'만
        ys, xs = np.where(i == c)
        py, px = int(ys.min()), int(xs.min())
        G = np.zeros((int(ys.max()) - py + 1, int(xs.max()) - px + 1), dtype=i.dtype)
        G[ys - py, xs - px] = c
        _cs.append(('cshape', _rk, G, (py, px)))
    for src, rank, G, gpos in list(_part_extract4_all(i2)) + _cs:
        nz = np.argwhere(G != 0)
        npx = len(nz)
        if npx < 2: continue
        gcols = frozenset(int(v) for v in G[G != 0])
        for tname, T in _D4_PART.items():
            V = T(G)
            vs = frozenset(map(tuple, np.argwhere(V != 0).tolist()))
            for on, osil, ocols, opos in oo:
                if on != npx or osil != vs: continue
                moved = (tname != 'id') or (tuple(gpos) != tuple(opos))
                recolored = (ocols != gcols)
                if crop or moved or recolored:
                    hits.append((src, rank, tname, moved, recolored))
    return hits

def d_part(pairs):
    ins = [np.asarray(e['input']) for e in pairs]
    outs = [np.asarray(e['output']) for e in pairs]
    # 전제: 모든 쌍에서 출력이 입력보다 작음 (잘라내기 문제)
    if not all(o.size < i.size and o.shape[0] <= i.shape[0]
               and o.shape[1] <= i.shape[1] for i, o in zip(ins, outs)):
        return None
    cands = []
    # A. 도형 선택 (지표 x 순위 x 렌더)
    for metric in ('px', 'bbox'):
        for rank in (0, 1, -1):
            for mode in ('raw', 'mask'):
                def fn(g, _m=metric, _r=rank, _md=mode):
                    g2, bg, objs = _part_objs(g)
                    t = _part_pick_rank(objs, _m, _r)
                    return _part_render(g2, bg, t[2], _md)
                cands.append(fn)
    # B. 고유 도형 (고유색 / 혼자 다른 실루엣)
    for picker in (_part_pick_unique_color, _part_pick_odd_shape):
        for mode in ('raw', 'mask'):
            def fn(g, _p=picker, _md=mode):
                g2, bg, objs = _part_objs(g)
                t = _p(objs)
                return _part_render(g2, bg, t[2], _md)
            cands.append(fn)
    # C. 색 bbox (모든 train 입력에 존재하는 색만 후보)
    common = set(np.unique(ins[0]).tolist())
    for i in ins[1:]:
        common &= set(np.unique(i).tolist())
    for c in sorted(int(v) for v in common):
        for inner in (False, True):
            def fn(g, _c=c, _in=inner):
                return _part_color_bbox(g, _c, _in)
            cands.append(fn)
    # D. [사용자 절차] shapes/regions 최대·2번째 x 변환 -> crop 답 (bg=0 계열)
    for src_ in ('shape', 'region'):
        for rank_ in (0, 1):
            for tname_, T_ in _D4_PART.items():
                def fn(g, _s=src_, _r=rank_, _T=T_):
                    cands4 = _part_extract4(g)
                    if (_s, _r) not in cands4: raise ValueError('cand missing')
                    return _T(cands4[(_s, _r)])
                cands.append(fn)
    # 필터: 모든 쌍에서 출력 크기 일치(전치 포함 - d4 후속용) 필수. 정확 일치는 앞에.
    exact, shape_only = [], []
    for fn in cands:
        ok_shape = ok_exact = True
        for i, o in zip(ins, outs):
            try:
                r = np.asarray(fn(i))
            except Exception:
                ok_shape = False; break
            if r.shape not in (o.shape, o.shape[::-1]):
                ok_shape = False; break
            if r.shape != o.shape or not np.array_equal(r, o):
                ok_exact = False
        if not ok_shape: continue
        (exact if ok_exact else shape_only).append(fn)
    fns = exact + shape_only
    return fns if fns else None

# ----------------------------------------------------------- 부분 v1.5 (사용자 절차)
#   extract_all_shapes(색깔 단위) / extract_all_regions(연결 단위) 에서
#   가장큰·2번째큰 -> 최대 4후보. 각각 6변환(90/180/270, x미러, y미러, xy미러)
#   + 무변환(id) -> 답 후보. 25% 크기 필터·슬라이드 ±3칸 매칭은 탐지용,
#   채택은 기존 train-exact 필터 + LOO 가 최종 심판.
_D4_PART = {
    'id':     lambda a: a.copy(),
    'rot90':  lambda a: np.rot90(a, -1),
    'rot180': lambda a: np.rot90(a, 2),
    'mx':     lambda a: np.flipud(a).copy(),
    'my':     lambda a: np.fliplr(a).copy(),
    'mxy':    lambda a: np.fliplr(np.flipud(a)).copy(),   # == rot180 (스펙 유지)
    'rot270': lambda a: np.rot90(a, 1),
}

def _part_big2(items):
    """가장 큰(0)·2번째 큰(1) 선택 (픽셀 수). 동률 순위는 모호 -> 제외."""
    srt = sorted(items, key=lambda d: (-len(d['pixels']), d['position']))
    sizes = [len(d['pixels']) for d in srt]
    out = {}
    if srt and (len(srt) < 2 or sizes[0] != sizes[1]):
        out[0] = srt[0]
    if len(srt) >= 2 and sizes[0] != sizes[1] and \
       (len(srt) < 3 or sizes[1] != sizes[2]):
        out[1] = srt[1]
    return out

_PART4_MEMO = {}
def _part_extract4(g):
    """[사용자 절차] shapes/regions 각각 최대·2번째 -> 최대 4후보 dict
    (그리드 단위 메모 - 후보 fn 28개가 같은 추출을 재사용)"""
    a = np.asarray(g)
    key = (a.shape, a.tobytes())
    if key in _PART4_MEMO: return _PART4_MEMO[key]
    E = DiagonalShapeExtractor()
    gl = a.tolist()
    res = {}
    for src, items in (('shape', E.extract_all_shapes(gl)),
                       ('region', E.extract_all_regions(gl))):
        for rank, d in _part_big2(items).items():
            res[(src, rank)] = np.asarray(d['grid'])
    if len(_PART4_MEMO) > 512: _PART4_MEMO.clear()
    _PART4_MEMO[key] = res
    return res

_PART4ALL_MEMO = {}
def _part_extract4_all(g):
    """탐지 전용: 동률 포함, shapes/regions 각각 상위 2크기군 전부 반환.
    반환: [(src, rank, grid)] - rank 는 크기 순위(0=최대군, 1=2번째군)."""
    a = np.asarray(g)
    key = (a.shape, a.tobytes())
    if key in _PART4ALL_MEMO: return _PART4ALL_MEMO[key]
    E = DiagonalShapeExtractor()
    gl = a.tolist()
    res = []
    for src, items in (('shape', E.extract_all_shapes(gl)),
                       ('region', E.extract_all_regions(gl))):
        if not items: continue
        sizes = sorted({len(d['pixels']) for d in items}, reverse=True)
        for rank, sz in enumerate(sizes[:2]):
            for d in items:
                if len(d['pixels']) == sz:
                    res.append((src, rank, np.asarray(d['grid']), d['position']))
    if len(_PART4ALL_MEMO) > 512: _PART4ALL_MEMO.clear()
    _PART4ALL_MEMO[key] = res
    return res

def _part24_match(i, o, tol=3, min_area=0.25):
    """[사용자 절차] 후보 x 변환을 output 위에서 x:0~len(x), y:0~len(y)
    슬라이드하며 일치 검사 (miss<=tol 허용, 크기 >= output의 25%).
    반환: [(src, rank, 변환, dy, dx, miss)] - 탐지/분류 신호용.
    - 배경 정규화: 감지 배경을 0으로 치환 후 추출 (추출기는 배경=0 가정)
    - 값 일치 또는 실루엣(색맹) 일치 - "가장큰 도형, 색깔" 유형 커버"""
    i = np.asarray(i); o = np.asarray(o)
    hits = []
    try:
        bg_i = _detect_bg2(i)
        i2 = np.where(i == bg_i, 0, i) if bg_i != 0 else i
        bg_o = _detect_bg2(o)
        o2 = np.where(o == bg_o, 0, o) if bg_o != 0 else o
    except Exception:
        return hits
    base = min(i2.size, o2.size)
    for src, rank, G, _pos in _part_extract4_all(i2):
        for tname, T in _D4_PART.items():
            V = T(G)
            vh, vw = V.shape
            if vh > o2.shape[0] or vw > o2.shape[1]: continue
            if V.size < min_area * base: continue        # 25% 필터 (min(입력,출력) 기준)
            nz = V != 0
            npx = int(nz.sum())
            if npx < 3: continue
            tol_sub = max(tol, int(0.25 * npx))          # 부분집합 매칭은 비례 허용
            for dy in range(o2.shape[0] - vh + 1):
                for dx in range(o2.shape[1] - vw + 1):
                    win = o2[dy:dy+vh, dx:dx+vw]
                    miss = int((win[nz] != V[nz]).sum())          # 1) 값 일치
                    if miss > tol:
                        miss = int(((win != 0) != nz).sum())      # 2) 실루엣 일치
                    if miss <= tol:
                        hits.append((src, rank, tname, dy, dx, miss)); continue
                    miss = int((win[nz] == 0).sum())              # 3) 부분집합(도형 존재)
                    if miss <= tol_sub:
                        hits.append((src, rank, tname, dy, dx, miss))
    return hits

# ----------------------------------------------------------- 도형 동작 (obj)
#   [동작 라이브러리 v1] 선택기 x 도출형 동작. 같은 크기 쌍 전용.
#   원칙: 동작 후보는 train 쌍에서 "도출"하고, 채택은 파이프라인의
#         train-exact + LOO 가 최종 심판 (개선도는 탐색 신호일 뿐).
#   동작 v1 (사용자 목록):
#     move_global   : 선택 도형을 전 쌍 동일 벡터로 이동          (이동 일치)
#     move_marker   : 마커(나머지 도형) 기준 상대 위치로 이동      (마커가 이동 암시)
#     d4_inplace    : 선택 도형만 제자리 회전/미러                (회전·미러 암시)
#     recolor_const : 선택 도형을 전 쌍 동일 색으로               (색깔)
#     recolor_marker: 마커의 색으로 재색칠                        (마커가 색 암시)
#     delete_sel    : 선택 도형 삭제
#     keep_only     : 선택 도형만 남김
_OBJ_SELS_MEMO = {}
def _obj_sels(g):
    """선택기: (이름, [(y,x,색)]) 목록. 동률은 각각 반환. (그리드 메모)"""
    g = np.asarray(g)
    _key = (g.shape, g.tobytes())
    if _key in _OBJ_SELS_MEMO: return _OBJ_SELS_MEMO[_key]
    bg = _detect_bg2(g)
    out = []
    try:
        objs = _objs8(g, bg)
    except Exception:
        objs = []
    if objs:
        sizes = sorted({len(t[2]) for t in objs}, reverse=True)
        for name, want in (('big', 0), ('second', 1), ('small', len(sizes) - 1)):
            if want >= len(sizes): continue
            for t in objs:
                if len(t[2]) == sizes[want]:
                    cells = [(y, x, int(g[y, x])) for y, x in t[2]]
                    out.append((name, cells))
    # 색깔별 최대 (배경 제외)
    cs = {int(c): int((g == c).sum()) for c in np.unique(g) if int(c) != bg}
    if cs:
        mx = max(cs.values())
        for c, n in cs.items():
            if n == mx:
                cells = [(int(y), int(x), c) for y, x in zip(*np.where(g == c))]
                out.append(('cbig', cells))
    if len(_OBJ_SELS_MEMO) > 512: _OBJ_SELS_MEMO.clear()
    _OBJ_SELS_MEMO[_key] = (out, bg)
    return out, bg

def _obj_find(sel_cells, g2):
    """g2 에서 sel 과 같은 (실루엣, 색배치) 도형의 평행이동 위치들을 찾음."""
    ys = [y for y, _x, _c in sel_cells]; xs = [x for _y, x, _c in sel_cells]
    y0, x0 = min(ys), min(xs)
    rel = sorted((y - y0, x - x0, c) for y, x, c in sel_cells)
    H, W = g2.shape
    h = max(ys) - y0 + 1; w = max(xs) - x0 + 1
    hits = []
    for py in range(H - h + 1):
        for px in range(W - w + 1):
            if all(0 <= py + dy < H and 0 <= px + dx < W and
                   g2[py + dy, px + dx] == c for dy, dx, c in rel):
                hits.append((py - y0, px - x0))       # 이동 벡터
    return hits

def _obj_apply_move(g, sel_cells, vec, bg):
    out = np.asarray(g).copy()
    for y, x, _c in sel_cells: out[y, x] = bg
    for y, x, c in sel_cells:
        ny, nx = y + vec[0], x + vec[1]
        if not (0 <= ny < out.shape[0] and 0 <= nx < out.shape[1]):
            raise ValueError('move out of grid')
        out[ny, nx] = c
    return out

def _obj_bbox_cells(cells):
    ys = [y for y, _x, _c in cells]; xs = [x for _y, x, _c in cells]
    return min(ys), min(xs), max(ys), max(xs)

def d_obj(pairs):
    ins = [np.asarray(e['input']) for e in pairs]
    outs = [np.asarray(e['output']) for e in pairs]
    if not all(i.shape == o.shape for i, o in zip(ins, outs)):
        return None
    if all(np.array_equal(i, o) for i, o in zip(ins, outs)):
        return None
    fns = []
    sel_names = ('big', 'second', 'small', 'cbig')
    for sname in sel_names:
        # --- 쌍별 관측: 선택 도형이 출력에서 어떻게 됐나 ---
        obs = []                     # (bg, sel, 이동벡터목록, 제자리색, 삭제여부)
        ok = True
        for i, o in zip(ins, outs):
            sels, bg = _obj_sels(i)
            cand = [c for n, c in sels if n == sname]
            if not cand: ok = False; break
            got = None
            for sel in cand:         # 동률 후보 중 관측 가능한 것
                vecs = _obj_find(sel, o)
                same_pos_col = None
                r0, c0, r1, c1 = _obj_bbox_cells(sel)
                oc = {int(o[y, x]) for y, x, _c in sel}
                if len(oc) == 1:
                    same_pos_col = oc.pop()
                deleted = all(o[y, x] == bg for y, x, _c in sel)
                got = (bg, sel, vecs, same_pos_col, deleted)
                if vecs or same_pos_col is not None or deleted: break
            if got is None: ok = False; break
            obs.append(got)
        if not ok: continue

        # 1) move_global: 전 쌍 공통 이동 벡터 (0,0 제외)
        common = None
        for _bg, _s, vecs, _sc, _d in obs:
            vs = {v for v in vecs if v != (0, 0)}
            common = vs if common is None else (common & vs)
            if not common: break
        if common:
            vec = sorted(common)[0]
            def fn(g, _sn=sname, _v=vec):
                sels, bg = _obj_sels(g)
                cand = [c for n, c in sels if n == _sn]
                if len(cand) != 1: raise ValueError('sel ambiguous')
                return _obj_apply_move(g, cand[0], _v, bg)
            fns.append(fn)

        # 2) move_marker: 이동 벡터가 (마커 bbox 시작 - 도형 bbox 시작) 과
        #    전 쌍 동일한 상대 오프셋 (마커 = small / cbig 이외 색)
        for mname in ('small',):
            offs = None; ok2 = True
            for (bgp, sel, vecs, _sc, _d), i in zip(obs, ins):
                sels, _b = _obj_sels(i)
                mk = [c for n, c in sels if n == mname]
                if len(mk) != 1 or not vecs: ok2 = False; break
                mr0, mc0, _r, _c = _obj_bbox_cells(mk[0])
                sr0, sc0, _r2, _c2 = _obj_bbox_cells(sel)
                rel = {(vy - (mr0 - sr0), vx - (mc0 - sc0)) for vy, vx in vecs}
                offs = rel if offs is None else (offs & rel)
                if not offs: ok2 = False; break
            if ok2 and offs:
                off = sorted(offs)[0]
                def fn(g, _sn=sname, _mn=mname, _o=off):
                    sels, bg = _obj_sels(g)
                    cand = [c for n, c in sels if n == _sn]
                    mk = [c for n, c in sels if n == _mn]
                    if len(cand) != 1 or len(mk) != 1:
                        raise ValueError('sel/marker ambiguous')
                    mr0, mc0, _r, _c = _obj_bbox_cells(mk[0])
                    sr0, sc0, _r2, _c2 = _obj_bbox_cells(cand[0])
                    vec = (mr0 - sr0 + _o[0], mc0 - sc0 + _o[1])
                    return _obj_apply_move(g, cand[0], vec, bg)
                fns.append(fn)

        # 3) recolor_const: 제자리 단색 재색칠, 전 쌍 동일 색
        cols = {sc for _b, _s, _v, sc, _d in obs if sc is not None}
        in_cols = {frozenset(c for _y, _x, c in s) for _b, s, _v, _sc, _d in obs}
        if len(cols) == 1 and all(sc is not None for _b, _s, _v, sc, _d in obs):
            c_t = cols.pop()
            if not all(fs == frozenset([c_t]) for fs in in_cols):   # 실제 변화
                def fn(g, _sn=sname, _c=c_t):
                    sels, bg = _obj_sels(g)
                    cand = [c for n, c in sels if n == _sn]
                    if len(cand) != 1: raise ValueError('sel ambiguous')
                    out = np.asarray(g).copy()
                    for y, x, _cc in cand[0]: out[y, x] = _c
                    return out
                fns.append(fn)

        # 4) recolor_marker: 재색칠 색 == 마커(small)의 색, 전 쌍 성립
        ok3 = all(sc is not None for _b, _s, _v, sc, _d in obs)
        if ok3:
            match = True
            for (_b, _s, _v, sc, _d), i in zip(obs, ins):
                sels, _bg = _obj_sels(i)
                mk = [c for n, c in sels if n == 'small']
                mc = {cc for _y, _x, cc in mk[0]} if len(mk) == 1 else set()
                if mc != {sc}: match = False; break
            if match:
                def fn(g, _sn=sname):
                    sels, bg = _obj_sels(g)
                    cand = [c for n, c in sels if n == _sn]
                    mk = [c for n, c in sels if n == 'small']
                    if len(cand) != 1 or len(mk) != 1:
                        raise ValueError('sel/marker ambiguous')
                    mc = {cc for _y, _x, cc in mk[0]}
                    if len(mc) != 1: raise ValueError('marker multicolor')
                    out = np.asarray(g).copy(); c_t = mc.pop()
                    for y, x, _cc in cand[0]: out[y, x] = c_t
                    return out
                fns.append(fn)

        # 5) d4_inplace: 선택 도형 bbox 만 D4 변환, 나머지 불변 (전 쌍 동일 변환)
        for tname, T in _D4_PART.items():
            if tname == 'id': continue
            ok4 = True
            for (bgp, sel, _v, _sc, _d), i, o in zip(obs, ins, outs):
                r0, c0, r1, c1 = _obj_bbox_cells(sel)
                blk = np.asarray(i)[r0:r1+1, c0:c1+1]
                tb = T(blk)
                if tb.shape != blk.shape: ok4 = False; break
                t_o = np.asarray(o).copy()
                exp = np.asarray(i).copy(); exp[r0:r1+1, c0:c1+1] = tb
                if not np.array_equal(exp, t_o): ok4 = False; break
            if ok4:
                def fn(g, _sn=sname, _T=T):
                    sels, bg = _obj_sels(g)
                    cand = [c for n, c in sels if n == _sn]
                    if len(cand) != 1: raise ValueError('sel ambiguous')
                    r0, c0, r1, c1 = _obj_bbox_cells(cand[0])
                    out = np.asarray(g).copy()
                    blk = out[r0:r1+1, c0:c1+1]
                    tb = _T(blk)
                    if tb.shape != blk.shape: raise ValueError('bbox not square')
                    out[r0:r1+1, c0:c1+1] = tb
                    return out
                fns.append(fn)

        # 7) move_dir: 선택 도형을 4방향으로 벽/장애물까지 슬라이드 (도출 불필요)
        for _d in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            def fn(g, _sn=sname, _dd=_d):
                sels, bg = _obj_sels(g)
                cand = [c for n, c in sels if n == _sn]
                if len(cand) != 1: raise ValueError('sel ambiguous')
                sel = cand[0]
                out = np.asarray(g).copy()
                cur = {(y, x) for y, x, _c in sel}
                for y, x, _c in sel: out[y, x] = bg
                vy = vx = 0
                while True:
                    ny, nx = vy + _dd[0], vx + _dd[1]
                    ok = True
                    for y, x, _c in sel:
                        ty, tx = y + ny, x + nx
                        if not (0 <= ty < out.shape[0] and 0 <= tx < out.shape[1]) \
                           or out[ty, tx] != bg:
                            ok = False; break
                    if not ok: break
                    vy, vx = ny, nx
                for y, x, c in sel: out[y + vy, x + vx] = c
                return out
            fns.append(fn)
        # 8) copy_to_markers: 나머지 도형(마커) 위치마다 선택 도형 사본 추가
        #    정렬 2종(bbox 좌상단 / 중심) x 마커 유지 -> exact 필터가 판정
        for _align in ('tl', 'center'):
            def fn(g, _sn=sname, _al=_align):
                sels, bg = _obj_sels(g)
                cand = [c for n, c in sels if n == _sn]
                if len(cand) != 1: raise ValueError('sel ambiguous')
                sel = cand[0]
                r0, c0, r1, c1 = _obj_bbox_cells(sel)
                selset = {(y, x) for y, x, _c in sel}
                out = np.asarray(g).copy()
                others = [c for n, c in sels
                          if n in ('small', 'second') and
                          not ({(y, x) for y, x, _c in c} & selset)]
                if not others: raise ValueError('no markers')
                seen = set()
                for mk in others:
                    m0 = _obj_bbox_cells(mk)[:2]
                    if m0 in seen: continue
                    seen.add(m0)
                    if _al == 'tl':
                        oy, ox = m0[0] - r0, m0[1] - c0
                    else:
                        my = (m0[0] + _obj_bbox_cells(mk)[2]) // 2
                        mx = (m0[1] + _obj_bbox_cells(mk)[3]) // 2
                        oy, ox = my - (r0 + r1) // 2, mx - (c0 + c1) // 2
                    for y, x, c in sel:
                        ty, tx = y + oy, x + ox
                        if not (0 <= ty < out.shape[0] and 0 <= tx < out.shape[1]):
                            raise ValueError('copy out of grid')
                        out[ty, tx] = c
                return out
            fns.append(fn)
        # 6) delete_sel / keep_only
        if all(d for _b, _s, _v, _sc, d in obs):
            def fn(g, _sn=sname):
                sels, bg = _obj_sels(g)
                cand = [c for n, c in sels if n == _sn]
                if len(cand) != 1: raise ValueError('sel ambiguous')
                out = np.asarray(g).copy()
                for y, x, _c in cand[0]: out[y, x] = bg
                return out
            fns.append(fn)

    # 9) recolor_by_shape: 도형 실루엣(모양)이 색을 암시 - canon -> 색 사상을
    #    전 쌍·전 도형에서 도출, 일관되면 전 도형 재색칠 fn
    try:
        cmap = {}
        ok9 = True
        for i, o in zip(ins, outs):
            bg = _detect_bg2(i)
            for t in _objs8(i, bg):
                canon = _d4_canon(t[0])
                ocs = {int(o[y, x]) for y, x in t[2]}
                if len(ocs) != 1: ok9 = False; break
                c_t = ocs.pop()
                if cmap.get(canon, c_t) != c_t: ok9 = False; break
                cmap[canon] = c_t
            if not ok9: break
        # 사상이 실제 변화를 만들 때만 (입력색 그대로면 무의미)
        if ok9 and cmap:
            def fn(g, _m=dict(cmap)):
                g = np.asarray(g)
                bg = _detect_bg2(g)
                out = g.copy()
                for t in _objs8(g, bg):
                    canon = _d4_canon(t[0])
                    if canon not in _m: raise ValueError('unknown shape')
                    for y, x in t[2]: out[y, x] = _m[canon]
                return out
            fns.append(fn)
    except Exception:
        pass
    # 전역 동작 v2 추가
    try:
        fns.extend(_obj_global_fns(pairs, ins, outs))
    except Exception:
        pass
    # 필터: 단독 정확 일치 우선, 아니면 크기 유지 fn 은 파이프라인용으로 뒤에
    exact, rest = [], []
    for fn in fns:
        try:
            good = all(np.array_equal(np.asarray(fn(i)), o)
                       for i, o in zip(ins, outs))
        except Exception:
            continue
        (exact if good else rest).append(fn)
    out = exact + rest
    return out if out else None

# ----------------------------------------------------------- 동작 v2 (전역 동작)
# [사용자 CoT 기록 - 나중에 LLM 프롬프트 제작용]
#  #28 0c786b71: 가장큰 도형=모양. 색깔은 좌상단부터 x축->y축 스캔 순서로 뽑음. (tile 이 해결)
#  #49 13f06aa5: 같은 모양 중 더 많은 도형의 색을 가장큰 도형에 색칠. 특이 셀에서 광선. (미구현)
#  #52 1478ab18: 나머지 도형 개수 x%4 가 회전각(1=90,2=180,3=270). (미구현)
#  #69 195ba7dc: 회전까지 뒤져 같은 모양을 찾아 가장큰 도형에 색칠. (panels 가 해결)
#  #80 1b8318e3: 2번째 큰 도형이 output 크기. 안의 점을 가장큰 도형 모양으로 교체. (미구현)
#  #85 1c56ad9f: 불완전 도형 완성 후 추가 = 행들이 상대 인덱스 주기로 좌/우 1칸 이동. (roll_rows)
#  #112 22168020: 행마다 같은 색 최좌-최우 사이 채움. (fill_between)
#  #114 22233c11: 대각 인접 같은 모양 쌍의 연장선에 새 색 블록 추가. (extend_diag)
#  #119 22a4bbc2: 위->아래 도형 순서에서 주기 k 마다 재색칠. (recolor_kth)
_OBJL_MEMO = {}
def _obj_global_fns(pairs, ins, outs):
    fns = []
    # A) fill_between: 각 행(열)에서 색 c 의 최좌(상)-최우(하) 구간을 c 로 채움
    def _fill(g, axis):
        g = np.asarray(g); out = g.copy()
        bg = _detect_bg2(g)
        it = out if axis == 0 else out.T
        src = g if axis == 0 else g.T
        for y in range(src.shape[0]):
            row = src[y]
            for c in np.unique(row):
                if int(c) == bg: continue
                xs = np.where(row == c)[0]
                if len(xs) >= 2:
                    it[y, xs.min():xs.max()+1] = c
        return out
    for axis in (0, 1):
        fns.append(lambda g, _a=axis: _fill(g, _a))

    # B) extend_diag: 대각 블록-인접(오프셋 = bbox 크기) 같은 모양·색 쌍의
    #    연장선 양끝에 새 색 블록 추가 (새 색은 train 에서 도출)
    def _objs_list(g):
        """4-연결·색 단위 도형. 그리드 메모 + 도형수 상한 (폭주 방지)"""
        g = np.asarray(g)
        key = (g.shape, g.tobytes())
        if key in _OBJL_MEMO:
            v = _OBJL_MEMO[key]
            if isinstance(v, Exception): raise v
            return v
        try:
            if 0 in np.unique(g):
                bg, g2 = 0, g
            else:
                bg = _detect_bg2(g)
                g2 = np.where(g == bg, 0, g)
            E = DiagonalShapeExtractor()
            regs = E.extract_all_regions(g2.tolist())
            if len(regs) > 40: raise ValueError('too many objects')
            res = []
            for d in regs:
                py, px = d['position']
                ys = [y for y, x in d['pixels']]; xs = [x for y, x in d['pixels']]
                sil = frozenset((y - py, x - px) for y, x in d['pixels'])
                col = {int(g2[y, x]) for y, x in d['pixels']}
                res.append((sil, col, (py, px),
                            max(ys) - py + 1, max(xs) - px + 1))
            out = (res, bg)
        except Exception as ex:
            if len(_OBJL_MEMO) > 512: _OBJL_MEMO.clear()
            _OBJL_MEMO[key] = ex
            raise
        if len(_OBJL_MEMO) > 512: _OBJL_MEMO.clear()
        _OBJL_MEMO[key] = out
        return out
    def _extend(g, cnew):
        g = np.asarray(g); out = g.copy()
        objs, bg = _objs_list(g)
        hit = False
        for i1 in range(len(objs)):
            for i2 in range(len(objs)):
                if i1 == i2: continue
                s1, c1, p1, h1, w1 = objs[i1]; s2, c2, p2, h2, w2 = objs[i2]
                if s1 != s2 or c1 != c2: continue
                dy, dx = p2[0] - p1[0], p2[1] - p1[1]
                if abs(dy) != h1 or abs(dx) != w1: continue
                for base, sgn in ((p1, -1), (p2, +1)):
                    ny, nx = base[0] + sgn * dy, base[1] + sgn * dx
                    for ry, rx in s1:
                        yy, xx = ny + ry, nx + rx
                        if 0 <= yy < out.shape[0] and 0 <= xx < out.shape[1]:
                            out[yy, xx] = cnew; hit = True
        if not hit: raise ValueError('no diag pair')
        return out
    _newc = set()
    for i, o in zip(ins, outs):
        d = o[(o != i)]
        _newc |= {int(v) for v in np.unique(d)}
    for cnew in sorted(_newc):
        fns.append(lambda g, _c=cnew: _extend(g, _c))

    # C) recolor_kth: 읽기 순서(위->아래, 좌->우) 도형 중 idx%k==r 을 색 c 로
    def _reading(g):
        objs, bg = _objs_list(g)
        return sorted(objs, key=lambda t: (t[2][0], t[2][1])), bg
    def _rk(g, k, r, c):
        g = np.asarray(g); out = g.copy()
        objs, bg = _reading(g)
        if len(objs) <= max(1, r): raise ValueError('few objs')
        for idx, (sil, col, pos, h, w) in enumerate(objs):
            if idx % k == r:
                for ry, rx in sil: out[pos[0] + ry, pos[1] + rx] = c
        return out
    # (k, r, c) 도출: 첫 쌍에서 바뀐 도형 인덱스들로 추정 -> 전 쌍 검산은 필터가
    try:
        i0, o0 = ins[0], outs[0]
        objs0, _bg = _reading(i0)
        chg = [idx for idx, (sil, col, pos, h, w) in enumerate(objs0)
               if any(o0[pos[0]+ry, pos[1]+rx] != i0[pos[0]+ry, pos[1]+rx]
                      for ry, rx in sil)]
        if chg:
            newcols = {int(o0[objs0[chg[0]][2][0]+ry, objs0[chg[0]][2][1]+rx])
                       for ry, rx in objs0[chg[0]][0]}
            if len(newcols) == 1:
                _c = newcols.pop()
                for k in range(2, 7):
                    if all(x % k == chg[0] % k for x in chg):
                        fns.append(lambda g, _k=k, _r=chg[0] % k, _cc=_c:
                                   _rk(g, _k, _r, _cc))
    except Exception:
        pass

    # D) roll_rows: 가장큰 도형 bbox 기준 상대 행 인덱스의 주기 수열대로 행을 밀기
    def _bigbbox(g):
        g = np.asarray(g); bg = _detect_bg2(g)
        objs = _objs8(g, bg)
        if not objs: raise ValueError('no obj')
        big = max(objs, key=lambda t: len(t[2]))
        ys = [y for y, x in big[2]]
        return min(ys), max(ys)
    try:
        seq = {}
        ok = True
        for i, o in zip(ins, outs):
            r0, r1 = _bigbbox(i)
            for y in range(i.shape[0]):
                ds = [d for d in (-2, -1, 0, 1, 2)
                      if np.array_equal(np.roll(i[y], d), o[y])]
                if not ds: ok = False; break
                d = 0 if 0 in ds else ds[0]
                if y < r0 or y > r1:
                    if d != 0: ok = False; break
                    continue
                rel = y - r0
                if seq.get(rel, d) != d: ok = False; break
                seq[rel] = d
            if not ok: break
        if ok and seq and any(v != 0 for v in seq.values()):
            n = max(seq) + 1
            full = [seq.get(t, 0) for t in range(n)]
            per = next((p for p in range(1, n + 1)
                        if all(full[t] == full[t % p] for t in range(n))), n)
            pat = full[:per]
            def fn(g, _p=pat):
                g = np.asarray(g); out = g.copy()
                r0, r1 = _bigbbox(g)
                for y in range(r0, r1 + 1):
                    d = _p[(y - r0) % len(_p)]
                    if d: out[y] = np.roll(g[y], d)
                return out
            fns.append(fn)
    except Exception:
        pass
    return fns


# ===== largest_fill v2: geom_d4(심) + 서브 솔버 7종 (다른 세션 검증본 이식) =====
# ============================================================================
# geom_d4 (재구성 심): D4 군 연산 + 양방향 배율 매칭
#   largest_fill_stage.py 사용처에서 시그니처 역산. 표적 검증으로 동치 확인.
# ============================================================================
import numpy as np

D4_NAMES = ['id', 'rot90', 'rot180', 'rot270', 'mx', 'my', 'mx_rot90', 'my_rot90']

def apply_d4_grid(grid, name):
    a = np.asarray(grid)
    if name == 'id':        r = a
    elif name == 'rot90':   r = np.rot90(a, -1)
    elif name == 'rot180':  r = np.rot90(a, 2)
    elif name == 'rot270':  r = np.rot90(a, 1)
    elif name == 'mx':      r = np.flipud(a)
    elif name == 'my':      r = np.fliplr(a)
    elif name == 'mx_rot90': r = np.rot90(np.flipud(a), -1)
    elif name == 'my_rot90': r = np.rot90(np.fliplr(a), -1)
    else: raise ValueError(name)
    return [list(map(int, row)) for row in r]

def normalize_cells(cells):
    ys = min(p[0] for p in cells); xs = min(p[1] for p in cells)
    return frozenset((y - ys, x - xs) for (y, x) in cells)

def _apply_d4_cells(cells, name):
    H = max(p[0] for p in cells) + 1
    W = max(p[1] for p in cells) + 1
    out = set()
    for (y, x) in cells:
        if name == 'id':        p = (y, x)
        elif name == 'rot90':   p = (x, H - 1 - y)
        elif name == 'rot180':  p = (H - 1 - y, W - 1 - x)
        elif name == 'rot270':  p = (W - 1 - x, y)
        elif name == 'mx':      p = (H - 1 - y, x)
        elif name == 'my':      p = (y, W - 1 - x)
        elif name == 'mx_rot90': p = (x, y)                     # flipud 후 rot90
        elif name == 'my_rot90': p = (W - 1 - x, H - 1 - y)
        else: raise ValueError(name)
        out.add(p)
    return normalize_cells(frozenset(out))

def d4_variants_cells(cells):
    cells = normalize_cells(frozenset(cells))
    return [(name, _apply_d4_cells(cells, name)) for name in D4_NAMES]

def canonical_cells(cells):
    return min((v for _n, v in d4_variants_cells(cells)), key=lambda s: sorted(s))

def match_d4_cells(a, b):
    """a 를 D4 변환해 b(정규화 실루엣)와 일치하면 변환 이름, 아니면 None"""
    b = normalize_cells(frozenset(b))
    for name, v in d4_variants_cells(a):
        if v == b: return name
    return None

def scale_cells(cells, k):
    """셀 집합 k배 확대 (각 셀 -> k x k 블록)"""
    out = set()
    for (y, x) in cells:
        for dy in range(k):
            for dx in range(k):
                out.add((y * k + dy, x * k + dx))
    return frozenset(out)

def _downscale_cells(cells, k):
    """k배 축소: k x k 블록이 전부 차있을 때만. 아니면 None"""
    cells = set(cells)
    H = max(p[0] for p in cells) + 1; W = max(p[1] for p in cells) + 1
    if H % k or W % k: return None
    out = set()
    for by in range(H // k):
        for bx in range(W // k):
            blk = [(by * k + dy, bx * k + dx) for dy in range(k) for dx in range(k)]
            inside = sum(1 for p in blk if p in cells)
            if inside == k * k: out.add((by, bx))
            elif inside != 0: return None
    return frozenset(out) if out else None

def scale_grid(grid, k):
    a = np.asarray(grid)
    return [list(map(int, row)) for row in np.kron(a, np.ones((k, k), dtype=a.dtype))]

def unscale_grid(grid, k):
    a = np.asarray(grid); H, W = a.shape
    if H % k or W % k: return None
    out = a.reshape(H // k, k, W // k, k)
    first = out[:, 0, :, 0]
    if not (out == first[:, None, :, None]).all(): return None   # 블록 불균일
    return [list(map(int, row)) for row in first]

def infer_scale(a_cells, b_cells, max_k=4):
    """b == a x k (정수 k, D4 허용) 이면 k, b == a / k 이면 (1,k) ... 간이판"""
    na, nb = len(a_cells), len(b_cells)
    for k in range(1, max_k + 1):
        if nb == na * k * k: return (k, 1)
        if na == nb * k * k: return (1, k)
    return None

def match_d4_ratio(comp, shape_cells, max_k=4):
    """홀(comp, 절대좌표)과 도형(shape_cells, 정규화)을 D4 + 배율로 매칭.
    반환 (d4이름, num, den): d4(shape) x (num/den) == normalize(comp).
    num/den 우선순위: 동배율(1,1) -> 확대(2,3,4) -> 축소(1/2,1/3,1/4)"""
    comp_n = normalize_cells(frozenset(comp))
    nc, ns = len(comp_n), len(shape_cells)
    ratios = [(1, 1)] + [(k, 1) for k in range(2, max_k + 1)] + \
             [(1, k) for k in range(2, max_k + 1)]
    for num, den in ratios:
        if nc * den * den != ns * num * num * (den * den) // (den * den):  # 자리표
            pass
        if den == 1:
            if nc != ns * num * num: continue
            for name, v in d4_variants_cells(shape_cells):
                if (scale_cells(v, num) if num > 1 else v) == comp_n:
                    return (name, num, den)
        else:
            if ns != nc * den * den: continue
            small = _downscale_cells(comp_n, 1)  # comp 그대로
            for name, v in d4_variants_cells(shape_cells):
                dv = _downscale_cells(v, den)
                if dv is not None and normalize_cells(dv) == comp_n:
                    return (name, num, den)
    return None

# ============================================================================
# largest_fill 스테이지: "가장 큰 도형 = 템플릿, 나머지 도형 = 채움"
#
# 분류: 일치(나머지 도형 셀 수 == 홀 셀 수) → 슬라이딩 패킹(exact packing)
#       과잉(나머지 도형 셀 수 >  홀 셀 수) → 홀 모양 매칭 + 같은 모양 중 다수결 색
#
# 게이트: 모든 train 쌍 train-exact 재현 시에만 답 채택 (기존 엔진 철학과 동일)
# 추출 3종을 순서대로 시도하고, train-exact를 통과하는 첫 종류를 채택
# ============================================================================
from collections import Counter, defaultdict
import numpy as np
from scipy import ndimage
# geom_d4: 아래 인라인 정의 사용 (원본 geom_d4.py 셀이 있으면 그 정의가 우선)

EMPTY = -1  # 내부 표현: 배경(빈칸). 0이 실제 색일 수 있으므로 0을 쓰지 않음


# 하위 호환 별칭 (기존 호출부 유지)
normalize = normalize_cells
canonical = canonical_cells


# ---------------------------------------------------------------------------
# 배경 검출 (엔진의 _detect_bg_conf 단순판: 최빈색)
# ---------------------------------------------------------------------------
def detect_bg(grid):
    return Counter(v for row in grid for v in row).most_common(1)[0][0]


# ---------------------------------------------------------------------------
# 도형 추출 3종 (사용자 고유 코드 의미 보존, bg 파라미터화)
#   diagonal : extract_tiny_diagonal_regions 방식 (8-conn, 다색 허용)
#   regions  : extract_all_regions 방식        (색깔별 4-conn)
#   shapes   : extract_all_shapes 방식         (색깔별 전체 셀 = 도형 1개)
# 반환 도형: {'cells': {(y,x): color}, 'pos', 'size', 'colors'}
# ---------------------------------------------------------------------------
def extract_shapes(grid, bg, kind):
    arr = np.array(grid)
    h, w = arr.shape
    shapes = []

    def make_shape(cells):
        ys = [p[0] for p in cells]; xs = [p[1] for p in cells]
        return {
            'cells': {(y, x): int(arr[y, x]) for (y, x) in cells},
            'pos': (min(ys), min(xs)),
            'size': (max(ys) - min(ys) + 1, max(xs) - min(xs) + 1),
            'colors': Counter(int(arr[y, x]) for (y, x) in cells),
        }

    if kind == 'diagonal':
        # 배경 제외 8-conn (다색 허용) — DiagonalShapeExtractor.extract_tiny_diagonal_regions
        structure = np.ones((3, 3), dtype=int)
        labeled, n = ndimage.label(arr != bg, structure=structure)
        for rid in range(1, n + 1):
            cells = [(int(y), int(x)) for y, x in np.argwhere(labeled == rid)]
            shapes.append(make_shape(cells))

    elif kind == 'regions':
        # 색깔별 4-conn — extract_all_regions
        for color in np.unique(arr):
            if color == bg:
                continue
            labeled, n = ndimage.label(arr == color)
            for rid in range(1, n + 1):
                cells = [(int(y), int(x)) for y, x in np.argwhere(labeled == rid)]
                shapes.append(make_shape(cells))

    elif kind == 'shapes':
        # 색깔별 전체 — extract_all_shapes
        for color in np.unique(arr):
            if color == bg:
                continue
            cells = [(int(y), int(x)) for y, x in np.argwhere(arr == color)]
            if cells:
                shapes.append(make_shape(cells))
    return shapes


# ---------------------------------------------------------------------------
# 정규화/캐노니컬 — geom_d4 모듈로 이관 (상단 별칭 normalize/canonical 사용)
# ---------------------------------------------------------------------------

# ---------------------------------------------------------------------------
# 캔버스 구성: 가장 큰 도형 bbox 크롭, 템플릿 셀만 유지(나머지는 EMPTY=홀)
# ---------------------------------------------------------------------------
def build_canvas(template):
    (y0, x0), (H, W) = template['pos'], template['size']
    canvas = [[EMPTY] * W for _ in range(H)]
    for (y, x), c in template['cells'].items():
        canvas[y - y0][x - x0] = c
    holes = {(r, c) for r in range(H) for c in range(W) if canvas[r][c] == EMPTY}
    return canvas, holes


# ---------------------------------------------------------------------------
# 일치 모드: exact-cover 백트래킹 패킹
#   각 도형을 (0..H-h, 0..W-w) 슬라이딩하며 홀에 정확히 얹히는 배치를 탐색
#   가장 앞선 미채움 홀 셀을 덮는 배치만 분기 → 결정적/고속
# ---------------------------------------------------------------------------
def pack_exact(canvas, holes, rest):
    H, W = len(canvas), len(canvas[0])
    # (정규모양, 색) 동일 도형은 개수로 묶음 — 교환 가능하므로 탐색 축소
    groups = Counter()
    for s in rest:
        groups[(normalize(frozenset(s['cells'].keys())), s['colors'].most_common(1)[0][0])] += 1
    groups = dict(groups)

    # 모양별 전체 배치 후보 사전 계산 (0..H-h, 0..W-w 슬라이딩)
    placements = {}
    for (norm, color) in groups:
        h = max(p[0] for p in norm) + 1
        w = max(p[1] for p in norm) + 1
        plist = []
        for dy in range(H - h + 1):
            for dx in range(W - w + 1):
                cells = frozenset((p[0] + dy, p[1] + dx) for p in norm)
                if cells <= holes:
                    plist.append(cells)
        placements[(norm, color)] = plist

    result = {}
    budget = [200000]  # 탐색 노드 상한 (폭주 방지)

    def dfs(unfilled, remaining):
        budget[0] -= 1
        if budget[0] <= 0:
            return False
        if not unfilled:
            return True
        target = min(unfilled)  # 가장 앞선 홀 셀
        for key, cnt in list(remaining.items()):
            if cnt == 0:
                continue
            for cells in placements[key]:
                if target in cells and cells <= unfilled:
                    remaining[key] -= 1
                    for cell in cells:
                        result[cell] = key[1]
                    if dfs(unfilled - cells, remaining):
                        return True
                    remaining[key] += 1
                    for cell in cells:
                        del result[cell]
        return False

    if not dfs(frozenset(holes), dict(groups)):
        return None
    out = [row[:] for row in canvas]
    for (r, c), color in result.items():
        out[r][c] = color
    return out


# ---------------------------------------------------------------------------
# 과잉 모드: 홀 연결요소별 모양 매칭(회전/미러 동치) 후 같은 모양 중 다수결 색
# ---------------------------------------------------------------------------
def fill_majority(canvas, holes, rest):
    H, W = len(canvas), len(canvas[0])
    mask = np.zeros((H, W), dtype=int)
    for (r, c) in holes:
        mask[r, c] = 1
    labeled, n = ndimage.label(mask)  # 홀 = 4-conn 연결요소

    # 나머지 도형: 캐노니컬 클래스별 색 카운트
    class_color_count = defaultdict(Counter)
    for s in rest:
        if len(s['colors']) != 1:
            continue  # 색깔 도형은 단색 전제
        color = next(iter(s['colors']))
        class_color_count[canonical(frozenset(s['cells'].keys()))][color] += 1

    out = [row[:] for row in canvas]
    for rid in range(1, n + 1):
        comp = frozenset((int(y), int(x)) for y, x in np.argwhere(labeled == rid))
        cc = canonical(comp)
        if cc not in class_color_count:
            return None  # 홀에 맞는 모양 없음 → 포기
        counts = class_color_count[cc].most_common()
        if len(counts) > 1 and counts[0][1] == counts[1][1]:
            return None  # 다수결 동률 → 포기
        for (r, c) in comp:
            out[r][c] = counts[0][0]
    return out


# ---------------------------------------------------------------------------
# 템플릿 선택
#   single : 가장 큰 도형 1개 (동률이면 포기)
#   merge  : 가장 큰 도형과 색 구성(집합)이 같은 도형을 모두 병합
#            (예: 112 train0 — 3색 템플릿이 좌우 두 조각으로 분리된 경우)
# ---------------------------------------------------------------------------
def pick_template(shapes, template_mode):
    shapes = sorted(shapes, key=lambda s: -len(s['cells']))
    if template_mode == 'single':
        template, rest = shapes[0], shapes[1:]
    else:  # merge
        sig = frozenset(shapes[0]['colors'].keys())
        merged_cells = {}
        rest = []
        for s in shapes:
            if frozenset(s['colors'].keys()) == sig:
                merged_cells.update(s['cells'])
            else:
                rest.append(s)
        ys = [p[0] for p in merged_cells]; xs = [p[1] for p in merged_cells]
        template = {
            'cells': merged_cells,
            'pos': (min(ys), min(xs)),
            'size': (max(ys) - min(ys) + 1, max(xs) - min(xs) + 1),
            'colors': Counter(merged_cells.values()),
        }
    # 템플릿이 나머지보다 확실히 커야 함 (동률 → 모호 → 포기)
    if rest and len(template['cells']) <= max(len(s['cells']) for s in rest):
        return None, None
    return template, rest


# ---------------------------------------------------------------------------
# 과잉 모드 2: 스케일 + D4 매칭 (69번 유형의 일반화)
#   홀 연결요소와 나머지 도형을 양방향 배율(2,3,4배 & 1/2,1/3,1/4배) + D4 변환
#   (90/180/270 회전, x/y 미러, 미러+회전)으로 매칭하여 색을 채움.
#   smode:
#     uniform      — 모든 홀이 같은 배율 r로 매칭, 출력 = 템플릿 × r (기존 69번 동작)
#     uniform_orig — 모든 홀이 같은 배율 r로 매칭, 출력 = 원본 크기(×1)
#     free_orig    — 홀마다 독립 배율 허용, 출력 = 원본 크기(×1)
#   매칭 안 된 잉여 도형은 노이즈로 무시. 색 동률이면 포기.
# ---------------------------------------------------------------------------
MAX_SCALE = 4  # 배율 상한: 4배 ~ 1/4배


def _hole_components(canvas, holes):
    H, W = len(canvas), len(canvas[0])
    mask = np.zeros((H, W), dtype=int)
    for (r, c) in holes:
        mask[r, c] = 1
    labeled, n = ndimage.label(mask)
    return [frozenset((int(y), int(x)) for y, x in np.argwhere(labeled == rid))
            for rid in range(1, n + 1)]


def _pick_color(votes):
    """엄격 다수결: 동률이면 None"""
    if not votes:
        return None
    top = votes.most_common()
    if len(top) > 1 and top[0][1] == top[1][1]:
        return None
    return top[0][0]


def fill_scaled_d4(canvas, holes, rest, smode='uniform'):
    comps = _hole_components(canvas, holes)
    if not comps:
        return None
    # 단색 도형만 색 후보
    mono = [(next(iter(s['colors'])), normalize_cells(frozenset(s['cells'].keys())))
            for s in rest if len(s['colors']) == 1]

    # 각 홀 × 도형: 양방향 배율 매칭 (셀 수 비율이 배율을 유일하게 결정)
    per_comp = []  # [ {(num,den): Counter(color)} ]
    for comp in comps:
        by_ratio = defaultdict(Counter)
        for color, cells in mono:
            m = match_d4_ratio(comp, cells, max_k=MAX_SCALE)
            if m:
                name, num, den = m
                by_ratio[(num, den)][color] += 1
        per_comp.append(by_ratio)

    if smode == 'free_orig':
        filled = [row[:] for row in canvas]
        for comp, by_ratio in zip(comps, per_comp):
            votes = Counter()
            for cnt in by_ratio.values():
                votes.update(cnt)
            color = _pick_color(votes)
            if color is None:
                return None
            for (r, c) in comp:
                filled[r][c] = color
        return filled

    # uniform / uniform_orig: 모든 홀 공통 배율 r
    common = set(per_comp[0])
    for by_ratio in per_comp[1:]:
        common &= set(by_ratio)
    # 결정적 순서: 동배율 → 확대(2,3,4) → 축소(1/2,1/3,1/4)
    for (num, den) in sorted(common, key=lambda r: (r[1], r[0])):
        assign = {}
        ok = True
        for ci, by_ratio in enumerate(per_comp):
            color = _pick_color(by_ratio[(num, den)])
            if color is None:
                ok = False
                break
            assign[ci] = color
        if not ok:
            continue
        filled = [row[:] for row in canvas]
        for ci, comp in enumerate(comps):
            for (r, c) in comp:
                filled[r][c] = assign[ci]
        if smode == 'uniform_orig':
            return filled
        # uniform: 출력에 배율 반영
        if den == 1:
            return filled if num == 1 else scale_grid(filled, num)
        out = unscale_grid(filled, den)  # 블록 불균일 시 None
        if out is not None:
            return out
    return None


# ---------------------------------------------------------------------------
# 단일 그리드 풀이
# ---------------------------------------------------------------------------
def solve_one(grid, kind, template_mode='single', smode='uniform'):
    bg = detect_bg(grid)
    shapes = extract_shapes(grid, bg, kind)
    if len(shapes) < 2:
        return None
    template, rest = pick_template(shapes, template_mode)
    if template is None:
        return None
    canvas, holes = build_canvas(template)
    rest_cells = sum(len(s['cells']) for s in rest)
    if not holes:
        return None
    if rest_cells == len(holes):        # 일치
        out = pack_exact(canvas, holes, rest)
    elif rest_cells > len(holes):       # 과잉
        out = fill_majority(canvas, holes, rest)
        if out is None:                 # 폴백: 양방향 배율 + D4 (회전/미러) 매칭
            out = fill_scaled_d4(canvas, holes, rest, smode)
    else:                               # 부족 → 이 스테이지 대상 아님
        return None
    if out is None or any(EMPTY in row for row in out):
        return None
    return out


# ---------------------------------------------------------------------------
# 태스크 풀이 + train-exact 게이트
# 반환: (test 답 리스트, 채택 kind, 모드) 또는 (None, None, None)
# ---------------------------------------------------------------------------
KINDS = ['diagonal', 'regions', 'shapes']
TEMPLATE_MODES = ['single', 'merge']
SCALE_MODES = ['uniform', 'uniform_orig', 'free_orig']


def solve_largest_fill(task):
    for kind in KINDS:
        for tmode in TEMPLATE_MODES:
            for smode in SCALE_MODES:
                ok = True
                for pair in task['train']:
                    pred = solve_one(pair['input'], kind, tmode, smode)
                    if pred is None or pred != pair['output']:
                        ok = False
                        break
                if not ok:
                    continue
                preds = []
                for t in task['test']:
                    p = solve_one(t['input'], kind, tmode, smode)
                    if p is None:
                        ok = False
                        break
                    preds.append(p)
                if ok:
                    return preds, f"{kind}/{tmode}/{smode}", 'train_exact'
    return None, None, None


# ============================================================================
# 스테이지 2: rotate_count — 나머지 도형 개수(셀 수)가 회전각을 의미 (52번 유형)
#   각 색 레이어가 자기 색 바깥 셀 수 x에 대해 90도 × (x % 4) 회전
#   (1→90, 2→180, 3→270, 0→0, 5→90, 7→270)
#   바깥에 없는 색은 0도. 캔버스는 정사각형이어야 90/270 가능.
# ============================================================================
def _rotate_layer_cells(cells, H, W, k, direction):
    """캔버스(H×W) 안에서 셀 좌표를 90도 × k 회전"""
    for _ in range(k % 4):
        if direction == 'cw':
            cells = [(x, H - 1 - y) for (y, x) in cells]
        else:
            cells = [(W - 1 - x, y) for (y, x) in cells]
        H, W = W, H
    return cells, H, W


def solve_rotation_count_one(grid, kind, direction):
    bg = detect_bg(grid)
    shapes = extract_shapes(grid, bg, kind)
    if len(shapes) < 2:
        return None
    shapes.sort(key=lambda s: -len(s['cells']))
    template, rest = shapes[0], shapes[1:]
    if rest and len(template['cells']) <= len(rest[0]['cells']):
        return None

    # 템플릿 bbox(확장 반경 G) 안팎으로 도형을 구분해 흡수 (고정점 반복)
    # 장식은 프레임에 붙어 있고(간격<=G-1), 회전 지시자는 멀리 떨어져 있음
    G = 2
    tcells = dict(template['cells'])
    changed = True
    while changed:
        changed = False
        ys = [p[0] for p in tcells]; xs = [p[1] for p in tcells]
        by0, by1, bx0, bx1 = min(ys) - G, max(ys) + G, min(xs) - G, max(xs) + G
        keep = []
        for s in rest:
            (sy, sx), (sh, sw) = s['pos'], s['size']
            if sy <= by1 and sy + sh - 1 >= by0 and sx <= bx1 and sx + sw - 1 >= bx0:
                tcells.update(s['cells'])
                changed = True
            else:
                keep.append(s)
        rest = keep
    ys = [p[0] for p in tcells]; xs = [p[1] for p in tcells]
    y0, x0 = min(ys), min(xs)
    H, W = max(ys) - y0 + 1, max(xs) - x0 + 1
    tpl_colors = set(tcells.values())

    # 색별 바깥 셀 수 → 회전 스텝
    counts = Counter()
    for s in rest:
        for c, n in s['colors'].items():
            counts[c] += n
    steps = {c: counts.get(c, 0) % 4 for c in tpl_colors}
    if any(k in (1, 3) for k in steps.values()) and H != W:
        return None  # 비정사각 캔버스에 90/270 불가

    out = [[bg] * W for _ in range(H)]
    for (y, x), c in tcells.items():
        cells, _, _ = _rotate_layer_cells([(y - y0, x - x0)], H, W, steps[c], direction)
        (ny, nx) = cells[0]
        if out[ny][nx] != bg:
            return None  # 레이어 충돌 → 포기
        out[ny][nx] = c
    return out


def solve_rotation_count(task):
    for kind in KINDS:
        for direction in ['cw', 'ccw']:
            ok = True
            for pair in task['train']:
                pred = solve_rotation_count_one(pair['input'], kind, direction)
                if pred is None or pred != pair['output']:
                    ok = False
                    break
            if not ok:
                continue
            preds = []
            for t in task['test']:
                p = solve_rotation_count_one(t['input'], kind, direction)
                if p is None:
                    ok = False
                    break
                preds.append(p)
            if ok:
                return preds, f"{kind}/{direction}", 'train_exact'
    return None, None, None


# ============================================================================
# 스테이지 3: border_orient — 외곽선 색 정렬 (119번 유형)
#   가장 큰 도형의 인접 마커선 2개(색 A,B)와 그리드 외곽 균일 테두리 2개(색 A,B)가
#   일치하도록 D4(회전+미러) 변환을 선택하여 내용을 재배향
# ============================================================================
_d4g = apply_d4_grid

# 변환별 (측면 → 측면) 매핑을 프로브로 사전 계산
def _side_map():
    probe = [[0, 1, 0], [4, 9, 2], [0, 3, 0]]  # T=1 R=2 B=3 L=4
    where = {}
    for name in D4_NAMES_ALL:
        g = _d4g(probe, name)
        m = {}
        for v, side in [(1, 'T'), (2, 'R'), (3, 'B'), (4, 'L')]:
            if g[0][1] == v: m[side] = 'T'
            elif g[1][2] == v: m[side] = 'R'
            elif g[2][1] == v: m[side] = 'B'
            elif g[1][0] == v: m[side] = 'L'
        where[name] = m
    return where


D4_NAMES_ALL = D4_NAMES
_SIDE_MAP = None


def _uniform_border_colors(grid, bg):
    """그리드 외곽 4변(모서리 제외)의 균일 비배경 색 → {측면: 색}"""
    H, W = len(grid), len(grid[0])
    out = {}
    sides = {'T': [grid[0][c] for c in range(1, W - 1)],
             'B': [grid[H - 1][c] for c in range(1, W - 1)],
             'L': [grid[r][0] for r in range(1, H - 1)],
             'R': [grid[r][W - 1] for r in range(1, H - 1)]}
    for side, vals in sides.items():
        u = set(vals)
        if len(u) == 1 and next(iter(u)) != bg:
            out[side] = next(iter(u))
    return out


def _side_of_color(grid, bg, color):
    """색 color가 과반(노이즈 허용)인 외곽 측면. 유일하지 않으면 None"""
    H, W = len(grid), len(grid[0])
    sides = {'T': [grid[0][c] for c in range(1, W - 1)],
             'B': [grid[H - 1][c] for c in range(1, W - 1)],
             'L': [grid[r][0] for r in range(1, H - 1)],
             'R': [grid[r][W - 1] for r in range(1, H - 1)]}
    scored = []
    for side, vals in sides.items():
        cnt = sum(1 for v in vals if v == color)
        if cnt * 2 >= len(vals):  # 과반 이상
            scored.append((cnt, side))
    if not scored:
        return None
    scored.sort(reverse=True)
    if len(scored) > 1 and scored[0][0] == scored[1][0]:
        return None  # 동률 → 모호
    return scored[0][1]


def solve_border_orient_one(grid, kind):
    global _SIDE_MAP
    if _SIDE_MAP is None:
        _SIDE_MAP = _side_map()
    bg = detect_bg(grid)
    H, W = len(grid), len(grid[0])

    # 템플릿: 그리드 경계에 닿지 않는 가장 큰 도형
    shapes = [s for s in extract_shapes(grid, bg, kind)
              if s['pos'][0] > 0 and s['pos'][1] > 0
              and s['pos'][0] + s['size'][0] < H and s['pos'][1] + s['size'][1] < W]
    if not shapes:
        return None
    shapes.sort(key=lambda s: -len(s['cells']))
    template = shapes[0]
    (y0, x0), (h, w) = template['pos'], template['size']

    # 마커: bbox 4변에 인접한 선분이 균일 비배경 색이면 마커
    markers = {}
    segs = {'T': [(y0 - 1, x) for x in range(x0, x0 + w)] if y0 > 0 else [],
            'B': [(y0 + h, x) for x in range(x0, x0 + w)] if y0 + h < H else [],
            'L': [(y, x0 - 1) for y in range(y0, y0 + h)] if x0 > 0 else [],
            'R': [(y, x0 + w) for y in range(y0, y0 + h)] if x0 + w < W else []}
    for side, seg in segs.items():
        if not seg:
            continue
        vals = {grid[y][x] for (y, x) in seg}
        if len(vals) == 1 and next(iter(vals)) != bg:
            markers[side] = next(iter(vals))
    if len(markers) != 2 or len(set(markers.values())) != 2:
        return None

    # 각 마커 색이 과반인 외곽 측면을 역탐색 (테두리 노이즈 허용)
    target = {}
    for mside, mcol in markers.items():
        bside = _side_of_color(grid, bg, mcol)
        if bside is None:
            return None
        target[mside] = bside
    if len(set(target.values())) != 2:
        return None

    # 내용 크롭 (원본 그대로)
    content = [row[x0:x0 + w] for row in grid[y0:y0 + h]]

    # 마커 측면 → 외곽 측면 색 일치시키는 D4 선택
    for name in D4_NAMES_ALL:
        m = _SIDE_MAP[name]
        if all(m[ms] == ts for ms, ts in target.items()):
            return _d4g(content, name)
    return None


def solve_border_orient(task):
    for kind in KINDS:
        ok = True
        for pair in task['train']:
            pred = solve_border_orient_one(pair['input'], kind)
            if pred is None or pred != pair['output']:
                ok = False
                break
        if not ok:
            continue
        preds = []
        for t in task['test']:
            p = solve_border_orient_one(t['input'], kind)
            if p is None:
                ok = False
                break
            preds.append(p)
        if ok:
            return preds, kind, 'train_exact'
    return None, None, None


# ============================================================================
# 스테이지 4: shape_swap — 지시자 모양 교체 + 색상쌍 매핑 (114번 유형)
#   1. 가장 큰 도형(프레임) bbox = 출력 영역
#   2. 프레임 안 solid k×k 단색 블록 → 바깥의 같은 색 지시자 도형(비solid,
#      bbox=k×k)의 모양으로 교체
#   3. 바깥의 2색 결합 도형 = 색상쌍: 다수 셀 색 → 소수 셀 색으로 변환
#      (동률이면 읽기 순서: 먼저 나오는 색 → 나중 색)
#   지시자/매핑이 하나라도 없으면 포기
# ============================================================================
def solve_shape_swap_one(grid, kind='diagonal'):
    bg = detect_bg(grid)
    shapes = extract_shapes(grid, bg, kind)
    if len(shapes) < 3:
        return None
    shapes.sort(key=lambda s: -len(s['cells']))
    frame, rest = shapes[0], shapes[1:]
    if rest and len(frame['cells']) <= len(rest[0]['cells']):
        return None
    (y0, x0), (H, W) = frame['pos'], frame['size']

    blocks, pairs = [], {}
    frag = {}  # 바깥 단색 도형: 색깔별 셀 병합 (지시자 파편화 대응)
    for s in rest:
        (sy, sx), (sh, sw) = s['pos'], s['size']
        inside = (sy > y0 and sx > x0 and
                  sy + sh - 1 < y0 + H - 1 and sx + sw - 1 < x0 + W - 1)
        if inside:
            # 내부: solid 정사각 단색 블록이어야 함
            if len(s['colors']) != 1 or sh != sw or len(s['cells']) != sh * sw:
                return None
            blocks.append(s)
        elif len(s['colors']) == 2:
            # 색상쌍: 다수 → 소수 (동률이면 읽기 순서)
            (c1, n1), (c2, n2) = s['colors'].most_common(2)
            if n1 == n2:
                first = min(s['cells'], key=lambda p: p)
                c1 = s['cells'][first]
                c2 = next(c for c in s['colors'] if c != c1)
            if c1 in pairs and pairs[c1] != c2:
                return None  # 매핑 충돌
            pairs[c1] = c2
        elif len(s['colors']) == 1:
            color = next(iter(s['colors']))
            frag.setdefault(color, {}).update(s['cells'])
        # 그 외 다색(3색 이상) 바깥 도형은 노이즈로 무시

    # 지시자: 색깔별 병합 결과
    indicators = {}
    for color, cells in frag.items():
        ys = [p[0] for p in cells]; xs = [p[1] for p in cells]
        ih, iw = max(ys) - min(ys) + 1, max(xs) - min(xs) + 1
        indicators[color] = (normalize_cells(frozenset(cells.keys())), ih, iw)

    if not blocks:
        return None
    out = [[bg] * W for _ in range(H)]
    for (y, x), c in frame['cells'].items():
        out[y - y0][x - x0] = c
    for b in blocks:
        color = next(iter(b['colors']))
        if color not in indicators or color not in pairs:
            return None
        norm, ih, iw = indicators[color]
        (sy, sx), (sh, sw) = b['pos'], b['size']
        if (ih, iw) != (sh, sw):
            return None  # 지시자 bbox와 블록 크기 불일치
        for (dy, dx) in norm:
            out[sy - y0 + dy][sx - x0 + dx] = pairs[color]
    return out


def solve_shape_swap(task):
    for kind in ['diagonal']:  # 색상쌍은 다색 결합 추출 필요
        ok = True
        for pair in task['train']:
            pred = solve_shape_swap_one(pair['input'], kind)
            if pred is None or pred != pair['output']:
                ok = False
                break
        if not ok:
            continue
        preds = []
        for t in task['test']:
            p = solve_shape_swap_one(t['input'], kind)
            if p is None:
                ok = False
                break
            preds.append(p)
        if ok:
            return preds, kind, 'train_exact'
    return None, None, None


# ============================================================================
# 스테이지 5: hole_transplant — 색깔 대응 이식 (28번 유형)
#   1. 단색 최대 도형 = 템플릿(전체 모양), 다색 최대 도형 = 소스(색깔 지도)
#   2. 템플릿 내부 홀(외부와 안 통하는 bg 영역)과 소스의 색 그룹(주색 아닌
#      셀들의 대각 연결요소)을 정규화 무게중심 최근접(헝가리안)으로 1:1 대응
#   3. 각 홀을 대응 색으로 채움. 개수 불일치/대응 모호 → 포기
# ============================================================================
from scipy.optimize import linear_sum_assignment


def _interior_holes(canvas_mask, H, W):
    """mask=True가 도형 셀. 외부와 연결되지 않은 빈 영역들(4-conn) 반환"""
    empty = np.array([[not canvas_mask[r][c] for c in range(W)] for r in range(H)])
    labeled, n = ndimage.label(empty)
    exterior = set()
    for r in range(H):
        for c in (0, W - 1):
            if labeled[r][c]: exterior.add(labeled[r][c])
    for c in range(W):
        for r in (0, H - 1):
            if labeled[r][c]: exterior.add(labeled[r][c])
    return [frozenset((int(y), int(x)) for y, x in np.argwhere(labeled == rid))
            for rid in range(1, n + 1) if rid not in exterior]


def solve_hole_transplant_one(grid, kind='diagonal'):
    bg = detect_bg(grid)
    shapes = extract_shapes(grid, bg, kind)
    monos = [s for s in shapes if len(s['colors']) == 1]
    multis = [s for s in shapes if len(s['colors']) >= 2]
    if not monos or not multis:
        return None
    template = max(monos, key=lambda s: len(s['cells']))
    source = max(multis, key=lambda s: len(s['cells']))
    (y0, x0), (H, W) = template['pos'], template['size']
    tmask = [[False] * W for _ in range(H)]
    for (y, x) in template['cells']:
        tmask[y - y0][x - x0] = True
    holes = _interior_holes(tmask, H, W)
    if not holes:
        return None

    # 소스 색 그룹: 주색 아닌 셀들의 "색깔별" 대각 연결요소
    main = source['colors'].most_common(1)[0][0]
    other = {(y, x): c for (y, x), c in source['cells'].items() if c != main}
    if not other:
        return None
    (sy0, sx0), (sH, sW) = source['pos'], source['size']
    groups = []
    for col in sorted(set(other.values())):
        m = np.zeros((sH, sW), dtype=int)
        for (y, x), c in other.items():
            if c == col:
                m[y - sy0][x - sx0] = 1
        labeled, n = ndimage.label(m, structure=np.ones((3, 3), dtype=int))
        for rid in range(1, n + 1):
            cells = [(int(y), int(x)) for y, x in np.argwhere(labeled == rid)]
            groups.append((cells, col))
    if len(groups) != len(holes):
        return None

    # 정규화 무게중심 최근접 (헝가리안) 대응
    def ncent(cells, h, w):
        ys = sum(p[0] for p in cells) / len(cells)
        xs = sum(p[1] for p in cells) / len(cells)
        return (ys / max(h - 1, 1), xs / max(w - 1, 1))
    hc = [ncent(list(h), H, W) for h in holes]
    gc = [ncent(cells, sH, sW) for cells, _ in groups]
    cost = np.array([[(a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2 for b in gc] for a in hc])
    ri, ci = linear_sum_assignment(cost)

    out = [[bg] * W for _ in range(H)]
    color0 = next(iter(template['colors']))
    for (y, x) in template['cells']:
        out[y - y0][x - x0] = color0
    for hi, gi in zip(ri, ci):
        col = groups[gi][1]
        for (r, c) in holes[hi]:
            out[r][c] = col
    return out


def solve_hole_transplant(task):
    ok = True
    for pair in task['train']:
        pred = solve_hole_transplant_one(pair['input'])
        if pred is None or pred != pair['output']:
            return None, None, None
    preds = []
    for t in task['test']:
        p = solve_hole_transplant_one(t['input'])
        if p is None:
            return None, None, None
        preds.append(p)
    return preds, 'diagonal', 'train_exact'


# ============================================================================
# 스테이지 6: motif_stamp — 점 위치에 모티프 도장 (80번 유형)
#   1. 속이 빈 직사각 액자 2개 탐지 (색 무관, 4-conn 완전 테두리)
#   2. 점 상자 B: 내부 장식이 전부 1셀 동일 색(점 색 d)
#      모티프 상자 A: 내부 장식이 모두 동일한 다셀 모티프 (색 보존)
#   3. 앵커 = 모티프에서 색 d인 유일 셀. 각 점에 앵커를 맞춰 모티프 도장
#   4. 출력 = B의 bbox 크롭 (액자 + 내부 + 도장)
# ============================================================================
def _find_frames(grid):
    """기하학적 스캔으로 완전한 직사각 테두리 탐지 (배경 무늬와 붙어도 무관).
    조건: 4변이 모두 같은 색 c로 균일하고, 내부에 c가 아닌 셀이 존재"""
    H, W = len(grid), len(grid[0])
    frames = []
    # 수평 최대 런 수집
    runs = []
    for r in range(H):
        c0 = 0
        while c0 < W:
            c1 = c0
            while c1 + 1 < W and grid[r][c1 + 1] == grid[r][c0]:
                c1 += 1
            if c1 - c0 + 1 >= 3:
                runs.append((r, c0, c1, grid[r][c0]))
            c0 = c1 + 1
    for (r, x0, x1, col) in runs:
        # 위 변 = 최대 런 전체. 세로 변 무결성을 유지하며 아래 변 탐색
        for rb in range(r + 2, H):
            if grid[rb - 1][x0] != col or grid[rb - 1][x1] != col:
                break  # 세로 변이 끊김
            if all(grid[rb][k] == col for k in range(x0, x1 + 1)):
                inner = [grid[rr][k] for rr in range(r + 1, rb) for k in range(x0 + 1, x1)]
                if inner:
                    base = Counter(inner).most_common(1)[0][0]
                    if base != col:  # 진짜 액자: 내부 베이스색이 테두리색과 다름
                        frames.append({'color': col, 'bbox': (r, x0, rb - r + 1, x1 - x0 + 1)})
    # 중복 제거 + 최대 액자 우선 (동일 bbox 중복 방지)
    seen = set()
    uniq = []
    for f in sorted(frames, key=lambda f: -(f['bbox'][2] * f['bbox'][3])):
        if f['bbox'] not in seen:
            seen.add(f['bbox'])
            uniq.append(f)
    return uniq


def _frame_decorations(grid, frame):
    """액자 내부의 (베이스색, 장식 연결요소[셀→색]) 반환"""
    y0, x0, h, w = frame['bbox']
    inner = [[grid[y][x] for x in range(x0 + 1, x0 + w - 1)] for y in range(y0 + 1, y0 + h - 1)]
    ih, iw = h - 2, w - 2
    base = Counter(v for row in inner for v in row).most_common(1)[0][0]
    m = np.array([[1 if inner[r][c] != base else 0 for c in range(iw)] for r in range(ih)])
    labeled, n = ndimage.label(m, structure=np.ones((3, 3), dtype=int))
    decos = []
    for rid in range(1, n + 1):
        cells = {(int(y), int(x)): inner[y][x] for y, x in np.argwhere(labeled == rid)}
        decos.append(cells)
    return base, decos


def solve_motif_stamp_one(grid):
    frames = _find_frames(grid)
    if len(frames) < 2:
        return None
    # 역할 판별: 점 상자(장식 전부 1셀) vs 모티프 상자(다셀 장식 존재)
    infos = []
    for f in frames:
        base, decos = _frame_decorations(grid, f)
        infos.append((f, base, decos))
    dot_boxes = [i for i in infos if i[2] and all(len(d) == 1 for d in i[2])]
    motif_boxes = [i for i in infos if i[2] and all(len(d) >= 2 for d in i[2])]
    if len(dot_boxes) != 1 or len(motif_boxes) != 1:
        return None
    (bf, bbase, bdecos) = dot_boxes[0]
    (af, abase, adecos) = motif_boxes[0]

    # 모티프 종류: 정규화(셀→색) 기준으로 그룹화 (여러 종류 허용)
    def norm_deco(deco):
        ys = [p[0] for p in deco]; xs = [p[1] for p in deco]
        return frozenset(((y - min(ys), x - min(xs)), c) for (y, x), c in deco.items())
    motifs = [dict(n) for n in {norm_deco(dc) for dc in adecos}]

    # 점 색 d → (모티프, 앵커): 색 d 셀을 정확히 1개 가진 유일한 모티프
    def motif_for(d):
        cands = []
        for m in motifs:
            anchors = [p for p, c in m.items() if c == d]
            if len(anchors) == 1:
                cands.append((m, anchors[0]))
        return cands[0] if len(cands) == 1 else None

    # 출력 = 점 상자 크롭 + 도장
    y0, x0, h, w = bf['bbox']
    out = [[grid[y0 + r][x0 + c] for c in range(w)] for r in range(h)]
    for deco in bdecos:
        (py, px) = next(iter(deco))       # 내부 좌표
        d = next(iter(deco.values()))     # 점 색
        sel = motif_for(d)
        if sel is None:
            return None
        motif, (ay, ax) = sel
        for (my, mx), c in motif.items():
            r = 1 + py + (my - ay)
            cc = 1 + px + (mx - ax)
            if not (1 <= r < h - 1 and 1 <= cc < w - 1):
                return None  # 모티프가 내부를 벗어남
            if out[r][cc] not in (bbase, d, c):
                return None  # 충돌
            out[r][cc] = c
    return out


def solve_motif_stamp(task):
    ok = True
    for pair in task['train']:
        pred = solve_motif_stamp_one(pair['input'])
        if pred is None or pred != pair['output']:
            return None, None, None
    preds = []
    for t in task['test']:
        p = solve_motif_stamp_one(t['input'])
        if p is None:
            return None, None, None
        preds.append(p)
    return preds, 'frames', 'train_exact'


# ============================================================================
# 스테이지 7: occlusion_restore — 창(액자) 뒤 가려진 도형 복원 (85번 유형)
#   1. 가장 큰 액자 = 창(window). 출력 = 창 내부 크롭
#   2. 창 밖의 불완전 직사각형(테두리 셀만 존재)을 완성:
#      - 단색: 은닉 테두리 셀을 그 색으로
#      - 2색 교대 패턴: 너비 홀수 → 패리티(둘레 교대) 연속
#                        너비 짝수 → 180도 스탬프(보이는 셀의 회전 복사)
#   3. 완성된 은닉 셀 중 창 내부에 들어오는 것만 그려 넣음
#   테두리 위에 있지 않은 컴포넌트는 노이즈로 무시
# ============================================================================
def solve_occlusion_restore_one(grid):
    H, W = len(grid), len(grid[0])
    bg = detect_bg(grid)
    frames = _find_frames(grid)
    if not frames:
        return None
    win = max(frames, key=lambda f: f['bbox'][2] * f['bbox'][3])
    fy, fx, fh, fw = win['bbox']

    # 창 영역 제외 후 비배경 컴포넌트 추출 (대각, 다색 허용)
    m = np.zeros((H, W), dtype=int)
    for r in range(H):
        for c in range(W):
            if grid[r][c] != bg and not (fy <= r < fy + fh and fx <= c < fx + fw):
                m[r][c] = 1
    labeled, n = ndimage.label(m, structure=np.ones((3, 3), dtype=int))

    # 컴포넌트 수집 후, 창에 의해 분리된 같은 도형의 조각 병합
    # (색 구성 동일 + 열 범위 정확히 일치 → 세로 병합 / 행 범위 일치 → 가로 병합)
    comps = []
    for rid in range(1, n + 1):
        cells = {(int(y), int(x)): grid[y][x] for y, x in np.argwhere(labeled == rid)}
        comps.append(cells)
    merged = True
    while merged:
        merged = False
        for i in range(len(comps)):
            for j in range(i + 1, len(comps)):
                a, b = comps[i], comps[j]
                if set(a.values()) != set(b.values()):
                    continue
                ax = (min(p[1] for p in a), max(p[1] for p in a))
                bx = (min(p[1] for p in b), max(p[1] for p in b))
                ay = (min(p[0] for p in a), max(p[0] for p in a))
                by = (min(p[0] for p in b), max(p[0] for p in b))
                if ax == bx or ay == by:  # 열 범위 or 행 범위 일치
                    comps[i] = {**a, **b}
                    del comps[j]
                    merged = True
                    break
            if merged:
                break

    out = [[grid[fy + 1 + r][fx + 1 + c] for c in range(fw - 2)] for r in range(fh - 2)]
    drew = False
    for cells in comps:
        ys = [p[0] for p in cells]; xs = [p[1] for p in cells]
        y0, y1, x0, x1 = min(ys), max(ys), min(xs), max(xs)
        border = {(r, c) for r in range(y0, y1 + 1) for c in range(x0, x1 + 1)
                  if r in (y0, y1) or c in (x0, x1)}
        if not all(p in border for p in cells):
            continue  # 직사각 테두리 도형 아님 → 노이즈
        hidden = border - set(cells)
        if not hidden:
            continue
        colors = sorted(set(cells.values()))
        fill = {}
        if len(colors) == 1:
            for p in hidden:
                fill[p] = colors[0]
        elif len(colors) == 2:
            # 보이는 셀의 패리티 위상 (일관성 필수)
            phase = {}
            for (r, c), v in cells.items():
                p = (r + c) % 2
                if phase.get(p, v) != v:
                    return None  # 패리티 비일관 → 포기
                phase[p] = v
            if len(phase) < 2:
                return None
            w_rect = x1 - x0 + 1
            if w_rect % 2 == 1:  # 너비 홀수 → 패리티 연속
                for (r, c) in hidden:
                    fill[(r, c)] = phase[(r + c) % 2]
            else:                # 너비 짝수 → 180도 스탬프
                for (r, c) in hidden:
                    src = (y0 + y1 - r, x0 + x1 - c)
                    if src in cells:
                        fill[(r, c)] = cells[src]
                    elif not (fy < r < fy + fh - 1 and fx < c < fx + fw - 1):
                        pass  # 창 밖 은닉 셀은 출력에 불필요
                    else:
                        return None  # 창 안인데 소스 없음 → 포기
        else:
            continue  # 3색 이상 → 노이즈
        for (r, c), v in fill.items():
            rr, cc = r - fy - 1, c - fx - 1
            if 0 <= rr < fh - 2 and 0 <= cc < fw - 2:
                out[rr][cc] = v
                drew = True
    return out if drew else None


def solve_occlusion_restore(task):
    for pair in task['train']:
        pred = solve_occlusion_restore_one(pair['input'])
        if pred is None or pred != pair['output']:
            return None, None, None
    preds = []
    for t in task['test']:
        p = solve_occlusion_restore_one(t['input'])
        if p is None:
            return None, None, None
        preds.append(p)
    return preds, 'frames', 'train_exact'


def d_lfill(pairs):
    """largest_fill v2: 서브 솔버 7종, 각각 고정 순서 폴백 체인 fn 으로 반환.
    채택은 파이프라인 train-exact + LOO 가 심판. (#69 유형 잔여 갭)"""
    ins = [np.asarray(e['input']) for e in pairs]
    outs = [np.asarray(e['output']) for e in pairs]
    if not all(o.size < i.size for i, o in zip(ins, outs)):
        return None
    _SUBS = [
        (solve_one, [(k, t, s) for k in KINDS for t in TEMPLATE_MODES
                     for s in SCALE_MODES]),
        (solve_rotation_count_one, [(k, d) for k in KINDS for d in ('cw', 'ccw')]),
        (solve_border_orient_one, [(k,) for k in KINDS]),
        (solve_shape_swap_one, [(k,) for k in KINDS]),
        (solve_hole_transplant_one, [(k,) for k in KINDS]),
        (solve_motif_stamp_one, [()]),
        (solve_occlusion_restore_one, [()]),
    ]
    keep = []
    for _solver, _combos in _SUBS:
        def fn(g, _s=_solver, _c=_combos):
            gl = np.asarray(g).tolist()
            for args in _c:
                try:
                    out = _s(gl, *args)
                except Exception:
                    out = None
                if out is not None:
                    return np.asarray(out)
            raise ValueError('lfill fail')
        try:
            if all(np.array_equal(np.asarray(fn(i)), o)
                   for i, o in zip(ins, outs)):
                keep.append(fn)
        except Exception:
            pass
    return keep or None

STAGES = {'color': d_color, 'd4': d_d4, 'tile': d_tile,
          'scale': d_scale, 'shrink': d_shrink, 'restore': d_restore,
          'panels': d_panels, 'flow': d_flow, 'part': d_part, 'obj': d_obj, 'lfill': d_lfill}

# ----------------------------------------------------------- 오더 파이프라인
_MAX_BRANCH = 200
TASK_TIME_CAP = 4.0     # solve_by_orders task당 상한(초) (복원 3단 폴백 여유)

# ----------------------------------------------------------- 추측(triage) 신호
#   비싼 stage 는 싼 신호를 통과한 task 에서만 시도. LOO 가 마지막 심판.
def _mirror_agreement(g, max_off=4):
    """기준선(중앙 +- max_off 오프셋, 행/열) 반쪽 셀 일치율의 최댓값"""
    g = np.asarray(g); H, W = g.shape
    best = 0.0
    for axis, L in (('col', W), ('row', H)):
        center = L - 1
        for p2 in range(max(1, center - max_off), min(2 * L - 2, center + max_off + 1)):
            idx = np.arange(L)
            j = p2 - idx
            m = (j >= 0) & (j < L) & (idx < j)      # 쌍 중복 제거
            if not m.any(): continue
            if axis == 'col':
                A = g[:, idx[m]]; B = g[:, j[m]]
            else:
                A = g[idx[m], :]; B = g[j[m], :]
            agree = float((A == B).mean())
            if agree > best: best = agree
    return best

def _has_rect_color(g):
    g = np.asarray(g)
    for c in (int(v) for v in np.unique(g)):
        cells = int((g == c).sum())
        if 0 < cells <= g.size // 2 and _occ_is_rects(g, c):
            return True
    return False

RESTORE_SMALL = 100      # 이하 크기는 무조건 허용 (도출 비용 미미)
RESTORE_AGREE = 0.55

def _maybe_restore(task):
    for e in task.get('train', []):
        g = np.asarray(e['input'])
        if g.size <= RESTORE_SMALL:
            continue
        if not _has_rect_color(g): return False
        if _mirror_agreement(g) < RESTORE_AGREE: return False
    return True

def _verify_restore_once(train):
    """복원 판정 2단: [1] 사분면 검사(check_restore_quadrant, ~ms) 로 후보 축소
    -> [2] d_restore 를 전체 train 쌍에 돌려 모든 쌍 정확 재현 시에만 True.
    (GT 5/5 정판, training 120 중 지목 1개 = 오탐 0)"""
    try:
        if not train:
            return False
        if not check_restore_quadrant(train):     # 1단: 회원님 사분면 사양
            return False
        fns = d_restore(train)                    # 2단: 풀이 본체로 확정
        if not fns:
            return False
        for fn in fns[:3]:
            ok = True
            for e in train:
                try:
                    r = fn(e['input'])
                    o = np.asarray(e['output'])
                    if np.asarray(r).shape != o.shape or \
                       not np.array_equal(np.asarray(r), o):
                        ok = False; break
                except Exception:
                    ok = False; break
            if ok:
                return True
        return False
    except Exception:
        return False

STAGE_TRIAGE = {'restore': lambda task: _verify_restore_once(task.get('train', []))}

def _d4_canon(shape):
    pts = sorted(shape)
    forms = []
    cur = list(pts)
    for _ in range(4):
        cur = [(x, -y) for y, x in cur]
        for v in (cur, [(y, -x) for y, x in cur]):
            my = min(p[0] for p in v); mx = min(p[1] for p in v)
            forms.append(tuple(sorted((p[0]-my, p[1]-mx) for p in v)))
    return min(forms)

def _objs8(g, bg=0):
    g = np.asarray(g); H, W = g.shape
    seen = np.zeros((H, W), bool); out = []
    for r in range(H):
        for c in range(W):
            if g[r, c] != bg and not seen[r, c]:
                st = [(r, c)]; seen[r, c] = True; cells = []
                while st:
                    y, x = st.pop(); cells.append((y, x))
                    for dy in (-1, 0, 1):
                        for dx in (-1, 0, 1):
                            ny, nx = y + dy, x + dx
                            if 0 <= ny < H and 0 <= nx < W and not seen[ny, nx] \
                               and g[ny, nx] != bg:
                                seen[ny, nx] = True; st.append((ny, nx))
                r0 = min(y for y, _ in cells); c0 = min(x for _, x in cells)
                sil = frozenset((y - r0, x - c0) for y, x in cells)
                colored = frozenset((y - r0, x - c0, int(g[y, x])) for y, x in cells)
                pos = frozenset(cells)
                out.append((sil, colored, pos))
    return out

def _detect_bg_conf(g):
    """배경색 감지 + 신뢰도. 반환 (bg, conf):
      1.0 모서리 4개 동일 / 0.9 3개 / 0.7 최다색이 모서리 2개+
      0.5 최다색이 전체의 50%+ / 0.4 테두리 최다색 60%+
      (None, 0.0) = 배경 규정 불가(애매) - 0을 강제하지 않음"""
    g = np.asarray(g); H, W = g.shape
    corners = [int(g[0, 0]), int(g[0, W-1]), int(g[H-1, 0]), int(g[H-1, W-1])]
    from collections import Counter as _C
    cnt = _C(corners)
    col, n = cnt.most_common(1)[0]
    if n == 4: return col, 1.0
    if n == 3: return col, 0.9
    vals, cnts = np.unique(g, return_counts=True)
    most = int(vals[np.argmax(cnts)])
    if cnt.get(most, 0) >= 2: return most, 0.7
    if cnts.max() / g.size >= 0.5: return most, 0.5
    border = np.concatenate([g[0], g[-1], g[:, 0], g[:, -1]])
    bv, bc = np.unique(border, return_counts=True)
    if bc.max() / border.size >= 0.6: return int(bv[np.argmax(bc)]), 0.4
    return None, 0.0

def _detect_bg2(g):
    """호환용: 색만 반환. 애매하면 '그리드 최다색' (0 강제 아님)."""
    bg, conf = _detect_bg_conf(g)
    if bg is None:
        vals, cnts = np.unique(np.asarray(g), return_counts=True)
        return int(vals[np.argmax(cnts)])
    return bg

def classify_task(task, labels=False):
    """규칙 가설을 확신 순서로 반환 (첫번째 규칙 = [0]).
    labels=True 면 (순위, 양성 라벨 목록) 반환 - 다중 분류 대조용.
    양성 라벨 = 신호가 실제로 잡힌 것만 (기본 후보 꼬리 제외)."""
    train = task.get('train', [])
    if not train: return []
    rank = []
    def add(name):
        if name not in rank: rank.append(name)
    ins = [np.asarray(e['input']) for e in train]
    outs = [np.asarray(e['output']) for e in train]
    same_size = all(i.shape == o.shape for i, o in zip(ins, outs))

    # ---- 크기 관계 ----
    if not same_size:
        ratios = []
        for i, o in zip(ins, outs):
            ih, iw = i.shape; oh, ow = o.shape
            if oh % ih == 0 and ow % iw == 0 and (oh//ih, ow//iw) != (1, 1):
                ratios.append(('up', oh//ih, ow//iw))
            elif ih % oh == 0 and iw % ow == 0 and (ih//oh, iw//ow) != (1, 1):
                ratios.append(('down', ih//oh, iw//ow))
            else:
                ratios.append(None)
        if all(r is not None and r == ratios[0] for r in ratios):
            if ratios[0][0] == 'up':
                add('tile'); add('scale')          # 2,3,4배 확대
            else:
                add('shrink'); add('panels')       # 축소/부분
        # 구분선 패널 (output == 패널 크기)
        seps = [_find_sep(i) for i in ins]
        if all(s and any(a.shape == o.shape for _d, _c, a, _b in s)
               for s, o in zip(seps, outs)):
            add('panels')
        # 출력이 입력보다 작음 -> 부분/패치 (복원 patch 포함)
        if all(o.size < i.size for i, o in zip(ins, outs)):
            if _verify_restore_once(train): add('restore')
            add('panels')
        # 크기 달라도: 도형(색+모양) 다중집합 보존 = 이동 / 감소 = 이동,삭제
        try:
            mv = de = True
            for i, o in zip(ins, outs):
                bi = _detect_bg2(i); bo = _detect_bg2(o)
                isig = sorted(hash(c) for _s, c, _p in _objs8(i, bi))
                osig = sorted(hash(c) for _s, c, _p in _objs8(o, bo))
                from collections import Counter as _C3
                ci, co = _C3(isig), _C3(osig)
                if isig != osig: mv = False
                if not (all(ci[k] >= v for k, v in co.items()) and
                        len(osig) < len(isig)): de = False
            if mv: add('move')
            elif de: add('move'); add('delete')
        except Exception:
            pass
        # 가장 큰 도형: 최대 입력 도형의 bbox == 출력 크기 (모든 쌍) -> 부분
        try:
            big_ok = True
            for i, o in zip(ins, outs):
                objs = _objs8(i, _detect_bg2(i))
                if not objs: big_ok = False; break
                big = max(objs, key=lambda t: len(t[2]))
                ys = [y for y, _x in big[2]]; xs = [x for _y, x in big[2]]
                if (max(ys) - min(ys) + 1, max(xs) - min(xs) + 1) != o.shape:
                    big_ok = False; break
            if big_ok:
                _rank_insert_part(rank)
            add('lfill')               # 템플릿 채움 (출력<입력 전용, 전제 자체 가드)
        except Exception:
            pass
    else:
        # ---- 같은 크기: 배경 분석 (사용자 판정식 + 신뢰도) ----
        bgc = [_detect_bg_conf(i) for i in ins]
        bg_reliable = all(c >= 0.5 and b is not None for b, c in bgc)
        bgs = [(b if b is not None else _detect_bg2(i))
               for (b, _c), i in zip(bgc, ins)]
        if bg_reliable:
            # 사용자 배경 판정 트리 (배경 칸 수의 방향이 첫 규칙을 정함)
            n_in = [int((i == b).sum()) for i, b in zip(ins, bgs)]
            n_out = [int((o == b).sum()) for o, b in zip(outs, bgs)]
            bg_same_mask = all(np.array_equal(i == b, o == b)
                               for i, o, b in zip(ins, outs, bgs))
            if all(o < i for i, o in zip(n_in, n_out)):
                # 배경 줄어듦: 입력 도형(색+모양)이 전부 보존되며 개수가 늘면 "추가"
                is_add = True
                try:
                    for i, o, b in zip(ins, outs, bgs):
                        io = _objs8(i, b); oo = _objs8(o, b)
                        isig = sorted(hash(c) for _s, c, _p in io)
                        osig = sorted(hash(c) for _s, c, _p in oo)
                        # 입력 도형 다중집합 <= 출력 다중집합 & 개수 증가
                        from collections import Counter as _C2
                        ci, co = _C2(isig), _C2(osig)
                        if not all(co[k] >= v for k, v in ci.items()) or \
                           len(osig) <= len(isig):
                            is_add = False; break
                except Exception:
                    is_add = False
                if is_add:
                    add('add'); add('pattern'); add('tile')
                else:
                    add('pattern'); add('tile')    # 배경 줄어듦 -> 패턴, 복제
            elif all(o > i for i, o in zip(n_in, n_out)):
                # 늘어남이어도 비배경 마스크가 거의 같으면 색깔이 우선
                near = all(float(((i == b) == (o == b)).mean()) >= 0.98
                           for i, o, b in zip(ins, outs, bgs))
                if near: add('color')
                add('delete')                  # 배경 늘어남 -> 도형 삭제
            elif all(o == i for i, o in zip(n_in, n_out)):
                if bg_same_mask:
                    add('color')               # 칸 동일 + 모양 동일 = 색깔
                else:
                    add('move')                # 칸 동일 + 모양 다름 = 이동
            else:
                # 칸 수 변화가 쌍마다 다르거나 소량: 마스크 근사 일치면 색깔
                near = all(float(((i == b) == (o == b)).mean()) >= 0.98
                           for i, o, b in zip(ins, outs, bgs))
                if near: add('color')
        # 배경 애매(bg_reliable=False)면 배경 근거 판정은 침묵 - 도형 비교로만 진행
        # 도형 실루엣(색무관) 다중집합 비교
        try:
            for i, o, b in zip(ins, outs, bgs):
                io = _objs8(i, b); oo = _objs8(o, b)
                isil = sorted(hash(s) for s, _c, _p in io)
                osil = sorted(hash(s) for s, _c, _p in oo)
                ipos = set(p for _s, _c, p in io)
                opos = set(p for _s, _c, p in oo)
                if isil == osil and ipos != opos:
                    add('move'); break
            for i, o, b in zip(ins, outs, bgs):
                io = _objs8(i, b); oo = _objs8(o, b)
                if set(p for _s,_c,p in io) == set(p for _s,_c,p in oo) \
                   and sorted(hash(c) for _s,c,_p in io) != sorted(hash(c) for _s,c,_p in oo):
                    add('color'); break
            n_in = [len(_objs8(i, b)) for i, b in zip(ins, bgs)]
            n_out = [len(_objs8(o, b)) for o, b in zip(outs, bgs)]
            if all(a < b for a, b in zip(n_in, n_out)): add('add')
            if all(a > b for a, b in zip(n_in, n_out)):
                add('delete'); add('move')     # 이동+생략 부류 동반
            # 실루엣 다중집합이 진부분집합으로 감소 -> 이동,삭제
            try:
                sub = True
                from collections import Counter as _C4
                for i, o, b in zip(ins, outs, bgs):
                    isil = _C4(sorted(hash(s) for s, _c, _p in _objs8(i, b)))
                    osil = _C4(sorted(hash(s) for s, _c, _p in _objs8(o, b)))
                    if not (all(isil[k] >= v for k, v in osil.items())
                            and sum(osil.values()) < sum(isil.values())):
                        sub = False; break
                if sub: add('move'); add('delete')
            except Exception:
                pass
        except Exception:
            pass
        # D4 직행
        for name, t in _D4.items():
            if all(t(i).shape == o.shape and np.array_equal(t(i), o)
                   for i, o in zip(ins, outs)):
                add('d4'); break
        if _verify_restore_once(train):
            rank.insert(0, 'restore')     # 실검증 통과 -> 최상위
    # ---- output 들 간 일치 (부분/상수 출력 계열) ----
    if len(outs) >= 2:
        if all(o.shape == outs[0].shape for o in outs): 
            if all(np.array_equal(o, outs[0]) for o in outs):
                add('const_out')
            # 부분 신호가 강하면(출력 D4 실루엣 동일 + 입력이 출력보다 작음) 최상위 승격
            try:
                sils0 = [sorted(hash(_d4_canon(s)) for s,_c,_p in _objs8(o, _detect_bg2(o)))
                         for o in outs]
                if all(s == sils0[0] for s in sils0) and \
                   all(o.size <= i.size for i, o in zip(ins, outs)):
                    _rank_insert_part(rank)
            except Exception:
                pass
            sils = [sorted(hash(_d4_canon(s)) for s,_c,_p in _objs8(o, _detect_bg2(o)))
                    for o in outs]
            if all(s == sils[0] for s in sils):
                add('part')                # output 모양 동일(회전/미러/색만) = 부분 문제
    sig = list(rank)                      # 양성 신호 스냅샷 (기본 후보 제외)
    # 남은 기본 후보
    for n in ['color', 'd4', 'tile', 'scale', 'shrink', 'panels']:
        add(n)
    # ---- 다중 라벨 (분류 메타) ----
    #   퍼즐 하나가 여러 분류에 속할 수 있음. stage 정렬 '뒤'에 부가되므로
    #   propose_orders 의 오더 정렬에는 영향 없음. 리포트/대조용.
    #   largest/second: 사용자 절차(_part24_match) - 가장큰/2번째큰 도형이
    #   (변환·이동 포함) output 안에 나타나면 해당 라벨.
    def add2(name):                       # 다중 라벨: 순위와 라벨 모두에
        add(name)
        if name not in sig: sig.append(name)
    try:
        from collections import Counter as _C
        _bgc = _C(_detect_bg2(i) for i in ins).most_common(1)[0][0]  # 합의 배경
        # 쌍별 감지와 합의 배경 둘 다 시도 (배경이 쌍마다 다른 태스크 보호)
        hits_all = [_part_obj_match(i, o) + _part_obj_match(i, o, bg=_bgc)
                    for i, o in zip(ins, outs)]
        _n = len(hits_all)
        _need = max(1, _n - 1)                                   # (n-1)쌍 완화
        if sum(1 for hs in hits_all if hs) >= _need:
            if sum(any(h[1] == 0 for h in hs) for hs in hits_all) >= _need:
                add2('largest')
            if sum(any(h[1] == 1 for h in hs) for hs in hits_all) >= _need:
                add2('second')
            _ROT = ('rot90', 'rot180', 'rot270')
            if sum(any(h[2] in _ROT for h in hs) for hs in hits_all) >= _need:
                add2('d4')
            if same_size and sum(any(h[2] == 'id' and h[3] for h in hs)
                                 for hs in hits_all) >= _need:
                add2('move')
    except Exception:
        pass
    # ---- [사용자 사양] 가장큰도형 추가 신호 (3출처 중 1개만 맞아도 인정) ----
    #   1) 배경 살리기: 합의배경 제외 '최대 색도형' 실루엣이 전 쌍 보존 (제자리 포함)  <- #52
    #   2) 최대 색 픽셀수 +-1 대응, 모든 쌍 통과 필수                                <- #85
    #   3) 같은 위치 진부분집합 확장 (도형이 자라나는 유형)                          <- #112
    #   4) 패널 접기: 패널 비영 마스크의 OR/AND == 출력 비영 마스크                  <- #69
    try:
        from collections import Counter as _C2
        _bgc2 = _C2(_detect_bg2(i) for i in ins).most_common(1)[0][0]
        def _maxc(g):
            cs = {int(c): int((g == c).sum()) for c in np.unique(g)
                  if int(c) != _bgc2}
            if not cs: return []
            mx = max(cs.values())
            return [c for c, n in cs.items() if n == mx]   # 동률 전부 (any 인정)
        def _sil(g, c):
            ys, xs = np.where(g == c)
            if len(ys) == 0: return None
            return frozenset((int(y) - int(ys.min()), int(x) - int(xs.min()))
                             for y, x in zip(ys, xs))
        ok_keep = ok_pm1 = ok_grow = True
        for i, o in zip(ins, outs):
            cc = _maxc(i)
            if not cc or i.shape != o.shape or np.array_equal(i, o):
                ok_keep = ok_pm1 = ok_grow = False; break
            if not any(_sil(i, c) == _sil(o, c) for c in cc):
                ok_keep = False
            if not any(abs(int((i == c).sum()) - int((o == c).sum())) <= 1
                       for c in cc):
                ok_pm1 = False
            def _grow(c):
                iy = {tuple(t) for t in np.argwhere(i == c).tolist()}
                oy = {tuple(t) for t in np.argwhere(o == c).tolist()}
                return iy < oy
            if not any(_grow(c) for c in cc):
                ok_grow = False
        if ok_keep or ok_pm1 or ok_grow:
            add2('largest')
        def _fold_ok(i, o):
            for axis in (0, 1):
                g = i if axis == 1 else i.T
                ot = o if axis == 1 else o.T
                W = g.shape[1]
                # 구분선 = "그 색의 모든 셀이 상수열에만 존재"하는 색의 열
                #   (패널 내부의 우연 상수열 오인 방지 - 195ba7dc 쌍1/2 교훈)
                _const = {}
                for j in range(W):
                    col = set(g[:, j].tolist())
                    if len(col) == 1:
                        _const.setdefault(int(col.pop()), []).append(j)
                seps = None
                for _sc, _js in _const.items():
                    if int((g == _sc).sum()) == len(_js) * g.shape[0]:
                        seps = _js; break
                if not seps: continue
                cuts = [-1] + seps + [W]
                panels = [g[:, a+1:b] for a, b in zip(cuts[:-1], cuts[1:])
                          if b - a - 1 > 0]
                if len(panels) < 2 or any(p.shape != ot.shape for p in panels):
                    continue
                # 배경 후보를 패널/출력 독립 조합으로 시도 (전 쌍 정확 일치 게이트)
                _obg = int(_C2(ot.ravel().tolist()).most_common(1)[0][0])
                _cands_b = {_bgc2, 0, _obg} | {int(v) for v in np.unique(ot)}
                for _b in {_bgc2, 0, _obg}:
                    u = (panels[0] != _b).copy(); n_ = u.copy()
                    for p in panels[1:]:
                        m = (p != _b); u = u | m; n_ = n_ & m
                    for _ob in _cands_b:
                        om = (ot != _ob)
                        if np.array_equal(u, om) or np.array_equal(n_, om):
                            return True
            return False
        if all(_fold_ok(i, o) for i, o in zip(ins, outs)):
            add2('largest')
    except Exception:
        pass
    try:
        # output유사: 같은 크기 + 셀 일치율 >= 70% (전 쌍), 또는
        #             출력이 입력의 부분그리드로 정확히 등장 (전 쌍)
        if same_size:
            if all(float(np.mean(i == o)) >= 0.7 for i, o in zip(ins, outs)):
                add2('outsim')
        elif all(o.shape[0] <= i.shape[0] and o.shape[1] <= i.shape[1]
                 for i, o in zip(ins, outs)):
            def _subgrid(i, o):
                H, W = i.shape; h, w = o.shape
                for dy in range(H - h + 1):
                    for dx in range(W - w + 1):
                        if np.array_equal(i[dy:dy+h, dx:dx+w], o): return True
                return False
            if all(_subgrid(i, o) for i, o in zip(ins, outs)):
                add2('outsim')
    except Exception:
        pass
    if labels:
        return rank, sig
    return rank

# ============================================================================
# [LLM 도구 1호: 분류/지각 tool]  analyze_problem(task) -> 사실 dict
#   [LLM 도구 2호: 실행/검산 tool] try_order(task, order) -> 판정 + 실패내역
# ============================================================================
def find_mirror_axes(grid, unknown=(0,), max_off=7):
    """[사용자 절차] 중간선 오프셋 스캔 축 탐지.
    좌우축: 정확히 반 가르는 중간선부터 y좌표 +1..+max_off, -1..-max_off 이동하며
    미러링 일치 '개수'를 셈 -> 최다 위치가 축. 상하축도 동일. 미지색은 제외.
    반환: {'좌우축': {'위치'(반칸 단위 float), '오프셋', '일치수', '검사쌍', '일치율'},
           '상하축': {...}, '사분면': (r축, c축) 교차 정보}"""
    g = np.asarray(grid)
    H, W = g.shape
    unk = set(int(u) for u in unknown)
    known = np.ones(g.shape, bool)
    for u in unk:
        known &= (g != u)
    def scan(L, axis):
        center_p2 = L - 1                      # 정확히 반 가르는 중간선 (2p 좌표)
        best = None
        order = [0] + [s * k for k in range(1, max_off + 1) for s in (1, -1)]
        for off in order:
            p2 = center_p2 + off
            if not (1 <= p2 <= 2 * L - 3): continue
            cnt = tot = 0
            for a in range(L):
                b = p2 - a
                if not (0 <= b < L) or a >= b: continue
                A = g[a] if axis == 'r' else g[:, a]
                B = g[b] if axis == 'r' else g[:, b]
                kA = known[a] if axis == 'r' else known[:, a]
                kB = known[b] if axis == 'r' else known[:, b]
                m = kA & kB
                tot += int(m.sum())
                cnt += int((A[m] == B[m]).sum())
            if best is None or cnt > best['일치수']:
                best = {'위치': p2 / 2.0, '오프셋': off, '일치수': cnt,
                        '검사쌍': tot, '일치율': round(cnt / tot, 3) if tot else 0.0}
        return best
    r_ax = scan(H, 'r')   # 상하 가르는 축 (행)
    c_ax = scan(W, 'c')   # 좌우 가르는 축 (열)
    return {'상하축': r_ax, '좌우축': c_ax,
            '사분면_중심': (r_ax['위치'] if r_ax else None,
                        c_ax['위치'] if c_ax else None)}

def check_restore_quadrant(train, accept=0.80, reject=0.70):
    """[사용자 사양] 복원 실행 없이 판정:
    가림색 후보(출력 근거: 소멸색/bbox일치) -> 축 스캔(find_mirror_axes)
    -> 사분면 양축 일치율로 판정.
      - 양축 모두 accept(80%) 이상 & 검사쌍 충분 -> 복원 (True)
      - 최선축이 reject(70%) 미만 -> 아님 (False)
      - 후보 없음/표본 부족/중간값 -> 포기 (False)
    모든 train 쌍이 통과해야 True."""
    if not train:
        return False
    for e in train:
        i = np.asarray(e['input']); o = np.asarray(e['output'])
        # 가림색 후보 (출력 근거)
        cands = []
        for c in np.unique(i).tolist():
            n_i = int((i == c).sum())
            if n_i == 0: continue
            if i.shape == o.shape:
                if int((o == c).sum()) == 0:
                    cands.append(int(c))                 # 소멸색
            else:
                ys, xs = np.where(i == c)
                if (int(ys.max()-ys.min()+1), int(xs.max()-xs.min()+1)) == o.shape:
                    cands.append(int(c))                 # bbox == 출력 크기
        if not cands:
            return False                                  # 포기
        ok_pair = False
        need = i.size // 8                                # 최소 검사쌍
        for occ in cands:
            ax = find_mirror_axes(i, unknown=(occ,))
            ra, ca = ax['상하축'], ax['좌우축']
            if not ra or not ca:
                continue                                  # 축 못 구함 -> 포기
            if ra['검사쌍'] < need or ca['검사쌍'] < need:
                continue                                  # 표본 부족 -> 포기
            lo = min(ra['일치율'], ca['일치율'])
            if lo < reject:
                continue                                  # 70% 미만 -> 아님
            if lo >= accept:
                ok_pair = True; break                     # 양축 80%+ -> 복원
            # 70~80% 사이 = 중간값 -> 이 후보로는 판정 보류
        if not ok_pair:
            return False
    return True

def analyze_problem(task):
    """문제의 지각 사실을 구조화해 반환 (LLM 지휘자의 눈)."""
    tr = task['train']
    ins = [np.asarray(e['input']) for e in tr]
    outs = [np.asarray(e['output']) for e in tr]
    F = {}
    F['train_쌍수'] = len(tr)
    F['크기'] = {'입력': [list(i.shape) for i in ins],
               '출력': [list(o.shape) for o in outs]}
    F['크기_같음'] = all(i.shape == o.shape for i, o in zip(ins, outs))
    ups = all(o.shape[0] % i.shape[0] == 0 and o.shape[1] % i.shape[1] == 0
              and o.shape != i.shape for i, o in zip(ins, outs))
    downs = all(i.shape[0] % o.shape[0] == 0 and i.shape[1] % o.shape[1] == 0
                and o.shape != i.shape for i, o in zip(ins, outs))
    F['정수배_확대'] = bool(ups); F['정수배_축소'] = bool(downs)
    # output 들 간 관계
    F['output들_크기일치'] = len(outs) >= 2 and all(o.shape == outs[0].shape for o in outs)
    F['output들_동일'] = len(outs) >= 2 and all(np.array_equal(o, outs[0]) for o in outs)
    try:
        sils = [sorted(hash(_d4_canon(s)) for s, _c, _p in _objs8(o, _detect_bg2(o)))
                for o in outs]
        F['output들_모양일치(D4)'] = len(outs) >= 2 and all(s == sils[0] for s in sils)
    except Exception:
        F['output들_모양일치(D4)'] = None
    F['output들_색깔일치'] = len(outs) >= 2 and all(
        set(np.unique(o).tolist()) == set(np.unique(outs[0]).tolist()) for o in outs)
    # 배경
    bgc = [_detect_bg_conf(i) for i in ins]
    F['배경'] = [{'색': b, '신뢰도': c} for b, c in bgc]
    F['배경_판정가능'] = all(c >= 0.5 and b is not None for b, c in bgc)
    if F['배경_판정가능'] and F['크기_같음']:
        bgs = [b for b, _c in bgc]
        d = [int((o == b).sum()) - int((i == b).sum())
             for i, o, b in zip(ins, outs, bgs)]
        F['배경칸_변화'] = d
        F['배경칸_방향'] = ('줄어듦(패턴/복제/추가)' if all(x < 0 for x in d) else
                        '늘어남(삭제)' if all(x > 0 for x in d) else
                        '동일' if all(x == 0 for x in d) else '혼재')
        F['배경모양_동일'] = all(np.array_equal(i == b, o == b)
                            for i, o, b in zip(ins, outs, bgs))
    # 도형 관계
    try:
        rows = []
        for i, o in zip(ins, outs):
            bi = _detect_bg2(i); bo = _detect_bg2(o)
            io = _objs8(i, bi); oo = _objs8(o, bo)
            rows.append({
                '입력도형수': len(io), '출력도형수': len(oo),
                '실루엣보존': sorted(hash(s) for s,_c,_p in io) ==
                          sorted(hash(s) for s,_c,_p in oo),
                '위치보존': set(p for _s,_c,p in io) == set(p for _s,_c,p in oo),
                '색보존': sorted(hash(c) for _s,c,_p in io) ==
                        sorted(hash(c) for _s,c,_p in oo)})
        F['도형'] = rows
    except Exception:
        F['도형'] = None
    if F['크기_같음']:
        F['고정셀_비율'] = round(float(np.mean(
            [float((i == o).mean()) for i, o in zip(ins, outs)])), 3)
    # 실검증 (해보고 아는 사실)
    F['실검증'] = {
        '복원': bool(_verify_restore_once(tr)),
        '그리드분할': all(bool(_find_sep(i)) and
            any(a.shape == o.shape for _d,_c,a,_b in _find_sep(i))
            for i, o in zip(ins, outs)),
        'D4직행': any(all(np.array_equal(t(i), o) if t(i).shape == o.shape else False
                       for i, o in zip(ins, outs)) for t in _D4.values()),
    }
    F['규칙_가설_순위'] = classify_task(task)[:5]
    return F

def analyze_problem_text(task):
    """LLM 프롬프트용 한국어 사실 요약."""
    F = analyze_problem(task)
    L = []
    L.append(f"train {F['train_쌍수']}쌍, 크기 {F['크기']['입력']} -> {F['크기']['출력']}")
    L.append(f"크기같음={F['크기_같음']} 정수배확대={F['정수배_확대']} 축소={F['정수배_축소']}")
    if '배경칸_방향' in F:
        L.append(f"배경: {F['배경'][0]['색']}(신뢰도 {F['배경'][0]['신뢰도']}), "
                 f"칸변화 {F['배경칸_변화']} -> {F['배경칸_방향']}, "
                 f"배경모양동일={F.get('배경모양_동일')}")
    else:
        L.append(f"배경 판정가능={F['배경_판정가능']}")
    if F.get('도형'):
        r = F['도형'][0]
        L.append(f"도형: {r['입력도형수']}->{r['출력도형수']}개, "
                 f"실루엣보존={r['실루엣보존']} 위치보존={r['위치보존']} 색보존={r['색보존']}")
    if '고정셀_비율' in F:
        L.append(f"고정셀비율={F['고정셀_비율']}")
    L.append(f"output들: 크기일치={F['output들_크기일치']} 동일={F['output들_동일']} "
             f"모양일치={F['output들_모양일치(D4)']} 색깔일치={F['output들_색깔일치']}")
    v = F['실검증']
    L.append(f"실검증: 복원={v['복원']} 그리드분할={v['그리드분할']} D4직행={v['D4직행']}")
    L.append(f"규칙 가설(순위): {F['규칙_가설_순위']}")
    return '\n'.join(L)

def try_order(task, order):
    """오더(단계 이름 리스트)를 실행+LOO 검산. LLM 지휘 루프의 실행 도구.
    반환: {'ok', 'pred'|None, 'fail'(실패 내역 문자열)}"""
    train = task.get('train', [])
    order = tuple(order)
    unknown = [n for n in order if n not in STAGES]
    if unknown:
        return {'ok': False, 'pred': None,
                'fail': f'없는 단계: {unknown}. 가능: {sorted(STAGES)}'}
    pipe = _derive_pipeline(order, train)
    if pipe is None:
        return {'ok': False, 'pred': None,
                'fail': f'{"->".join(order)}: train에서 도출 실패 (규칙 불성립)'}
    for i in range(len(train)):
        rest = [e for j, e in enumerate(train) if j != i]
        p = _derive_pipeline(order, rest)
        bad = None
        if p is None:
            bad = f'fold{i}: 도출 실패'
        else:
            try:
                r = p(train[i]['input'])
                exp = np.asarray(train[i]['output'])
                if np.asarray(r).shape != exp.shape:
                    bad = f'fold{i}: 크기 불일치 {np.asarray(r).shape} vs {exp.shape}'
                elif not np.array_equal(np.asarray(r), exp):
                    bad = f'fold{i}: {int((np.asarray(r) != exp).sum())}px 불일치'
            except Exception as e:
                bad = f'fold{i}: 적용 예외 {e}'
        if bad:
            return {'ok': False, 'pred': None, 'fail': f'{"->".join(order)} LOO {bad}'}
    try:
        return {'ok': True, 'pred': pipe(task['test'][0]['input']).tolist(), 'fail': ''}
    except Exception as e:
        return {'ok': False, 'pred': None, 'fail': f'test 적용 예외 {e}'}

def propose_orders(task):
    """분류기 순위로 _ORDERS 재정렬 (첫 stage 가 상위 규칙인 오더 먼저)"""
    rank = classify_task(task)
    pos = {n: i for i, n in enumerate(rank)}
    return sorted(_ORDERS, key=lambda o: (pos.get(o[0], 99), len(o)))

def _pairs_key(cur):
    return tuple((bytes(np.asarray(e['input']).astype(np.int8).tobytes()),
                  bytes(np.asarray(e['output']).astype(np.int8).tobytes()))
                 for e in cur)

def _derive_pipeline(order, pairs, _cache=None, _deadline=None):
    """단계를 순서대로 도출·적용 (도출기가 후보 목록을 주면 분기 탐색).
    각 단계는 '이전 단계 적용 후' 쌍에서 도출."""
    import time as _t
    budget = [_MAX_BRANCH]
    cache = _cache if _cache is not None else {}
    def rec(idx, cur, fns):
        if budget[0] <= 0: return None
        if _deadline is not None and _t.time() > _deadline: return None
        if idx == len(order):
            for e in cur:
                if not np.array_equal(np.asarray(e['input']), np.asarray(e['output'])):
                    return None
            return list(fns)
        ck = (order[idx], _pairs_key(cur))
        if ck in cache:
            got = cache[ck]
        else:
            got = STAGES[order[idx]](cur)
            cache[ck] = got
        if got is None: return None
        cands = got if isinstance(got, list) else [got]
        for fn in cands:
            budget[0] -= 1
            if budget[0] <= 0: return None
            nxt = []
            ok = True
            for e in cur:
                try:
                    nxt.append({'input': fn(e['input']).tolist(),
                                'output': e['output']})
                except Exception:
                    ok = False; break
            if not ok: continue
            r = rec(idx + 1, nxt, fns + [fn])
            if r is not None: return r
        return None
    fns = rec(0, [{'input': e['input'], 'output': e['output']} for e in pairs], [])
    if fns is None: return None
    def pipeline(g):
        for fn in fns: g = fn(g)
        return np.asarray(g)
    return pipeline

def _all_orders(max_len=3):
    names = list(STAGES)
    out = []
    for L in range(1, max_len + 1):
        out.extend(permutations(names, L))
    return out

_ORDERS = _all_orders()

def solve_by_orders(task):
    """오더 전수 + LOO 채택. 반환: {'pred', 'order', 'loo_conf'} or None"""
    import time as _t
    train = task.get('train', [])
    if len(train) < 2: return None
    test_in = task['test'][0]['input']
    cache = {}
    deadline = _t.time() + TASK_TIME_CAP
    # 추측 신호: 비싼 stage 는 신호 통과시에만 시도
    blocked = {name for name, sig in STAGE_TRIAGE.items() if not sig(task)}
    for order in propose_orders(task):
        if _t.time() > deadline: return None
        if blocked and any(n in blocked for n in order): continue
        pipe = _derive_pipeline(order, train, _cache=cache, _deadline=deadline)
        if pipe is None: continue
        ok = True
        for i in range(len(train)):
            rest = [e for j, e in enumerate(train) if j != i]
            p = _derive_pipeline(order, rest, _cache=cache, _deadline=deadline)
            if p is None: ok = False; break
            try:
                if not np.array_equal(p(train[i]['input']),
                                      np.asarray(train[i]['output'])):
                    ok = False; break
            except Exception:
                ok = False; break
        if not ok: continue
        try:
            return {'pred': pipe(test_in).tolist(),
                    'order': '->'.join(order), 'loo_conf': 1.0}
        except Exception:
            continue
    return None'''
_RULE_STACK = r'''# ============================================================================
# [규칙 스택 통합 파일] 오더 엔진 셀(셀88) "바로 다음" 셀에 이 파일 전체를 붙임.
#   구성: 도구36 -> 2b -> 2c -> 2d -> 2e무브 -> 2h -> 2i(가드판) -> 2j
#         -> 2k -> 2l -> 2m -> 2n(승격 연결)  = 도구 총 69개
#   전제: 앞 셀에 오더 엔진(STAGES, try_order, _derive_pipeline)이 로드돼 있어야 함
#   끝나면: solve_by_orders 가 규칙 스택으로 교체되고
#          ENGINE_NOPOOL_ALLOW 기본값이 안전 도구 58종으로 설정됨
#   실측: training 104/1000 (오답 2), evaluation 13유닛 13정답 0오답 (338s)
# ============================================================================


# ==========================================================================
# ==== [1/12: 도구 11->36 세분화] ====
# ==========================================================================

# ============================================================
# [도구 확장 셀: 11개 -> 36개] 셀1(탐색_셀1_준비.py) "다음", instruct 셀 "이전"
#   엔진 내부 서브 솔버를 개별 호출 가능한 도구로 승격.
#   - d4    1개 -> 7개 (회전/미러 각각)
#   - flow  1개 -> 5개 (방향별 낙하 4 + 뻗음 1)
#   - part  1개 -> 10개 (선택 기준별)
#   - lfill 1개 -> 7개 (서브 솔버별)
#   - 나머지 7개(color/tile/scale/shrink/restore/panels/obj)는 그대로
#   STAGES 에 추가 등록 -> try_order/_derive_pipeline 이 그대로 인식.
#   instruct 메뉴는 TOOLS_FINE (36개)을 쓴다.
# ============================================================
import numpy as np

assert 'STAGES' in globals() and 'solve_by_orders' in globals(), \
    "셀1에 오더 엔진이 없음 - 탐색_셀1_준비.py(엔진 본체 포함)를 셀1로 쓰세요"

FINE = {}

# ---------------- d4: 변환별 7개 ----------------
def _mk_d4(name):
    t = _D4[name]
    def deriv(pairs, _t=t):
        try:
            ok = all(
                np.asarray(_t(np.asarray(e['input']))).shape ==
                np.asarray(e['output']).shape and
                np.array_equal(np.asarray(_t(np.asarray(e['input']))),
                               np.asarray(e['output']))
                for e in pairs)
        except Exception:
            return None
        return (lambda g, _f=_t: np.asarray(_f(np.asarray(g)))) if ok else None
    return deriv
for _n in _D4:                       # rot90/rot180/rot270/flipH/flipV/transpose/anti
    FINE[f'd4_{_n}'] = _mk_d4(_n)

# ---------------- flow: 방향별 4 + 뻗음 1 ----------------
def _mk_flow_dir(d):
    def deriv(pairs, _d=d):
        ins = [np.asarray(e['input']) for e in pairs]
        outs = [np.asarray(e['output']) for e in pairs]
        if any(i.shape != o.shape for i, o in zip(ins, outs)):
            return None
        fns = []
        try:
            if all(np.array_equal(_fall_all(i, _d), o)
                   for i, o in zip(ins, outs)):
                fns.append(lambda g, _dd=_d: _fall_all(g, _dd))
        except Exception:
            pass
        cands = set()
        for i in ins:
            cands |= {int(v) for v in np.unique(i).tolist() if v != 0}
        for c in sorted(cands):
            try:
                if all(np.array_equal(_flow_fall(i, c, _d), o)
                       for i, o in zip(ins, outs)):
                    fns.append(lambda g, _c=c, _dd=_d: _flow_fall(g, _c, _dd))
            except Exception:
                pass
        return fns or None
    return deriv
for _d in ('down', 'up', 'left', 'right'):
    FINE[f'flow_{_d}'] = _mk_flow_dir(_d)

def _flow_extend_deriv(pairs):
    ins = [np.asarray(e['input']) for e in pairs]
    outs = [np.asarray(e['output']) for e in pairs]
    if any(i.shape != o.shape for i, o in zip(ins, outs)):
        return None
    cands = set()
    for i in ins:
        cands |= {int(v) for v in np.unique(i).tolist() if v != 0}
    fns = []
    for c in sorted(cands):
        for mode, d in _FLOW_LAWS:
            if mode != 'extend':
                continue
            try:
                if all(np.array_equal(_flow_extend(i, c, d), o)
                       for i, o in zip(ins, outs)):
                    fns.append(lambda g, _c=c, _d=d: _flow_extend(g, _c, _d))
            except Exception:
                pass
    return fns or None
FINE['flow_extend'] = _flow_extend_deriv

# ---------------- part: 선택 기준별 10개 ----------------
def _part_gate_filter(cands, pairs):
    """d_part 말미와 동일: 출력<입력 게이트 + 크기 필터(정확 일치 우선)"""
    ins = [np.asarray(e['input']) for e in pairs]
    outs = [np.asarray(e['output']) for e in pairs]
    if not all(o.size < i.size and o.shape[0] <= i.shape[0]
               and o.shape[1] <= i.shape[1] for i, o in zip(ins, outs)):
        return None
    exact, shape_only = [], []
    for fn in cands:
        ok_shape = ok_exact = True
        for i, o in zip(ins, outs):
            try:
                r = np.asarray(fn(i))
            except Exception:
                ok_shape = False; break
            if r.shape not in (o.shape, o.shape[::-1]):
                ok_shape = False; break
            if r.shape != o.shape or not np.array_equal(r, o):
                ok_exact = False
        if not ok_shape:
            continue
        (exact if ok_exact else shape_only).append(fn)
    fns = exact + shape_only
    return fns or None

def _mk_part_rank(metric, rank):
    def deriv(pairs, _m=metric, _r=rank):
        cands = []
        for mode in ('raw', 'mask'):
            def fn(g, _mm=_m, _rr=_r, _md=mode):
                g2, bg, objs = _part_objs(g)
                t = _part_pick_rank(objs, _mm, _rr)
                return _part_render(g2, bg, t[2], _md)
            cands.append(fn)
        return _part_gate_filter(cands, pairs)
    return deriv
FINE['part_big_px'] = _mk_part_rank('px', 0)      # 픽셀수 최대 도형
FINE['part_second_px'] = _mk_part_rank('px', 1)   # 픽셀수 2번째
FINE['part_small_px'] = _mk_part_rank('px', -1)   # 픽셀수 최소
FINE['part_big_bbox'] = _mk_part_rank('bbox', 0)  # bbox 최대
FINE['part_second_bbox'] = _mk_part_rank('bbox', 1)
FINE['part_small_bbox'] = _mk_part_rank('bbox', -1)

def _mk_part_picker(picker):
    def deriv(pairs, _p=picker):
        cands = []
        for mode in ('raw', 'mask'):
            def fn(g, _pp=_p, _md=mode):
                g2, bg, objs = _part_objs(g)
                t = _pp(objs)
                return _part_render(g2, bg, t[2], _md)
            cands.append(fn)
        return _part_gate_filter(cands, pairs)
    return deriv
FINE['part_unique_color'] = _mk_part_picker(_part_pick_unique_color)  # 고유색 도형
FINE['part_odd_shape'] = _mk_part_picker(_part_pick_odd_shape)        # 혼자 다른 실루엣

def _part_colorbbox_deriv(pairs):
    ins = [np.asarray(e['input']) for e in pairs]
    common = set(np.unique(ins[0]).tolist())
    for i in ins[1:]:
        common &= set(np.unique(i).tolist())
    cands = []
    for c in sorted(int(v) for v in common):
        for inner in (False, True):
            def fn(g, _c=c, _in=inner):
                return _part_color_bbox(g, _c, _in)
            cands.append(fn)
    return _part_gate_filter(cands, pairs)
FINE['part_color_bbox'] = _part_colorbbox_deriv   # 색 c 의 bbox 잘라내기

def _part_user4_deriv(pairs):
    """[사용자 절차] shapes/regions 최대·2번째 x D4 변환 -> crop"""
    cands = []
    for src_ in ('shape', 'region'):
        for rank_ in (0, 1):
            for tname_, T_ in _D4_PART.items():
                def fn(g, _s=src_, _r=rank_, _T=T_):
                    c4 = _part_extract4(g)
                    if (_s, _r) not in c4:
                        raise ValueError('cand missing')
                    return _T(c4[(_s, _r)])
                cands.append(fn)
    return _part_gate_filter(cands, pairs)
FINE['part_user4'] = _part_user4_deriv            # 가장큰/2번째큰 (사용자 절차)

# ---------------- lfill: 서브 솔버별 7개 ----------------
_LF_SPECS = [
    ('lfill_fill',   solve_one,
     [(k, t, s) for k in KINDS for t in TEMPLATE_MODES for s in SCALE_MODES]),
    ('lfill_rot',    solve_rotation_count_one,
     [(k, d) for k in KINDS for d in ('cw', 'ccw')]),
    ('lfill_border', solve_border_orient_one, [(k,) for k in KINDS]),
    ('lfill_swap',   solve_shape_swap_one,    [(k,) for k in KINDS]),
    ('lfill_hole',   solve_hole_transplant_one, [(k,) for k in KINDS]),
    ('lfill_motif',  solve_motif_stamp_one,   [()]),
    ('lfill_occl',   solve_occlusion_restore_one, [()]),
]
def _mk_lfill(solver, combos):
    def deriv(pairs, _s=solver, _c=combos):
        ins = [np.asarray(e['input']) for e in pairs]
        outs = [np.asarray(e['output']) for e in pairs]
        if not all(o.size < i.size for i, o in zip(ins, outs)):
            return None
        def fn(g, _ss=_s, _cc=_c):
            gl = np.asarray(g).tolist()
            for args in _cc:
                try:
                    out = _ss(gl, *args)
                except Exception:
                    out = None
                if out is not None:
                    return np.asarray(out)
            raise ValueError('lfill fail')
        try:
            if all(np.array_equal(np.asarray(fn(i)), o)
                   for i, o in zip(ins, outs)):
                return [fn]
        except Exception:
            pass
        return None
    return deriv
for _nm, _sv, _cb in _LF_SPECS:
    FINE[_nm] = _mk_lfill(_sv, _cb)

# ---------------- 쪼개지 않은 7개는 그대로 승격 ----------------
for _nm in ('color', 'tile', 'scale', 'shrink', 'restore', 'panels', 'obj'):
    FINE[_nm] = STAGES[_nm]

STAGES.update(FINE)                  # 파이프라인/try_order 가 새 이름 인식
TOOLS_FINE = sorted(FINE)            # instruct 메뉴 (36개)
print(f"도구 확장 완료: {len(TOOLS_FINE)}개")
print(", ".join(TOOLS_FINE))


# ==========================================================================
# ==== [2/12: 새 함수 v1 11종 + usercode] ====
# ==========================================================================

# ============================================================
# [셀2b - 함수 추가] 셀2(도구36_확장셀) "다음"에 실행
#   추가 1: 새 규칙 함수 11종 (numpy 전용, 파라미터는 train 쌍에서 도출)
#   추가 2: usercode = 회원님 엔진 전체 (part 파일의 engine_answer 호출)
#   -> 도구 총 36 + 11 + 1 = 48개
# ============================================================
import numpy as np

assert 'STAGES' in globals(), "셀1(탐색_셀1_준비.py) 먼저"
assert 'TOOLS_FINE' in globals(), "셀2(도구36_확장셀.py) 먼저"

def _NB_bg(g):
    v, c = np.unique(g, return_counts=True)
    return int(v[c.argmax()])

def _NB_objs(g, bg):
    g = np.asarray(g); H, W = g.shape
    seen = np.zeros((H, W), bool); out = []
    for r in range(H):
        for c in range(W):
            if seen[r, c] or g[r, c] == bg:
                continue
            col = int(g[r, c]); st = [(r, c)]; cells = [(r, c)]
            seen[r, c] = True
            while st:
                y, x = st.pop()
                for dy, dx in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                    ny, nx = y + dy, x + dx
                    if 0 <= ny < H and 0 <= nx < W and not seen[ny, nx] \
                            and g[ny, nx] == col:
                        seen[ny, nx] = True
                        st.append((ny, nx)); cells.append((ny, nx))
            out.append((col, cells))
    return out

def _NB_sil(cells):
    ys = [y for y, _ in cells]; xs = [x for _, x in cells]
    y0, x0 = min(ys), min(xs)
    return frozenset((y - y0, x - x0) for y, x in cells)

def dn_const_out(pairs):
    outs = [np.asarray(e['output']) for e in pairs]
    if len(outs) < 2:
        return None
    if all(o.shape == outs[0].shape and np.array_equal(o, outs[0])
           for o in outs[1:]):
        O = outs[0].copy()
        return lambda g, _O=O: _O.copy()
    return None

def dn_crop_content(pairs):
    def mk(bgmode):
        def fn(g, _m=bgmode):
            g = np.asarray(g)
            bg = _NB_bg(g) if _m == 'mode' else 0
            ys, xs = np.where(g != bg)
            if len(ys) == 0:
                raise ValueError('empty')
            return g[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
        return fn
    return [mk('mode'), mk('zero')]

def dn_dedup_rowcol(pairs):
    def fn(g):
        g = np.asarray(g)
        rows = [0] + [i for i in range(1, g.shape[0])
                      if not np.array_equal(g[i], g[i - 1])]
        g2 = g[rows]
        cols = [0] + [j for j in range(1, g2.shape[1])
                      if not np.array_equal(g2[:, j], g2[:, j - 1])]
        return g2[:, cols]
    return fn

def dn_block_mode(pairs):
    ks = []
    for e in pairs:
        i = np.asarray(e['input']); o = np.asarray(e['output'])
        if o.shape[0] == 0 or o.shape[1] == 0: return None
        if i.shape[0] % o.shape[0] or i.shape[1] % o.shape[1]: return None
        k, m = i.shape[0] // o.shape[0], i.shape[1] // o.shape[1]
        if (k, m) == (1, 1): return None
        ks.append((k, m))
    if len(set(ks)) != 1:
        return None
    k, m = ks[0]
    def fn(g, _k=k, _m=m):
        g = np.asarray(g)
        H, W = g.shape[0] // _k, g.shape[1] // _m
        out = np.zeros((H, W), int)
        for y in range(H):
            for x in range(W):
                b = g[y*_k:(y+1)*_k, x*_m:(x+1)*_m]
                v, c = np.unique(b, return_counts=True)
                out[y, x] = int(v[c.argmax()])
        return out
    return fn

def dn_resize_nn(pairs):
    rs = []
    for e in pairs:
        i = np.asarray(e['input']); o = np.asarray(e['output'])
        if i.shape == o.shape or i.shape[0] == 0 or i.shape[1] == 0:
            return None
        rs.append((o.shape[0] / i.shape[0], o.shape[1] / i.shape[1]))
    if any(abs(a - rs[0][0]) > 1e-9 or abs(b - rs[0][1]) > 1e-9
           for a, b in rs[1:]):
        return None
    ry, rx = rs[0]
    def fn(g, _ry=ry, _rx=rx):
        g = np.asarray(g)
        H = int(round(g.shape[0] * _ry)); W = int(round(g.shape[1] * _rx))
        if H < 1 or W < 1: raise ValueError('size')
        ys = (np.arange(H) * g.shape[0] / H).astype(int)
        xs = (np.arange(W) * g.shape[1] / W).astype(int)
        return g[np.ix_(ys, xs)]
    return fn

def dn_period_fill(pairs):
    ins = [np.asarray(e['input']) for e in pairs]
    outs = [np.asarray(e['output']) for e in pairs]
    holes = set(int(v) for v in np.unique(ins[0]))
    for i, o in zip(ins, outs):
        if i.shape != o.shape:
            return None
        holes &= {int(c) for c in np.unique(i)
                  if (i == c).any() and not (o == c).any()}
    cand = sorted(holes) if holes else [0]
    def mk(hole):
        def fn(g, _h=hole):
            g = np.asarray(g).copy(); H, W = g.shape
            m = (g != _h)
            if m.all() or not m.any(): raise ValueError('no hole')
            def find_p(L, axis):
                for p in range(1, L):
                    ok = True
                    for a in range(L):
                        b = a % p
                        if axis == 0:
                            va, vb, ma, mb = g[a], g[b], m[a], m[b]
                        else:
                            va, vb, ma, mb = g[:, a], g[:, b], m[:, a], m[:, b]
                        both = ma & mb
                        if both.any() and not (va[both] == vb[both]).all():
                            ok = False; break
                    if ok: return p
                return L
            py, px = find_p(H, 0), find_p(W, 1)
            if py == H and px == W: raise ValueError('no period')
            for y in range(H):
                for x in range(W):
                    if not m[y, x]:
                        vals = g[y % py::py, x % px::px]
                        vv = vals[(vals != _h)]
                        if len(vv) == 0: raise ValueError('unfillable')
                        v, c = np.unique(vv, return_counts=True)
                        g[y, x] = int(v[c.argmax()])
            return g
        return fn
    return [mk(h) for h in cand[:3]]

def dn_connect_dots(pairs):
    def mk(bgmode):
        def fn(g, _m=bgmode):
            g = np.asarray(g).copy()
            bg = _NB_bg(g) if _m == 'mode' else 0
            for gg in (g, g.T):
                H, W = gg.shape
                for y in range(H):
                    row = gg[y]
                    idx = [x for x in range(W) if row[x] != bg]
                    for a, b in zip(idx, idx[1:]):
                        if row[a] == row[b] and b > a + 1 and \
                                (row[a+1:b] == bg).all():
                            row[a+1:b] = row[a]
            return g
        return fn
    return [mk('mode'), mk('zero')]

def dn_fill_holes(pairs):
    def mk(fill_mode, const_c=None):
        def fn(g, _fm=fill_mode, _cc=const_c):
            g = np.asarray(g).copy(); H, W = g.shape
            bg = _NB_bg(g)
            seen = np.zeros((H, W), bool)
            filled = False
            for r in range(H):
                for c in range(W):
                    if seen[r, c] or g[r, c] != bg:
                        continue
                    st = [(r, c)]; cells = [(r, c)]; seen[r, c] = True
                    border = False; neigh = set()
                    while st:
                        y, x = st.pop()
                        if y in (0, H-1) or x in (0, W-1): border = True
                        for dy, dx in ((1,0),(-1,0),(0,1),(0,-1)):
                            ny, nx = y+dy, x+dx
                            if not (0 <= ny < H and 0 <= nx < W): continue
                            if g[ny, nx] == bg:
                                if not seen[ny, nx]:
                                    seen[ny, nx] = True
                                    st.append((ny, nx)); cells.append((ny, nx))
                            else:
                                neigh.add(int(g[ny, nx]))
                    if border: continue
                    if _fm == 'auto' and len(neigh) == 1:
                        v = neigh.pop()
                    elif _fm == 'const' and _cc is not None:
                        v = _cc
                    else:
                        continue
                    for y, x in cells: g[y, x] = v
                    filled = True
            if not filled: raise ValueError('no hole')
            return g
        return fn
    fns = [mk('auto')]
    try:
        newc = None
        for e in pairs:
            i = np.asarray(e['input']); o = np.asarray(e['output'])
            if i.shape != o.shape: newc = None; break
            d = {int(c) for c in np.unique(o)} - {int(c) for c in np.unique(i)}
            if len(d) == 1:
                c = d.pop()
                if newc in (None, c): newc = c
                else: newc = None; break
        if newc is not None:
            fns.append(mk('const', newc))
    except Exception:
        pass
    return fns

def dn_panels_reduce(pairs):
    def split(g):
        g = np.asarray(g)
        for ax in (1, 0):
            gg = g if ax == 1 else g.T
            H, W = gg.shape
            segs = None
            for c in set(int(v) for v in np.unique(gg)):
                js = [j for j in range(W) if (gg[:, j] == c).all()]
                if js and int((gg == c).sum()) == len(js) * H:
                    cuts = [-1] + js + [W]
                    ps = [gg[:, a+1:b] for a, b in zip(cuts[:-1], cuts[1:])
                          if b - a - 1 > 0]
                    if len(ps) >= 2 and all(p.shape == ps[0].shape for p in ps):
                        segs = ps; break
            if segs:
                return [p if ax == 1 else p.T for p in segs]
        return None
    def mk(op):
        def fn(g, _op=op):
            ps = split(g)
            if not ps or len(ps) < 2: raise ValueError('no panels')
            bgs = [_NB_bg(p) for p in ps]
            ms = [p != b for p, b in zip(ps, bgs)]
            cnt = np.zeros(ms[0].shape, int)
            for m in ms: cnt += m
            k = len(ms)
            if _op == 'or':   mask = cnt > 0
            elif _op == 'and': mask = cnt == k
            elif _op == 'xor': mask = (cnt % 2) == 1
            else:              mask = cnt > k / 2
            return mask.astype(int)      # 색은 뒤 단계(color)가 입힘
        return fn
    return [mk(o) for o in ('or', 'and', 'xor', 'maj')]

def _recolor_derive(pairs, keyfn):
    mp = {}
    for e in pairs:
        i = np.asarray(e['input']); o = np.asarray(e['output'])
        if i.shape != o.shape: return None
        bg = _NB_bg(i)
        for col, cells in _NB_objs(i, bg):
            oc = {int(o[y, x]) for y, x in cells}
            if len(oc) != 1: return None
            k = keyfn(col, cells)
            v = oc.pop()
            if k in mp and mp[k] != v: return None
            mp[k] = v
    if not mp: return None
    def fn(g, _mp=dict(mp), _kf=keyfn):
        g = np.asarray(g).copy(); bg = _NB_bg(g)
        for col, cells in _NB_objs(g, bg):
            k = _kf(col, cells)
            if k not in _mp: raise ValueError('unseen key')
            for y, x in cells: g[y, x] = _mp[k]
        return g
    return fn

def dn_recolor_size(pairs):
    return _recolor_derive(pairs, lambda col, cells: len(cells))

def dn_recolor_shape(pairs):
    return _recolor_derive(pairs, lambda col, cells: _NB_sil(cells))

_NEW = {'const_out': dn_const_out, 'crop_content': dn_crop_content,
        'dedup_rowcol': dn_dedup_rowcol, 'block_mode': dn_block_mode,
        'resize_nn': dn_resize_nn, 'period_fill': dn_period_fill,
        'connect_dots': dn_connect_dots, 'fill_holes': dn_fill_holes,
        'panels_reduce': dn_panels_reduce, 'recolor_size': dn_recolor_size,
        'recolor_shape': dn_recolor_shape}

# ---------------- usercode = 회원님 엔진 전체 (engine_answer) ----------------
#   part 파일(arc_engine_part0.py 등)이 정의한 engine_answer 를 도구로 등록.
#   fn(g) = engine_answer({'train': (도출에 쓰인 쌍들), 'test': [{'input': g}]})
#   -> try_order 의 LOO 와 그대로 호환 (fold 마다 재도출됨)
if 'engine_answer' in globals():
    def d_usercode(pairs):
        _tr = [{'input': np.asarray(e['input']).tolist(),
                'output': np.asarray(e['output']).tolist()} for e in pairs]
        def fn(g, _t=_tr):
            r = engine_answer({'train': _t, 'test': [{'input':
                               np.asarray(g).tolist()}]})
            p = r.get('pred') if isinstance(r, dict) else None
            if p is None:
                raise ValueError('usercode no pred')
            return np.asarray(p)
        return fn
    _NEW['usercode'] = d_usercode
    print("usercode(회원님 엔진 engine_answer) 등록됨")
else:
    print("경고: engine_answer 없음 -> usercode 도구 생략"
          " (part 파일에 엔진 API 셀이 포함돼야 함)")

STAGES.update(_NEW)
TOOLS_FINE = sorted(set(TOOLS_FINE) | set(_NEW))
print(f"함수 추가 완료 -> 도구 총 {len(TOOLS_FINE)}개")


# ==========================================================================
# ==== [3/12: 새 함수 v2 11종] ====
# ==========================================================================

# ============================================================
# [셀2c - 함수 추가 v2] 셀2b "다음"에 실행. 새 규칙 함수 11종 -> 도구 총 58개
#   sym_overlay    : 배경 칸을 거울상(좌우/상하)에서 가져와 채움 (대칭 완성)
#   add_border     : 테두리 두르기 (두께/색은 train 에서 도출)
#   remove_border  : 테두리 벗기기 (두께 도출)
#   panel_pick     : 패널 분할 -> 하나 고르기 (혼자 다른 것/비배경 최다/최소/다수파)
#   pick_color     : 출력 = 상수색 그리드 (최다/최소 비배경색, 최대도형색)
#   scale_dynamic  : 쌍마다 다른 배율 - 법칙 도출 (색 수 / 도형 수)
#   crop_half      : 절반/사분면 잘라내기 (8방향 후보)
#   hollow         : 도형 속 비우기 (내부 칸 -> 배경)
#   tile_crop_const: 출력 크기가 상수면 입력을 반복해 그 크기로 채움
#   keep_maxcolor  : 최다 비배경색만 남기기 (나머지 -> 배경)
#   keep_mincolor  : 최소 비배경색만 남기기
# ============================================================
import numpy as np

assert 'STAGES' in globals(), "셀1 먼저"
assert 'TOOLS_FINE' in globals(), "셀2/셀2b 먼저"

def _V2_bg(g):
    v, c = np.unique(g, return_counts=True)
    return int(v[c.argmax()])

def _V2_objs(g, bg):
    g = np.asarray(g); H, W = g.shape
    seen = np.zeros((H, W), bool); out = []
    for r in range(H):
        for c in range(W):
            if seen[r, c] or g[r, c] == bg:
                continue
            col = int(g[r, c]); st = [(r, c)]; cells = [(r, c)]
            seen[r, c] = True
            while st:
                y, x = st.pop()
                for dy, dx in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                    ny, nx = y + dy, x + dx
                    if 0 <= ny < H and 0 <= nx < W and not seen[ny, nx] \
                            and g[ny, nx] == col:
                        seen[ny, nx] = True
                        st.append((ny, nx)); cells.append((ny, nx))
            out.append((col, cells))
    return out

def dv_sym_overlay(pairs):
    def mk(axis, bgmode):
        def fn(g, _a=axis, _m=bgmode):
            g = np.asarray(g).copy()
            bg = _V2_bg(g) if _m == 'mode' else 0
            m = g[:, ::-1] if _a == 'h' else g[::-1, :]
            hole = (g == bg) & (m != bg)
            if not hole.any(): raise ValueError('no hole')
            g[hole] = m[hole]
            return g
        return fn
    return [mk(a, m) for a in ('h', 'v') for m in ('mode', 'zero')]

def dv_add_border(pairs):
    ws = set(); cs = set()
    for e in pairs:
        i = np.asarray(e['input']); o = np.asarray(e['output'])
        dh, dw = o.shape[0] - i.shape[0], o.shape[1] - i.shape[1]
        if dh <= 0 or dh != dw or dh % 2: return None
        w = dh // 2
        if not np.array_equal(o[w:-w, w:-w], i): return None
        bord = o.copy(); bord[w:-w, w:-w] = -1
        u = set(int(v) for v in np.unique(bord)) - {-1}
        if len(u) != 1: return None
        ws.add(w); cs.add(u.pop())
    if len(ws) != 1 or len(cs) != 1: return None
    w, c = ws.pop(), cs.pop()
    def fn(g, _w=w, _c=c):
        g = np.asarray(g)
        out = np.full((g.shape[0] + 2*_w, g.shape[1] + 2*_w), _c, int)
        out[_w:-_w, _w:-_w] = g
        return out
    return fn

def dv_remove_border(pairs):
    ws = set()
    for e in pairs:
        i = np.asarray(e['input']); o = np.asarray(e['output'])
        dh, dw = i.shape[0] - o.shape[0], i.shape[1] - o.shape[1]
        if dh <= 0 or dh != dw or dh % 2: return None
        w = dh // 2
        if not np.array_equal(i[w:-w, w:-w], o): return None
        ws.add(w)
    if len(ws) != 1: return None
    w = ws.pop()
    def fn(g, _w=w):
        g = np.asarray(g)
        if g.shape[0] <= 2*_w or g.shape[1] <= 2*_w: raise ValueError('small')
        return g[_w:-_w, _w:-_w]
    return fn

def _V2_split(g):
    g = np.asarray(g)
    for ax in (1, 0):
        gg = g if ax == 1 else g.T
        H, W = gg.shape
        for c in set(int(v) for v in np.unique(gg)):
            js = [j for j in range(W) if (gg[:, j] == c).all()]
            if js and int((gg == c).sum()) == len(js) * H:
                cuts = [-1] + js + [W]
                ps = [gg[:, a+1:b] for a, b in zip(cuts[:-1], cuts[1:])
                      if b - a - 1 > 0]
                if len(ps) >= 2 and all(p.shape == ps[0].shape for p in ps):
                    return [p if ax == 1 else p.T for p in ps]
    return None

def dv_panel_pick(pairs):
    def mk(rule):
        def fn(g, _r=rule):
            ps = _V2_split(g)
            if not ps or len(ps) < 2: raise ValueError('no panels')
            keys = [p.tobytes() for p in ps]
            if _r == 'odd':
                from collections import Counter as _C
                cnt = _C(keys)
                odd = [p for p, k in zip(ps, keys) if cnt[k] == 1]
                if len(odd) != 1: raise ValueError('no odd')
                return odd[0]
            if _r == 'major':
                from collections import Counter as _C
                cnt = _C(keys)
                k, n = cnt.most_common(1)[0]
                if n < 2: raise ValueError('no major')
                return ps[keys.index(k)]
            nb = [int((p != _V2_bg(p)).sum()) for p in ps]
            j = nb.index(max(nb) if _r == 'most' else min(nb))
            return ps[j]
        return fn
    return [mk(r) for r in ('odd', 'most', 'min', 'major')]

def dv_pick_color(pairs):
    outs = [np.asarray(e['output']) for e in pairs]
    if not all(len(np.unique(o)) == 1 for o in outs): return None
    shp = outs[0].shape
    same_shp = all(o.shape == shp for o in outs)
    def law_color(g, law):
        g = np.asarray(g); bg = _V2_bg(g)
        cs = {int(c): int((g == c).sum()) for c in np.unique(g) if int(c) != bg}
        if not cs: raise ValueError('no color')
        if law == 'most': return max(cs, key=cs.get)
        if law == 'min':  return min(cs, key=cs.get)
        objs = _V2_objs(g, bg)
        if not objs: raise ValueError('no obj')
        return max(objs, key=lambda t: len(t[1]))[0]
    def mk(law, shape_mode):
        def fn(g, _l=law, _s=shape_mode):
            c = law_color(g, _l)
            s = shp if _s == 'const' else (1, 1)
            return np.full(s, c, int)
        return fn
    fns = [mk(l, 'one') for l in ('most', 'min', 'big')]
    if same_shp and shp != (1, 1):
        fns += [mk(l, 'const') for l in ('most', 'min', 'big')]
    return fns

def dv_scale_dynamic(pairs):
    def k_of(g, law):
        g = np.asarray(g); bg = _V2_bg(g)
        if law == 'colors':
            return max(1, len({int(c) for c in np.unique(g)} - {bg}))
        return max(1, len(_V2_objs(g, bg)))
    for law in ('colors', 'objs'):
        ok = True
        for e in pairs:
            i = np.asarray(e['input']); o = np.asarray(e['output'])
            k = k_of(i, law)
            if o.shape != (i.shape[0] * k, i.shape[1] * k): ok = False; break
        if ok:
            def fn(g, _l=law):
                g = np.asarray(g); k = k_of(g, _l)
                return np.kron(g, np.ones((k, k), int))
            return fn
    return None

def dv_crop_half(pairs):
    def mk(which):
        def fn(g, _w=which):
            g = np.asarray(g); H, W = g.shape
            h, w = H // 2, W // 2
            if _w == 'top': return g[:h]
            if _w == 'bot': return g[H - h:]
            if _w == 'left': return g[:, :w]
            if _w == 'right': return g[:, W - w:]
            if _w == 'tl': return g[:h, :w]
            if _w == 'tr': return g[:h, W - w:]
            if _w == 'bl': return g[H - h:, :w]
            return g[H - h:, W - w:]
        return fn
    return [mk(w) for w in ('top', 'bot', 'left', 'right',
                            'tl', 'tr', 'bl', 'br')]

def dv_hollow(pairs):
    def fn(g):
        g = np.asarray(g); bg = _V2_bg(g)
        out = g.copy(); H, W = g.shape
        ch = False
        for y in range(1, H - 1):
            for x in range(1, W - 1):
                if g[y, x] != bg and \
                   g[y-1, x] == g[y+1, x] == g[y, x-1] == g[y, x+1] == g[y, x]:
                    out[y, x] = bg; ch = True
        if not ch: raise ValueError('no interior')
        return out
    return fn

def dv_tile_crop_const(pairs):
    outs = [np.asarray(e['output']) for e in pairs]
    shp = outs[0].shape
    if not all(o.shape == shp for o in outs): return None
    if not all(np.asarray(e['input']).size < outs[0].size for e in pairs):
        return None
    def fn(g, _s=shp):
        g = np.asarray(g)
        ky = -(-_s[0] // g.shape[0]); kx = -(-_s[1] // g.shape[1])
        return np.tile(g, (ky, kx))[:_s[0], :_s[1]]
    return fn

def _keep_color(g, mode):
    g = np.asarray(g).copy(); bg = _V2_bg(g)
    cs = {int(c): int((g == c).sum()) for c in np.unique(g) if int(c) != bg}
    if len(cs) < 2: raise ValueError('few colors')
    c = max(cs, key=cs.get) if mode == 'max' else min(cs, key=cs.get)
    g[(g != c) & (g != bg)] = bg
    return g

def dv_keep_maxcolor(pairs):
    return lambda g: _keep_color(g, 'max')

def dv_keep_mincolor(pairs):
    return lambda g: _keep_color(g, 'min')

_NEW2 = {'sym_overlay': dv_sym_overlay, 'add_border': dv_add_border,
         'remove_border': dv_remove_border, 'panel_pick': dv_panel_pick,
         'pick_color': dv_pick_color, 'scale_dynamic': dv_scale_dynamic,
         'crop_half': dv_crop_half, 'hollow': dv_hollow,
         'tile_crop_const': dv_tile_crop_const,
         'keep_maxcolor': dv_keep_maxcolor, 'keep_mincolor': dv_keep_mincolor}
STAGES.update(_NEW2)
TOOLS_FINE = sorted(set(TOOLS_FINE) | set(_NEW2))
print(f"함수 v2 추가 {len(_NEW2)}종 -> 도구 총 {len(TOOLS_FINE)}개")


# ==========================================================================
# ==== [4/12: 설명서 함수 dsc110] ====
# ==========================================================================

# ============================================================
# [셀2d - 설명서 함수 1차] 셀2c "다음"에 실행. arc 설명서의 A급 절차를
#   함수로 구현해 도구에 추가. (이 파일에 dscN 이 계속 늘어남)
#   dsc110: [설명서 #110] 소멸색 도형 -> 구멍 수 -> 범례색 재색칠
#     범례 = 최소변 3 인 사각형꼴(칸수+구멍칸 == bbox) 성분. 매핑 없으면 배경.
#     검증: eval #110 (1등 실패분) LOO 통과 + test 입력 2개 모두 정답.
# ============================================================
import numpy as np

assert 'STAGES' in globals(), "셀1 먼저"
assert 'TOOLS_FINE' in globals(), "셀2/셀2b/셀2c 먼저"

def _D110_comps(mask):
    H, W = mask.shape
    seen = np.zeros((H, W), bool); out = []
    for r in range(H):
        for c in range(W):
            if mask[r, c] and not seen[r, c]:
                st = [(r, c)]; seen[r, c] = True; cells = [(r, c)]
                while st:
                    y, x = st.pop()
                    for dy in (-1, 0, 1):
                        for dx in (-1, 0, 1):
                            if dy == dx == 0: continue
                            ny, nx = y + dy, x + dx
                            if 0 <= ny < H and 0 <= nx < W and mask[ny, nx] \
                                    and not seen[ny, nx]:
                                seen[ny, nx] = True
                                st.append((ny, nx)); cells.append((ny, nx))
                out.append(cells)
    return out

def _D110_hinfo(cells):
    ys = [y for y, _ in cells]; xs = [x for _, x in cells]
    y0, y1, x0, x1 = min(ys), max(ys), min(xs), max(xs)
    m = np.zeros((y1 - y0 + 3, x1 - x0 + 3), bool)
    for y, x in cells: m[y - y0 + 1, x - x0 + 1] = True
    seen = np.zeros_like(m); st = [(0, 0)]; seen[0, 0] = True
    while st:
        y, x = st.pop()
        for dy, dx in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            ny, nx = y + dy, x + dx
            if 0 <= ny < m.shape[0] and 0 <= nx < m.shape[1] \
                    and not m[ny, nx] and not seen[ny, nx]:
                seen[ny, nx] = True; st.append((ny, nx))
    inner = (~m) & (~seen)
    return len(_D110_comps(inner)), int(inner.sum()), y1 - y0 + 1, x1 - x0 + 1

def dsc110(pairs):
    ins = [np.asarray(e['input']) for e in pairs]
    outs = [np.asarray(e['output']) for e in pairs]
    tgts = set()
    for i, o in zip(ins, outs):
        if i.shape != o.shape: return None
        d = {int(c) for c in np.unique(i)} - {int(c) for c in np.unique(o)}
        if len(d) != 1: return None
        tgts |= d
    if len(tgts) != 1: return None
    tgt = tgts.pop()
    def fn(g, _t=tgt):
        g = np.asarray(g).copy()
        v, cnt = np.unique(g, return_counts=True)
        bg = int(v[cnt.argmax()])
        legend = {}
        for c in v.tolist():
            c = int(c)
            if c in (bg, _t): continue
            for cells in _D110_comps(g == c):
                h, hc, bh, bw = _D110_hinfo(cells)
                if len(cells) + hc == bh * bw and min(bh, bw) == 3:
                    legend[h] = c
        for cells in _D110_comps(g == _t):
            h, _hc, _bh, _bw = _D110_hinfo(cells)
            newc = legend.get(h, bg)
            for y, x in cells: g[y, x] = newc
        return g
    return fn

_DSC = {'dsc110': dsc110}
STAGES.update(_DSC)
TOOLS_FINE = sorted(set(TOOLS_FINE) | set(_DSC))
print(f"설명서 함수 {len(_DSC)}종 추가 -> 도구 총 {len(TOOLS_FINE)}개")


# ==========================================================================
# ==== [5/12: 무브 엔진 moveX] ====
# ==========================================================================

# ============================================================
# [무브 엔진 셀] 셀2c/2d 다음에 실행. move 사양서(사용자) 구현 -> 도구 'moveX'
#   구조:
#   1) fix/moving 분해: 전 train 공통 - 제자리에 그대로면 fix, 아니면 mover
#   2) mover 선택기 (test 에서 input 만으로 결정 가능한 것만):
#      moving색 집합 / 특정색 부착 / 가장 큰 도형 / 2번째 / 전부
#   3) 이동 법칙 메뉴:
#      상수 벡터 / 색깔별 벡터 / 상하좌우 슬라이드(장애물·벽·특정색·같은색 정지,
#      가까운·먼 순서) / 자기 높이·너비 배수 / 특정색 중심 미러(행·열·점) /
#      표면적 최소(접촉 최대) 위치 탐색
#   채택: (선택기 x 법칙) 조합을 전 train 에 시뮬레이션 -> 정확 재현만 후보.
#   최종 심판은 기존 try_order 의 LOO.
# ============================================================
import numpy as np

assert 'STAGES' in globals(), "셀1 먼저"

def _ME_bg(g):
    v, c = np.unique(g, return_counts=True)
    return int(v[c.argmax()])

def _ME_objs(g, bg):
    """색깔단위 4연결 성분 -> [(색, frozenset((y,x)))]"""
    g = np.asarray(g); H, W = g.shape
    seen = np.zeros((H, W), bool); out = []
    for r in range(H):
        for c in range(W):
            if seen[r, c] or g[r, c] == bg:
                continue
            col = int(g[r, c]); st = [(r, c)]; cells = [(r, c)]
            seen[r, c] = True
            while st:
                y, x = st.pop()
                for dy, dx in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                    ny, nx = y + dy, x + dx
                    if 0 <= ny < H and 0 <= nx < W and not seen[ny, nx] \
                            and g[ny, nx] == col:
                        seen[ny, nx] = True
                        st.append((ny, nx)); cells.append((ny, nx))
            out.append((col, frozenset(cells)))
    return out

def _ME_movers(i, o, bg):
    """제자리 보존이면 fix, 아니면 mover. 반환 (movers, fixed)"""
    movers = []; fixed = []
    for col, cells in _ME_objs(i, bg):
        if all(int(o[y, x]) == col for y, x in cells):
            fixed.append((col, cells))
        else:
            movers.append((col, cells))
    return movers, fixed

_DIRS = {'up': (-1, 0), 'down': (1, 0), 'left': (0, -1), 'right': (0, 1)}

# ---------------- 선택기 (input 만으로 결정) ----------------
def _sel_apply(spec, g, bg):
    kind = spec[0]
    objs = _ME_objs(g, bg)
    if not objs:
        return []
    if kind == 'colors':                       # moving 색 집합
        return [ob for ob in objs if ob[0] in spec[1]]
    if kind == 'attached':                     # 특정색에 붙은 도형
        g = np.asarray(g); H, W = g.shape
        out = []
        for col, cells in objs:
            if col == spec[1]:
                continue
            near = any(0 <= y+dy < H and 0 <= x+dx < W and
                       g[y+dy, x+dx] == spec[1]
                       for y, x in cells
                       for dy, dx in ((1,0),(-1,0),(0,1),(0,-1)))
            if near:
                out.append((col, cells))
        return out
    if kind == 'largest':
        return [max(objs, key=lambda ob: len(ob[1]))]
    if kind == 'second':
        srt = sorted(objs, key=lambda ob: -len(ob[1]))
        return [srt[1]] if len(srt) >= 2 else []
    if kind == 'all':
        return objs
    return []

# ---------------- 법칙 시뮬레이션 ----------------
def _paint(g, bg, movers, targets):
    """movers 지우고 targets((색, cells)) 그리기. 경계 밖이면 None"""
    g = np.asarray(g).copy(); H, W = g.shape
    for col, cells in movers:
        for y, x in cells:
            g[y, x] = bg
    for col, cells in targets:
        for y, x in cells:
            if not (0 <= y < H and 0 <= x < W):
                return None
        for y, x in cells:
            g[y, x] = col
    return g

def _shift(cells, v):
    return frozenset((y + v[0], x + v[1]) for y, x in cells)

def _law_apply(law, g, bg, movers):
    """law -> [(색, 새 cells)] or None"""
    g = np.asarray(g); H, W = g.shape
    kind = law[0]
    if kind == 'vec':                          # 상수 벡터
        return [(c, _shift(s, law[1])) for c, s in movers]
    if kind == 'cvec':                         # 색깔별 벡터
        mp = law[1]
        out = []
        for c, s in movers:
            if c not in mp:
                return None
            out.append((c, _shift(s, mp[c])))
        return out
    if kind == 'own':                          # 자기 높이/너비 배수
        d, k, unit = law[1], law[2], law[3]
        dy, dx = _DIRS[d]
        out = []
        for c, s in movers:
            h = max(y for y, _ in s) - min(y for y, _ in s) + 1
            w = max(x for _, x in s) - min(x for _, x in s) + 1
            m = (h if unit == 'h' else w) * k
            out.append((c, _shift(s, (dy * m, dx * m))))
        return out
    if kind == 'slide':                        # 방향 슬라이드
        d, stop, order = law[1], law[2], law[3]
        dy, dx = _DIRS[d]
        # 진행 방향 앞쪽 것부터 (order='near') / 뒤쪽부터 ('far')
        def keyf(ob):
            lead = max((y * dy + x * dx) for y, x in ob[1])
            return -lead if order == 'near' else lead
        work = np.asarray(g).copy()
        mv = sorted(movers, key=keyf)
        for c, s in mv:
            for y, x in s:
                work[y, x] = bg
        out = []
        for c, s in mv:
            cur = s
            while True:
                nxt = _shift(cur, (dy, dx))
                if any(not (0 <= y < H and 0 <= x < W) for y, x in nxt):
                    break                       # 벽
                blocked = any(work[y, x] != bg for y, x in nxt - cur)
                if stop == 'obstacle' and blocked:
                    break
                if isinstance(stop, int):       # 특정색에 닿으면 정지
                    touch = any(0 <= y < H and 0 <= x < W and
                                work[y, x] == stop for y, x in nxt - cur)
                    if blocked:
                        break
                    if touch:
                        break
                if stop == 'same' and blocked:
                    hit_same = any(work[y, x] == c for y, x in nxt - cur)
                    break_all = True
                    if hit_same:
                        break
                    break
                if blocked:
                    break
                cur = nxt
            for y, x in cur:
                work[y, x] = c
            out.append((c, cur))
        return out
    if kind == 'tetris':                       # 낙하-끼움 (회전 허용, #385형)
        # 각 mover 가 아래로 떨어져 가장 깊이 안착하는 자리에 놓임.
        # 동률이면 무회전 우선, 그다음 왼쪽 우선. 처리 순서 = 위(왼쪽)부터.
        work = np.asarray(g).copy()
        for c, s2 in movers:
            for y, x in s2:
                work[y, x] = bg
        def variants(s2):
            pts = [(y, x) for y, x in s2]
            y0 = min(y for y, _ in pts); x0 = min(x for _, x in pts)
            base = [(y - y0, x - x0) for y, x in pts]
            seen = []
            for rk in range(4):
                p = base
                for _ in range(rk):
                    p = [(x, -y) for y, x in p]
                    my = min(y for y, _ in p); mx = min(x for _, x in p)
                    p = [(y - my, x - mx) for y, x in p]
                key = frozenset(p)
                if key not in [frozenset(q) for q in seen]:
                    seen.append(p)
            return seen
        out = []
        mv = sorted(movers, key=lambda ob: (-len(ob[1]), min(ob[1])))
        for c, s2 in mv:
            best = None
            for vi, var in enumerate(variants(s2)):
                vh = max(y for y, _ in var) + 1
                vw = max(x for _, x in var) + 1
                for ox in range(W - vw + 1):
                    # 낙하: 가능한 가장 깊은 oy
                    fy = None
                    for oy in range(H - vh + 1):
                        cells = [(y + oy, x + ox) for y, x in var]
                        if all(work[y, x] == bg for y, x in cells):
                            fy = oy
                        else:
                            break               # 더 못 내려감
                    if fy is None:
                        continue
                    bottom = fy + vh - 1
                    cells = [(y + fy, x + ox) for y, x in var]
                    cs = set(cells)
                    contact = sum(1 for y, x in cells
                                  for dy2, dx2 in ((1,0),(-1,0),(0,1),(0,-1))
                                  if (y+dy2, x+dx2) not in cs and
                                  0 <= y+dy2 < H and 0 <= x+dx2 < W and
                                  work[y+dy2, x+dx2] != bg)
                    key = (-bottom, -contact, vi, ox)
                    if best is None or key < best[0]:
                        best = (key, cells)
            if best is None:
                return None
            for y, x in best[1]:
                work[y, x] = c
            out.append((c, frozenset(best[1])))
        return out
    if kind == 'edge':                         # 가장자리/모서리 일치 (사용자)
        # 선택된 도형(특정색/도형/패턴)을 지정 위치에 붙임 (평행이동만)
        pos = law[1]
        out = []
        for c, s2 in movers:
            ys = [y for y, _ in s2]; xs = [x for _, x in s2]
            y0, y1 = min(ys), max(ys); x0, x1 = min(xs), max(xs)
            dy = dx = 0
            if 't' in pos: dy = -y0
            if 'b' in pos: dy = (H - 1) - y1
            if 'l' in pos: dx = -x0
            if 'r' in pos: dx = (W - 1) - x1
            if pos == 'cy': dy = (H - (y1 - y0 + 1)) // 2 - y0
            if pos == 'cx': dx = (W - (x1 - x0 + 1)) // 2 - x0
            out.append((c, _shift(s2, (dy, dx))))
        return out
    if kind == 'domino':                       # 도미노 넘어짐 (사용자 사양)
        # 기둥(1칸 폭 세로, 바닥에 닿음)들이 dir 로 연쇄로 넘어짐.
        # 간격 0칸이면 다음 기둥도 넘어지고, 1칸 이상 남으면 멈춤.
        d = law[1]
        trigc = law[2] if len(law) > 2 else None
        Hh, Ww = g.shape
        pillars = []
        for c, s2 in movers:
            xs2 = {x for _y, x in s2}
            ys2 = {y for y, _x in s2}
            if len(xs2) != 1 or max(ys2) != Hh - 1:
                return None                     # 기둥 아님
            pillars.append((c, s2, next(iter(xs2)), len(ys2)))
        if not pillars:
            return None
        pillars.sort(key=lambda p: p[2], reverse=(d == 'left'))
        out = []
        first = True
        chain = True
        reach = None
        for c, s2, x, h in pillars:
            if d == 'right':
                span = list(range(x, x + h))
                trig = first or c == trigc or \
                    (chain and reach is not None and x <= reach + 1)
            else:
                span = list(range(x - h + 1, x + 1))
                trig = first or c == trigc or \
                    (chain and reach is not None and x >= reach - 1)
            if trig and all(0 <= sx < Ww for sx in span):
                out.append((c, frozenset((Hh - 1, sx) for sx in span)))
                reach = (span[-1] if d == 'right' else span[0])
                chain = True
            else:
                out.append((c, s2))            # 서 있음 -> 연쇄 끊김
                chain = False
                reach = None
            first = False
        return out
    if kind == 'mirror':                       # 특정색 중심 미러
        m, mode = law[1], law[2]
        ys, xs = np.where(np.asarray(g) == m)
        if len(ys) == 0:
            return None
        out = []
        for c, s in movers:
            if mode == 'row':
                if len(set(ys.tolist())) != 1:
                    return None
                r = ys[0]
                out.append((c, frozenset((2 * r - y, x) for y, x in s)))
            elif mode == 'col':
                if len(set(xs.tolist())) != 1:
                    return None
                cc = xs[0]
                out.append((c, frozenset((y, 2 * cc - x) for y, x in s)))
            else:                              # point (180도)
                if len(ys) != 1:
                    return None
                py, px = int(ys[0]), int(xs[0])
                out.append((c, frozenset((2 * py - y, 2 * px - x)
                                         for y, x in s)))
        return out
    if kind == 'surface':                      # 표면적 최소 = 접촉 최대 위치
        work = np.asarray(g).copy()
        for c, s in movers:
            for y, x in s:
                work[y, x] = bg
        out = []
        for c, s in movers:
            best = None
            ys = [y for y, _ in s]; xs = [x for _, x in s]
            h = max(ys) - min(ys) + 1; w = max(xs) - min(xs) + 1
            base = frozenset((y - min(ys), x - min(xs)) for y, x in s)
            for oy in range(H - h + 1):
                for ox in range(W - w + 1):
                    cand = frozenset((y + oy, x + ox) for y, x in base)
                    if any(work[y, x] != bg for y, x in cand):
                        continue
                    contact = sum(1 for y, x in cand
                                  for dy, dx in ((1,0),(-1,0),(0,1),(0,-1))
                                  if 0 <= y+dy < H and 0 <= x+dx < W
                                  and (y+dy, x+dx) not in cand
                                  and work[y+dy, x+dx] != bg)
                    dist = abs(oy - min(ys)) + abs(ox - min(xs))
                    key = (-contact, dist)
                    if best is None or key < best[0]:
                        best = (key, cand)
            if best is None:
                return None
            for y, x in best[1]:
                work[y, x] = c
            out.append((c, best[1]))
        return out
    return None

def _ME_law_rank(law):
    """특이도: 낮을수록 먼저. 구조 법칙 우선, 관측 벡터는 뒤로(과적합 위험)."""
    k = law[0]
    if k == 'tetris': return 0
    if k == 'domino' and len(law) > 2 and law[2] is not None: return 1
    if k == 'mirror': return 2
    if k == 'edge': return 3
    if k == 'own': return 4
    if k == 'slide' and isinstance(law[2], int): return 5
    if k == 'slide': return 6
    if k == 'domino': return 7
    if k == 'cvec': return 8
    if k == 'vec': return 9
    return 10                                  # surface

def _me_program_fn(sel, law, bg_mode):
    def fn(g, _s=sel, _l=law, _bm=bg_mode):
        g = np.asarray(g)
        bg = _ME_bg(g) if _bm == 'mode' else 0
        movers = _sel_apply(_s, g, bg)
        if not movers:
            raise ValueError('no movers')
        tg = _law_apply(_l, g, bg, movers)
        if tg is None:
            raise ValueError('law fail')
        r = _paint(g, bg, movers, tg)
        if r is None:
            raise ValueError('out of grid')
        return r
    return fn

def dm_moveX(pairs):
    """(선택기 x 법칙) 전수 -> 전 train 정확 재현 조합만 후보 반환"""
    import time as _t
    dl = _t.time() + 6.0
    ins = [np.asarray(e['input']) for e in pairs]
    outs = [np.asarray(e['output']) for e in pairs]
    if any(i.shape != o.shape for i, o in zip(ins, outs)):
        return None
    if all(np.array_equal(i, o) for i, o in zip(ins, outs)):
        return None
    cands = []
    for bg_mode in ('mode', 'zero'):
        bgs = [(_ME_bg(i) if bg_mode == 'mode' else 0) for i in ins]
        # moving 색 집합 (전 쌍 관측)
        mv_colors = set()
        obs_ok = True
        for i, o, b in zip(ins, outs, bgs):
            mv, fx = _ME_movers(i, o, b)
            if not mv:
                obs_ok = False
                break
            mv_colors |= {c for c, _s in mv}
        if not obs_ok:
            continue
        all_colors = set()
        for i in ins:
            all_colors |= {int(v) for v in np.unique(i)}
        sels = [('all',)]
        for m in sorted(all_colors):
            sels.append(('attached', m))
        sels += [('largest',), ('second',),
                 ('colors', frozenset(mv_colors))]   # colors 는 fold 취약 -> 마지막
        # 법칙 후보
        laws = [('surface',), ('tetris',)]
        for _p in ('t', 'b', 'l', 'r', 'tl', 'tr', 'bl', 'br',
                   'cy', 'cx'):
            laws.append(('edge', _p))
        for _dd in ('right', 'left'):
            laws.append(('domino', _dd, None))
            for _tc in sorted(mv_colors):      # 트리거 = mover 색만
                laws.append(('domino', _dd, _tc))
        for d in _DIRS:
            for order in ('near', 'far'):
                laws.append(('slide', d, 'obstacle', order))
                for m in sorted(all_colors):
                    laws.append(('slide', d, m, order))
            for k in (1, 2, 3):
                for u in ('h', 'w'):
                    laws.append(('own', d, k, u))
        for m in sorted(all_colors):
            for mode in ('row', 'col', 'point'):
                laws.append(('mirror', m, mode))
        # 관측 벡터 기반 법칙 (쌍1 mover 의 이동 후보)
        i0, o0, b0 = ins[0], outs[0], bgs[0]
        mv0, _ = _ME_movers(i0, o0, b0)
        vecs = set()
        for c, s in mv0[:4]:
            base = sorted(s)
            H, W = o0.shape
            for oy in range(-H + 1, H):
                for ox in range(-W + 1, W):
                    ns = [(y + oy, x + ox) for y, x in base]
                    if all(0 <= y < H and 0 <= x < W and
                           int(o0[y, x]) == c for y, x in ns):
                        vecs.add((oy, ox))
            if len(vecs) > 40:
                break
        for v in sorted(vecs):
            if v != (0, 0):
                laws.append(('vec', v))
        # 색깔별 벡터 (쌍1 관측으로 사전 구성)
        cvec = {}
        cv_ok = True
        for c, s in mv0:
            found = None
            base = sorted(s)
            H, W = o0.shape
            for oy in range(-H + 1, H):
                if found:
                    break
                for ox in range(-W + 1, W):
                    ns = [(y + oy, x + ox) for y, x in base]
                    if all(0 <= y < H and 0 <= x < W and
                           int(o0[y, x]) == c for y, x in ns):
                        found = (oy, ox)
                        break
            if found is None:
                cv_ok = False
                break
            if c in cvec and cvec[c] != found:
                cv_ok = False
                break
            cvec[c] = found
        if cv_ok and cvec:
            laws.append(('cvec', cvec))
        # 전수 검증
        for sel in sels:
            if _t.time() > dl:
                break
            for law in laws:
                if _t.time() > dl:
                    break
                fn = _me_program_fn(sel, law, bg_mode)
                try:
                    if all(np.array_equal(fn(i), o)
                           for i, o in zip(ins, outs)):
                        cands.append((_ME_law_rank(law), len(cands), fn))
                except Exception:
                    continue
                if len(cands) >= 12:
                    break
            if len(cands) >= 12:
                break
        if len(cands) >= 12:
            break
    if not cands:
        return None
    cands.sort(key=lambda x: (x[0], x[1]))     # 특이도 -> 발견 순
    return [fn for _r, _k, fn in cands[:6]]

STAGES['moveX'] = dm_moveX
if 'TOOLS_FINE' in globals():
    TOOLS_FINE = sorted(set(TOOLS_FINE) | {'moveX'})
print("무브 엔진 등록: moveX (도구", len(globals().get('TOOLS_FINE', [])), "개)")


# ==========================================================================
# ==== [6/12: 특징 규칙 featrule 2종 [조심]] ====
# ==========================================================================

# ============================================================
# [셀2h - 특징 규칙 도구] featrule / featrule_iter (도구 60 -> 62)
#   위치: 규칙기반 노트북에서 셀2e(무브 엔진) "다음", 분류기 셀 "전".
#   표적: 근접 1~10% "국소 변화" 문제 (identity가 최근접인 197개 계열)
#
#   원리: train 쌍(같은 크기)에서 "바뀌는 칸"의 특징 -> 새 값 규칙 도출.
#     특징 = 자기값 + 상하좌우 인접값 + 4방향 최근접 비배경(색, 인접여부)
#     색 정규화 = 배경 B / 등장순 c0,c1.. (색이 바뀌는 fold에도 일반화)
#     출력 인코딩 = 특징 슬롯 참조(장거리 색 복사 가능) 또는 절대색
#     안전장치 = 규칙 함수성 + 안바뀐 칸과 충돌 금지 + train 완전 재현 + LOO
#
#   실측 (training 1000, LOO 채택 기준):
#     정답 6 (신규 4: 50cb2852, 6f8cd79b, 84f2aca1, a406ac07)
#     오답채택 1 (bb43febb) -> [조심] 계열: 호출 순서 뒤로 둘 것
#     evaluation 120: 0 (표적 유형이 eval에 적음)
# ============================================================
import numpy as np

def _FR_bg(g):
    v, c = np.unique(g, return_counts=True)
    return int(v[c.argmax()])

def _FR_feat(g, y, x, bg):
    H, W = g.shape
    def at(yy, xx):
        return int(g[yy, xx]) if 0 <= yy < H and 0 <= xx < W else -1
    f = [int(g[y, x]), at(y-1, x), at(y+1, x), at(y, x-1), at(y, x+1)]
    for dy, dx in ((-1, 0), (1, 0), (0, -1), (0, 1)):
        yy, xx = y + dy, x + dx; col = -1; d = 0
        while 0 <= yy < H and 0 <= xx < W:
            d += 1
            if g[yy, xx] != bg: col = int(g[yy, xx]); break
            yy += dy; xx += dx
        f.append(col); f.append(1 if d == 1 and col != -1 else 0)
    return f            # 길이 13: [자기,인접4, (색,인접?)x4]

_SLOT = [0, 1, 2, 3, 4, 5, 7, 9, 11]   # 색이 들어있는 슬롯 위치

def _FR_key(f, bg):
    canon = {}; out = []
    for i, v in enumerate(f):
        if i in (6, 8, 10, 12):          # 거리1 플래그는 그대로
            out.append(v); continue
        if v == -1: out.append('X')
        elif v == bg: out.append('B')
        else:
            if v not in canon: canon[v] = f'c{len(canon)}'
            out.append(canon[v])
    return tuple(out), canon

def _FR_encode_val(v, f, bg, canon):
    if v == bg: return ('B',)
    for si in _SLOT:
        if f[si] == v: return ('S', si)   # 슬롯 참조
    return ('A', v)                        # 절대색 (마지막 수단)

def _FR_decode_val(enc, f, bg):
    if enc[0] == 'B': return bg
    if enc[0] == 'S':
        v = f[enc[1]]
        return v if v != -1 else None
    return enc[1]

def _FR_build(pairs):
    rules = {}; keep = set()
    for e in pairs:
        i = np.asarray(e['input']); o = np.asarray(e['output'])
        if i.shape != o.shape: return None
        bg = _FR_bg(i); H, W = i.shape
        for y in range(H):
            for x in range(W):
                f = _FR_feat(i, y, x, bg)
                k, canon = _FR_key(f, bg)
                if int(o[y, x]) != int(i[y, x]):
                    enc = _FR_encode_val(int(o[y, x]), f, bg, None)
                    if k in rules and rules[k] != enc: return None
                    rules[k] = enc
                else:
                    keep.add(k)
    if not rules: return None
    if any(k in keep for k in rules): return None      # 충돌
    return rules

def _FR_apply(g, rules):
    g = np.asarray(g); bg = _FR_bg(g); H, W = g.shape
    out = g.copy()
    for y in range(H):
        for x in range(W):
            f = _FR_feat(g, y, x, bg)
            k, _ = _FR_key(f, bg)
            if k in rules:
                v = _FR_decode_val(rules[k], f, bg)
                if v is not None: out[y, x] = v
    return out

def dv_featrule(pairs):
    rules = _FR_build(pairs)
    if rules is None: return None
    for e in pairs:                          # train 완전 재현 필수
        if not np.array_equal(_FR_apply(e['input'], rules),
                              np.asarray(e['output'])): return None
    return lambda g, _r=rules: _FR_apply(g, _r)

def dv_featrule_iter(pairs):
    rules = _FR_build(pairs)
    if rules is None: return None
    def fn(g, _r=rules):
        cur = np.asarray(g)
        for _ in range(64):
            nxt = _FR_apply(cur, _r)
            if np.array_equal(nxt, cur): break
            cur = nxt
        return cur
    for e in pairs:
        if not np.array_equal(fn(e['input']), np.asarray(e['output'])): return None
    return fn

if 'STAGES' in globals():
    STAGES['featrule'] = dv_featrule
    STAGES['featrule_iter'] = dv_featrule_iter
    TOOLS_FINE = sorted(set(TOOLS_FINE) | {'featrule', 'featrule_iter'})
    print(f"특징 규칙 도구 2종 추가 -> 도구 총 {len(TOOLS_FINE)}개")


# ==========================================================================
# ==== [7/12: 도형 diff v4 (복잡도 가드판)] ====
# ==========================================================================

# ============================================================
# [셀2i v4 - 도형 diff 도구] objdiff (기존 셀2i "교체용")
#   위치: 셀2h(특징 규칙) 다음, 셀2j(objmove) 전.
#
# ── v3 -> v4 변경점 ────────────────────────────────────────
#   1) 쌍둥이(twin) 속성 신설: "같은 실루엣 + 다른 색" 도형 관계.
#      - 실루엣 비교는 D4 정규형(회전/반전 무시). 실측: 짝꿍이 뒤집혀
#        놓인 문제(2a5f8217 test)가 있어 원형 비교로는 놓침.
#      - 틀 안(inbb != -1)에 있는 쌍둥이(=견본)를 우선 선택.
#      - 색이 여러 후보면 가장 가까운 쌍둥이 선택.
#      - has_twin(존재 bool) / twin(쌍둥이 색) / twin_inbb(쌍둥이를 품은
#        틀의 색) / touching_any(접촉 여부 bool) 속성 추가.
#   2) 관계 재색 모드 2종:
#      - twinc    : 새 색 = 쌍둥이의 색 (틀 안 견본 색을 받는 문제)
#      - twinhost : 새 색 = 쌍둥이를 품은 틀의 색 (63613498 계열)
#   3) 운명 키 순서를 "일반화 우선"으로 재배열:
#      bool/관계 키(크기순위·구멍·has_twin 등)를 앞에, 색/실루엣 값 키를
#      뒤에. 실측: 후보 상한 때문에 관계 키가 시도조차 안 되던 문제 수정.
#
# ── 실측 (LOO 채택 기준) ──────────────────────────────────
#   training 1000: 정답 11, 오답채택 0
#     v3 대비 신규 +1: 845d6e51 (쌍둥이 관계)
#     (63613498 은 원형-실루엣 매칭 구성에서만 잡히는데, 그 구성은
#      2a5f8217 오답채택을 되살려서 폐기 - 안전 우선. 다음 라운드 과제)
#   evaluation 120: 0
# ============================================================
import numpy as np
from collections import Counter

def _OD_bg(g):
    v, c = np.unique(g, return_counts=True)
    return int(v[c.argmax()])

def _OD_objs(g, bg):
    g = np.asarray(g); H, W = g.shape
    seen = np.zeros((H, W), bool); out = []
    for r in range(H):
        for c in range(W):
            if seen[r, c] or g[r, c] == bg: continue
            col = int(g[r, c]); st = [(r, c)]; cells = []
            seen[r, c] = True
            while st:
                y, x = st.pop(); cells.append((y, x))
                for dy, dx in ((1,0),(-1,0),(0,1),(0,-1)):
                    ny, nx = y+dy, x+dx
                    if 0 <= ny < H and 0 <= nx < W and not seen[ny, nx] \
                            and g[ny, nx] == col:
                        seen[ny, nx] = True; st.append((ny, nx))
            out.append((col, frozenset(cells)))
    return out

def _OD_sil(cells):
    y0 = min(y for y, _ in cells); x0 = min(x for _, x in cells)
    return frozenset((y-y0, x-x0) for y, x in cells)

def _OD_holes(cells):
    ys = [y for y, _ in cells]; xs = [x for _, x in cells]
    y0, y1 = min(ys), max(ys); x0, x1 = min(xs), max(xs)
    H, W = y1-y0+3, x1-x0+3
    m = np.zeros((H, W), bool)
    for y, x in cells: m[y-y0+1, x-x0+1] = True
    seen = np.zeros((H, W), bool); st = [(0, 0)]; seen[0, 0] = True
    while st:
        y, x = st.pop()
        for dy, dx in ((1,0),(-1,0),(0,1),(0,-1)):
            ny, nx = y+dy, x+dx
            if 0 <= ny < H and 0 <= nx < W and not seen[ny, nx] and not m[ny, nx]:
                seen[ny, nx] = True; st.append((ny, nx))
    holes = 0
    for y in range(H):
        for x in range(W):
            if not m[y, x] and not seen[y, x]:
                holes += 1
                st = [(y, x)]; seen[y, x] = True
                while st:
                    yy, xx = st.pop()
                    for dy, dx in ((1,0),(-1,0),(0,1),(0,-1)):
                        ny, nx = yy+dy, xx+dx
                        if 0 <= ny < H and 0 <= nx < W and not seen[ny, nx] \
                                and not m[ny, nx]:
                            seen[ny, nx] = True; st.append((ny, nx))
    return holes

def _OD_props(objs, shape):
    H, W = shape
    sizes = sorted({len(cs) for _, cs in objs}, reverse=True)
    ccnt = Counter(c for c, _ in objs)
    out = []
    for col, cs in objs:
        ys = [y for y, _ in cs]; xs = [x for _, x in cs]
        p = {'color': col, 'size': len(cs),
             'szrank': sizes.index(len(cs)) + 1,
             'uniqcol': ccnt[col] == 1,
             'holes': _OD_holes(cs),
             'border': min(ys)==0 or min(xs)==0 or max(ys)==H-1 or max(xs)==W-1,
             'sil': _OD_sil(cs),
             'hw': (max(ys)-min(ys)+1, max(xs)-min(xs)+1)}
        out.append((col, cs, p))
    # 관계 속성: touch(8방향 인접 타도형 색 집합), inbb(자기를 bbox에 품는 타도형 색)
    boxes = []
    for col, cs in objs:
        ys = [y for y, _ in cs]; xs = [x for _, x in cs]
        boxes.append((col, cs, min(ys), max(ys), min(xs), max(xs)))
    for k, (col, cs, p) in enumerate(out):
        near = set()
        cset = objs[k][1]
        for col2, cs2 in objs:
            if cs2 is cset: continue
            if any(abs(y-y2) <= 1 and abs(x-x2) <= 1
                   for y, x in cset for y2, x2 in cs2):
                near.add(col2)
        p['touch'] = frozenset(near)
        ys = [y for y, _ in cset]; xs = [x for _, x in cset]
        y0, y1, x0, x1 = min(ys), max(ys), min(xs), max(xs)
        host = -1; harea = None
        for col2, cs2, by0, by1, bx0, bx1 in boxes:
            if cs2 is cset: continue
            if by0 <= y0 and y1 <= by1 and bx0 <= x0 and x1 <= bx1:
                a = (by1-by0+1) * (bx1-bx0+1)
                if harea is None or a < harea: host, harea = col2, a
        p['inbb'] = host
    # ---- 쌍둥이(twin) 속성: "같은 실루엣 + 다른 색" 도형 관계 ----
    #   twin      = 같은 실루엣인 다른색 도형의 색 (그 색이 유일할 때만, 아니면 -1)
    #   has_twin  = 쌍둥이 존재 여부 (bool - 색이 바뀌어도 일반화되는 형태)
    #   touching_any = 다른 도형과 하나라도 접촉? (틀 안 견본 vs 자유 도형 구분)
    #   쓰임새: "틀 안 견본과 같은 모양인 자유 도형을 재색" (776ffc46, 63613498)
    # 쌍둥이 매칭은 D4 정규형 실루엣으로 (회전/반전돼 있어도 짝꿍 인정.
    #   2a5f8217 test 에서 쌍꿍이 좌우반전으로 놓인 사례 실측)
    def _sil8(sil):
        cells = sorted(sil)
        best = None
        for t in range(8):
            pts = []
            for y, x in cells:
                yy, xx = (y, x)
                if t & 1: yy, xx = xx, yy            # 전치
                if t & 2: yy = -yy                    # 상하 반전
                if t & 4: xx = -xx                    # 좌우 반전
                pts.append((yy, xx))
            y0 = min(p_[0] for p_ in pts); x0 = min(p_[1] for p_ in pts)
            cn = tuple(sorted((y-y0, x-x0) for y, x in pts))
            if best is None or cn < best: best = cn
        return best
    sil_groups = {}       # 원형(그대로) 실루엣 그룹
    sil8_groups = {}      # D4 정규형 그룹 (폴백용)
    for k, (col, cs, p) in enumerate(out):
        p['sil8'] = _sil8(p['sil'])
        sil_groups.setdefault(p['sil'], []).append(k)
        sil8_groups.setdefault(p['sil8'], []).append(k)
    for k, (col, cs, p) in enumerate(out):
        # 짝꿍 판정은 D4 정규형(회전/반전 무시)으로 통일.
        #   원형-우선 2단 방식도 실험했으나 오답채택이 되살아나 폐기 (실측)
        cands = [j for j in sil8_groups[p['sil8']]
                 if j != k and out[j][0] != col]
        # 틀 안(inbb != -1)에 있는 쌍둥이(=견본) 우선.
        #   776ffc46: 자유 1-블록의 쌍둥이 = 틀 안 2-블록 (견본)
        #   63613498: 9-도형의 쌍둥이 = 5-모서리 bbox 안 3-도형 (견본)
        framed = [j for j in cands if out[j][2]['inbb'] != -1]
        pick = framed if framed else cands
        if pick:
            cols = {out[j][0] for j in pick}
            if len(cols) == 1:
                j0 = pick[0]                         # 색이 유일하면 그것
            else:
                # 색이 여러 개면 "가장 가까운" 쌍둥이를 고른다
                #   (2a5f8217 처럼 짝꿍이 옆에 붙어 있는 문제 - 거리로 결정)
                def _d(j):
                    return min(max(abs(y-y2), abs(x-x2))
                               for y, x in cs for y2, x2 in out[j][1])
                j0 = min(pick, key=_d)
            p['twin'] = out[j0][0]                   # 쌍둥이의 색
            p['twin_inbb'] = out[j0][2]['inbb']      # 쌍둥이를 품은 틀의 색 (-1=없음)
        else:
            p['twin'] = -1; p['twin_inbb'] = -1
        p['has_twin'] = p['twin'] != -1
        p['touching_any'] = len(p['touch']) > 0
    return out

_OD_KEYS = [
    # ---- 일반화 우선 순서 ----
    # 앞쪽 = 값이 색에 안 묶이는 키 (fold 에서 색이 바뀌어도 일반화)
    # 뒤쪽 = 색/실루엣 값 기반 키 (암기성 - LOO 에서 잘 깨지므로 후순위)
    'szrank', 'uniqcol', 'holes', 'border',
    'has_twin',                      # 쌍둥이(같은 실루엣, 다른 색) 존재 (bool)
    ('has_twin', 'touching_any'),    # 쌍둥이 있음 x 접촉 여부
    ('has_twin', 'inbb'),            # 쌍둥이 있음 x 내가 틀 안인가
    'twin_inbb',                     # 내 쌍둥이가 틀 안인가 (틀 색)
    ('has_twin', 'twin_inbb'),
    ('has_twin', 'border'),
    ('size', 'border'),
    'size', 'hw', 'touch', 'inbb',
    # ---- 색 기반 (암기성) ----
    'color', 'sil',
    ('color', 'size'), ('color', 'sil'), ('color', 'touch'),
    ('color', 'inbb'), ('color', 'border'),
    ('has_twin', 'touch')]

def _OD_get(p, key):
    if isinstance(key, tuple):
        return tuple(p[k] for k in key)
    return p[key]

def _OD_diff(pairs):
    """쌍마다 [(col, cells, props, fate, param)] / fate: K,D,R(newc),M(vec)"""
    out = []
    for e in pairs:
        i = np.asarray(e['input']); o = np.asarray(e['output'])
        if i.shape != o.shape: return None
        bg = _OD_bg(i)
        if _OD_bg(o) != bg: return None
        oo = _OD_objs(o, bg)
        omap = {k: False for k in oo}
        recs = []
        for (col, cs, p) in _OD_props(_OD_objs(i, bg), i.shape):
            if (col, cs) in omap and not omap[(col, cs)]:
                omap[(col, cs)] = True; recs.append((col, cs, p, 'K', None)); continue
            hit = next(((c2, cs2) for (c2, cs2) in omap
                        if cs2 == cs and not omap[(c2, cs2)] and c2 != col), None)
            if hit:
                omap[hit] = True; recs.append((col, cs, p, 'R', hit[0])); continue
            sil = _OD_sil(cs)
            hit = next(((c2, cs2) for (c2, cs2) in omap
                        if not omap[(c2, cs2)] and c2 == col
                        and len(cs2) == len(cs) and _OD_sil(cs2) == sil), None)
            if hit:
                omap[hit] = True
                dy = min(y for y,_ in hit[1]) - min(y for y,_ in cs)
                dx = min(x for _,x in hit[1]) - min(x for _,x in cs)
                recs.append((col, cs, p, 'M', (dy, dx))); continue
            if all(o[y, x] == bg for y, x in cs):
                recs.append((col, cs, p, 'D', None)); continue
            return None
        if not all(omap.values()): return None
        out.append((np.asarray(e['input']), recs))
    return out

def _OD_sim_slide(g, bg, movers, d):
    """movers(셀집합 리스트)를 d 방향으로 순차 적층 슬라이드. 결과 위치 리스트."""
    g = np.asarray(g).copy(); H, W = g.shape
    dy, dx = d
    for cs in movers:
        for y, x in cs: g[y, x] = bg
    def depth(cs):
        if dy == 1: return -max(y for y, _ in cs)
        if dy == -1: return min(y for y, _ in cs)
        if dx == 1: return -max(x for _, x in cs)
        return min(x for _, x in cs)
    order = sorted(range(len(movers)), key=lambda j: depth(movers[j]))
    res = [None] * len(movers)
    for j in order:
        cur = set(movers[j])
        while True:
            nxt = {(y+dy, x+dx) for y, x in cur}
            if any(not (0 <= y < H and 0 <= x < W) for y, x in nxt): break
            if any(g[y, x] != bg for y, x in nxt): break
            cur = nxt
        res[j] = frozenset(cur)
        for y, x in cur: g[y, x] = 9 if bg != 9 else 8   # 장애물 표시(색 무관)
    return res

def dv_objdiff(pairs):
    diff = _OD_diff(pairs)
    if diff is None: return None
    fates = [f for _, recs in diff for (_,_,_,f,_) in recs]
    if set(fates) == {'K'}: return None
    # ---- 2단 파라미터 학습 재료 ----
    R = [(col, p, prm) for _, recs in diff for (col,_,p,f,prm) in recs if f=='R']
    M = [(prm) for _, recs in diff for (_,_,_,f,prm) in recs if f=='M']
    # recolor 모드
    rmodes = []
    if R:
        cs = {prm for _,_,prm in R}
        if len(cs) == 1: rmodes.append(('const', cs.pop()))
        cmap = {}
        ok = all(not (col in cmap and cmap[col] != prm) and not cmap.update({col: prm})
                 for col, _, prm in R)
        cmap = {}; ok = True
        for col, _, prm in R:
            if col in cmap and cmap[col] != prm: ok = False; break
            cmap[col] = prm
        if ok: rmodes.append(('cmap', dict(cmap)))
        for pk in ('size', 'holes', 'sil'):
            pmap = {}; ok = True
            for _, p, prm in R:
                v = p[pk]
                if v in pmap and pmap[v] != prm: ok = False; break
                pmap[v] = prm
            if ok: rmodes.append(('pmap', pk, dict(pmap)))
        # 관계 재색 1: 새 색 = 쌍둥이(같은 실루엣, 다른 색)의 색
        #   예) 776ffc46: 틀 안 2-블록이 견본 -> 같은 모양 1-블록들이 2로
        if all(p.get('twin', -1) == prm for _, p, prm in R):
            rmodes.append(('twinc',))
        # 관계 재색 2: 새 색 = 쌍둥이를 "품은 틀"의 색 (twin_inbb)
        #   예) 63613498: 견본 3-도형이 5-모서리 bbox 안 -> 9-도형이 5로
        if all(p.get('twin_inbb', -1) == prm for _, p, prm in R):
            rmodes.append(('twinhost',))
    else: rmodes = [None]
    # move 모드
    mmodes = []
    if M:
        vs = set(M)
        if len(vs) == 1: mmodes.append(('vec', vs.pop()))
        dirs = set()
        for dy, dx in M:
            if dy and dx: dirs.add(None)
            elif dy: dirs.add((1,0) if dy>0 else (-1,0))
            elif dx: dirs.add((0,1) if dx>0 else (0,-1))
        if len(dirs) == 1 and None not in dirs:
            mmodes.append(('slide', dirs.pop()))
        if None not in dirs:
            mmodes.append(('slide_near', None))
    else: mmodes = [None]
    if R and not rmodes: return None
    if M and not mmodes: return None

    fns = []
    for key in _OD_KEYS:
        fmap = {}; ok = True
        for _, recs in diff:
            for (_,_,p,f,_) in recs:
                v = _OD_get(p, key)
                if v in fmap and fmap[v] != f: ok = False; break
                fmap[v] = f
            if not ok: break
        if not ok or set(fmap.values()) == {'K'}: continue
        for rm in (rmodes if R else [None]):
            for mm in (mmodes if M else [None]):
                def fn(g, _fm=dict(fmap), _k=key, _rm=rm, _mm=mm):
                    g = np.asarray(g); bg = _OD_bg(g)
                    objs = _OD_props(_OD_objs(g, bg), g.shape)
                    out = g.copy()
                    movers = []; mcols = []
                    for col, cs, p in objs:
                        f = _fm.get(_OD_get(p, _k), 'K')
                        if f == 'K': continue
                        for y, x in cs: out[y, x] = bg
                        if f == 'D': continue
                        if f == 'R':
                            if _rm is None: raise ValueError('no rmode')
                            if _rm[0] == 'const': nc = _rm[1]
                            elif _rm[0] == 'cmap': nc = _rm[1].get(col)
                            elif _rm[0] == 'twinc':
                                # 새 색 = 쌍둥이의 색 (test 그리드에서 실시간 계산)
                                nc = p.get('twin', -1)
                                if nc == -1: raise ValueError('twin 없음')
                            elif _rm[0] == 'twinhost':
                                # 새 색 = 쌍둥이를 품은 틀의 색
                                nc = p.get('twin_inbb', -1)
                                if nc == -1: raise ValueError('twin 틀 없음')
                            else: nc = _rm[2].get(p[_rm[1]])
                            if nc is None: raise ValueError('rmap miss')
                            for y, x in cs: out[y, x] = nc
                        elif f == 'M':
                            if _mm is None: raise ValueError('no mmode')
                            movers.append(cs); mcols.append(col)
                    if movers:
                        if _mm[0] == 'vec':
                            dy, dx = _mm[1]; H, W = g.shape
                            for cs, col in zip(movers, mcols):
                                ncs = [(y+dy, x+dx) for y, x in cs]
                                if any(not (0<=y<H and 0<=x<W) for y, x in ncs):
                                    raise ValueError('oob')
                                for y, x in ncs: out[y, x] = col
                        elif _mm[0] == 'slide':
                            base = out.copy()
                            res = _OD_sim_slide(base, bg, movers, _mm[1])
                            for cs, col in zip(res, mcols):
                                for y, x in cs: out[y, x] = col
                        else:                       # slide_near
                            base = out.copy()
                            H, W = base.shape
                            for cs, col in zip(movers, mcols):
                                bestd = None; bestk = None
                                for dv2 in ((-1,0),(1,0),(0,-1),(0,1)):
                                    dy, dx = dv2; cur = set(cs); k = 0
                                    hit = False
                                    while True:
                                        nxt = {(y+dy, x+dx) for y, x in cur}
                                        if any(not (0<=y<H and 0<=x<W)
                                               for y, x in nxt): break
                                        if any(base[y, x] != bg for y, x in nxt):
                                            hit = True; break
                                        cur = nxt; k += 1
                                    if hit:
                                        if bestk is None or k < bestk:
                                            bestk, bestd = k, dv2
                                        elif k == bestk: bestd = 'tie'
                                if bestd is None or bestd == 'tie':
                                    raise ValueError('slide_near ambiguous')
                                dy, dx = bestd; cur = set(cs)
                                for _ in range(bestk):
                                    cur = {(y+dy, x+dx) for y, x in cur}
                                for y, x in cs:
                                    if (y, x) not in cur: out[y, x] = bg
                                for y, x in cur: out[y, x] = col
                                base = out.copy()
                    return out
                good = True
                for e in pairs:
                    try:
                        if not np.array_equal(fn(e['input']),
                                              np.asarray(e['output'])):
                            good = False; break
                    except Exception:
                        good = False; break
                if good: fns.append(fn)
                if len(fns) >= 12: break
            if len(fns) >= 12: break
        if len(fns) >= 12: break
    return fns or None

if 'STAGES' in globals():
    STAGES['objdiff'] = dv_objdiff
    TOOLS_FINE = sorted(set(TOOLS_FINE) | {'objdiff'})
    print(f"도형 diff v4 추가 -> 도구 총 {len(TOOLS_FINE)}개")


# ==========================================================================
# ==== [8/12: objmove (도형diff x 무브법칙)] ====
# ==========================================================================

# ============================================================
# [셀2j - objmove 도구] objdiff(누가) x 무브엔진(어떻게) 결합 (도구 +1)
#
# ── 만든 이유 ──────────────────────────────────────────────
#   무브엔진(moveX)의 법칙(tetris/domino/mirror/edge/slide/vec...)은
#   강력하지만, "어느 도형이 움직이는가"의 선택기가 5종뿐이라
#   (all/attached/largest/second/colors) 속성 기반 선택이 안 됨.
#   반대로 objdiff는 속성 키(크기순위/구멍수/테두리접촉/복합키...)로
#   움직일 도형을 정확히 찾지만, 이동 법칙이 3종뿐
#   (공통벡터/공통방향 슬라이드/slide_near).
#   => 이 도구는 objdiff의 1단(운명 사전 = 누가)과
#      무브엔진의 법칙 전수(= 어떻게)를 이어붙인 다리(bridge).
#
# ── 동작 순서 ──────────────────────────────────────────────
#   1) objdiff의 _OD_diff 로 train 쌍마다 도형 대응을 만들어
#      각 도형의 운명(K=유지, M=이동)을 얻는다.
#      (재색 R/삭제 D 가 섞인 문제는 objdiff 담당이므로 여기선 포기)
#   2) objdiff 1단과 동일하게 "속성 키 -> 운명" 사전(fate map)을 찾는다.
#      예: szrank(크기순위)=1 -> M, 나머지 -> K
#          holes(구멍수)=0 -> M / border(테두리접촉)=False -> M ...
#   3) 무브엔진의 법칙 후보 전수를 만들고(_law_apply 재사용),
#      "fate map 이 고른 movers + 법칙" 조합이 모든 train 쌍을
#      정확 재현하는지 검사한다. 통과한 조합만 후보.
#   4) test 적용: test 도형들의 속성 -> fate map 으로 movers 선정
#      -> 법칙 적용 -> movers 지우고 새 위치에 그림.
#
# ── 안전장치 ──────────────────────────────────────────────
#   - 도형 대응 실패(복제/변형/배경변화/설명불가) 시 도출 포기
#   - train "완전 재현" 조합만 후보 (부분 일치 없음)
#   - 채택 심판은 프레임워크 공통 train-exact + LOO
#   - 법칙 순위는 무브엔진과 동일하게 구조 우선(_ME_law_rank):
#     tetris/domino/mirror 같은 구조 법칙 > vec 같은 암기성 법칙.
#     (vec 를 먼저 채택하면 fold 마다 벡터가 달라 LOO 에서 깨지는
#      실측 문제가 있었음 - 무브엔진 개발 때 확인)
#
# ── 셀 순서 ───────────────────────────────────────────────
#   셀2e(무브 엔진)와 셀2i(도형 diff) "다음"에 실행해야 함.
#   (아래 assert 가 의존성을 검사)
# ============================================================
import numpy as np

# 의존성 검사: 무브엔진(셀2e)과 도형 diff(셀2i)가 먼저 로드돼야 함
assert '_law_apply' in globals(), "셀2e(무브 엔진) 먼저 실행"
assert '_ME_law_rank' in globals(), "셀2e(무브 엔진) 먼저 실행"
assert '_OD_diff' in globals(), "셀2i(도형 diff) 먼저 실행"
assert '_OD_props' in globals(), "셀2i(도형 diff) 먼저 실행"

def _OM_anchors(g, bg, movers):
    """비무버(닻 후보) 도형 목록. 관계 이동 법칙의 기준점들."""
    mv_cells = set()
    for _c, cs in movers:
        mv_cells |= set(cs)
    out = []
    for col, cs in _OD_objs(np.asarray(g), bg):
        if set(cs) & mv_cells:
            continue
        out.append((col, cs))
    return out

def _OM_anchor_for(mover_cs, anchors, amode):
    """이 mover 의 닻 하나 고르기.
      'big'  = 전역 최대 비무버 도형 (기존 동작 - 상자 1개 문제)
      'near' = 이 mover 에서 가장 가까운 비무버 도형
               (1b8318e3 처럼 55블록이 여러 개고 점이 제 블록을 찾아가는 문제)
    거리 = 셀 간 체비셰프 최소거리."""
    if not anchors:
        return None
    if amode == 'big':
        return max(anchors, key=lambda a: len(a[1]))
    def dist(a):
        return min(max(abs(y-ay), abs(x-ax))
                   for y, x in mover_cs for ay, ax in a[1])
    return min(anchors, key=dist)

def _OM_law_apply(law, g, bg, movers):
    """신규 관계 법칙(anchor_go)은 여기서 처리, 나머지는 무브엔진 위임.

    anchor_go (닻 방향 직진): 각 mover 가 닻 쪽으로 한 칸씩 직진.
      방향 = 범위 겹침 기준:
        mover 의 행 범위가 닻의 행 범위와 겹치면 세로성분 0 (수평 직진)
        열 범위가 겹치면 가로성분 0 (수직 직진)
        둘 다 안 겹치면 대각선 (부호 = 닻 중심 쪽)
      정지 조건 3변형 (law[1]):
        'orth' = 닻과 상하좌우 인접(4방향 접촉) 직전 정지  (자석-직각)
        'cheb' = 닻과 대각 포함 인접(8방향 접촉) 직전 정지 (자석-대각)
        'exit' = 닻 반대편으로 "도약": 닻 중심 방향 부호 x bbox 스팬만큼
                 축별 평행이동. (상자 안 점이 반대편 밖 대각 위치로 나가는
                 문제 - 비정사각 상자면 dy 와 dx 가 달라서 스텝 이동이 아닌
                 도약 공식이어야 함을 952a094c 쌍2 에서 실측)
      실측 근거: a48eeaf7(cheb), 952a094c(exit) 훈련 그림에서 도출
    """
    kind = law[0]
    if kind != 'anchor_go':
        return _law_apply(law, g, bg, movers)          # 무브엔진 위임
    stop = law[1]
    amode = law[2] if len(law) > 2 else 'big'          # 닻 선택 모드
    g = np.asarray(g); H, W = g.shape
    anchors = _OM_anchors(g, bg, movers)
    if not anchors:
        return None
    out = []
    for c, cs in movers:
        # mover 마다 닻을 고른다 ('big'=전역 최대 1개, 'near'=최근접)
        anc = _OM_anchor_for(cs, anchors, amode)
        if anc is None:
            return None
        aset = set(anc[1])
        ays = [y for y, _ in aset]; axs = [x for _, x in aset]
        ay0, ay1, ax0, ax1 = min(ays), max(ays), min(axs), max(axs)
        mys = [y for y, _ in cs]; mxs = [x for _, x in cs]
        my0, my1 = min(mys), max(mys); mx0, mx1 = min(mxs), max(mxs)
        # 방향 계산 - 모드별로 다름:
        #  접촉 모드(orth/cheb): 범위가 겹치는 축은 0 (직진), 아니면 닻 쪽 부호
        #    (밖에 있는 점이 닻과 같은 행/열이면 그 축으로만 직진하는 실측 규칙)
        #  관통 모드(exit): 닻 중심 쪽 부호 (중심을 지나 반대편으로 관통.
        #    mover 가 닻 "안"에 있으면 범위가 겹쳐 부호가 0 이 되므로 중심 기준)
        if stop == 'exit':
            # 도약 공식: 방향 부호(닻 중심 쪽) x 축별 bbox 스팬
            cy2, cx2 = ay0 + ay1, ax0 + ax1            # 중심 x2 (정수 유지)
            my2, mx2 = my0 + my1, mx0 + mx1
            sy = 0 if cy2 == my2 else (1 if cy2 > my2 else -1)
            sx = 0 if cx2 == mx2 else (1 if cx2 > mx2 else -1)
            if sy == 0 and sx == 0:
                return None
            dy = sy * (ay1 - ay0); dx = sx * (ax1 - ax0)
            ncs = frozenset((y + dy, x + dx) for y, x in cs)
            if any(not (0 <= y < H and 0 <= x < W) for y, x in ncs):
                return None                             # 판 밖 -> 실패
            if ncs & aset:
                return None                             # 닻과 겹침 -> 실패
            out.append((c, ncs)); continue
        else:
            row_ov = not (my1 < ay0 or ay1 < my0)      # 행 범위 겹침?
            col_ov = not (mx1 < ax0 or ax1 < mx0)      # 열 범위 겹침?
            sy = 0 if row_ov else (1 if ay0 > my1 else -1)
            sx = 0 if col_ov else (1 if ax0 > mx1 else -1)
        if sy == 0 and sx == 0:
            return None                                 # 이미 겹침 -> 정의 불가
        cur = set(cs); ok = False
        def touching(cells):
            if stop == 'orth':
                return any((y+dy, x+dx) in aset for y, x in cells
                           for dy, dx in ((1,0),(-1,0),(0,1),(0,-1)))
            return any(abs(y-ay) <= 1 and abs(x-ax) <= 1
                       for y, x in cells for ay, ax in aset)
        if stop in ('orth', 'cheb') and touching(cur):
            return None                                 # 이미 접촉 상태
        for _ in range(H + W):
            nxt = {(y+sy, x+sx) for y, x in cur}
            if any(not (0 <= y < H and 0 <= x < W) for y, x in nxt):
                return None                             # 정지 전에 판 밖
            if stop in ('orth', 'cheb'):
                if nxt & aset:
                    return None                         # 관통 금지 (접촉 모드)
                cur = nxt
                if touching(cur):
                    ok = True; break                    # 닿았다 -> 정지
            else:                                       # 'exit' = 관통 모드
                cur = nxt
                entered = True                          # bbox 밖 판정
                if all(not (ay0 <= y <= ay1 and ax0 <= x <= ax1)
                       for y, x in cur) and not (cur & aset):
                    # 완전히 bbox 밖 + 닻과 안 겹침 -> 그런데 "출발부터 밖"이면
                    # 관통이 아님: 최소 1칸 이동 후 bbox 를 넘어야 함
                    start_out = all(not (ay0 <= y <= ay1 and ax0 <= x <= ax1)
                                    for y, x in cs)
                    if start_out:
                        # 출발이 밖이면: bbox 를 지나쳐 반대편으로 나갔을 때만 정지
                        crossed = ((sy > 0 and min(y for y, _ in cur) > ay1) or
                                   (sy < 0 and max(y for y, _ in cur) < ay0) or
                                   (sy == 0)) and                                   ((sx > 0 and min(x for _, x in cur) > ax1) or
                                   (sx < 0 and max(x for _, x in cur) < ax0) or
                                   (sx == 0))
                        if not crossed:
                            continue
                    ok = True; break
        if not ok:
            return None
        out.append((c, frozenset(cur)))
    return out

def _OM_laws(ins, outs, bgs, movers_per_pair):
    """무브엔진 dm_moveX 와 같은 법칙 후보 목록을 만든다.
    (dm_moveX 안에 인라인으로 있어 재사용이 안 되므로 같은 규칙으로 재구성)
    movers_per_pair: 쌍마다 [(색, cells)] - fate map 이 고른 movers 관측값
    """
    mv_colors = set()
    for mv in movers_per_pair:
        mv_colors |= {c for c, _s in mv}
    all_colors = set()
    for i in ins:
        all_colors |= {int(v) for v in np.unique(i)}

    laws = [('surface',), ('tetris',),
            # anchor_go: (정지조건, 닻모드) 전 조합.
            #   닻모드 big = 전역 최대 도형 (상자 1개 문제: 952a094c 등)
            #   닻모드 near = mover 별 최근접 도형 (블록 여러 개: 1b8318e3 등)
            ('anchor_go', 'orth', 'big'),
            ('anchor_go', 'cheb', 'big'),
            ('anchor_go', 'exit', 'big'),
            ('anchor_go', 'orth', 'near'),
            ('anchor_go', 'cheb', 'near'),
            ('anchor_go', 'exit', 'near')]
    for _p in ('t', 'b', 'l', 'r', 'tl', 'tr', 'bl', 'br', 'cy', 'cx'):
        laws.append(('edge', _p))                      # 10위치 가장자리 정렬
    for _dd in ('right', 'left'):
        laws.append(('domino', _dd, None))             # 도미노(연쇄 밀기)
        for _tc in sorted(mv_colors):                  # 트리거색 = mover 색만
            laws.append(('domino', _dd, _tc))          #  (배경색 트리거는
    for d in _DIRS:                                    #   스퓨리어스 실측)
        for order in ('near', 'far'):
            laws.append(('slide', d, 'obstacle', order))   # 장애물까지 슬라이드
            for m in sorted(all_colors):
                laws.append(('slide', d, m, order))        # 특정색까지 슬라이드
        for k in (1, 2, 3):
            for u in ('h', 'w'):
                laws.append(('own', d, k, u))          # 자기 크기 배수만큼 이동
    for m in sorted(all_colors):
        for mode in ('row', 'col', 'point'):
            laws.append(('mirror', m, mode))           # 거울(행/열/점 대칭)

    # 관측 벡터 법칙: 쌍1 movers 가 실제로 놓일 수 있는 평행이동 전수
    i0, o0, b0 = ins[0], outs[0], bgs[0]
    mv0 = movers_per_pair[0]
    vecs = set()
    for c, s in list(mv0)[:4]:
        base = sorted(s); H, W = o0.shape
        for oy in range(-H + 1, H):
            for ox in range(-W + 1, W):
                ns = [(y + oy, x + ox) for y, x in base]
                if all(0 <= y < H and 0 <= x < W and
                       int(o0[y, x]) == c for y, x in ns):
                    vecs.add((oy, ox))
        if len(vecs) > 40:
            break
    for v in sorted(vecs):
        if v != (0, 0):
            laws.append(('vec', v))                    # 상수 벡터 (순위 최하)

    # 색깔별 벡터: 색 c 의 도형은 항상 벡터 v_c 로 이동 (쌍1 관측)
    cvec = {}; cv_ok = True
    for c, s in mv0:
        found = None; base = sorted(s); H, W = o0.shape
        for oy in range(-H + 1, H):
            if found: break
            for ox in range(-W + 1, W):
                ns = [(y + oy, x + ox) for y, x in base]
                if all(0 <= y < H and 0 <= x < W and
                       int(o0[y, x]) == c for y, x in ns):
                    found = (oy, ox); break
        if found is None: cv_ok = False; break
        if c in cvec and cvec[c] != found: cv_ok = False; break
        cvec[c] = found
    if cv_ok and cvec:
        laws.append(('cvec', dict(cvec)))
    # 구조 우선 순위로 정렬 (무브엔진과 동일 기준.
    #  신규 관계 법칙은 mirror(2)급 = 구조 법칙으로 취급)
    def _rank(law):
        if law[0] == 'anchor_go':
            return 2
        try:
            return _ME_law_rank(law)
        except Exception:
            return 9
    laws.sort(key=_rank)
    return laws

def dv_objmove(pairs):
    import time as _t
    dl = _t.time() + 6.0                               # 문제당 상한

    # ---- 1) 도형 대응: 운명(K/M)만 있는 문제로 한정 ----
    diff = _OD_diff(pairs)                             # 셀2i 재사용
    if diff is None:
        return None                                    # 대응 실패 -> 포기
    fates = {f for _, recs in diff for (_, _, _, f, _) in recs}
    if 'M' not in fates or not fates <= {'K', 'M'}:
        return None                                    # R/D 섞임 -> objdiff 몫

    ins = [g for g, _ in diff]
    outs = [np.asarray(e['output']) for e in pairs]
    bgs = [_OD_bg(i) for i in ins]

    # ---- 2) fate map 후보: 속성 키가 K/M 을 일관되게 결정하는지 ----
    fate_keys = []
    for key in _OD_KEYS:                               # 단일 + 복합 키 전수
        fmap = {}; ok = True
        for _, recs in diff:
            for (_, _, p, f, _) in recs:
                v = _OD_get(p, key)
                if v in fmap and fmap[v] != f: ok = False; break
                fmap[v] = f
            if not ok: break
        if ok and 'M' in fmap.values():
            fate_keys.append((key, dict(fmap)))
    if not fate_keys:
        return None                                    # 누가 움직이는지 설명 불가

    # ---- 3) (fate map x 법칙) 전수 검증 ----
    # 관측 movers: fate map 과 무관하게 diff 에서 직접 (모든 키 공통)
    movers_pp = []
    for _, recs in diff:
        movers_pp.append([(col, cs) for (col, cs, _, f, _) in recs if f == 'M'])
    laws = _OM_laws(ins, outs, bgs, movers_pp)

    fns = []
    for key, fmap in fate_keys:
        if _t.time() > dl: break
        for law in laws:
            if _t.time() > dl: break
            good = True
            for (i, recs), o, bg, mv_obs in zip(diff, outs, bgs, movers_pp):
                # train 에서는 fate map 선택 == 관측 movers (사전 구성상 동일)
                try:
                    tg = _OM_law_apply(law, i, bg, mv_obs)  # 신규+셀2e 법칙
                    if tg is None: good = False; break
                    res = _paint(i, bg, mv_obs, tg)          # 지우고 그리기
                    if res is None or not np.array_equal(res, o):
                        good = False; break
                except Exception:
                    good = False; break
            if not good:
                continue
            # ---- 4) test 적용 함수: 속성 -> fate map -> movers -> 법칙 ----
            def fn(g, _fm=dict(fmap), _k=key, _law=law):
                g = np.asarray(g); bg = _OD_bg(g)
                objs = _OD_props(_OD_objs(g, bg), g.shape)   # 셀2i 속성 재사용
                movers = [(col, cs) for col, cs, p in objs
                          if _fm.get(_OD_get(p, _k), 'K') == 'M']
                if not movers:
                    raise ValueError('movers 없음')          # 미지 속성값 -> 포기
                tg = _OM_law_apply(_law, g, bg, movers)
                if tg is None:
                    raise ValueError('법칙 적용 불가')
                res = _paint(g, bg, movers, tg)
                if res is None:
                    raise ValueError('경계 밖')
                return res
            fns.append(fn)
            break                       # 이 fate map 은 최상위 법칙 하나만
        if len(fns) >= 6:
            break
    return fns or None

# ---- 도구 등록 ----
STAGES['objmove'] = dv_objmove
TOOLS_FINE = sorted(set(TOOLS_FINE) | {'objmove'})
print(f"objmove(도형diff x 무브법칙) 추가 -> 도구 총 {len(TOOLS_FINE)}개")


# ==========================================================================
# ==== [9/12: grid_mirror (거울 완성)] ====
# ==========================================================================

# ============================================================
# [셀2k - 그리드/패턴 도구] grid_mirror (모티프 성분 거울 완성)
#
# ── 최종 규칙 (2546ccf6 train+test 실측으로 3회 수정 끝에 확정) ──
#   구분선이 판을 패널로 나눌 때, "무늬(모티프)"는 무늬가 있는
#   패널들의 연결 성분(상하좌우 이웃)이다. 각 모티프는 자기
#   밴드-사각형(행밴드 x 열밴드 bbox) 안에서 구분선을 거울삼아
#   대칭이다. 완성 = 그 bbox 안 거울 궤도 합집합.
#
#   실패 교훈 3가지 (모두 실측):
#     v1 전면 반사      -> 먼 패널까지 번짐 (train 재현 실패)
#     v2 인접 짝 완성   -> 궤도 4번째 칸(대각) 누락
#     v3 고정 2x2 그룹  -> 서로 다른 모티프의 절반씩 걸친 "가짜
#        그룹"이 활성화되어 무늬가 남의 밴드로 번짐 (test 오답)
#   => 완성 단위를 "연결 성분의 bbox"로 (이 파일).
#      성분 bbox 가 3밴드 이상으로 크면 그 성분은 포기(보수적).
#
#   안전장치: 궤도 안 색 충돌 시 성분 포기 + train 완전 재현 + LOO
#
# ── 위치와 실측 ───────────────────────────────────────────
#   위치: 셀2j(objmove) 다음, 탐색 셀 앞.
#   실측 (프레임워크 동일 "첫 후보" LOO 기준):
#     training 1000: 정답 1 (2546ccf6), 오답채택 0 / evaluation: 0
#   추가 교훈: 성분 연결에 "같은 색 집합" 조건 필수 -
#     다른 색 무늬 패널이 대각 사슬로 이어져 한 성분이 되면
#     bbox 가 커져 전부 포기되는 버그 실측.
#   남은 그리드 표적(15113be4, 42a15761, cbded52d 등)은 규칙이
#   달라 별도 도구 필요 - 다음 라운드.
# ============================================================
import numpy as np
from collections import Counter

def _GM_bg(g):
    v, c = np.unique(g, return_counts=True)
    return int(v[c.argmax()])

def _GM_seps(g):
    """구분선 검출: 같은 색으로 꽉 찬 행/열 중 최다 색. None=없음"""
    g = np.asarray(g); H, W = g.shape
    rows = [y for y in range(H) if (g[y] == g[y, 0]).all()]
    cols = [x for x in range(W) if (g[:, x] == g[0, x]).all()]
    colors = [int(g[y, 0]) for y in rows] + [int(g[0, x]) for x in cols]
    if not colors: return None
    sep_color = Counter(colors).most_common(1)[0][0]
    rs = [y for y in rows if int(g[y, 0]) == sep_color]
    cs = [x for x in cols if int(g[0, x]) == sep_color]
    if not rs or not cs: return None           # 양방향 격자만 (보수적)
    return rs, cs, sep_color

def _GM_bands(size, seps):
    cuts = [-1] + sorted(seps) + [size]
    return [(a + 1, b - 1) for a, b in zip(cuts[:-1], cuts[1:]) if b - 1 >= a + 1]

def dk_grid_mirror(pairs):
    for e in pairs:
        i = np.asarray(e['input']); o = np.asarray(e['output'])
        if i.shape != o.shape: return None
        sp = _GM_seps(i)
        if sp is None: return None
        # 밴드 사전 검사: 구분선이 가장자리에만 있으면 밴드가 2개 미만이라
        #   fn 이 도출 중 예외를 던져 탐색 셀 전체가 죽는 사고 실측(캐글 7차).
        #   도출기는 "절대" 예외를 밖으로 내면 안 됨 -> 여기서 미리 거른다.
        if len(_GM_bands(i.shape[0], sp[0])) < 2: return None
        if len(_GM_bands(i.shape[1], sp[1])) < 2: return None
    def fn(g):
        g = np.asarray(g).copy(); H, W = g.shape
        bg = _GM_bg(g)
        sp = _GM_seps(g)
        if sp is None: raise ValueError('구분선 없음')
        rs, cs, _sc = sp
        rb = _GM_bands(H, rs); cb = _GM_bands(W, cs)
        if len(rb) < 2 or len(cb) < 2: raise ValueError('밴드 부족')
        # ---- 1) 패널별 무늬 색 집합 (원본 기준) ----
        #   성분 연결에 "같은 색 집합" 조건이 필수임을 실측:
        #   2546ccf6 에서 3-무늬와 1-무늬 패널이 대각 사슬로 이어져
        #   한 성분(bbox 4x3)이 되는 바람에 전부 포기되던 버그.
        PCOL = {}
        for bi, (y0, y1) in enumerate(rb):
            for bj, (x0, x1) in enumerate(cb):
                s = frozenset(int(g[y, x]) for y in range(y0, y1 + 1)
                              for x in range(x0, x1 + 1) if g[y, x] != bg)
                if s: PCOL[(bi, bj)] = s
        # ---- 2) 무늬 패널의 연결 성분 (상하좌우 + 같은 색 집합) ----
        seen = set(); comps = []
        for key in PCOL:
            if key in seen: continue
            st = [key]; seen.add(key); comp = []
            while st:
                a, b = st.pop(); comp.append((a, b))
                for da, db in ((1,0),(-1,0),(0,1),(0,-1)):
                    nk = (a + da, b + db)
                    if nk in PCOL and nk not in seen \
                            and PCOL[nk] == PCOL[key]:
                        seen.add(nk); st.append(nk)
            comps.append(comp)
        # ---- 3) 성분별 bbox 궤도 완성 ----
        for comp in comps:
            bis = sorted({a for a, _ in comp}); bjs = sorted({b for _, b in comp})
            bi0, bi1 = bis[0], bis[-1]; bj0, bj1 = bjs[0], bjs[-1]
            if bi1 - bi0 > 1 or bj1 - bj0 > 1:
                continue                        # 3밴드 이상 성분은 포기 (보수적)
            # bbox 안의 거울선: 두 행밴드 사이 구분선 / 두 열밴드 사이 구분선
            rline = rb[bi0][1] + 1 if bi1 > bi0 else None
            cline = cb[bj0][1] + 1 if bj1 > bj0 else None
            if rline is None and cline is None:
                continue                        # 1패널 성분 - 완성할 거울 없음
            y0, y1 = rb[bi0][0], rb[bi1][1]
            x0, x1 = cb[bj0][0], cb[bj1][1]
            fills = []; ok = True
            for y in range(y0, y1 + 1):
                if rline is not None and y >= rline: break   # 반쪽만 돌면 충분
                for x in range(x0, x1 + 1):
                    if cline is not None and x >= cline: break
                    orbit = [(y, x)]
                    if rline is not None:
                        my = 2 * rline - y
                        if y0 <= my <= y1: orbit.append((my, x))
                    if cline is not None:
                        mx = 2 * cline - x
                        if x0 <= mx <= x1: orbit.append((y, mx))
                    if rline is not None and cline is not None:
                        my = 2 * rline - y; mx = 2 * cline - x
                        if y0 <= my <= y1 and x0 <= mx <= x1:
                            orbit.append((my, mx))
                    orbit = [p for p in orbit
                             if g[p] != _sc or True]          # (구분선 칸 없음)
                    vals = {int(g[p]) for p in orbit} - {bg}
                    if len(vals) > 1:
                        ok = False; break                     # 궤도 색 충돌
                    if len(vals) == 1:
                        v = vals.pop()
                        for p in orbit:
                            if g[p] == bg: fills.append((p, v))
                if not ok: break
            if ok:
                for p, v in fills: g[p] = v
        return g
    try:                                        # train 완전 재현 필수
        for e in pairs:                         # (도출 중 예외 = 도출 실패로
            if not np.array_equal(fn(e['input']),   # 처리 - 절대 밖으로 안 냄)
                                  np.asarray(e['output'])):
                return None
    except Exception:
        return None
    return fn

if 'STAGES' in globals():
    STAGES['grid_mirror'] = dk_grid_mirror
    TOOLS_FINE = sorted(set(TOOLS_FINE) | {'grid_mirror'})
    print(f"그리드/패턴 도구 추가 -> 도구 총 {len(TOOLS_FINE)}개")


# ==========================================================================
# ==== [10/12: template_complete + snake_advance] ====
# ==========================================================================

# ============================================================
# [셀2l - 견본 완성 도구] template_complete (도구 +1)
#
# ── 규칙 (fd096ab6 실측으로 도출) ─────────────────────────
#   판에 "완성된 견본 도형"(칸 수 최대인 색 모티프) 하나와,
#   그 견본의 "부분 조각"인 다른 색 모티프들이 있다.
#   조각을 견본 모양에 평행이동으로 겹쳐 맞춘 뒤(부분집합 정합),
#   빠진 칸을 그 조각의 색으로 채워 견본 모양을 완성한다.
#
#   모티프 = "색 단위" 셀 집합 (연결성 무시 - 견본에 대각 연결
#   조각이 흔해서 4-연결로 나누면 견본이 쪼개지는 실측 문제)
#
#   정합 안전장치:
#     - 조각 -> 견본 정합 평행이동이 "유일"해야 채움
#       (2곳 이상 맞으면 모호 -> 그 조각 포기)
#     - 조각이 견본과 같은 크기면 이미 완성 -> 건드리지 않음
#   도출 안전장치:
#     - 도출 중 어떤 예외도 밖으로 내지 않음 (grid_mirror 사고 교훈)
#     - train 완전 재현 + LOO (프레임워크 공통)
# ============================================================
import numpy as np

def _TC_bg(g):
    v, c = np.unique(g, return_counts=True)
    return int(v[c.argmax()])

def _TC_motifs(g, bg):
    """모티프 = "같은 색 8-연결 성분" 목록 [(색, 셀집합)].
    v1은 색 단위로 묶었는데, 같은 색 조각이 2개인 test 에서
    합쳐져 정합이 깨지는 실측 버그 (fd096ab6 test) -> 8-연결로 분리.
    (견본의 대각 연결 조각은 8-연결이라 그대로 한 덩어리 유지)"""
    g = np.asarray(g); H, W = g.shape
    seen = np.zeros((H, W), bool); out = []
    for r in range(H):
        for c in range(W):
            if seen[r, c] or g[r, c] == bg: continue
            col = int(g[r, c]); st = [(r, c)]; cells = []
            seen[r, c] = True
            while st:
                y, x = st.pop(); cells.append((y, x))
                for dy in (-1, 0, 1):
                    for dx in (-1, 0, 1):
                        ny, nx = y + dy, x + dx
                        if 0 <= ny < H and 0 <= nx < W \
                                and not seen[ny, nx] and g[ny, nx] == col:
                            seen[ny, nx] = True; st.append((ny, nx))
            out.append((col, frozenset(cells)))
    return out

def _TC_rel(cells):
    """왼위(bbox 최소점) 기준 상대 좌표"""
    y0 = min(y for y, _ in cells); x0 = min(x for _, x in cells)
    return frozenset((y - y0, x - x0) for y, x in cells), (y0, x0)

def dk_template_complete(pairs):
    try:
        for e in pairs:
            i = np.asarray(e['input']); o = np.asarray(e['output'])
            if i.shape != o.shape: return None
        def fn(g):
            g = np.asarray(g).copy(); H, W = g.shape
            bg = _TC_bg(g)
            mot = _TC_motifs(g, bg)
            if len(mot) < 2: raise ValueError('모티프 부족')
            # 견본 = 칸 수 최대 성분 (동률이면 모호 -> 포기)
            mot.sort(key=lambda t: -len(t[1]))
            if len(mot[0][1]) == len(mot[1][1]):
                raise ValueError('견본 모호')
            tcol, tcells = mot[0]
            trel, _ = _TC_rel(tcells)
            def try_fit(c, cs):
                """조각 cs 를 견본에 유일 정합시켜 완성. 성공 여부 반환."""
                if len(cs) >= len(trel): return False   # 이미 완성 크기
                crel, (cy0, cx0) = _TC_rel(cs)
                fits = []
                for oy in range(0, max(y for y, _ in trel) + 1):
                    for ox in range(0, max(x for _, x in trel) + 1):
                        if all((y + oy, x + ox) in trel for y, x in crel):
                            fits.append((oy, ox))
                if len(fits) != 1: return False         # 모호/불가 -> 포기
                oy, ox = fits[0]
                ay, ax = cy0 - oy, cx0 - ox             # 견본 앵커 절대 위치
                cells = [(ay + y, ax + x) for y, x in trel]
                if any(not (0 <= y < H and 0 <= x < W) for y, x in cells):
                    return False
                if any(g[y, x] != bg and (y, x) not in cs for y, x in cells):
                    return False                        # 다른 무늬와 충돌
                for y, x in cells: g[y, x] = c
                return True
            # 조각 = 색 단위 (견본 성분 제외).
            #   1순위: 색 전체를 하나의 조각으로 정합
            #     (떨어져 있어도 한 조각인 문제: fd096ab6 쌍1의 6-조각 실측)
            #   2순위(폴백): 8-연결 성분별로 각각 정합
            #     (같은 색 조각이 여러 개인 문제: fd096ab6 test 의 2-조각 실측)
            from collections import defaultdict
            bycol = defaultdict(set)
            for c, cs in mot[1:]:
                bycol[c] |= set(cs)
            for c, allcells in bycol.items():
                if try_fit(c, frozenset(allcells)):
                    continue
                for c2, cs in mot[1:]:
                    if c2 == c:
                        try_fit(c, cs)
            return g
        for e in pairs:                                 # train 완전 재현
            if not np.array_equal(fn(e['input']), np.asarray(e['output'])):
                return None
        return fn
    except Exception:
        return None                                     # 도출기는 예외 금지

if 'STAGES' in globals():
    STAGES['template_complete'] = dk_template_complete
    TOOLS_FINE = sorted(set(TOOLS_FINE) | {'template_complete'})
    print(f"견본 완성 도구 추가 -> 도구 총 {len(TOOLS_FINE)}개")

# ============================================================
# [추가 도구] snake_advance (뱀 전진) - 696d4842 계열
#   판에 "경로"(꺾인 선, 끝점 2개)와 "표식"(다른 색 1칸)이 있다.
#   경로의 한 끝점이 표식과 같은 행/열에 있고 사이가 배경이면:
#     - 끝점에서 표식 직전까지 K칸을 경로 색으로 연장 (전진)
#     - 반대쪽 끝(꼬리)에서 경로를 따라 K칸을 표식 색으로 재색
#   = 뱀이 K칸 전진하고, 지나온 꼬리 자리에 표식 색 흔적이 남는 그림.
#   도출 안전장치: 예외 밖으로 내지 않음 + train 완전 재현 + LOO
# ============================================================
def _SA_comps(g, bg):
    """같은 색 4-연결 성분 [(색, 셀리스트)]"""
    g = np.asarray(g); H, W = g.shape
    seen = np.zeros((H, W), bool); out = []
    for r in range(H):
        for c in range(W):
            if seen[r, c] or g[r, c] == bg: continue
            col = int(g[r, c]); st = [(r, c)]; cells = []
            seen[r, c] = True
            while st:
                y, x = st.pop(); cells.append((y, x))
                for dy, dx in ((1,0),(-1,0),(0,1),(0,-1)):
                    ny, nx = y + dy, x + dx
                    if 0 <= ny < H and 0 <= nx < W and not seen[ny, nx] \
                            and g[ny, nx] == col:
                        seen[ny, nx] = True; st.append((ny, nx))
            out.append((col, cells))
    return out

def _SA_path_order(cells):
    """경로 셀들을 끝점부터 순서대로. 경로가 아니면 None.
    (각 셀 이웃 수: 끝점 2개=1, 나머지=2 인 단순 폴리라인만 인정)"""
    cset = set(cells)
    nbr = {}
    for y, x in cells:
        nbr[(y, x)] = [(y+dy, x+dx) for dy, dx in ((1,0),(-1,0),(0,1),(0,-1))
                       if (y+dy, x+dx) in cset]
    ends = [p for p in cells if len(nbr[p]) == 1]
    if len(ends) != 2: return None
    if any(len(nbr[p]) > 2 for p in cells): return None
    order = [ends[0]]; prev = None
    while len(order) < len(cells):
        cur = order[-1]
        nxt = [p for p in nbr[cur] if p != prev]
        if len(nxt) != 1: return None
        prev = cur; order.append(nxt[0])
    return order

def dk_snake_advance(pairs):
    try:
        for e in pairs:
            i = np.asarray(e['input']); o = np.asarray(e['output'])
            if i.shape != o.shape: return None
        def fn(g):
            g = np.asarray(g).copy(); H, W = g.shape
            bg = _TC_bg(g)
            comps = _SA_comps(g, bg)
            markers = [(c, cs[0]) for c, cs in comps if len(cs) == 1]
            paths = [(c, cs) for c, cs in comps if len(cs) >= 3]
            did = False
            for pc, cells in paths:
                order = _SA_path_order(cells)
                if order is None: continue
                # 두 끝점 각각에 대해 같은 행/열 표식 탐색
                for head_i in (0, -1):
                    hy, hx = order[head_i]
                    hit = None
                    for mc, (my, mx) in markers:
                        if mc == pc: continue
                        if my == hy or mx == hx:
                            dy = 0 if my == hy else (1 if my > hy else -1)
                            dx = 0 if mx == hx else (1 if mx > hx else -1)
                            # 사이가 전부 배경인지 + 전진 방향이 경로의
                            # 머리 진행 방향과 일직선인지 확인
                            steps = []
                            y, x = hy + dy, hx + dx
                            okpath = True
                            while (y, x) != (my, mx):
                                if not (0 <= y < H and 0 <= x < W) \
                                        or g[y, x] != bg:
                                    okpath = False; break
                                steps.append((y, x)); y += dy; x += dx
                            if okpath and steps:
                                hit = (mc, steps); break
                    if hit is None: continue
                    mc, steps = hit
                    K = len(steps)
                    if K >= len(order): continue        # 꼬리보다 길면 포기
                    for y, x in steps: g[y, x] = pc     # 전진
                    tail = order[::-1] if head_i == 0 else order
                    for y, x in tail[:K]: g[y, x] = mc  # 꼬리 재색
                    did = True
                    break
            if not did: raise ValueError('뱀 전진 없음')
            return g
        for e in pairs:                                 # train 완전 재현
            if not np.array_equal(fn(e['input']), np.asarray(e['output'])):
                return None
        return fn
    except Exception:
        return None

if 'STAGES' in globals():
    STAGES['snake_advance'] = dk_snake_advance
    TOOLS_FINE = sorted(set(TOOLS_FINE) | {'snake_advance'})
    print(f"뱀 전진 도구 추가 -> 도구 총 {len(TOOLS_FINE)}개")


# ==========================================================================
# ==== [11/12: wave_decay + midpoint_mark] ====
# ==========================================================================

# ============================================================
# [셀2m - 고유 규칙 도구 2종] wave_decay + midpoint_mark
#
# ── wave_decay (침식 파동) - af726779 계열 ─────────────────
#   입력: 배경 판에 "패턴이 있는 줄" 하나 (행 또는 열).
#   규칙(실측 도출): 다음 세대 = "폭 1짜리 틈"
#     = 마크가 아니면서 양옆(x-1, x+1)이 모두 마크인 위치.
#   세대는 원 줄에서 2줄 간격으로 (공간이 넓은 쪽으로) 진행하고,
#   색은 [신규색, 원색, 신규색, ...] 교대. 신규색은 train 출력에만
#   나타나는 색으로 도출. 세대가 비거나 판을 벗어나면 정지.
#   * 처음엔 '보색' 규칙, '이웃 규칙' 단독을 시도했으나 각각 다른
#     쌍에서 깨졌고, 둘의 교집합 = "폭 1 틈"이 전 쌍 일치 (실측)
#
# ── midpoint_mark (중심 표시) - 11e1fe23 계열 ─────────────
#   입력: 외딴 점 여러 개 (각각 1칸).
#   규칙(실측 도출): 점들의 bbox 중심에 "표시색" 1칸을 찍고,
#   각 점의 색을 중심에서 그 점 방향(부호 단위 벡터)으로 한 칸
#   옆에 찍는다. 표시색(5)은 train 출력에만 나타나는 색으로 도출.
#
# ── 공통 안전장치 ─────────────────────────────────────────
#   도출기는 예외를 밖으로 내지 않음 (grid_mirror 사고 교훈)
#   + train 완전 재현 + LOO (프레임워크 공통)
# ============================================================
import numpy as np

def _WM_bg(g):
    v, c = np.unique(g, return_counts=True)
    return int(v[c.argmax()])

def _WM_newcolor(pairs):
    """train 출력에만 나타나는 색 (모든 쌍 공통, 유일해야 함)"""
    common = None
    for e in pairs:
        i = np.asarray(e['input']); o = np.asarray(e['output'])
        ex = set(int(v) for v in np.unique(o)) - set(int(v) for v in np.unique(i))
        common = ex if common is None else (common & ex)
    if common and len(common) == 1:
        return common.pop()
    return None

# ---------------- wave_decay ----------------
def dk_wave_decay(pairs):
    try:
        newc = _WM_newcolor(pairs)
        if newc is None: return None
        for e in pairs:
            if np.asarray(e['input']).shape != np.asarray(e['output']).shape:
                return None
        def fn(g):
            g = np.asarray(g).copy(); H, W = g.shape
            bg = _WM_bg(g)
            ys, xs = np.where(g != bg)
            if len(ys) == 0: raise ValueError('패턴 없음')
            if len(set(ys.tolist())) == 1:              # 가로 줄 패턴
                r = int(ys[0]); horiz = True
                marks = set(xs.tolist()); L = W
                src = int(g[r, xs[0]])
            elif len(set(xs.tolist())) == 1:            # 세로 줄 패턴
                r = int(xs[0]); horiz = False
                marks = set(ys.tolist()); L = H
                src = int(g[ys[0], r])
            else:
                raise ValueError('한 줄 아님')
            size = H if horiz else W
            step = 2 if (size - 1 - r) >= r else -2     # 넓은 쪽으로 진행
            colors = [newc, src]                        # 교대 시작 = 신규색
            k = 0; pos = r
            while True:
                # 다음 세대 = "폭 1짜리 틈" (양옆이 모두 마크인 비마크)
                nxt = {x for x in range(1, L - 1)
                       if x not in marks and (x - 1) in marks
                       and (x + 1) in marks}
                pos += step
                if not nxt or not (0 <= pos < size): break
                col = colors[k % 2]; k += 1
                for x in nxt:
                    if horiz: g[pos, x] = col
                    else:     g[x, pos] = col
                marks = nxt
            if k == 0: raise ValueError('세대 없음')
            return g
        for e in pairs:
            if not np.array_equal(fn(e['input']), np.asarray(e['output'])):
                return None
        return fn
    except Exception:
        return None

# ---------------- midpoint_mark ----------------
def dk_midpoint_mark(pairs):
    try:
        markc = _WM_newcolor(pairs)
        if markc is None: return None
        for e in pairs:
            if np.asarray(e['input']).shape != np.asarray(e['output']).shape:
                return None
        def fn(g):
            g = np.asarray(g).copy(); H, W = g.shape
            bg = _WM_bg(g)
            ys, xs = np.where(g != bg)
            pts = [(int(y), int(x), int(g[y, x]))
                   for y, x in zip(ys.tolist(), xs.tolist())]
            if not (2 <= len(pts) <= 6): raise ValueError('점 수 부적합')
            pset = {(y, x) for y, x, _ in pts}
            for y, x, _ in pts:
                if any((y+dy, x+dx) in pset
                       for dy, dx in ((1,0),(-1,0),(0,1),(0,-1))):
                    raise ValueError('점 아님')
            y0 = min(y for y, _, _ in pts); y1 = max(y for y, _, _ in pts)
            x0 = min(x for _, x, _ in pts); x1 = max(x for _, x, _ in pts)
            if (y0 + y1) % 2 or (x0 + x1) % 2:
                raise ValueError('중심이 정수 아님')
            cy, cx = (y0 + y1) // 2, (x0 + x1) // 2
            g[cy, cx] = markc
            for y, x, col in pts:
                sy = 0 if y == cy else (1 if y > cy else -1)
                sx = 0 if x == cx else (1 if x > cx else -1)
                ny, nx = cy + sy, cx + sx
                if not (0 <= ny < H and 0 <= nx < W):
                    raise ValueError('판 밖')
                g[ny, nx] = col
            return g
        for e in pairs:
            if not np.array_equal(fn(e['input']), np.asarray(e['output'])):
                return None
        return fn
    except Exception:
        return None

if 'STAGES' in globals():
    STAGES['wave_decay'] = dk_wave_decay
    STAGES['midpoint_mark'] = dk_midpoint_mark
    TOOLS_FINE = sorted(set(TOOLS_FINE) | {'wave_decay', 'midpoint_mark'})
    print(f"고유 규칙 도구 2종 추가 -> 도구 총 {len(TOOLS_FINE)}개")


# ==========================================================================
# ==== [12/12: 승격 연결 (solve_by_orders 교체)] ====
# ==========================================================================

# ============================================================
# [셀2n - 승격 연결 셀] 규칙 스택(도구 69) -> 제출 블록 연결
#   위치: 모든 도구 셀(오더엔진 -> 도구36 -> 2b~2m) "다음",
#         3단계 제출 블록 "앞"에 새 셀로 추가.
#
# ── 하는 일 ───────────────────────────────────────────────
#   1) 셀3c(레퍼런스+점수 탐색)의 문제당 풀이 로직을 함수화한
#      rule_solve(task) 를 만들고,
#   2) 제출 블록의 build_engine_answers_live 가 부르는 이름인
#      solve_by_orders 를 이 함수로 "덮어쓴다".
#      (구 오더 엔진의 순열 탐색 대신 실측 100+/1000 스택이 응답)
#      반환형은 동일: {'pred', 'order', 'loo_conf'} / 실패시 None
#   3) ENGINE_NOPOOL_ALLOW 기본값을 "실측 오답 0 도구 전체"로 확장.
#      - 제출 블록 정책: 규칙 답이 LLM 후보 풀에 있으면 무조건 승격,
#        풀에 없으면 src 도구가 이 목록에 있을 때만 승격.
#      - 실측 근거: training 1000에서 오답채택은 조심계열 2건뿐
#        (1f85a75f part_big_bbox, bb43febb featrule) -> 조심계열 제외.
#        restore 는 조심계열이지만 eval 복원 2건 검증됐던 기존 허용 유지.
#
# ── 안전 원칙 ─────────────────────────────────────────────
#   attempt_1 = kgmon top1 불가침 (제출 블록이 보장)
#   채택 = train-exact + LOO (try_order 공통 심판)
#   이 셀은 rerun/오프라인 모두에서 함수 정의만 함 (실행은 제출 블록이)
# ============================================================
import os as _os, time as _time
import numpy as _np2
from collections import Counter as _Counter2

assert 'try_order' in globals(), "오더 엔진 셀 먼저"
assert 'TOOLS_FINE' in globals() and len(TOOLS_FINE) >= 60, \
    f"도구 셀(2~2m) 먼저 (현재 {len(globals().get('TOOLS_FINE', []))}개)"

# ---- 셀3c 과 동일한 레퍼런스/점수 장치 ----
_RS_REF74 = {
 "up|same|lo":      {"tile": 13, "resize_nn": 3},
 "smaller|diff|lo": {"panels": 13},
 "same|same|lo":    {"obj": 2, "d4_rot180": 2, "d4_transpose": 2,
                     "flow_down": 1, "flow_up": 1, "d4_flipH": 1,
                     "d4_flipV": 1, "flow_left": 1, "color": 1, "d4_rot90": 1},
 "same|sub|lo":     {"period_fill": 3, "obj": 2, "restore": 1},
 "same|diff|lo":    {"recolor_size": 2, "color": 2, "recolor_shape": 1},
 "same|sup|lo":     {"fill_holes": 2, "obj": 1},
 "down|sub|lo":     {"block_mode": 1, "lfill_fill": 1, "part_user4": 1},
 "mixed|sub|lo":    {"restore": 2, "part_unique_color": 1},
 "smaller|sub|lo":  {"part_user4": 1, "part_big_px": 1},
 "smaller|same|hi": {"crop_content": 1, "dedup_rowcol": 1},
 "same|same|hi":    {"connect_dots": 2},
 "smaller|same|lo": {"dedup_rowcol": 2},
 "down|same|lo":    {"resize_nn": 1},
 "mixed|same|hi":   {"dedup_rowcol": 1},
 "same|sub|hi":     {"restore": 1},
 "smaller|sub|hi":  {"restore": 1},
}
_RS_MARG = {"tile": 13, "panels": 13, "obj": 5, "restore": 5, "resize_nn": 4,
            "dedup_rowcol": 4, "period_fill": 3, "color": 3, "fill_holes": 2,
            "part_user4": 2, "connect_dots": 2, "d4_rot180": 2,
            "recolor_size": 2, "d4_transpose": 2}
# 조심계열(거짓 상승/오답채택 실측) - 풀 밖 승격 금지 목록의 기반
_RS_RISKY = {"obj", "scale", "recolor_shape", "part_big_bbox",
             "part_big_px", "part_small_bbox", "part_small_px",
             "part_unique_color", "part_user4", "featrule", "featrule_iter"}

def _RS_feats3(train):
    ins = [_np2.asarray(p['input']) for p in train]
    outs = [_np2.asarray(p['output']) for p in train]
    rels = []
    for i, o in zip(ins, outs):
        if i.shape == o.shape: rels.append('same')
        elif o.shape[0] % i.shape[0] == 0 and o.shape[1] % i.shape[1] == 0:
            rels.append('up')
        elif i.shape[0] % o.shape[0] == 0 and i.shape[1] % o.shape[1] == 0:
            rels.append('down')
        elif o.size < i.size: rels.append('smaller')
        else: rels.append('other')
    size_rel = rels[0] if all(r == rels[0] for r in rels) else 'mixed'
    pal = []
    for i, o in zip(ins, outs):
        si, so = set(i.ravel().tolist()), set(o.ravel().tolist())
        pal.append('same' if si == so else
                   ('sub' if so < si else ('sup' if so > si else 'diff')))
    palette = pal[0] if all(p == pal[0] for p in pal) else 'mixed'
    def sym(g):
        return max(float((g == g[:, ::-1]).mean()),
                   float((g == g[::-1, :]).mean()))
    symin = 'hi' if min(sym(i) for i in ins) >= 0.8 else 'lo'
    return f"{size_rel}|{palette}|{symin}"

def _RS_order(train):
    b = _RS_REF74.get(_RS_feats3(train), {})
    first = sorted(b, key=lambda t: -b[t])
    rest = sorted((t for t in TOOLS_FINE if t not in b),
                  key=lambda t: -_RS_MARG.get(t, 0))
    return [t for t in first if t in TOOLS_FINE] + rest

def _RS_s0(train):
    s, n = 0.0, 0
    for p in train:
        i = _np2.asarray(p['input']); o = _np2.asarray(p['output'])
        if i.shape != o.shape: return 0.0
        s += float((i == o).mean()); n += 1
    return s / max(1, n)

def _RS_rise(nme, train):
    pipe = _derive_pipeline((nme,), train, _cache={},
                            _deadline=_time.time() + 1.0)
    if pipe is None: return None
    s, n = 0.0, 0
    for p in train:
        try:
            o = _np2.asarray(pipe(p['input'])); tg = _np2.asarray(p['output'])
        except Exception:
            return None
        if o.shape != tg.shape: return None
        s += float((o == tg).mean()); n += 1
    return s / max(1, n)

def rule_solve(task, _cap=6.0):
    """셀3c 로직의 문제당 버전. 반환 {'pred','order','loo_conf'} / None.
    1) 레퍼런스 순서 단일 호출 -> 2) 점수 오르는 도구 접두 2단 (빔 8)"""
    try:
        train = task.get('train', [])
        if len(train) < 2: return None
        dl = _time.time() + _cap
        order = _RS_order(train)
        for nme in order:                          # 1) 단일
            if _time.time() > dl: return None
            try:
                r = try_order(task, [nme])
            except Exception:
                continue
            if r and r.get('ok'):
                return {'pred': r['pred'], 'order': nme, 'loo_conf': 1.0}
        s0 = _RS_s0(train)                         # 2) 2단
        pref = []
        for nme in order:
            if _time.time() > dl: return None
            s = _RS_rise(nme, train)
            if s is not None and s > s0 + 0.02:
                pref.append((nme, s))
        pref.sort(key=lambda x: -x[1])
        for a, _s in pref[:8]:
            for b_ in order:
                if b_ == a: continue
                if _time.time() > dl: return None
                try:
                    r = try_order(task, [a, b_])
                except Exception:
                    continue
                if r and r.get('ok'):
                    return {'pred': r['pred'], 'order': f"{a}->{b_}",
                            'loo_conf': 1.0}
        return None
    except Exception:
        return None                                # 절대 예외를 밖으로 안 냄

# ---- 제출 블록 연결: 같은 이름으로 덮어쓰기 ----
solve_by_orders = rule_solve
print(f"승격 연결: solve_by_orders <- 규칙 스택 (도구 {len(TOOLS_FINE)}개)")

# ---- 풀 밖 승격 허용 목록: 실측 오답 0 도구 전체 (+기존 검증된 restore) ----
_safe = sorted((set(TOOLS_FINE) - _RS_RISKY) | {"restore"})
_os.environ.setdefault("ENGINE_NOPOOL_ALLOW", ",".join(_safe))
print(f"ENGINE_NOPOOL_ALLOW 기본값: 안전 도구 {len(_safe)}개"
      f" (조심계열 {len(_RS_RISKY)}종 제외, restore 유지)")'''
for name, src in (('order_engine.py', _ORDER_ENGINE), ('rule_stack_all.py', _RULE_STACK)):
    open(name, 'w', encoding='utf-8').write(src)
    print('엔진 파일 기록:', os.path.abspath(name), len(src.splitlines()), '줄')


엔진 파일 기록: /kaggle/working/order_engine.py 3322 줄
엔진 파일 기록: /kaggle/working/rule_stack_all.py 3002 줄


In [3]:
# [셀2] ★제출판(v57) 설정 + 데이터 + 더미 제출 + 규칙함수 선해결(v56 그대로) + 대상 풀
#   v56(LB 2.5) 골격: rerun → test 240 / 오프라인 → train 4 + eval 1 리허설, 더미 submission 즉시 저장,
#   칸수 오름차순(easy-first), 규칙 스택 선해결(CPU, GPU 적재 전), 2패스(근접 문제 재개방).
#   LLM 경로만 v56 의 'K16 규칙 투표' → v2 '서술 에이전트(다중턴 샌드박스)' 로 교체.
import os, sys, json, glob, time, re, math, random, shutil, hashlib, subprocess, threading, queue, io, traceback
import numpy as np
from collections import Counter, deque, defaultdict
from concurrent.futures import ThreadPoolExecutor
import io as _io, contextlib as _ctx

# ---- 플래그 (duck-merge-kit 명명 유지 — ablation 은 여기만 바꾼다) ----
DUCK_FLAGS = {
    'DUCK_SANDBOX_HELPERS': '1',     # 격자 프리미티브 + ★서술함수 API (관찰카드·정합체크·4축/silx 등급·단계·반례)
    'DUCK_MACROS': '1',              # task_report / try_transforms / check(등급 되먹임) / submit — MUST 없음
    'DUCK_BACKGROUND_HINT': '1',     # 배경 = 증거 기반 (비운 칸 > 테두리 > 면적), 조건부 안내
    'DUCK_LENS_HINT': '1',           # ★렌즈 카탈로그(8관점 + 10가족 표준/변형) + 증거탐지 + 씨앗 few-shot (서술의 통찰)
    'DUCK_MULTI_AVATAR_HINT': '0',   # 제거 — ARC-2 에 조종 객체 없음 (기록용)
}
SEED_K           = 3         # 첫 메시지에 인용할 씨앗 서술 수 (자기 자신 태스크 제외)
DESC_REQUIRED    = True      # ★서술 필수 매 턴 (DESCRIPTION 블록 없으면 경고, 코드는 실행)
for _k, _v in DUCK_FLAGS.items():
    os.environ[_k] = _v

# ---- 런 모드 (v56 규약: rerun = 제출, 아니면 오프라인 리허설) ----
RERUN            = TRUE_SUBMISSION                      # 셀1 (KAGGLE_IS_COMPETITION_RERUN)
OFFLINE_IDS      = {'training': ['fc754716', 'feca6190', 'ff28f65a', 'fe45cba4'],   # v56 리허설 fc754716(LLM 8/8) + 소형 3
                    'evaluation': ['7b5033c1']}                                      # v56 실측: eval 에서 LLM 이 푼 문제
SMOKE            = False     # ★True: STUB 엔진(가짜 LLM) — CPU 만으로 배관 검증 (환경 ARC2_SMOKE=1 도 동일)
if os.environ.get('ARC2_SMOKE') == '1':
    SMOKE = True
STUB_LLM         = SMOKE

# ---- 에이전트 노브 (제출판: v56 실측 처리량 ~250 tok/s 집계 기준으로 240 태스크 × 5턴 ≈ 5h) ----
MAX_TURNS_P1     = 5         # 1패스 태스크당 모델 턴 (마지막 턴은 강제 submit)
MAX_TURNS_P2     = 4         # 2패스(근접 문제 재개방) 추가 턴
MAX_TURNS        = MAX_TURNS_P1                          # (호환용 — 셀5 기본값)
MIN_TURNS_P1     = 3         # 예산 압박 시 1패스 턴을 여기까지 줄인다 (적응)
CONCURRENCY      = 28        # 락스텝 동시 태스크 (vLLM max_num_seqs=32 안)
THINK_BUDGET     = 3072      # 턴당 사고 캡 (v56 3584 보다 조금 낮춤: 턴이 5번이다). 잘리면 </think> 강제 후 답 (2패스)
TOK_ANSWER       = 1280      # 답(서술 + 코드) 토큰
MAX_MODEL_LEN    = 32768     # ★16384(검증값) 에서 상향 시도 — 적재 실패 시 16384 로 자동 재시도 (셀7)
THINK_MIN, ANSWER_MIN = 1024, 512   # 컨텍스트 압박 시 히스토리를 잘라 최소 이만큼은 확보
TEMP, TOP_P, TOP_K = 0.6, 0.95, 20  # Qwen3 thinking 권장값
EXEC_TIMEOUT_S   = 90        # 한 턴 코드 실행 상한 (초과 → 샌드박스 재시작; check 1회 ≤ 5s×쌍수)
SOLVE_TIMEOUT_S  = 5         # check() 안에서 solve() 1회 상한
OUT_CAP          = 3600      # 모델에게 돌려주는 샌드박스 출력 상한 (문자) — check() 등급 되먹임 포함
INIT_TIMEOUT_S   = 90        # 워커 부팅 + task_report
PASS1_FRAC       = 0.70      # 예산의 70% 까지 1패스(새 태스크 착수), 나머지는 근접 문제 재개방(2패스)
NEAR_MIN_PAIRS   = 1         # 2패스 재개방 기준: train 쌍 1개 이상 통과 …
NEAR_MIN_NSCORE  = 0.30      # … 또는 정규화 점수 0.30 이상 (널 기준선 대비)
RULE_CAP_S       = 6.0       # 규칙 스택 rule_solve 유닛당 캡 (v56: eval 120 평균 2.6s/태스크)
RULE_BUDGET_MIN  = 25        # 규칙 선해결 전체 예산 (v56 그대로)
BUDGET_MIN       = 684 if RERUN else 60   # 셀1 시각 기준 자동 스톱: 제출 11.4h (v56 검증 마진) / 리허설 60분 (적재 8분 + 5문제 1·2패스)
MAX_SIDE         = 30
SEED             = 42
random.seed(SEED)
TASK_LIMIT       = None
if SMOKE:
    MAX_TURNS_P1, MAX_TURNS_P2, MAX_TURNS, MIN_TURNS_P1 = 2, 2, 2, 2
    if not RERUN and os.environ.get('ARC2_TARGET') and os.environ.get('ARC2_FULL') != '1':
        TASK_LIMIT = 6

for _k, _v in json.loads(os.environ.get('ARC2_KNOBS', '{}')).items():   # (로컬 검증용) 노브 덮어쓰기 — 캐글에서는 비어 있다
    globals()[_k] = _v

WORK = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.path.abspath('./working')
AG = f'{WORK}/agent'
SB_DIR = f'{WORK}/arc_sandbox'
for _d in (WORK, AG, f'{AG}/logs', f'{AG}/fail_bank', SB_DIR):
    os.makedirs(_d, exist_ok=True)
SUBMISSION_PATH = f'{WORK}/submission.json'
MANIFEST_PATH = f'{WORK}/run_manifest.json'
LEDGER_F = f'{AG}/ledger.json'


def LOG(*a):
    print(f'[{time.strftime("%H:%M:%S")}]', *a, flush=True)


def manifest_update(**fields):
    try:
        data = json.load(open(MANIFEST_PATH)) if os.path.exists(MANIFEST_PATH) else {}
        data.update(fields)
        json.dump(data, open(MANIFEST_PATH, 'w'), ensure_ascii=False, indent=1, default=str)
    except Exception as e:
        LOG('manifest write skipped:', repr(e)[:100])


try:
    _t0 = json.load(open(f'{WORK}/_t0.json'))['t0']
except Exception:
    _t0 = time.time()
START = time.monotonic()
DEADLINE = START + max(120, BUDGET_MIN * 60 - (time.time() - _t0))
PASS1_DEADLINE = START + (DEADLINE - START) * PASS1_FRAC
LOG(f'모드: {"★제출(대회 재실행 → test)" if RERUN else "리허설(오프라인 train 4 + eval 1, 채점)"} / 자동 스톱까지 {(DEADLINE - START) / 60:.0f}분 '
    f'(1패스 착수 마감 {(PASS1_DEADLINE - START) / 60:.0f}분) / SMOKE={SMOKE}')


# ---- 데이터 (캐글 arc-prize-2026 포맷: arc-agi_<split>_challenges.json + _solutions.json) ----
def _glob_depth(root, name, maxdepth=6):
    """깊이 제한 탐색 — /kaggle/input 은 FUSE 라 glob('**') 가 모델/휠 트리 전체를 훑는다 (실측 수 분)."""
    out = []
    for d in range(0, maxdepth + 1):
        out += glob.glob('/'.join([root] + ['*'] * d + [name]))
    return sorted(set(out))


def _find_file(name):
    roots = [os.environ.get('ARC2_DATA_DIR', '')] + ['/kaggle/input']
    for r in roots:
        if not r or not os.path.isdir(r):
            continue
        c = _glob_depth(r, name)
        if c:
            return c[0]
    return None


def load_split(split):
    """-> (challenges dict tid->{'train','test'}, solutions dict tid->[grids] | {})"""
    fp = _find_file(f'arc-agi_{split}_challenges.json')
    assert fp, f'★arc-agi_{split}_challenges.json 없음 — Add Input 에 대회 데이터(arc-prize-2026) 추가'
    ch = json.load(open(fp))
    sp = _find_file(f'arc-agi_{split}_solutions.json')
    sol = json.load(open(sp)) if sp else {}
    LOG(f'{split}: 태스크 {len(ch)} / 정답 {len(sol)}  ({fp})')
    return ch, sol


def task_cells(t):
    return sum(len(p['input']) * len(p['input'][0]) + len(p['output']) * len(p['output'][0]) for p in t['train']) + \
        sum(len(x['input']) * len(x['input'][0]) for x in t['test'])


def ok_size(t):
    return all(len(g) <= MAX_SIDE and len(g[0]) <= MAX_SIDE for p in t['train'] for g in (p['input'], p['output'])) and \
        all(len(x['input']) <= MAX_SIDE and len(x['input'][0]) <= MAX_SIDE for x in t['test'])


SPLIT_OF = {}
if RERUN:
    CH, SOL = load_split('test')                      # 240 태스크, 정답 없음
    SPLIT_OF = {t: 'test' for t in CH}
elif os.environ.get('ARC2_TARGET'):                   # (로컬 검증용) 스플릿 통째로
    CH, SOL = load_split(os.environ['ARC2_TARGET'])
    SPLIT_OF = {t: os.environ['ARC2_TARGET'] for t in CH}
else:
    CH, SOL = {}, {}
    for _split, _ids in OFFLINE_IDS.items():
        _ch, _sol = load_split(_split)
        _miss = [t for t in _ids if t not in _ch]
        if _miss:
            LOG(f'  ★{_split} 에 없는 id (건너뜀): {_miss}')
        for _t in _ids:
            if _t in _ch:
                CH[_t] = _ch[_t]; SPLIT_OF[_t] = _split
                if _t in _sol:
                    SOL[_t] = _sol[_t]
LOG(f'태스크 {len(CH)}개 로드 (rerun={RERUN}, 채점 {"O" if SOL else "X"})')

# ---- ★더미 제출 즉시 저장 (전 태스크 identity) = 어떤 시점에 죽어도 유효 (v56) ----
SUB_LOCK = threading.Lock()
LEDGER = {}
SUB = {tid: [{'attempt_1': [list(map(int, r)) for r in te['input']], 'attempt_2': [list(map(int, r)) for r in te['input']]}
             for te in t['test']] for tid, t in CH.items()}


def write_submission():
    with SUB_LOCK:
        tmp = SUBMISSION_PATH + '.tmp'
        json.dump(SUB, open(tmp, 'w'))
        os.replace(tmp, SUBMISSION_PATH)
        json.dump(LEDGER, open(LEDGER_F, 'w'), ensure_ascii=False, indent=1)


write_submission()
LOG(f'더미 submission.json 저장 ({len(SUB)}태스크) → {SUBMISSION_PATH}')

_order = [t for t in sorted(CH, key=lambda k: task_cells(CH[k])) if ok_size(CH[t])]   # 칸수 오름차순(easy-first)
if TASK_LIMIT:
    _order = _order[:TASK_LIMIT]
LOG(f'실행 순서: {len(_order)}태스크 (칸수 오름차순), 예산 {BUDGET_MIN}분')

# ============================================================================
# ★[v56 그대로] 규칙함수 선해결 — 오더엔진+도구69 (CPU, GPU 적재 전)
#   실측(ARC-AGI-2 eval 120 전수): 채택 11 / 전유닛 정답 11 / 오답 0 / 310s.
#   rule_solve 는 test[0] 만 예측 → test 유닛마다 task 를 쪼개 호출(104엔진 실측 교훈).
#   채택 시 attempt_1 = 규칙 답, attempt_2 = identity. LLM 풀에서 제외(GPU 절약).
#   실패해도 무해: 엔진 로드 실패 시 LLM 단독으로 그대로 진행.
# ============================================================================
RULE_SOLVED, _ENS, _ebuf = [], {}, _io.StringIO()
RUN_ID = time.strftime('%m%d_%H%M')
try:
    _tr = time.monotonic()
    _eng_dir = next((d for d in ('.', '/kaggle/working', WORK) if os.path.exists(f'{d}/order_engine.py')), None)
    assert _eng_dir, '★order_engine.py 없음 — 셀1.5(엔진 내장)를 먼저 실행'
    with _ctx.redirect_stdout(_ebuf):
        exec(open(f'{_eng_dir}/order_engine.py', encoding='utf-8').read(), _ENS)
        exec(open(f'{_eng_dir}/rule_stack_all.py', encoding='utf-8').read(), _ENS)
    LOG(f'★규칙 스택 로드: 도구 {len(_ENS.get("TOOLS_FINE", []))}종 ({time.monotonic() - _tr:.0f}s)')
except Exception as _e:
    LOG(f'★규칙 스택 로드 실패 — LLM 단독 진행: {repr(_e)[:120]}'); _ENS = {}
if _ENS:
    _tr = time.monotonic()
    for _i, _tid in enumerate(_order):
        if time.monotonic() - _tr > RULE_BUDGET_MIN * 60 or time.monotonic() > DEADLINE - 60:
            LOG(f'규칙 선해결 예산 마감 — {_i}/{len(_order)} 시도'); break
        _t = CH[_tid]; _preds = []
        for _te in _t['test']:
            try:
                with _ctx.redirect_stdout(_ebuf):
                    _r = _ENS['rule_solve']({'train': _t['train'], 'test': [{'input': _te['input']}]}, _cap=RULE_CAP_S)
            except Exception:
                _r = None
            if _r is None:
                _preds = None; break
            _preds.append(np.asarray(_r['pred']).tolist())
        if _preds:
            _units = [{'attempt_1': _preds[_k], 'attempt_2': [list(map(int, _row)) for _row in _t['test'][_k]['input']]}
                      for _k in range(len(_preds))]
            _sc = None
            if _tid in SOL:
                _sc = sum(int(_units[_k]['attempt_1'] == SOL[_tid][_k] or _units[_k]['attempt_2'] == SOL[_tid][_k])
                          for _k in range(len(SOL[_tid]))) / max(1, len(SOL[_tid]))
            with SUB_LOCK:
                SUB[_tid] = _units
                LEDGER[_tid] = {'status': ('solved' if _sc == 1.0 else 'partial' if _sc else 'failed') if _sc is not None else 'done',
                                'score': _sc, 'reason': 'rule', 'by': 'rule', 'rule_order': _r.get('order'), 'turns': 0,
                                'n_train': len(_t['train']), 'split': SPLIT_OF.get(_tid), 'run': RUN_ID}
            RULE_SOLVED.append(_tid)
            write_submission()
            LOG(f'  ★규칙해결 [{len(RULE_SOLVED)}] {_tid} ({_r["order"]})' + (f' 채점 {_sc}' if _sc is not None else ''))
    LOG(f'★규칙 선해결: {len(RULE_SOLVED)}/{len(_order)} 채택 ({(time.monotonic() - _tr) / 60:.1f}분) — LLM 은 나머지만')
    _ENS = {}                       # 엔진 네임스페이스 해제 (메모리)

POOL = [(tid, {'train': CH[tid]['train'], 'test': CH[tid]['test'], 'tid': tid}, SOL.get(tid)) for tid in _order if tid not in RULE_SOLVED]   # tid: 자기 씨앗 제외용
LOG(f'★LLM 대상 {len(POOL)} 태스크 (전체 {len(CH)}, 규칙해결 {len(RULE_SOLVED)}, 30초과 {len(CH) - len(_order) if not TASK_LIMIT else "-"}) / '
    f'정답 보유 {sum(1 for _, _, s in POOL if s)} / 턴 {MAX_TURNS_P1}+{MAX_TURNS_P2} / 동시 {CONCURRENCY}')


[14:27:12] 모드: 리허설(오프라인 train 4 + eval 1, 채점) / 자동 스톱까지 60분 (1패스 착수 마감 42분) / SMOKE=False
[14:27:12] training: 태스크 1000 / 정답 1000  (/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_training_challenges.json)
[14:27:12] evaluation: 태스크 120 / 정답 120  (/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json)
[14:27:12] 태스크 5개 로드 (rerun=False, 채점 O)
[14:27:12] 더미 submission.json 저장 (5태스크) → /kaggle/working/submission.json
[14:27:12] 실행 순서: 5태스크 (칸수 오름차순), 예산 60분
[14:27:13] ★규칙 스택 로드: 도구 69종 (1s)
[14:27:15] ★규칙 선해결: 0/5 채택 (0.0분) — LLM 은 나머지만
[14:27:15] ★LLM 대상 5 태스크 (전체 5, 규칙해결 0, 30초과 0) / 정답 보유 5 / 턴 5+4 / 동시 28


In [4]:
# [셀3] 샌드박스 헬퍼 소스 (SANDBOX_HELPERS 강화판) — 워커가 exec 하는 순수 격자 프리미티브 48종
#   ARC-3 킷 sandbox_helpers.py 의 계약(표준 라이브러리 + numpy, 순수 함수, 30x30 에서 ms) 그대로.
HELPERS_SRC = r'''# ===================================================================================
# ARC-2 SANDBOX HELPERS (SANDBOX_HELPERS 강화판) — 순수 격자 프리미티브. 게임 규칙 없음.
#   설계 계약 (duck-merge-kit sandbox_helpers.py 와 동일):
#     * 표준 라이브러리 + numpy 만. 이 소스는 그대로 샌드박스 서브프로세스에 exec 된다.
#     * 순수 함수(입력 격자/노드 → 값). 태스크 상태(TRAIN/TEST)는 매크로가 읽는다.
#     * 30x30 에서 전부 ms 단위. 출력은 JSON-safe / 한 줄 요약 가능하게.
#   ARC-3 → ARC-2 대응: frame.ascii → np.ndarray, segmentation node → objects() 노드,
#     match_nodes(before, after) → match_nodes(objects(input), objects(output)) : 규칙이
#     각 객체에 무엇을 했는지(이동/재색/출현/소멸)를 쌍마다 그대로 보여준다.
# ===================================================================================
import math, itertools, json, re
from collections import Counter, deque, defaultdict
import numpy as np

MAX_SIDE = 30
D4_KEYS = ('id', 'rot90', 'rot180', 'rot270', 'flipH', 'flipV', 'transpose', 'anti')
_D4 = {'id': lambda g: g, 'rot90': lambda g: np.rot90(g, 1), 'rot180': lambda g: np.rot90(g, 2),
       'rot270': lambda g: np.rot90(g, 3), 'flipH': lambda g: g[:, ::-1], 'flipV': lambda g: g[::-1, :],
       'transpose': lambda g: g.T, 'anti': lambda g: np.rot90(g.T, 2)}
_N4 = ((-1, 0), (1, 0), (0, -1), (0, 1))
_N8 = _N4 + ((-1, -1), (-1, 1), (1, -1), (1, 1))


def G(x):
    """any grid-like -> 2D int ndarray (copy)."""
    a = np.array(x, dtype=int)
    if a.ndim == 0:
        a = a.reshape(1, 1)
    elif a.ndim == 1:
        a = a.reshape(1, -1)
    return a


def grid_str(g):
    """grid -> rows of digits (no spaces). -1 prints as '.'"""
    return '\n'.join(''.join('.' if v < 0 else str(int(v)) for v in row) for row in G(g))


def show(g, label=None):
    """print a grid as digit rows (compact). Never returns the grid."""
    g = G(g)
    if label:
        print(f'{label} ({g.shape[0]}x{g.shape[1]}):')
    print(grid_str(g))


def valid_grid(x):
    """True if x is a 2D grid of ints 0-9 with sides 1..30."""
    try:
        a = np.asarray(x)
        if a.dtype.kind not in 'iub' or a.ndim != 2 or a.size == 0:
            return False
        if a.shape[0] > MAX_SIDE or a.shape[1] > MAX_SIDE:
            return False
        return bool(a.min() >= 0 and a.max() <= 9)
    except Exception:
        return False


# ----------------------------------------------------------------------------- background
def _mode(g):
    c = Counter(G(g).ravel().tolist())
    return int(c.most_common(1)[0][0]), c


def _border_color(g):
    g = G(g)
    b = np.concatenate([g[0, :], g[-1, :], g[:, 0], g[:, -1]])
    u = np.unique(b)
    return int(u[0]) if len(u) == 1 else None


def background(grid):
    """background color of ONE grid: mode color, but 0 wins when 0 covers >= 50% of the mode
    count, and a uniform border color wins over a slightly larger interior mode."""
    g = G(grid)
    m, c = _mode(g)
    bc = _border_color(g)
    if bc is not None and bc != m and c.get(bc, 0) >= 0.5 * c[m]:
        return bc
    if m != 0 and c.get(0, 0) >= 0.5 * c[m]:
        return 0
    return m


def infer_background(pairs):
    """Task-level background from train pairs. Evidence order: (1) the color revealed in cells an
    object VACATED (input object cell -> different output color), (2) uniform border color,
    (3) area in inputs AND outputs. Returns {'bg', 'confidence', 'evidence': {...}}."""
    ins = [G(p['input']) for p in pairs]
    outs = [G(p['output']) for p in pairs]
    cols = sorted(set(int(v) for g in ins for v in np.unique(g)))
    score = {c: 0.0 for c in cols}
    ev = {'mode_in': [], 'mode_out': [], 'border_in': [], 'vacated': Counter(), 'per_input': []}
    for gi, go in zip(ins, outs):
        mi, ci = _mode(gi)
        mo, co = _mode(go)
        ev['mode_in'].append(mi)
        ev['mode_out'].append(mo)
        bc = _border_color(gi)
        ev['border_in'].append(bc)
        ev['per_input'].append(background(gi))
        for c in cols:
            score[c] += ci.get(c, 0) / gi.size + 0.5 * co.get(c, 0) / go.size
        if bc is not None:
            score[bc] += 0.6
        # vacated-cell evidence (same size only): input object cells (color != mode) whose
        # output color changed -> the output color there is what was "underneath".
        if gi.shape == go.shape:
            chg = (gi != go) & (gi != mi)
            if chg.any():
                for v, n in Counter(go[chg].tolist()).items():
                    ev['vacated'][int(v)] += int(n)
    for c in cols:
        score[c] += 0.3 * (c == 0)
    if ev['vacated']:
        v, n = ev['vacated'].most_common(1)[0]
        if n >= 3 and v in score:
            score[v] += 0.8
    ranked = sorted(score.items(), key=lambda kv: -kv[1])
    bg = ranked[0][0]
    gap = ranked[0][1] - (ranked[1][1] if len(ranked) > 1 else -9)
    area = min(Counter(g.ravel().tolist()).get(bg, 0) / g.size for g in ins)
    if area >= 0.5 and gap >= 0.4:
        conf = 'high'
    elif area >= 0.3 or gap >= 0.3:
        conf = 'medium'
    else:
        conf = 'low'
    if len(set(ev['per_input'])) > 1:
        conf = 'low'
    ev['vacated'] = dict(ev['vacated'].most_common(3))
    ev['score'] = {int(k): round(v, 2) for k, v in ranked[:3]}
    return {'bg': int(bg), 'confidence': conf, 'evidence': ev}


# ----------------------------------------------------------------------------- components / objects
def components(mask, conn=8):
    """bool mask -> list of cell lists (connected components)."""
    m = np.asarray(mask, dtype=bool)
    H, W = m.shape
    seen = np.zeros((H, W), dtype=bool)
    nb = _N8 if conn == 8 else _N4
    out = []
    for r in range(H):
        for c in range(W):
            if m[r, c] and not seen[r, c]:
                seen[r, c] = True
                q = deque([(r, c)])
                cells = []
                while q:
                    y, x = q.popleft()
                    cells.append((y, x))
                    for dy, dx in nb:
                        ny, nx = y + dy, x + dx
                        if 0 <= ny < H and 0 <= nx < W and m[ny, nx] and not seen[ny, nx]:
                            seen[ny, nx] = True
                            q.append((ny, nx))
                out.append(cells)
    return out


def _holes_in_crop(crop):
    m = crop == -1
    n = 0
    for cells in components(m, conn=4):
        if not any(y == 0 or x == 0 or y == m.shape[0] - 1 or x == m.shape[1] - 1 for y, x in cells):
            n += 1
    return n


def _make_node(g, cells, idx):
    ys = [y for y, _ in cells]
    xs = [x for _, x in cells]
    r0, r1, c0, c1 = min(ys), max(ys), min(xs), max(xs)
    crop = np.full((r1 - r0 + 1, c1 - c0 + 1), -1, dtype=int)
    for y, x in cells:
        crop[y - r0, x - c0] = g[y, x]
    colors = sorted(set(int(g[y, x]) for y, x in cells))
    sil = (crop != -1)
    sil_key = f'{sil.shape[0]}x{sil.shape[1]}:' + np.packbits(sil.ravel()).tobytes().hex()
    color = colors[0] if len(colors) == 1 else -1
    h = f'{color}|{sil_key}' if color != -1 else f'multi|{crop.shape[0]}x{crop.shape[1]}:' + (crop + 1).astype(np.uint8).tobytes().hex()
    H, W = g.shape
    return {'id': idx, 'color': color, 'colors': colors, 'pixels': len(cells), 'cells': cells,
            'bbox': [r0, c0, r1, c1], 'center': [(r0 + r1) / 2.0, (c0 + c1) / 2.0],
            'centroid': [round(sum(ys) / len(ys), 2), round(sum(xs) / len(xs), 2)],
            'hash': h, 'sil': sil_key, 'holes': _holes_in_crop(crop),
            'touches_border': bool(r0 == 0 or c0 == 0 or r1 == H - 1 or c1 == W - 1),
            'is_rect': len(cells) == crop.size, 'is_dot': len(cells) == 1,
            'is_line': (r0 == r1 or c0 == c1) and len(cells) == crop.size, 'crop': crop}


def objects(grid, bg=None, conn=8, by_color=True, min_pixels=1):
    """Connected non-background components as node dicts (reading order).
    by_color=True: one color per node; False: any non-bg colors glued together (multi-color node,
    color=-1, colors=[...]). node keys: id,color,colors,pixels,cells,bbox,center,centroid,hash
    (color+shape),sil (shape only),holes,touches_border,is_rect,is_line,is_dot,crop(-1=empty)."""
    g = G(grid)
    if bg is None:
        bg = background(g)
    nodes = []
    if by_color:
        for c in sorted(set(g.ravel().tolist()) - {bg}):
            for cells in components(g == c, conn):
                if len(cells) >= min_pixels:
                    nodes.append(cells)
    else:
        for cells in components(g != bg, conn):
            if len(cells) >= min_pixels:
                nodes.append(cells)
    nodes.sort(key=lambda cs: (min(y for y, _ in cs), min(x for _, x in cs)))
    return [_make_node(g, cs, i) for i, cs in enumerate(nodes)]


def summ(node):
    """one-line JSON-safe summary of a node (no cells/crop)."""
    return {'id': node['id'], 'color': node['color'] if node['color'] != -1 else node['colors'],
            'px': node['pixels'], 'bbox': node['bbox'], 'holes': node['holes'],
            'rect': node['is_rect'], 'border': node['touches_border']}


def bbox(node):
    return list(node['bbox'])


def center(node):
    """bbox center (row, col) floats."""
    return tuple(node['center'])


def _pt(o):
    if isinstance(o, dict):
        return center(o)
    return (float(o[0]), float(o[1]))


def manhattan(a, b):
    (ar, ac), (br, bc) = _pt(a), _pt(b)
    return abs(ar - br) + abs(ac - bc)


def nodes_by_color(nodes, color):
    return [n for n in nodes if n['color'] == color or (n['color'] == -1 and color in n['colors'])]


def touching(nodes, i, j):
    """True if node i and node j share a 4-connected edge."""
    a = set(nodes[i]['cells']) if isinstance(i, int) else set(i['cells'])
    b = nodes[j]['cells'] if isinstance(j, int) else j['cells']
    for y, x in b:
        for dy, dx in _N4:
            if (y + dy, x + dx) in a:
                return True
    return False


def neighbors(nodes, node_id):
    """ids of nodes 4-adjacent to node_id."""
    return [n['id'] for n in nodes if n['id'] != node_id and touching(nodes, node_id, n['id'])]


def same_shape(n1, n2, ignore_color=True, allow_d4=False):
    a, b = n1['crop'], n2['crop']
    if ignore_color:
        a, b = np.where(a != -1, 1, -1), np.where(b != -1, 1, -1)
    cands = [f(a) for f in _D4.values()] if allow_d4 else [a]
    return any(t.shape == b.shape and np.array_equal(t, b) for t in cands)


# ----------------------------------------------------------------------------- matching / diffs
def match_nodes(nodes_in, nodes_out, shape_change_radius=8):
    """Match objects of an input grid to objects of its output grid (3 stages, as in ARC-3):
    1) unique hash (color+shape) on both sides -> matched; 2) identical twins -> greedy nearest
    center; 3) leftovers: same silhouette (shape only) -> 'recolored' (same place) or
    'moved_recolored'; the rest -> appeared / disappeared, with same-color nearby pairs listed in
    shape_change_candidates. Returns dict with lists: stable(count), moved, recolored, appeared,
    disappeared, shape_change_candidates."""
    A, B = list(nodes_in), list(nodes_out)
    by_h_a, by_h_b = defaultdict(list), defaultdict(list)
    for n in A:
        by_h_a[n['hash']].append(n)
    for n in B:
        by_h_b[n['hash']].append(n)
    pairs, used_a, used_b = [], set(), set()
    for h, al in by_h_a.items():
        bl = by_h_b.get(h, [])
        if not bl:
            continue
        cands = sorted(((manhattan(a, b), ia, ib) for ia, a in enumerate(al) for ib, b in enumerate(bl)),
                       key=lambda t: t[0])
        ta, tb = set(), set()
        for d, ia, ib in cands:
            if ia in ta or ib in tb:
                continue
            ta.add(ia)
            tb.add(ib)
            pairs.append((al[ia], bl[ib], 'same'))
            used_a.add(id(al[ia]))
            used_b.add(id(bl[ib]))
    # stage 3a: silhouette-only match among leftovers (recolor)
    rest_a = [n for n in A if id(n) not in used_a]
    rest_b = [n for n in B if id(n) not in used_b]
    by_s_b = defaultdict(list)
    for n in rest_b:
        by_s_b[n['sil']].append(n)
    for a in rest_a:
        bl = [b for b in by_s_b.get(a['sil'], []) if id(b) not in used_b]
        if not bl:
            continue
        b = min(bl, key=lambda x: manhattan(a, x))
        pairs.append((a, b, 'recolor'))
        used_a.add(id(a))
        used_b.add(id(b))
    stable, moved, recolored = 0, [], []
    for a, b, kind in pairs:
        (ar, ac), (br, bc) = center(a), center(b)
        dr, dc = br - ar, bc - ac
        rec = {'in_id': a['id'], 'out_id': b['id'], 'color': a['color'] if a['color'] != -1 else a['colors'],
               'px': a['pixels'], 'from': list(a['bbox'][:2]), 'to': list(b['bbox'][:2]),
               'shift': [int(round(dr)), int(round(dc))]}
        if kind == 'recolor':
            rec['color_to'] = b['color'] if b['color'] != -1 else b['colors']
            rec['kind'] = 'recolored' if (dr == 0 and dc == 0) else 'moved_recolored'
            recolored.append(rec)
        elif dr == 0 and dc == 0:
            stable += 1
        else:
            moved.append(rec)
    disappeared = [n for n in A if id(n) not in used_a]
    appeared = [n for n in B if id(n) not in used_b]
    cands = []
    for dn in disappeared:
        for an in appeared:
            if dn['color'] != an['color']:
                continue
            d = manhattan(dn, an)
            if d <= shape_change_radius:
                cands.append({'in_id': dn['id'], 'out_id': an['id'], 'color': dn['color'],
                              'distance': round(d, 1), 'px_before': dn['pixels'], 'px_after': an['pixels']})
    cands.sort(key=lambda x: x['distance'])
    return {'stable': stable, 'moved': moved, 'recolored': recolored,
            'appeared': [summ(n) for n in appeared], 'disappeared': [summ(n) for n in disappeared],
            'shape_change_candidates': cands}


def match_summary(m):
    """match_nodes result -> one line."""
    s = f"stable {m['stable']}, moved {len(m['moved'])}, recolored {len(m['recolored'])}, appeared {len(m['appeared'])}, disappeared {len(m['disappeared'])}"
    ex = []
    for r in m['moved'][:2]:
        ex.append(f"moved color {r['color']} px{r['px']} shift {r['shift']}")
    for r in m['recolored'][:2]:
        ex.append(f"{r['kind']} {r['color']}->{r['color_to']} px{r['px']} at {r['from']}")
    for r in m['appeared'][:2]:
        ex.append(f"appeared color {r['color']} px{r['px']} bbox {r['bbox']}")
    for r in m['disappeared'][:2]:
        ex.append(f"disappeared color {r['color']} px{r['px']} bbox {r['bbox']}")
    if m['shape_change_candidates']:
        c = m['shape_change_candidates'][0]
        ex.append(f"shape-change? color {c['color']} px {c['px_before']}->{c['px_after']} d={c['distance']}")
    return s + ((' | ' + '; '.join(ex)) if ex else '')


def diff_cells(a, b, max_cells=40):
    """cell-level diff of two same-size grids: {'changed','bbox','cells':[[r,c,from,to]...],
    'transitions':{'from->to':n},'truncated'}; if shapes differ -> {'changed':None,'shape_a','shape_b'}"""
    a, b = G(a), G(b)
    if a.shape != b.shape:
        return {'changed': None, 'shape_a': list(a.shape), 'shape_b': list(b.shape)}
    rs, cs = np.nonzero(a != b)
    cells = [[int(r), int(c), int(a[r, c]), int(b[r, c])] for r, c in zip(rs, cs)]
    trans = Counter(f'{x[2]}->{x[3]}' for x in cells)
    box = [int(rs.min()), int(cs.min()), int(rs.max()), int(cs.max())] if len(rs) else None
    return {'changed': len(cells), 'bbox': box, 'cells': cells[:max_cells],
            'transitions': dict(trans.most_common(8)), 'truncated': len(cells) > max_cells}


def color_map(pairs):
    """cell-wise color mapping input->output over same-size pairs: {'map':{in:out},'consistent',
    'conflicts':[[color,{out:n}]], 'changed':[colors that map to a different color]}"""
    tab = defaultdict(Counter)
    same = True
    for p in pairs:
        gi, go = G(p['input']), G(p['output'])
        if gi.shape != go.shape:
            same = False
            continue
        for vi, vo in zip(gi.ravel().tolist(), go.ravel().tolist()):
            tab[int(vi)][int(vo)] += 1
    mp, conflicts = {}, []
    for c, cnt in sorted(tab.items()):
        v, n = cnt.most_common(1)[0]
        mp[c] = v
        if len(cnt) > 1:
            conflicts.append([c, dict(cnt.most_common(3))])
    return {'map': mp, 'consistent': same and not conflicts, 'same_size': same,
            'conflicts': conflicts, 'changed': [c for c, v in mp.items() if c != v]}


def compare(pred, target, max_cells=12):
    """{'match','shape_ok','wrong','wrong_cells':[[r,c,pred,target]...],'per_color':{target_color:n_wrong}}"""
    if pred is None:
        return {'match': False, 'shape_ok': False, 'wrong': None, 'wrong_cells': [], 'per_color': {}, 'error': 'no output'}
    p, t = G(pred), G(target)
    if p.shape != t.shape:
        return {'match': False, 'shape_ok': False, 'wrong': None, 'wrong_cells': [],
                'per_color': {}, 'shape_pred': list(p.shape), 'shape_target': list(t.shape)}
    rs, cs = np.nonzero(p != t)
    cells = [[int(r), int(c), int(p[r, c]), int(t[r, c])] for r, c in zip(rs, cs)]
    return {'match': len(cells) == 0, 'shape_ok': True, 'wrong': len(cells), 'wrong_cells': cells[:max_cells],
            'per_color': dict(Counter(x[3] for x in cells).most_common(5)), 'total': int(t.size)}


# ----------------------------------------------------------------------------- structure
def separators(grid, bg=None, include_bg=False):
    """full rows/cols of ONE non-background color -> (rows, cols, color); ([], [], None) if none.
    include_bg=True also accepts lines of the background color."""
    g = G(grid)
    if bg is None:
        bg = background(g)
    H, W = g.shape
    rows = [(y, int(g[y, 0])) for y in range(H) if (g[y] == g[y, 0]).all()]
    cols = [(x, int(g[0, x])) for x in range(W) if (g[:, x] == g[0, x]).all()]
    cnt = Counter([c for _, c in rows if include_bg or c != bg] + [c for _, c in cols if include_bg or c != bg])
    if not cnt:
        return [], [], None
    col = cnt.most_common(1)[0][0]
    return [y for y, c in rows if c == col], [x for x, c in cols if c == col], col


def panels(grid, bg=None):
    """cells between separator lines -> [(bbox, subgrid)] in reading order (whole grid if none)."""
    g = G(grid)
    H, W = g.shape
    rows, cols, _ = separators(g, bg)
    ys = [-1] + rows + [H]
    xs = [-1] + cols + [W]
    out = []
    for i in range(len(ys) - 1):
        for j in range(len(xs) - 1):
            r0, r1, c0, c1 = ys[i] + 1, ys[i + 1] - 1, xs[j] + 1, xs[j + 1] - 1
            if r1 >= r0 and c1 >= c0:
                out.append(([r0, c0, r1, c1], g[r0:r1 + 1, c0:c1 + 1].copy()))
    return out


def enclosed_regions(grid, bg=None, conn=4):
    """background pockets NOT touching the border (holes) -> list of cell lists."""
    g = G(grid)
    if bg is None:
        bg = background(g)
    H, W = g.shape
    out = []
    for cells in components(g == bg, conn):
        if not any(y == 0 or x == 0 or y == H - 1 or x == W - 1 for y, x in cells):
            out.append(cells)
    return out


def fill_region(grid, cells, color):
    g = G(grid)
    for y, x in cells:
        g[y, x] = color
    return g


def _passable_fn(passable):
    if callable(passable):
        return passable
    if isinstance(passable, (int, np.integer)):
        return lambda v: v == passable
    s = set(int(v) for v in passable)
    return lambda v: v in s


def reachable_cells(grid, start, passable, conn=4):
    """cells reachable from start over cells whose color satisfies `passable` (int / set / fn).
    The start cell always counts."""
    g = G(grid)
    H, W = g.shape
    ok = _passable_fn(passable)
    start = (int(start[0]), int(start[1]))
    if not (0 <= start[0] < H and 0 <= start[1] < W):
        return set()
    nb = _N8 if conn == 8 else _N4
    seen = {start}
    q = deque([start])
    while q:
        y, x = q.popleft()
        for dy, dx in nb:
            ny, nx = y + dy, x + dx
            if 0 <= ny < H and 0 <= nx < W and (ny, nx) not in seen and ok(int(g[ny, nx])):
                seen.add((ny, nx))
                q.append((ny, nx))
    return seen


def bfs_path(grid, start, goal, passable, conn=4):
    """shortest path start->goal over passable colors (goal cell always allowed) -> [(r,c)...]|None"""
    g = G(grid)
    H, W = g.shape
    ok = _passable_fn(passable)
    start, goal = (int(start[0]), int(start[1])), (int(goal[0]), int(goal[1]))
    if not (0 <= start[0] < H and 0 <= start[1] < W and 0 <= goal[0] < H and 0 <= goal[1] < W):
        return None
    if start == goal:
        return [start]
    nb = _N8 if conn == 8 else _N4
    prev = {start: None}
    q = deque([start])
    while q:
        y, x = q.popleft()
        for dy, dx in nb:
            n = (y + dy, x + dx)
            if not (0 <= n[0] < H and 0 <= n[1] < W) or n in prev:
                continue
            if n != goal and not ok(int(g[n])):
                continue
            prev[n] = (y, x)
            if n == goal:
                path = [n]
                while prev[path[-1]] is not None:
                    path.append(prev[path[-1]])
                return path[::-1]
            q.append(n)
    return None


def flood_fill(grid, r, c, color, conn=4):
    """recolor the connected same-color region containing (r,c)."""
    g = G(grid)
    src = int(g[r, c])
    if src == color:
        return g
    for cells in components(g == src, conn):
        if (r, c) in cells:
            for y, x in cells:
                g[y, x] = color
            break
    return g


def ray(grid, r, c, dr, dc, color, stop=None, bg=None, overwrite=False):
    """draw from the cell AFTER (r,c) in direction (dr,dc) until a non-bg cell (or a color in
    `stop`) blocks; overwrite=True draws through. -> (grid, hit) hit=(r,c,color)|None"""
    g = G(grid)
    if bg is None:
        bg = background(g)
    H, W = g.shape
    y, x = r + dr, c + dc
    while 0 <= y < H and 0 <= x < W:
        v = int(g[y, x])
        blocked = (v in stop) if stop is not None else (v != bg)
        if blocked and not overwrite:
            return g, (y, x, v)
        g[y, x] = color
        y += dr
        x += dc
    return g, None


def symmetries(grid):
    """which symmetries hold exactly: subset of ['H','V','D','A','R90','R180']"""
    g = G(grid)
    out = []
    if np.array_equal(g, g[:, ::-1]):
        out.append('H')
    if np.array_equal(g, g[::-1, :]):
        out.append('V')
    if g.shape[0] == g.shape[1]:
        if np.array_equal(g, g.T):
            out.append('D')
        if np.array_equal(g, np.rot90(g.T, 2)):
            out.append('A')
        if np.array_equal(g, np.rot90(g, 1)):
            out.append('R90')
    if np.array_equal(g, np.rot90(g, 2)):
        out.append('R180')
    return out


def symmetrize(grid, axes='HV', bg=None):
    """fill background cells from their mirror images (axes subset of 'H','V','D','A')."""
    g = G(grid)
    if bg is None:
        bg = background(g)
    for _ in range(2):
        for ax in axes:
            if ax == 'H':
                m = g[:, ::-1]
            elif ax == 'V':
                m = g[::-1, :]
            elif ax == 'D' and g.shape[0] == g.shape[1]:
                m = g.T
            elif ax == 'A' and g.shape[0] == g.shape[1]:
                m = np.rot90(g.T, 2)
            else:
                continue
            g = np.where(g == bg, m, g)
    return g


# ----------------------------------------------------------------------------- geometry / editing
def d4(grid, k):
    """k in id/rot90/rot180/rot270/flipH/flipV/transpose/anti"""
    return _D4[k](G(grid)).copy()


def d4_all(grid):
    return {k: f(G(grid)).copy() for k, f in _D4.items()}


def crop(grid, box):
    """box=[r0,c0,r1,c1] inclusive"""
    r0, c0, r1, c1 = box
    return G(grid)[r0:r1 + 1, c0:c1 + 1].copy()


def paste(grid, sub, r, c, transparent=-1):
    """overwrite sub onto grid at (r,c); cells equal to `transparent` are skipped; clipped at edges."""
    g, s = G(grid), G(sub)
    H, W = g.shape
    for dy in range(s.shape[0]):
        for dx in range(s.shape[1]):
            y, x = r + dy, c + dx
            v = s[dy, dx]
            if 0 <= y < H and 0 <= x < W and v != transparent:
                g[y, x] = v
    return g


def scale(grid, k):
    """each cell -> k×k block"""
    return np.kron(G(grid), np.ones((k, k), dtype=int))


def downscale(grid, k):
    """k×k blocks -> one cell (block mode)"""
    g = G(grid)
    H, W = g.shape[0] // k, g.shape[1] // k
    out = np.zeros((H, W), dtype=int)
    for y in range(H):
        for x in range(W):
            out[y, x] = _mode(g[y * k:(y + 1) * k, x * k:(x + 1) * k])[0]
    return out


def tile(grid, ny, nx):
    return np.tile(G(grid), (ny, nx))


def recolor(grid, mapping, default=None):
    """mapping {from:to} (or (from,to) pair). Colors not in mapping keep (or -> default)."""
    g = G(grid)
    if not isinstance(mapping, dict):
        mapping = {mapping[0]: mapping[1]}
    out = g.copy() if default is None else np.full_like(g, default)
    for s, d in mapping.items():
        out[g == s] = d
    return out


def gravity(grid, direction='down', bg=None):
    """let every non-bg cell fall in `direction` ('down','up','left','right') within its column/row."""
    g = G(grid)
    if bg is None:
        bg = background(g)
    if direction in ('left', 'right'):
        return gravity(g.T, 'up' if direction == 'left' else 'down', bg).T
    out = np.full_like(g, bg)
    for x in range(g.shape[1]):
        col = [v for v in g[:, x].tolist() if v != bg]
        if direction == 'down':
            out[g.shape[0] - len(col):, x] = col
        else:
            out[:len(col), x] = col
    return out


def node_grid(node, shape=None, bg=0):
    """grid containing only this node (crop, or placed at its bbox inside `shape`)."""
    if shape is None:
        c = node['crop'].copy()
        c[c == -1] = bg
        return c
    g = np.full(shape, bg, dtype=int)
    return paste(g, node['crop'], node['bbox'][0], node['bbox'][1])


def erase(grid, node, bg=None):
    g = G(grid)
    if bg is None:
        bg = background(g)
    for y, x in node['cells']:
        g[y, x] = bg
    return g


def move(grid, node, dr, dc, bg=None):
    """erase node and redraw it shifted by (dr,dc)."""
    g = erase(grid, node, bg)
    return paste(g, node['crop'], node['bbox'][0] + dr, node['bbox'][1] + dc)


def fill_holes(grid, node, color):
    """fill the enclosed empty pockets inside node's crop with color."""
    g = G(grid)
    r0, c0 = node['bbox'][0], node['bbox'][1]
    m = node['crop'] == -1
    for cells in components(m, conn=4):
        if not any(y == 0 or x == 0 or y == m.shape[0] - 1 or x == m.shape[1] - 1 for y, x in cells):
            for y, x in cells:
                g[r0 + y, c0 + x] = color
    return g


def outline_cells(node):
    """cells of the node that touch (4-conn) a cell outside the node -> list (grid coords)."""
    s = set(node['cells'])
    return [(y, x) for y, x in node['cells'] if any((y + dy, x + dx) not in s for dy, dx in _N4)]


def find(pat, big, ignore=-1):
    """first (r,c) where pattern (cells == ignore are wildcards) occurs inside big, else None."""
    p, b = G(pat), G(big)
    ph, pw = p.shape
    H, W = b.shape
    if ph > H or pw > W:
        return None
    m = p != ignore
    for y in range(H - ph + 1):
        for x in range(W - pw + 1):
            if np.array_equal(b[y:y + ph, x:x + pw][m], p[m]):
                return (y, x)
    return None


def find_all(pat, big, ignore=-1):
    p, b = G(pat), G(big)
    ph, pw = p.shape
    H, W = b.shape
    m = p != ignore
    out = []
    for y in range(H - ph + 1):
        for x in range(W - pw + 1):
            if np.array_equal(b[y:y + ph, x:x + pw][m], p[m]):
                out.append((y, x))
    return out


def count_colors(grid, exclude_bg=True, bg=None):
    """{color: n} descending"""
    g = G(grid)
    c = Counter(g.ravel().tolist())
    if exclude_bg:
        c.pop(background(g) if bg is None else bg, None)
    return {int(k): int(v) for k, v in c.most_common()}


def new_grid(h, w, color=0):
    return np.full((h, w), color, dtype=int)


def size_relation(pairs):
    """how output size relates to input size across pairs: {'kind': same|scale|downscale|tile|
    fixed|ratio|smaller|larger|mixed, ...}. For smaller/larger it also reports whether the
    output occurs inside the input (crop) / the input inside the output."""
    rel = []
    for p in pairs:
        (hi, wi), (ho, wo) = G(p['input']).shape, G(p['output']).shape
        rel.append(((hi, wi), (ho, wo)))
    if all(a == b for a, b in rel):
        return {'kind': 'same'}
    ins_, outs = [a for a, _ in rel], [b for _, b in rel]
    fixed_out = len(set(outs)) == 1
    if fixed_out and len(set(ins_)) > 1:
        return {'kind': 'fixed', 'shape': list(outs[0])}
    rat = [(b[0] / a[0], b[1] / a[1]) for a, b in rel]
    if len(set(rat)) == 1:
        rh, rw = rat[0]
        extra = {'also_fixed_shape': list(outs[0])} if fixed_out else {}
        if rh == rw and rh > 1 and float(rh).is_integer():
            return {'kind': 'scale', 'k': int(rh), **extra}
        if rh == rw and rh < 1 and float(1 / rh).is_integer():
            return {'kind': 'downscale', 'k': int(round(1 / rh)), **extra}
        if rh >= 1 and rw >= 1 and float(rh).is_integer() and float(rw).is_integer():
            return {'kind': 'tile', 'ny': int(rh), 'nx': int(rw), **extra}
        if not fixed_out:
            return {'kind': 'ratio', 'ratio': [rh, rw]}
    if fixed_out:
        return {'kind': 'fixed', 'shape': list(outs[0])}
    shapes = [f'{a[0]}x{a[1]}->{b[0]}x{b[1]}' for a, b in rel]
    if all(b[0] <= a[0] and b[1] <= a[1] for a, b in rel):
        pos = [find(p['output'], p['input']) for p in pairs]
        return {'kind': 'smaller', 'shapes': shapes,
                'output_found_inside_input_at': [list(x) if x else None for x in pos]}
    if all(b[0] >= a[0] and b[1] >= a[1] for a, b in rel):
        pos = [find(p['input'], p['output']) for p in pairs]
        return {'kind': 'larger', 'shapes': shapes,
                'input_found_inside_output_at': [list(x) if x else None for x in pos]}
    return {'kind': 'mixed', 'shapes': shapes}


SANDBOX_HELPERS = [
    'G', 'grid_str', 'show', 'valid_grid', 'background', 'infer_background', 'components', 'objects', 'summ',
    'bbox', 'center', 'manhattan', 'nodes_by_color', 'touching', 'neighbors', 'same_shape', 'match_nodes',
    'match_summary', 'diff_cells', 'color_map', 'compare', 'separators', 'panels', 'enclosed_regions',
    'fill_region', 'reachable_cells', 'bfs_path', 'flood_fill', 'ray', 'symmetries', 'symmetrize', 'd4',
    'd4_all', 'crop', 'paste', 'scale', 'downscale', 'tile', 'recolor', 'gravity', 'node_grid', 'erase',
    'move', 'fill_holes', 'outline_cells', 'find', 'find_all', 'count_colors', 'new_grid', 'size_relation',
]
'''
compile(HELPERS_SRC, 'helpers.py', 'exec')
_ns = {'__name__': '__selftest__'}
exec(HELPERS_SRC, _ns)
# ---- 자가검증 (즉시 실행) ----
_g = _ns['G']([[0,0,0,0],[0,1,1,0],[0,1,0,0],[2,0,0,0]])
assert _ns['background'](_g) == 0
_ob = _ns['objects'](_g)
assert [o['color'] for o in _ob] == [1, 2] and _ob[0]['pixels'] == 3 and _ob[0]['bbox'] == [1, 1, 2, 2]
_m = _ns['match_nodes'](_ob, _ns['objects'](_ns['move'](_g, _ob[0], 0, 1)))
assert _m['moved'] and _m['moved'][0]['shift'] == [0, 1] and _m['stable'] == 1, _m
_m2 = _ns['match_nodes'](_ob, _ns['objects'](_ns['recolor'](_g, {1: 3})))
assert _m2['recolored'] and _m2['recolored'][0]['color_to'] == 3, _m2
assert _ns['diff_cells'](_g, _ns['recolor'](_g, {1: 3}))['transitions'] == {'1->3': 3}
assert _ns['size_relation']([{'input': _g, 'output': _ns['scale'](_g, 2)}])['kind'] == 'scale'
_w = _ns['G']([[0,1,0],[0,1,0],[0,1,0]])
assert _ns['bfs_path'](_g, (0, 0), (3, 3), {0}) is not None and _ns['bfs_path'](_w, (0, 0), (0, 2), {0}) is None
assert len(_ns['enclosed_regions'](_ns['G']([[1,1,1],[1,0,1],[1,1,1]]), 0)) == 1
assert _ns['infer_background']([{'input': _g, 'output': _g}])['bg'] == 0
print(f'셀3 OK — 헬퍼 {len(_ns["SANDBOX_HELPERS"])}종 / 소스 {len(HELPERS_SRC):,}자')


셀3 OK — 헬퍼 50종 / 소스 30,937자


In [5]:
# [셀4] 매크로 + ★서술함수 소스 + 워커 스크립트 — task_report / try_transforms / check(등급 되먹임) / submit + 관찰카드·정합체크·4축·silx·단계·반례·렌즈증거·씨앗
#   매크로·서술함수는 TRAIN/TEST/BG/REPORT/_STATE 를 읽는다 (킷 sandbox_macros 의 리바인딩 대응: 같은 네임스페이스에 exec).
MACROS_SRC = r'''# ===================================================================================
# ARC-2 SANDBOX MACROS (MACROS 강화판) — TRAIN/TEST/BG/REPORT/_STATE 를 읽는 고수준 매크로.
#   설계 계약 (duck-merge-kit sandbox_macros.py 의 교훈 반영):
#     * 매크로는 프리미티브를 조합하고, 결과를 TRAIN 에 대해 **검증**해서 한 줄 요약만 찍는다.
#     * ARC-3 의 MUST 강제(opening_scan 이 첫 호출) 대신 — task_report() 는 하네스가 0턴에
#       모델 대신 실행해 결과를 첫 메시지에 넣는다 (모델 턴 0 소비, MUST 충돌 0).
#     * check() 는 게임 액션이 아니라 CPU 만 쓰므로 무료 — submit 전 필수. submit 은 train
#       전쌍 통과가 아니면 거절(force 만 예외). 매크로가 '분산 증폭기'가 되지 않도록,
#       try_transforms 의 후보는 전부 train 으로 채점된 뒤에만 보고된다.
# ===================================================================================
import signal as _signal
import time as _time

TRAIN, TEST, BG, REPORT = [], [], 0, {}
BANK = {}
_STATE = {'candidates': [], 'submitted': None, 'turn': 0, 'code': '', 'events': [], 'seen': set(), 'exclude_seeds': set()}
SEED_K = 3           # 첫 메시지에 인용할 씨앗 서술 수 (자기 자신 제외)


class _SolveTimeout(Exception):
    pass


def _alarm(signum, frame):
    raise _SolveTimeout()


def _run_solve(solve, grid, timeout=5):
    """(output ndarray | None, error str | None) — runs solve on a COPY with a timeout."""
    try:
        _signal.signal(_signal.SIGALRM, _alarm)
        _signal.alarm(int(timeout))
    except Exception:
        pass
    try:
        out = solve(G(grid).copy())
        try:
            _signal.alarm(0)
        except Exception:
            pass
        if out is None:
            return None, 'solve returned None'
        a = np.asarray(out)
        if a.dtype == object:
            return None, 'output is ragged (rows of different length)'
        if a.dtype.kind == 'f' and a.size and np.all(np.mod(a, 1) == 0):
            a = a.astype(int)
        a = a.astype(int) if a.dtype.kind in 'iub' else None
        if a is None or a.ndim != 2 or a.size == 0:
            return None, f'output must be a 2D int grid (got shape {np.asarray(out).shape}, dtype {np.asarray(out).dtype})'
        if a.shape[0] > MAX_SIDE or a.shape[1] > MAX_SIDE:
            return None, f'output too large {a.shape} (max 30x30)'
        if a.min() < 0 or a.max() > 9:
            return None, f'output has values outside 0-9 (min {int(a.min())}, max {int(a.max())})'
        return a, None
    except _SolveTimeout:
        return None, f'timeout (> {timeout}s)'
    except Exception as e:
        try:
            _signal.alarm(0)
        except Exception:
            pass
        return None, f'{type(e).__name__}: {str(e)[:160]}'


def _init_task(task):
    """called once by the worker: sets TRAIN/TEST/BG and REPORT (runs task_report quietly)."""
    global TRAIN, TEST, BG, REPORT
    TRAIN = [{'input': G(p['input']), 'output': G(p['output'])} for p in task['train']]
    TEST = [{'input': G(t['input'])} for t in task['test']]
    for p in TRAIN + TEST:                     # read-only: in-place edits raise instead of corrupting the task
        for a in p.values():
            a.flags.writeable = False
    BG = infer_background(TRAIN)['bg']
    _STATE['exclude_seeds'] = {task.get('tid')} if task.get('tid') else set()
    _STATE['tid'] = task.get('tid')
    for _c in (_NULL, _SIL_NULL) if '_NULL' in globals() else ():
        _c.clear()
    return BG


# ----------------------------------------------------------------------------- report
def _fmt_shape(g):
    return f'{g.shape[0]}x{g.shape[1]}'


def task_report(verbose=True):
    """Opening scan of the task (run by the harness at turn 0): sizes + size rule, background
    evidence, colors, cell-wise color map, change statistics, object matching per pair,
    separators/panels, symmetries, whole-grid D4, transform bank top. Sets REPORT."""
    global REPORT, BG
    R = {}
    L = []
    ins = [p['input'] for p in TRAIN]
    outs = [p['output'] for p in TRAIN]
    n = len(TRAIN)
    sizes = ', '.join(f'{_fmt_shape(i)}->{_fmt_shape(o)}' for i, o in zip(ins, outs))
    R['sizes'] = sizes
    R['size'] = size_relation(TRAIN)
    L.append(f"pairs: {n} train / {len(TEST)} test | sizes: {sizes} | test in: {', '.join(_fmt_shape(t['input']) for t in TEST)}")
    L.append(f"size rule: {json.dumps(R['size'])}")
    bgi = infer_background(TRAIN)
    BG = bgi['bg']
    R['background'] = bgi
    ev = bgi['evidence']
    same = R['size']['kind'] == 'same'
    # per-grid background actually used for object extraction below: the task-level BG when the
    # evidence is solid, otherwise each grid's own background (backgrounds differ per pair).
    pair_bg = [BG if bgi['confidence'] != 'low' else background(i) for i in ins]
    out_bg = [pb if same else background(o) for pb, o in zip(pair_bg, outs)]
    R['pair_bg'] = pair_bg
    L.append(f"background BG={BG} (confidence {bgi['confidence']}) evidence: mode_in {ev['mode_in']} mode_out {ev['mode_out']} border {ev['border_in']} vacated {ev['vacated']} score {ev['score']}"
             + (f" | per-grid bg differs -> objects below use per-pair bg {pair_bg}; inside solve use background(grid)" if bgi['confidence'] == 'low' else ''))
    ci = [sorted(set(i.ravel().tolist())) for i in ins]
    co = [sorted(set(o.ravel().tolist())) for o in outs]
    allin = sorted(set(c for cc in ci for c in cc))
    allout = sorted(set(c for cc in co for c in cc))
    new = sorted(set(allout) - set(allin))
    gone = sorted(set(allin) - set(allout))
    R['colors'] = {'in': allin, 'out': allout, 'new_in_output': new, 'vanished': gone,
                   'per_pair_in': ci, 'per_pair_out': co}
    L.append(f"colors: in {allin} out {allout} | new in output {new} | vanished {gone} | per pair in->out: " +
             '; '.join(f'{a}->{b}' for a, b in zip(ci, co)))
    if same:
        fixed = [c for c in allin if all(np.array_equal(i == c, o == c) for i, o in zip(ins, outs))]
        R['fixed_colors'] = fixed
        cm = color_map(TRAIN)
        R['color_map'] = cm
        chg = [int((i != o).sum()) for i, o in zip(ins, outs)]
        pct = [round(100.0 * c / i.size, 1) for c, i in zip(chg, ins)]
        R['changed_cells'] = chg
        trans = Counter()
        for i, o in zip(ins, outs):
            trans.update(diff_cells(i, o, max_cells=0)['transitions'])
        R['transitions'] = dict(trans.most_common(8))
        L.append(f"fixed colors (cells identical in & out): {fixed} | cells changed per pair: {chg} ({pct}%) | transitions from->to {R['transitions']}")
        if cm['consistent'] and cm['changed']:
            L.append(f"cell-wise color map is CONSISTENT: {cm['map']} (changed {cm['changed']}) -> a pure recolor may solve it (BANK['colormap'])")
        elif cm['changed']:
            L.append(f"cell-wise color map: majority {cm['map']} but INCONSISTENT for colors {[c for c, _ in cm['conflicts']]} -> the change depends on position/shape, not on color alone")
        else:
            L.append("cell-wise color map: majority identity -> changes are position/shape dependent (which cells change? see transitions and object matching)")
    # objects + matching
    mlines, objs_in, objs_out = [], [], []
    dense = False
    for k, (i, o) in enumerate(zip(ins, outs)):
        oi = objects(i, pair_bg[k], 8, True)
        oo = objects(o, out_bg[k], 8, True)
        objs_in.append(oi)
        objs_out.append(oo)
        if len(oi) > 40 or len(oo) > 40:
            dense = True
            mlines.append(f"  pair{k}: objects {len(oi)}->{len(oo)} (dense -- matching skipped; use objects_report/diff_pair selectively)")
            continue
        m = match_nodes(oi, oo)
        mlines.append(f"  pair{k}: objects {len(oi)}->{len(oo)} | {match_summary(m)}")
    R['objects_in'] = [len(x) for x in objs_in]
    R['objects_out'] = [len(x) for x in objs_out]
    L.append("objects (single-color, 8-conn, non-bg) and input->output matching (match_nodes):")
    L.extend(mlines[:4])
    if n > 4:
        L.append(f"  ... {n - 4} more pairs (diff_pair(i) / objects_report(i) show them)")
    # multi-color glued objects
    glued = [objects(i, pair_bg[k], 8, False) for k, i in enumerate(ins)]
    R['objects_glued'] = [len(x) for x in glued]
    L.append(f"multi-color glued objects per input (by_color=False): {R['objects_glued']} | largest px: {[max((o['pixels'] for o in x), default=0) for x in glued]}")
    # separators / panels
    seps = [separators(i, pair_bg[k]) for k, i in enumerate(ins)]
    if all(s[2] is not None and (s[0] or s[1]) for s in seps):
        pn = [len(panels(i, pair_bg[k])) for k, i in enumerate(ins)]
        R['separators'] = {'color': [s[2] for s in seps], 'rows': [s[0] for s in seps], 'cols': [s[1] for s in seps], 'panels': pn}
        L.append(f"separator lines (full non-bg lines): colors {[s[2] for s in seps]} rows {[s[0] for s in seps]} cols {[s[1] for s in seps]} -> panels per input {pn} (panels(grid) returns [(bbox, subgrid)])")
    else:
        R['separators'] = None
    # symmetries / D4
    syi = [symmetries(i) for i in ins]
    syo = [symmetries(o) for o in outs]
    R['symmetries'] = {'in': syi, 'out': syo}
    if any(syi) or any(syo):
        L.append(f"exact symmetries: inputs {syi} outputs {syo}")
    d4rel = []
    for i, o in zip(ins, outs):
        r = next((k for k, f in _D4.items() if f(i).shape == o.shape and np.array_equal(f(i), o)), None)
        d4rel.append(r)
    R['d4_relation'] = d4rel
    if any(d4rel):
        L.append(f"whole-grid D4 relation input->output: {d4rel}")
    # enclosed regions
    enc = [len(enclosed_regions(i, pair_bg[k])) for k, i in enumerate(ins)]
    R['enclosed_in'] = enc
    if any(enc) and not dense:
        L.append(f"enclosed background pockets (holes) per input: {enc} (enclosed_regions(grid) lists their cells)")
    # transform bank
    try:
        top = try_transforms(top=6, verbose=False)
        R['bank_top'] = top
        try:
            _seed_bank_candidates(top)
        except Exception:
            pass
        if top:
            L.append("transform bank (fit on train; BANK[name](grid) reusable): " +
                     ' | '.join(f"{t['name']} {t['pairs_ok']}/{n} acc {t['acc']:.2f}" for t in top))
            if top[0]['pairs_ok'] == n:
                L.append(f"** BANK['{top[0]['name']}'] reproduces ALL train pairs — verify with check(BANK['{top[0]['name']}']) and submit if the test output looks right **")
    except Exception as e:
        L.append(f"transform bank failed: {type(e).__name__}: {str(e)[:80]}")
    # ---- 서술 층 (narrate.py 가 같은 네임스페이스에 exec 되어 있을 때): 관찰카드 · 렌즈 증거 · 널 기준선 · 씨앗 인용 ----
    if 'observe_card' in globals():
        try:
            card = observe_card()
            R['card'] = card
            L.append(card_text(card))
        except Exception as e:
            L.append(f'observation card failed: {type(e).__name__}: {str(e)[:80]}')
        try:
            ev = lens_evidence()
            R['lens_evidence'] = ev
            if ev:
                L.append('LENS EVIDENCE (measured preconditions of catalog lenses -- candidates to test, not verdicts):')
                for e in ev:
                    L.append(f"  - {e['lens']}: {e['evidence']}")
            else:
                L.append('LENS EVIDENCE: no catalog-lens precondition fires -> reason from the 8 perspectives and the object matching above')
        except Exception as e:
            R['lens_evidence'] = []
            L.append(f'lens evidence failed: {type(e).__name__}: {str(e)[:80]}')
        try:
            s0, s0s = null_base(), sil_null()
            R['null'] = {'axes': s0, 'silx': s0s}
            L.append(f"null baselines (what identity / background-fill would score here): 4-axis {s0} (= normalized 0), silhouette {s0s}"
                     + (' -> the silhouette barely changes in this task: the MASK stage is nearly free, the work is in COLOR' if s0s >= 0.95 else ''))
        except Exception as e:
            L.append(f'null baseline failed: {type(e).__name__}: {str(e)[:80]}')
        try:
            shots = seed_shots(R.get('card') or observe_card(), R.get('lens_evidence', []), exclude=_STATE.get('exclude_seeds', set()), k=SEED_K)
            R['seed_shots'] = [t for t, _, _ in shots]
            if shots:
                L.append('SIMILAR SOLVED PUZZLES (verified OBJECTS/STEPS descriptions of OTHER tasks -- same style, different content):')
                for tid_, lens_, rule_ in shots:
                    L.append(f'  [{lens_}] ' + rule_.replace('\n', '\n      '))
        except Exception as e:
            L.append(f'seed shots failed: {type(e).__name__}: {str(e)[:80]}')
    REPORT = R
    text = '== TASK REPORT ==\n' + '\n'.join(L)
    if verbose:
        print(text)
    return R


# ----------------------------------------------------------------------------- transform bank
def _eval_fn(fn, pairs):
    ok, acc = 0, 0.0
    for p in pairs:
        try:
            out = fn(p['input'].copy())
            if out is None:
                continue
            out = G(out)
            t = p['output']
            if out.shape == t.shape:
                a = float((out == t).mean())
                acc += a
                ok += int(a == 1.0)
        except Exception:
            continue
    return ok, acc / max(1, len(pairs))


def _build_bank(pairs, bg):
    B = {}
    ins = [p['input'] for p in pairs]
    outs = [p['output'] for p in pairs]
    n = len(pairs)
    B['identity'] = lambda g: g
    for k in D4_KEYS[1:]:
        B[f'd4_{k}'] = (lambda kk: (lambda g: d4(g, kk)))(k)
    rel = size_relation(pairs)
    cm = color_map(pairs)
    if cm['same_size'] and cm['changed']:
        mp = dict(cm['map'])
        B['colormap'] = lambda g, mp=mp: recolor(g, mp)
        for k in D4_KEYS[1:]:
            B[f'd4_{k}+colormap'] = (lambda kk, mp=mp: (lambda g: recolor(d4(g, kk), mp)))(k)
    if rel['kind'] == 'scale':
        k = rel['k']
        B[f'scale_x{k}'] = lambda g, k=k: scale(g, k)
    if rel['kind'] == 'downscale':
        k = rel['k']
        B[f'downscale_x{k}'] = lambda g, k=k: downscale(g, k)
    if rel['kind'] == 'tile':
        ny, nx = rel['ny'], rel['nx']
        B[f'tile_{ny}x{nx}'] = lambda g, ny=ny, nx=nx: tile(g, ny, nx)
        if ny == 2 and nx == 2:
            B['tile_mirror_2x2'] = lambda g: np.block([[g, g[:, ::-1]], [g[::-1, :], g[::-1, ::-1]]])
            B['tile_rot_2x2'] = lambda g: np.block([[g, np.rot90(g, -1)], [np.rot90(g, 1), np.rot90(g, 2)]]) if g.shape[0] == g.shape[1] else None
        if ny == 1 and nx == 2:
            B['concat_mirror_h'] = lambda g: np.hstack([g, g[:, ::-1]])
        if ny == 2 and nx == 1:
            B['concat_mirror_v'] = lambda g: np.vstack([g, g[::-1, :]])
        if ny == 3 and nx == 3:
            B['tile_by_self_mask'] = lambda g: np.kron((g != bg).astype(int), g)
    if rel['kind'] in ('smaller', 'mixed', 'fixed'):
        def _crop_nonbg(g):
            b = background(g)
            rs, cs = np.nonzero(g != b)
            if len(rs) == 0:
                return None
            return g[rs.min():rs.max() + 1, cs.min():cs.max() + 1]
        B['crop_nonbg'] = _crop_nonbg

        def _crop_obj(g, which, by_color):
            ob = objects(g, None, 8, by_color)
            if not ob:
                return None
            o = (max if which == 'largest' else min)(ob, key=lambda x: x['pixels'])
            return crop(g, o['bbox'])
        for which in ('largest', 'smallest'):
            for byc in (True, False):
                B[f'crop_{which}_obj{"_color" if byc else "_glued"}'] = (lambda w, bc: (lambda g: _crop_obj(g, w, bc)))(which, byc)

        def _crop_unique_color(g):
            ob = objects(g, None, 8, True)
            cnt = Counter(o['color'] for o in ob)
            u = [o for o in ob if cnt[o['color']] == 1]
            if len(u) != 1:
                return None
            return crop(g, u[0]['bbox'])
        B['crop_unique_color_obj'] = _crop_unique_color
        # panel logic
        if all(separators(i, bg)[2] is not None for i in ins):
            outcols = sorted(set(int(v) for o in outs for v in np.unique(o)) - {bg})
            paint = outcols[0] if len(outcols) == 1 else None

            def _panel_op(g, op, paint=paint):
                ps = [p for _, p in panels(g)]
                if len(ps) < 2 or len(set(p.shape for p in ps)) != 1:
                    return None
                b = background(g)
                masks = [p != b for p in ps]
                if op == 'and':
                    m = np.logical_and.reduce(masks)
                elif op == 'or':
                    m = np.logical_or.reduce(masks)
                elif op == 'xor':
                    m = np.logical_xor.reduce(masks)
                elif op == 'nor':
                    m = ~np.logical_or.reduce(masks)
                elif op == 'first':
                    return ps[0]
                elif op == 'last':
                    return ps[-1]
                elif op == 'overlay_first':
                    out = ps[-1].copy()
                    for p in ps[-2::-1]:
                        out = np.where(p != b, p, out)
                    return out
                elif op == 'overlay_last':
                    out = ps[0].copy()
                    for p in ps[1:]:
                        out = np.where(p != b, p, out)
                    return out
                else:
                    return None
                c = paint if paint is not None else b
                return np.where(m, c, b)
            for op in ('and', 'or', 'xor', 'nor', 'first', 'last', 'overlay_first', 'overlay_last'):
                B[f'panel_{op}'] = (lambda o: (lambda g: _panel_op(g, o)))(op)
    if rel['kind'] == 'fixed':
        if all(np.array_equal(outs[0], o) for o in outs):
            B['const_output'] = lambda g, o=outs[0]: o.copy()
        if tuple(rel['shape']) == (1, 1):
            B['mode_color_1x1'] = lambda g: np.array([[Counter(v for v in g.ravel().tolist() if v != background(g)).most_common(1)[0][0]]])
            B['largest_obj_color_1x1'] = lambda g: np.array([[max(objects(g, None, 8, True), key=lambda o: o['pixels'])['color']]])
            B['rarest_color_1x1'] = lambda g: np.array([[Counter(v for v in g.ravel().tolist() if v != background(g)).most_common()[-1][0]]])
    if rel['kind'] == 'same':
        for d in ('down', 'up', 'left', 'right'):
            B[f'gravity_{d}'] = (lambda dd: (lambda g: gravity(g, dd, background(g))))(d)
        for ax in ('H', 'V', 'HV', 'D', 'A', 'HVDA'):
            B[f'symmetrize_{ax}'] = (lambda a: (lambda g: symmetrize(g, a, background(g))))(ax)
        # fill enclosed pockets with a learned color
        fillc = Counter()
        for i, o in zip(ins, outs):
            for cells in enclosed_regions(i, bg):
                fillc.update(int(o[y, x]) for y, x in cells)
        if fillc:
            c = fillc.most_common(1)[0][0]
            if c != bg:
                def _fill_enclosed(g, c=c):
                    out = g.copy()
                    for cells in enclosed_regions(g, background(g)):
                        out = fill_region(out, cells, c)
                    return out
                B[f'fill_enclosed_{c}'] = _fill_enclosed
        cols_in = sorted(set(int(v) for i in ins for v in np.unique(i)) - {bg})
        for c in cols_in:
            B[f'remove_color_{c}'] = (lambda cc: (lambda g: recolor(g, {cc: background(g)})))(c)
            B[f'keep_only_{c}'] = (lambda cc: (lambda g: np.where(g == cc, g, background(g))))(c)
        # object holes filled with learned color
        holec = Counter()
        for i, o in zip(ins, outs):
            for ob in objects(i, bg, 8, False):
                if ob['holes']:
                    r0, c0 = ob['bbox'][0], ob['bbox'][1]
                    m = ob['crop'] == -1
                    for cells in components(m, 4):
                        if not any(y == 0 or x == 0 or y == m.shape[0] - 1 or x == m.shape[1] - 1 for y, x in cells):
                            holec.update(int(o[r0 + y, c0 + x]) for y, x in cells)
        if holec:
            c = holec.most_common(1)[0][0]
            if c != bg:
                def _fill_obj_holes(g, c=c):
                    out = g.copy()
                    for ob in objects(g, None, 8, False):
                        if ob['holes']:
                            out = fill_holes(out, ob, c)
                    return out
                B[f'fill_object_holes_{c}'] = _fill_obj_holes
        # connect same-color cells on a row/col with a line of that color
        def _connect(g):
            out = g.copy()
            b = background(g)
            for c in sorted(set(g.ravel().tolist()) - {b}):
                pts = list(zip(*np.nonzero(g == c)))
                for (y0, x0), (y1, x1) in itertools.combinations(pts, 2):
                    if y0 == y1:
                        seg = out[y0, min(x0, x1):max(x0, x1) + 1]
                        if ((seg == b) | (seg == c)).all():
                            out[y0, min(x0, x1):max(x0, x1) + 1] = c
                    elif x0 == x1:
                        seg = out[min(y0, y1):max(y0, y1) + 1, x0]
                        if ((seg == b) | (seg == c)).all():
                            out[min(y0, y1):max(y0, y1) + 1, x0] = c
            return out
        B['connect_same_color_hv'] = _connect
        # outline / hollow objects
        def _hollow(g):
            out = g.copy()
            b = background(g)
            for ob in objects(g, b, 8, True):
                keep = set(outline_cells(ob))
                for y, x in ob['cells']:
                    if (y, x) not in keep:
                        out[y, x] = b
            return out
        B['hollow_objects'] = _hollow
        # largest / smallest object only
        def _keep_obj(g, which):
            b = background(g)
            ob = objects(g, b, 8, True)
            if not ob:
                return None
            o = (max if which == 'largest' else min)(ob, key=lambda x: x['pixels'])
            return node_grid(o, g.shape, b)
        B['keep_largest_obj'] = lambda g: _keep_obj(g, 'largest')
        B['keep_smallest_obj'] = lambda g: _keep_obj(g, 'smallest')
        # recolor objects by size rank (learned): color of each output object as function of pixel count
        sz = {}
        okmap = True
        for i, o in zip(ins, outs):
            for ob in objects(i, bg, 8, False):
                oc = Counter(int(o[y, x]) for y, x in ob['cells']).most_common(1)[0][0]
                if sz.setdefault(ob['pixels'], oc) != oc:
                    okmap = False
        if okmap and sz and len(set(sz.values())) > 1:
            def _recolor_by_size(g, sz=sz):
                out = g.copy()
                for ob in objects(g, background(g), 8, False):
                    if ob['pixels'] in sz:
                        for y, x in ob['cells']:
                            out[y, x] = sz[ob['pixels']]
                return out
            B['recolor_objects_by_size'] = _recolor_by_size
    return B


def try_transforms(top=8, verbose=True):
    """fit ~50 parametrized whole-grid transforms on TRAIN, rank by (pairs exact, cell accuracy).
    All candidates stay callable as BANK[name](grid). Prints the top list."""
    global BANK
    BANK = _build_bank(TRAIN, BG)
    res = []
    for name, fn in BANK.items():
        try:
            ok, acc = _eval_fn(fn, TRAIN)
        except Exception:
            continue
        if ok > 0 or acc >= 0.5:
            res.append({'name': name, 'pairs_ok': ok, 'acc': round(acc, 3)})
    res.sort(key=lambda r: (-r['pairs_ok'], -r['acc']))
    res = res[:top]
    if verbose:
        n = len(TRAIN)
        if not res:
            print(f'try_transforms: none of {len(BANK)} bank transforms reaches 50% cell accuracy — the rule is object/position specific')
        else:
            print(f'try_transforms ({len(BANK)} candidates): ' + ' | '.join(f"{r['name']} {r['pairs_ok']}/{n} acc {r['acc']:.2f}" for r in res))
    return res


def _seed_bank_candidates(top, max_n=2, min_pairs=1):
    """(v58) train 쌍을 min_pairs 개 이상 맞춘 은행 변환 상위 max_n 개를 test 예측과 함께 후보로 등록.
    모델이 유효한 solve 를 못 내면 attempt 가 identity(항상 오답) 대신 이 예측이 된다 (비용 0)."""
    k = 0
    for t in top or []:
        if k >= max_n:
            break
        if t.get('pairs_ok', 0) < min_pairs:
            continue
        fn = BANK.get(t['name'])
        if fn is None:
            continue
        preds = []
        for te in TEST:
            out, err = _run_solve(fn, te['input'], 5)
            preds.append(None if out is None else out.tolist())
        if any(p is None for p in preds):
            continue
        res = {'pairs_ok': t['pairs_ok'], 'n': len(TRAIN), 'acc': t.get('acc', 0.0), 'test_preds': preds,
               'nscore': None, 'stage': None, 'silx': None}
        _register('bank:' + t['name'], res)
        k += 1
    return k


# ----------------------------------------------------------------------------- check / submit
def _register(name, res, force=False):
    key = json.dumps(res['test_preds'])
    rec = {'name': name, 'turn': _STATE['turn'], 'code': _STATE['code'], 'pairs_ok': res['pairs_ok'],
           'n': res['n'], 'acc': res['acc'], 'all_pass': res['pairs_ok'] == res['n'] and res['n'] > 0,
           'test_preds': res['test_preds'], 'test_ok': all(p is not None for p in res['test_preds']),
           'forced': force, 'nscore': res.get('nscore'), 'stage': res.get('stage'), 'silx': res.get('silx'),
           'desc': _STATE.get('desc', '')}
    if key in _STATE['seen']:
        # same predictions as before: keep the best score but do not duplicate
        for c in _STATE['candidates']:
            if json.dumps(c['test_preds']) != key:
                continue
            if (c['pairs_ok'], c['acc']) < (rec['pairs_ok'], rec['acc']):
                c.update(pairs_ok=rec['pairs_ok'], acc=rec['acc'], all_pass=rec['all_pass'], code=rec['code'], turn=rec['turn'],
                         nscore=rec['nscore'], stage=rec['stage'], silx=rec['silx'], desc=rec['desc'])
            elif rec.get('desc') and not c.get('desc'):
                c.update(desc=rec['desc'], code=rec['code'], turn=rec['turn'], model_desc=True)
        return rec
    _STATE['seen'].add(key)
    _STATE['candidates'].append(rec)
    _STATE['events'].append({'type': 'candidate', 'name': name, 'pairs_ok': rec['pairs_ok'], 'n': rec['n'], 'acc': rec['acc']})
    return rec


def check(solve, verbose=True, name='check', timeout=5):
    """Run solve on every TRAIN pair (timeout each) and on TEST inputs. Prints k/n, per-pair
    wrong-cell counts with the first wrong cells [r,c,pred,target], and predicted test shapes.
    Registers the candidate (attempts are built from registered candidates). Returns dict."""
    res = {'pairs_ok': 0, 'n': len(TRAIN), 'per_pair': [], 'acc': 0.0, 'test_preds': [], 'test_shapes': []}
    lines = []
    accs = []
    outs = []
    for i, p in enumerate(TRAIN):
        out, err = _run_solve(solve, p['input'], timeout)
        outs.append(out)
        if out is None:
            res['per_pair'].append({'i': i, 'ok': False, 'error': err})
            lines.append(f'  pair{i}: ERROR {err}')
            accs.append(0.0)
            continue
        c = compare(out, p['output'])
        if c['match']:
            res['pairs_ok'] += 1
            res['per_pair'].append({'i': i, 'ok': True})
            lines.append(f'  pair{i}: OK')
            accs.append(1.0)
        elif not c['shape_ok']:
            res['per_pair'].append({'i': i, 'ok': False, 'shape_pred': c['shape_pred'], 'shape_target': c['shape_target']})
            lines.append(f"  pair{i}: SHAPE {c['shape_pred'][0]}x{c['shape_pred'][1]} but expected {c['shape_target'][0]}x{c['shape_target'][1]}")
            accs.append(0.0)
        else:
            res['per_pair'].append({'i': i, 'ok': False, 'wrong': c['wrong'], 'wrong_cells': c['wrong_cells'][:6]})
            accs.append(1.0 - c['wrong'] / c['total'])
            lines.append(f"  pair{i}: {c['wrong']}/{c['total']} cells wrong; first [r,c,pred,target]: {c['wrong_cells'][:6]} | wrong per target color {c['per_color']}")
    res['acc'] = round(sum(accs) / max(1, len(accs)), 3)
    for j, t in enumerate(TEST):
        out, err = _run_solve(solve, t['input'], timeout)
        res['test_preds'].append(None if out is None else out.tolist())
        res['test_shapes'].append(None if out is None else list(out.shape))
        if out is None:
            lines.append(f'  test{j}: ERROR {err}')
    # ---- 등급 채점 + 단계식 되먹임 (서술함수: 4축·널정규화·silx·단계·반례) ----
    graded = []
    if 'score_preds' in globals():
        try:
            tot, axes = score_preds(outs)
            s0 = null_base()
            nn = nscore(tot, s0)
            sv = [score_silx(o, p['output']) for o, p in zip(outs, TRAIN)]
            sil = round(sum(sv) / max(1, len(sv)), 2)
            nsil = nscore_sil(sil, sil_null())
            res.update(nscore=nn, raw=tot, axes=axes, silx=sil, nsilx=nsil)
            if res['pairs_ok'] < res['n']:
                stage, facts, advice = stage_feedback(outs, axes)
                res['stage'] = stage
                graded.append(f"GRADED: normalized {nn} (raw {tot}; size {axes[0]} shape {axes[1]} color {axes[2]} position {axes[3]}) | silhouette {sil} (normalized {nsil}) | STAGE: {stage}")
                graded += ['  ' + f for f in facts[:3]]
                if stage in ('MASK', 'COLOR') and axes[3] <= min(axes) and axes[3] < 0.9:
                    for o, p in zip(outs, TRAIN):
                        if o is not None and G(o).shape == p['output'].shape and not np.array_equal(G(o), p['output']):
                            pf = pos_facts(o, p['output'], '  position')
                            if pf:
                                graded.append(pf)
                            break
                graded.append('  -> ' + advice)
                ce = counterexample(outs)
                if ce:
                    graded.append(CE_HEADER)
                    graded.append(ce)
            else:
                res['stage'] = 'DONE'
        except Exception as e:
            graded.append(f'(graded scoring failed: {type(e).__name__}: {str(e)[:80]})')
    rec = _register(name, res)
    if verbose:
        head = f"check: {res['pairs_ok']}/{res['n']} train pairs pass (cell acc {res['acc']:.2f}); test output shapes {res['test_shapes']}"
        print(head)
        for ln in lines:
            print(ln)
        for ln in graded:
            print(ln)
        if res['pairs_ok'] == res['n'] and all(p is not None for p in res['test_preds']):
            print('  -> all train pairs pass. Call submit(solve) to finish.')
    return res


def submit(solve, force=False, note=''):
    """Final answer. Runs check(); accepted only if ALL train pairs pass (or force=True, meant for
    your last turn). Accepting ends the task; the harness fills attempt_2 from your other
    registered candidates."""
    res = check(solve, verbose=True, name='submit')
    ok_all = res['pairs_ok'] == res['n'] and res['n'] > 0
    if not all(p is not None for p in res['test_preds']):
        print('submit REFUSED: solve fails on a TEST input (see errors above) — fix it first' + (' (force ignored)' if force else ''))
        return False
    if ok_all or force:
        rec = _register('submit', res, force=force)
        _STATE['submitted'] = {'test_preds': res['test_preds'], 'pairs_ok': res['pairs_ok'], 'n': res['n'],
                               'forced': (not ok_all), 'note': note, 'turn': _STATE['turn'], 'code': _STATE['code']}
        _STATE['events'].append({'type': 'submit', 'forced': (not ok_all), 'pairs_ok': res['pairs_ok'], 'n': res['n']})
        print(f"SUBMITTED{' (forced, %d/%d)' % (res['pairs_ok'], res['n']) if not ok_all else ' — all train pairs pass'}. Task complete.")
        return True
    print(f"submit REFUSED: {res['pairs_ok']}/{res['n']} train pairs pass. Fix the rule and check() again; use submit(solve, force=True) only on your final turn.")
    return False


# ----------------------------------------------------------------------------- inspection macros
def show_pair(i, diff=False):
    """print input | output of TRAIN[i] side by side (digits)."""
    p = TRAIN[i]
    a, b = grid_str(p['input']).split('\n'), grid_str(p['output']).split('\n')
    w = max(len(r) for r in a)
    print(f"pair{i} input {_fmt_shape(p['input'])} | output {_fmt_shape(p['output'])}")
    for r in range(max(len(a), len(b))):
        print((a[r] if r < len(a) else '').ljust(w) + '  |  ' + (b[r] if r < len(b) else ''))
    if diff:
        diff_pair(i)


def show_test(j=0):
    t = TEST[j]['input']
    show(t, f'test{j} input')


def diff_pair(i, max_cells=30):
    """diff of TRAIN[i]: changed cells / transitions + object matching summary."""
    p = TRAIN[i]
    d = diff_cells(p['input'], p['output'], max_cells)
    if d['changed'] is None:
        print(f"pair{i}: sizes differ {d['shape_a']} -> {d['shape_b']} (no cell diff)")
    else:
        print(f"pair{i}: {d['changed']} cells changed, bbox {d['bbox']}, transitions {d['transitions']}")
        print(f"  cells [r,c,from,to]: {d['cells']}{' ...' if d['truncated'] else ''}")
    m = match_nodes(objects(p['input'], BG, 8, True), objects(p['output'], BG, 8, True))
    print('  objects: ' + match_summary(m))
    return d


def objects_report(i=0, which='both', by_color=True, max_nodes=12):
    """list objects of TRAIN[i] input/output (id,color,px,bbox,holes,rect,border)."""
    p = TRAIN[i]
    for lab, g in (('input', p['input']), ('output', p['output'])):
        if which != 'both' and which != lab:
            continue
        ob = objects(g, BG, 8, by_color)
        print(f"pair{i} {lab}: {len(ob)} objects" + (f' (showing {max_nodes})' if len(ob) > max_nodes else ''))
        for o in ob[:max_nodes]:
            print('  ', summ(o))


SANDBOX_MACROS = ['task_report', 'try_transforms', 'check', 'submit', 'show_pair', 'show_test', 'diff_pair', 'objects_report']
'''
NARRATE_SRC = r'''# ===================================================================================
# ARC-2 서술함수 (NARRATION FUNCTIONS) — 관찰카드 · 정합체크(게이트1) · 등급 채점(4축+silx) ·
#   단계식 되먹임(SIZE→MASK→COLOR) · 반례 · 렌즈 증거 탐지 · 씨앗 few-shot.
#   helpers.py + macros.py 위에 같은 네임스페이스로 exec 된다 (TRAIN/TEST/BG/REPORT 사용).
#   출처: 풀이전용판 셀2 observe/qwen_card/claim_of/verify_claims, 셀4b(4축·널정규화·단계식·
#         best_shift·pos_facts·등급형 모양축), 셀4d(score_silx·sil_null·nscore_sil·sil_facts),
#         v10.2(counterexample·size_check·LENSES 8), 채굴_씨앗15(SEEDS·LENS_STANDARDS).
#   score_size / score_color 는 원본 셀4 코드가 프로젝트에 없어 여기서 정의(v2 정의).
# ===================================================================================

# ----------------------------------------------------------------------------- 관찰카드
def _reappear(node, out, bg):
    """입력 도형이 출력에 재출현하나: 'in_place' | 'in_place_recolored' | 'moved' | 'moved+<d4>' |
    'silhouette' | 'silhouette+<d4>' | None"""
    r0, c0, r1, c1 = node['bbox']
    crop = node['crop']
    m = crop != -1
    if out.shape[0] > r1 and out.shape[1] > c1:
        sub = out[r0:r1 + 1, c0:c1 + 1]
        if np.array_equal(sub[m], crop[m]):
            return 'in_place'
        if np.array_equal(sub != bg, m):
            return 'in_place_recolored'
    for k, f in _D4.items():
        if find(f(crop), out) is not None:
            return 'moved' if k == 'id' else f'moved+{k}'
    sil = np.where(m, 1, -1)
    ob = np.where(out != bg, 1, 0)
    for k, f in _D4.items():
        if find(f(sil), ob) is not None:
            return 'silhouette' if k == 'id' else f'silhouette+{k}'
    return None


def observe_card():
    """관찰카드 — 사람 서술의 머리말(크기/일치/overwrite/배경)이 검증 가능하도록 측정한 사실 묶음."""
    ins = [p['input'] for p in TRAIN]
    outs = [p['output'] for p in TRAIN]
    same = all(i.shape == o.shape for i, o in zip(ins, outs))
    card = {'size': 'same' if same else 'diff', 'n_train': len(TRAIN),
            'sizes': [f'{i.shape[0]}x{i.shape[1]}->{o.shape[0]}x{o.shape[1]}' for i, o in zip(ins, outs)]}
    bgs = [background(i) for i in ins]
    card['bg'] = bgs
    ci = [sorted(set(i.ravel().tolist())) for i in ins]
    co = [sorted(set(o.ravel().tolist())) for o in outs]
    allc = sorted(set(c for cc in ci + co for c in cc))
    fixed, layer_same = [], []
    if same:
        for c in allc:
            if all(np.array_equal(i == c, o == c) for i, o in zip(ins, outs)):
                fixed.append(int(c))
        cov, pnt, nchg = [], [], []
        for i, o in zip(ins, outs):
            d = i != o
            cov.append(sorted(set(int(v) for v in i[d])))
            pnt.append(sorted(set(int(v) for v in o[d])))
            nchg.append(int(d.sum()))
        card['overwrite'] = {'covered': cov, 'painted': pnt, 'changed': nchg,
                             'covered_all': sorted(set(c for cc in cov for c in cc)),
                             'painted_all': sorted(set(c for cc in pnt for c in cc))}
        card['survival'] = [round(float((i == o).mean()), 3) for i, o in zip(ins, outs)]
        card['fg_survival'] = [round(float(((i == o) & (i != b)).sum() / max(1, int((i != b).sum()))), 3)
                               for i, o, b in zip(ins, outs, bgs)]
    card['fixed_colors'] = fixed
    card['fixed_colors_fg'] = [c for c in fixed if c not in bgs]
    fo, re_, nobj, nobj_c, match_pp = [], [], [], [], []
    for i, o, b in zip(ins, outs, bgs):
        objs = objects(i, b, 8, False)
        nobj.append(len(objs))
        nobj_c.append(len(objects(i, b, 8, True)))
        mp = Counter()
        for ob in objs:
            r = _reappear(ob, o, b)
            rec = {'colors': ob['colors'], 'px': ob['pixels'], 'bbox': ob['bbox'], 'holes': ob['holes'],
                   'dot': ob['is_dot'], 'line': ob['is_line'], 'rect': ob['is_rect']}
            if r == 'in_place':
                fo.append(rec); mp['in_place'] += 1
            elif r == 'in_place_recolored':
                mp['in_place_recolored'] += 1
            elif r:
                rec['how'] = r; re_.append(rec); mp['reappears_elsewhere'] += 1
            else:
                mp['gone'] += 1
        match_pp.append(dict(mp))
    card['objects_glued'] = nobj
    card['objects_by_color'] = nobj_c
    card['object_fate_per_pair'] = match_pp
    card['fixed_shapes'] = {'n': len(fo), 'examples': fo[:3]}
    card['reappearing_shapes'] = {'n': len(re_), 'examples': re_[:3]}
    holes = []
    for i, b in zip(ins, bgs):
        hh = Counter(ob['holes'] for ob in objects(i, b, 8, True))
        holes.append(sorted(((int(k), int(v)) for k, v in hh.items()), reverse=True)[:4])
    card['holes_by_color_objects'] = holes
    card['d4_in_out'] = [next((k for k, f in _D4.items() if f(i).shape == o.shape and np.array_equal(f(i), o)), None)
                         for i, o in zip(ins, outs)]
    card['d4_out_out'] = [next((k for k, f in _D4.items() if f(outs[0]).shape == o.shape and np.array_equal(f(outs[0]), o)), None)
                          for o in outs[1:]]
    seps = [separators(i, b) for i, b in zip(ins, bgs)]
    card['separators'] = [s[2] for s in seps] if all(s[2] is not None for s in seps) else None
    card['panels'] = [len(panels(i, b)) for i, b in zip(ins, bgs)] if card['separators'] else None
    tags = []
    if same and not all(np.array_equal(i, o) for i, o in zip(ins, outs)):
        silh = all(np.array_equal(i != b, o != b) for i, o, b in zip(ins, outs, bgs))
        cm = color_map(TRAIN)
        if silh or cm['consistent']:
            tags.append('color_only')
    if len(outs) >= 2 and (all(o.shape[0] == 1 for o in outs) or all(o.shape[1] == 1 for o in outs)):
        tags.append('outsim')
    if same and card['fixed_colors_fg']:
        tags.append('has_fixed_color')
    if card['fixed_shapes']['n'] and same:
        tags.append('has_fixed_shape')
    if card['reappearing_shapes']['n']:
        tags.append('shapes_reappear')
    if any(card['d4_in_out']):
        tags.append('global_d4')
    card['tags'] = tags
    card['sig'] = {'size': card['size'], 'holes': any(k > 0 for h in holes for k, _ in h),
                   'd4': bool(any(card['d4_in_out'])), 'reappear': card['reappearing_shapes']['n'] > 0,
                   'fixed_shape': card['fixed_shapes']['n'] > 0}
    return card


def card_text(card):
    """관찰카드 → 모델용 영어 사실문 (qwen_card 대응)."""
    L = [f"OBSERVATION CARD (measured on the training pairs; use these facts in HEAD):"]
    L.append(f"- size: {'SAME' if card['size'] == 'same' else 'DIFFERENT'} {card['sizes']} | background per input {card['bg']}")
    if card['fixed_colors']:
        L.append(f"- fixed colors (every cell of that color identical in input and output): {card['fixed_colors']} (non-background: {card['fixed_colors_fg']})")
    if card.get('overwrite'):
        ow = card['overwrite']
        L.append(f"- overwrite: cells changed per pair {ow['changed']}; colors covered (overwritten) {ow['covered_all']}; colors painted {ow['painted_all']}; "
                 f"input-cell survival {card['survival']} (foreground survival {card['fg_survival']})")
    L.append(f"- objects per input: glued (any colors touching) {card['objects_glued']}, single-color {card['objects_by_color']}; "
             f"per-pair fate of glued input objects {card['object_fate_per_pair']}")
    fs, rs = card['fixed_shapes'], card['reappearing_shapes']
    L.append(f"- shapes fixed in place: {fs['n']}" + (f" e.g. {[(e['colors'], e['px'], e['bbox']) for e in fs['examples'][:2]]}" if fs['n'] else '')
             + f" | shapes reappearing elsewhere/transformed: {rs['n']}" + (f" e.g. {[(e['colors'], e['px'], e.get('how')) for e in rs['examples'][:3]]}" if rs['n'] else ''))
    L.append(f"- holes per input as [(holes, n_objects)] of single-color objects: {card['holes_by_color_objects']}")
    if any(card['d4_in_out']):
        L.append(f"- whole-grid D4 relation input->output: {card['d4_in_out']}")
    if any(card['d4_out_out']):
        L.append(f"- outputs are D4-related to output 0: {card['d4_out_out']}")
    if card['separators']:
        L.append(f"- separator lines of color {card['separators']} -> panels per input {card['panels']}")
    L.append(f"- tags: {card['tags'] or ['none']}")
    return '\n'.join(L)


# ----------------------------------------------------------------------------- 서술 파싱 + 정합 체크 (게이트1)
_DESC_KEYS = ('HEAD', 'MECH', 'STEPS', 'PARAM', 'SIZES')


def parse_description(text):
    """DESCRIPTION 블록 → {'HEAD','MECH','STEPS','PARAM','SIZES','sizes':[(h,w)..]|None, 'found': bool}"""
    d = {k: '' for k in _DESC_KEYS}
    d['found'] = False
    if not text:
        d['sizes'] = None
        return d
    body = text
    m = re.search(r'DESCRIPTION\s*:?\s*\n(.*?)(?:\n```|\Z)', text, re.S | re.I)
    if m:
        body = m.group(1)
    cur = None
    for line in body.splitlines():
        mm = re.match(r'^\s*\**\s*(HEAD|MECH|STEPS|PARAM|SIZES)\s*\**\s*:\s*(.*)$', line, re.I)
        if mm:
            cur = mm.group(1).upper()
            d[cur] = mm.group(2).strip()
            d['found'] = True
            continue
        if cur and line.strip() and not line.strip().startswith('```'):
            d[cur] += ('\n' if d[cur] else '') + line.strip()
    pairs = re.findall(r'(\d{1,2})\s*[xX×*]\s*(\d{1,2})', d['SIZES'].split('\n')[0]) if d['SIZES'] else []
    d['sizes'] = [(int(a), int(b)) for a, b in pairs] or None
    return d


def verify_description(d):
    """게이트1: HEAD/SIZES 의 주장을 관찰카드와 대조. 크기·SIZES 는 하드(REJECT), 배경·일치·overwrite 는 소프트.
    -> {'lines': [...], 'verdict': 'PASS'|'REJECT'|'NO_DESCRIPTION', 'reasons': [...]}"""
    card = REPORT.get('card') or observe_card()
    lines, hard = [], []
    if not d.get('found'):
        return {'lines': ['no DESCRIPTION block found'], 'verdict': 'NO_DESCRIPTION', 'reasons': ['missing']}
    head = (d.get('HEAD') or '').lower()
    # size same / different
    if re.search(r'\b(same|equal|unchanged|identical)\b.*\bsize|size[^.;,]*\b(same|equal|unchanged|identical)\b', head):
        claim = 'same'
    elif re.search(r'\b(different|differs|changes|smaller|larger|bigger|crop|scaled|shrink|grow)\b', head):
        claim = 'diff'
    else:
        claim = None
    if claim:
        ok = claim == card['size']
        lines.append(f"size {'✓' if ok else '✗'} (claimed {claim}, measured {card['size']} {card['sizes']})")
        if not ok:
            hard.append(f"HEAD says size {claim} but measured {card['size']}")
    # SIZES per example
    want = [(len(p['output']), len(p['output'][0])) for p in TRAIN]
    if d.get('sizes'):
        bad = [(i, d['sizes'][i], want[i]) for i in range(min(len(d['sizes']), len(want))) if d['sizes'][i] != want[i]]
        if bad:
            i, s, w = bad[0]
            lines.append(f"SIZES ✗ (Example {i} predicted {s[0]}x{s[1]} but it is {w[0]}x{w[1]})")
            hard.append(f"SIZES: your rule predicts Example {i} output {s[0]}x{s[1]} but it is {w[0]}x{w[1]} — the mechanism is wrong, not a detail")
        else:
            lines.append('SIZES ✓')
    else:
        lines.append('SIZES ? (not given — give HxW per example; it is a free self-check)')
    # background
    mb = re.search(r'background\D{0,12}(\d)', head)
    if mb:
        c = int(mb.group(1))
        ok = all(b == c for b in card['bg'])
        lines.append(f"background {'✓' if ok else '✗'} (claimed {c}, measured per input {card['bg']})")
    # match N (shapes stay in place)
    mm_ = re.search(r'(\d+)\s*(?:shapes?|objects?)?\s*(?:stay|remain|fixed|unchanged|keep|kept|match)', head) or \
        re.search(r'(?:stay|remain|fixed|unchanged|keep|kept|match)\D{0,20}?(\d+)\s*(?:shapes?|objects?)', head)
    if mm_:
        n = int(mm_.group(1))
        meas = card['fixed_shapes']['n']
        per = [v.get('in_place', 0) for v in card['object_fate_per_pair']]
        ok = (n == meas) or all(k == n for k in per)
        lines.append(f"fixed shapes {'✓' if ok else '✗'} (claimed {n}, measured in-place shapes total {meas}, per pair {per})")
    # overwrite N
    mo = re.search(r'overwrit\w*\D{0,20}?(\d+)', head)
    if mo and card.get('overwrite'):
        n = int(mo.group(1))
        cov = card['overwrite']['covered_all']
        ok = len(cov) == n or (n == 1 and set(cov) <= set(card['bg']))
        lines.append(f"overwrite {'✓' if ok else '✗'} (claimed {n}, measured covered colors {cov})")
    # MECH lens
    mech = (d.get('MECH') or '').lower()
    known = [k for k in LENS_FAMILIES if k in mech] + [k for k in _PERSPECTIVE_KEYS if k in mech]
    if mech:
        ev = {e['lens'] for e in REPORT.get('lens_evidence', [])}
        if known:
            sup = [k for k in known if k in ev]
            lines.append(f"MECH lens {known}" + (f" — supported by measured evidence" if sup else " — no measured evidence for it (allowed; state why)"))
        else:
            lines.append("MECH: no catalog lens named (allowed: 'free' — but then STEPS must be explicit)")
    steps = d.get('STEPS') or ''
    nsteps = len(re.findall(r'(?:^|\n)\s*\d+[.)]', steps))
    if nsteps == 0 and steps:
        nsteps = 1
    lines.append(f"STEPS: {nsteps} step(s)" + (' — too few to be a mechanism' if nsteps < 2 else '') + (' — over 6, compress' if nsteps > 6 else ''))
    verdict = 'REJECT' if hard else 'PASS'
    return {'lines': lines, 'verdict': verdict, 'reasons': hard}


# ----------------------------------------------------------------------------- 4축 (크기/모양/색/배치) + 널 정규화
def score_size(pred, out):
    """크기 축 (v2 정의): 같으면 1.0, 전치면 0.7, 아니면 변 차이 비율로 감점"""
    p, o = G(pred).shape, G(out).shape
    if p == o:
        return 1.0
    if (p[1], p[0]) == o:
        return 0.7
    dh = abs(p[0] - o[0]) / max(p[0], o[0])
    dw = abs(p[1] - o[1]) / max(p[1], o[1])
    return round(max(0.0, 1.0 - (dh + dw)), 2)


def _hist(g, bg=None):
    g = G(g)
    b = background(g) if bg is None else bg
    c = Counter(int(v) for v in g.ravel().tolist() if v != b)
    return dict(sorted(c.items()))


def score_color(pred, out):
    """색깔 축 (v2 정의): 비배경 색 집합 Jaccard 0.5 + 정규화 히스토그램 일치 0.5"""
    hp, ho = _hist(pred), _hist(out)
    if not ho:
        return 1.0 if not hp else 0.0
    sp, so = set(hp), set(ho)
    jac = len(sp & so) / max(1, len(sp | so))
    tp, to = max(1, sum(hp.values())), max(1, sum(ho.values()))
    l1 = sum(abs(hp.get(c, 0) / tp - ho.get(c, 0) / to) for c in sp | so)
    return round(0.5 * jac + 0.5 * max(0.0, 1.0 - 0.5 * l1), 2)


def _masks(g):
    g = G(g)
    b = background(g)
    return [(tuple(o['colors']), (o['crop'] != -1).astype(np.int8)) for o in objects(g, b, 8, True)]


def _iou1(m1, m2):
    if m1.shape == m2.shape:
        uni = int((m1 | m2).sum())
        return int((m1 & m2).sum()) / uni if uni else 0.0
    H = max(m1.shape[0], m2.shape[0])
    W = max(m1.shape[1], m2.shape[1])
    A = np.zeros((H, W), np.int8)
    B = np.zeros((H, W), np.int8)
    A[:m1.shape[0], :m1.shape[1]] = m1
    B[:m2.shape[0], :m2.shape[1]] = m2
    uni = int((A | B).sum())
    return int((A & B).sum()) / uni if uni else 0.0


def _iou_d4(m1, m2):
    best = _iou1(m1, m2)
    for k, f in _D4.items():
        if k == 'id':
            continue
        best = max(best, 0.7 * _iou1(f(m1), m2))
    return best


def score_shape(pred, out, max_cands=12):
    """등급형 모양 축: 색 우선 + 실루엣 부분 IoU(D4 0.7배) 그리디 매칭 (셀4b 등급형).
    v2 속도 보강: 동일 (색,마스크) 는 해시로 즉시 매칭, 나머지는 칸수가 가까운 후보 max_cands 개만 IoU 비교
    (30x30 에 객체 200+ 인 격자에서 7s → 0.2s; 결과는 원본과 같거나 근사)."""
    P, O = _masks(pred), _masks(out)
    if not O:
        return 1.0 if not P else 0.5
    nP, nO = len(P), len(O)
    pool = defaultdict(list)
    for idx, (col, m) in enumerate(P):
        pool[(col, m.shape, m.tobytes())].append(idx)
    used, rest_O, tot = set(), [], 0.0
    for col, mo in O:
        key = (col, mo.shape, mo.tobytes())
        if pool.get(key):
            used.add(pool[key].pop())
            tot += 1.0
        else:
            rest_O.append((col, mo))
    P_rest = [(P[i][0], P[i][1], int(P[i][1].sum())) for i in range(nP) if i not in used]
    for col, mo in rest_O:
        if not P_rest:
            break
        px = int(mo.sum())
        order = sorted(range(len(P_rest)), key=lambda i: (abs(P_rest[i][2] - px), 0 if P_rest[i][0] == col else 1))[:max_cands]
        j = max(order, key=lambda i: (1.0 if P_rest[i][0] == col else 0.7) * _iou_d4(P_rest[i][1], mo))
        tot += (1.0 if P_rest[j][0] == col else 0.7) * _iou_d4(P_rest[j][1], mo)
        P_rest.pop(j)
    return round(tot / max(nO, nP), 2)


def _objsig(g):
    g = G(g)
    b = background(g)
    H, W = g.shape
    D = {}
    for o in objects(g, b, 8, True):
        m = (o['crop'] != -1).astype(np.int8)
        sig = (tuple(o['colors']), m.tobytes() + bytes(m.shape))
        y0, x0, y1, x1 = o['bbox']
        D.setdefault(sig, []).append((((y0 + y1) / 2) / max(1, H - 1), ((x0 + x1) / 2) / max(1, W - 1)))
    return D


def score_pos(pred, out):
    """배치 축: 색+실루엣이 같은 객체끼리 최근접 매칭 → 정규화 중심 거리 (반 격자 이상이면 0)"""
    Dp, Do = _objsig(pred), _objsig(out)
    npd = sum(len(v) for v in Dp.values())
    nod = sum(len(v) for v in Do.values())
    if nod == 0:
        return 1.0 if npd == 0 else 0.0
    tot = 0.0
    for sig, pos_o in Do.items():
        pos_p = list(Dp.get(sig, []))
        for cy, cx in pos_o:
            if not pos_p:
                continue
            j = min(range(len(pos_p)), key=lambda i: (pos_p[i][0] - cy) ** 2 + (pos_p[i][1] - cx) ** 2)
            d = ((pos_p[j][0] - cy) ** 2 + (pos_p[j][1] - cx) ** 2) ** 0.5
            pos_p.pop(j)
            tot += max(0.0, 1.0 - d / 0.5)
    return round(tot / max(npd, nod), 2)


def score_pos_d4(pred, out):
    base = score_pos(pred, out)
    if base >= 0.99:
        return base
    p = G(pred)
    best = base
    for k, f in _D4.items():
        if k == 'id':
            continue
        try:
            v = 0.7 * score_pos(f(p), out)
        except Exception:
            continue
        if v > best:
            best = v
    return round(best, 2)


def score4(pred, out):
    if pred is None:
        return 0.0, (0.0, 0.0, 0.0, 0.0)
    sz = score_size(pred, out)
    sh = score_shape(pred, out)
    co = score_color(pred, out)
    po = score_pos_d4(pred, out)
    return round((sz + sh + co + po) / 4, 2), (sz, sh, co, po)


def score_preds(preds):
    """train 예측 리스트(None 허용) → (총점, (크기, 모양, 색, 배치))"""
    T = []
    A = [[], [], [], []]
    for o, p in zip(preds, TRAIN):
        s, ax = score4(o, p['output'])
        T.append(s)
        for i in range(4):
            A[i].append(ax[i])
    n = max(1, len(T))
    return round(sum(T) / n, 2), tuple(round(sum(a) / n, 2) for a in A)


_NULL = {}


def null_base():
    """이 태스크에서 '아무것도 이해 못한' 예측(identity / 배경채움)이 받는 4축 점수 — 정규화의 0점"""
    if 'v' not in _NULL:
        s1, _ = score_preds([p['input'] for p in TRAIN])
        s2, _ = score_preds([np.full(p['input'].shape, background(p['input']), dtype=int) for p in TRAIN])
        _NULL['v'] = max(s1, s2)
    return _NULL['v']


def nscore(s, s0):
    return round(max(0.0, (s - s0) / max(0.05, 1.0 - s0)), 2)


KEEP_N = 0.30      # 빔 채택 임계 (정규화)
FINAL_N = 0.80     # 정밀 잔차 진입 임계 (정규화)


# ----------------------------------------------------------------------------- 실루엣 압축정합 축 (셀4d 이식)
def _silm(g):
    g = G(g)
    return (g != background(g)).astype(np.int8)


def _sil_bands(m, h):
    H, W = m.shape
    out = []
    for y in range(H - h + 1):
        v = np.zeros(W, dtype=np.uint8)
        for i in range(h):
            v |= (m[y + i].astype(np.uint8) << i)
        out.append(v.tobytes())
    return out


class _SilDict:
    def __init__(self, m):
        self.m = m
        self.H, self.W = m.shape
        self.cache = {}

    def band(self, h):
        if h not in self.cache:
            bs = _sil_bands(self.m, h) if h <= self.H else []
            self.cache[h] = (b'\xff' + b'\xff'.join(bs) + b'\xff') if bs else b''
        return self.cache[h]

    def has(self, bb, h):
        d = self.band(h)
        return bool(d) and bb in d


def _sil_maxw(ob, x, dic, h, wmax):
    lo, hi = 0, wmax
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if dic.has(ob[x:x + mid], h):
            lo = mid
        else:
            hi = mid - 1
    return lo


def sil_cover(mo, mp, margin=4.0, hmax=7):
    """정답 실루엣 mo 를 예측 mp 의 2D 블록으로 탐욕 커버. 매치 인정 = 블록 자기정보량 >= log2(사전크기)+margin"""
    H, W = mo.shape
    dic = _SilDict(mp)
    obands = {h: _sil_bands(mo, h) for h in range(1, min(hmax, H) + 1)}
    p1 = min(max(float(mp.mean()), 0.03), 0.97)
    i1, i0 = -math.log2(p1), -math.log2(1.0 - p1)
    thr = math.log2(max(2, mp.shape[0] * mp.shape[1])) + margin
    ones = np.pad(mo, ((1, 0), (1, 0))).cumsum(0).cumsum(1)

    def n1(y, x, h, w):
        return int(ones[y + h, x + w] - ones[y, x + w] - ones[y + h, x] + ones[y, x])
    cov = np.zeros((H, W), dtype=bool)
    for y in range(H):
        for x in range(W):
            if cov[y, x]:
                continue
            best = None
            for h in range(min(hmax, H - y), 0, -1):
                w = _sil_maxw(obands[h][y], x, dic, h, W - x)
                if w == 0:
                    continue
                k1 = n1(y, x, h, w)
                if k1 * i1 + (h * w - k1) * i0 >= thr and (best is None or h * w > best[0]):
                    best = (h * w, h, w)
            if best:
                _, h, w = best
                cov[y:y + h, x:x + w] = True
    return cov


def _sil_f1(mo, mp, margin=4.0):
    co = sil_cover(mo, mp, margin)
    cp = sil_cover(mp, mo, margin)
    r = float(((mo == 1) & co).sum()) / max(1.0, float((mo == 1).sum()))
    p = float(((mp == 1) & cp).sum()) / max(1.0, float((mp == 1).sum()))
    return 2 * p * r / max(1e-9, p + r)


def _sil_pos_f1(mo, mp, rng=2):
    H, W = mo.shape
    best = 0.0
    for dy in range(-rng, rng + 1):
        for dx in range(-rng, rng + 1):
            if abs(dy) >= H or abs(dx) >= W:      # 작은 격자: 이동량이 격자보다 크면 건너뜀 (원본 버그 수정)
                continue
            a = np.zeros_like(mp)
            a[max(0, dy):H + min(0, dy), max(0, dx):W + min(0, dx)] = mp[max(0, -dy):H + min(0, -dy), max(0, -dx):W + min(0, -dx)]
            inter = float(((a == 1) & (mo == 1)).sum())
            r = inter / max(1.0, float((mo == 1).sum()))
            p = inter / max(1.0, float((a == 1).sum()))
            f = 2 * p * r / max(1e-9, p + r)
            if f > best:
                best = f
    return best


_SIL_SELF = {}


def score_silx(pred, out, margin=4.0):
    """실루엣 압축정합 점수 [0,1] — 색깔 무시, 이동·부분모양 관용, 랜덤·배경 차단 (셀4d)"""
    if pred is None:
        return 0.0
    mo, mp = _silm(out), _silm(pred)
    if mo.sum() == 0:
        return 1.0 if mp.sum() == 0 else 0.0
    if mp.sum() == 0:
        return 0.0
    if mo.shape == mp.shape and np.array_equal(mo, mp):
        return 1.0
    key = (mo.shape, mo.tobytes())
    if key not in _SIL_SELF:
        _SIL_SELF[key] = _sil_f1(mo, mo, margin)
    self_s = _SIL_SELF[key]
    s = _sil_f1(mo, mp, margin)
    if self_s >= 0.99:
        return round(s, 2)
    s = min(1.0, s / max(self_s, 0.05))
    if mo.shape == mp.shape:
        s = max(s, _sil_pos_f1(mo, mp))
    return round(s, 2)


_SIL_NULL = {}


def sil_null():
    """identity/배경채움 실루엣 점수 — 이 축의 정규화 0점. >=0.95 면 마스크 단계는 무정보(색깔 단계로 직행)."""
    if 'v' not in _SIL_NULL:
        v1 = [score_silx(p['input'], p['output']) for p in TRAIN]
        v2 = [score_silx(np.full(p['input'].shape, background(p['input']), dtype=int), p['output']) for p in TRAIN]
        n = max(1, len(v1))
        _SIL_NULL['v'] = max(round(sum(v1) / n, 2), round(sum(v2) / n, 2))
    return _SIL_NULL['v']


def nscore_sil(s, s0):
    return round(max(0.0, (s - s0) / max(0.05, 1.0 - s0)), 2)


def sil_facts(pred, out, lab):
    """덮이지 않은 정답 영역(누락 모양) / 덮이지 않은 예측 영역(창작 모양) 사실문"""
    try:
        mo, mp = _silm(out), _silm(pred)
        if mo.sum() == 0 or mp.sum() == 0:
            return ''
        co = sil_cover(mo, mp)
        cp = sil_cover(mp, mo)
        F = []
        miss = (mo == 1) & ~co
        if miss.sum():
            ys, xs = np.where(miss)
            F.append(f"answer has shape material around rows {ys.min()}-{ys.max()} cols {xs.min()}-{xs.max()} "
                     f"({int(miss.sum())} cells) that appears NOWHERE in your output (silhouette, color ignored)")
        extra = (mp == 1) & ~cp
        if extra.sum():
            ys, xs = np.where(extra)
            F.append(f"your output invents shape material around rows {ys.min()}-{ys.max()} cols {xs.min()}-{xs.max()} "
                     f"({int(extra.sum())} cells) that appears nowhere in the answer")
        return (lab + ': ' + '; '.join(F)) if F else ''
    except Exception:
        return ''


# ----------------------------------------------------------------------------- 마스크 단계 도구 + 배치 사실문
def _bmask(g):
    g = G(g)
    return (g != background(g)).astype(np.int8)


def best_shift(mp, mo, rng=4):
    """비배경 마스크 상관: 예측을 (dy,dx) 이동시켜 IoU 최대가 되는 오프셋 (eval 실측 35/35 복구)"""
    if mp.shape != mo.shape:
        return (0, 0, 0.0)
    H, W = mo.shape
    best = (0, 0, -1.0)
    for dy in range(-rng, rng + 1):
        for dx in range(-rng, rng + 1):
            if abs(dy) >= H or abs(dx) >= W:      # 작은 격자 가드 (원본 버그 수정)
                continue
            a = np.zeros_like(mo)
            a[max(0, dy):H + min(0, dy), max(0, dx):W + min(0, dx)] = mp[max(0, -dy):H + min(0, -dy), max(0, -dx):W + min(0, -dx)]
            uni = int((a | mo).sum())
            iou = int((a & mo).sum()) / uni if uni else 0.0
            if iou > best[2]:
                best = (dy, dx, iou)
    return best


def pos_facts(pred, out, lab):
    """배치가 최저 축일 때: 어떤 도형이 어디 있어야 하는데 어디 있는지"""
    Dp, Do = _objsig(pred), _objsig(out)
    bycol = {}
    for sig, ps in Dp.items():
        bycol.setdefault(sig[0], []).extend(ps)
    H, W = G(out).shape
    F = []
    for sig, pos_o in list(Do.items())[:4]:
        col = sig[0]
        pos_p = Dp.get(sig)
        note = ''
        if not pos_p:
            pos_p = bycol.get(col)
            note = ' (your shape of that color is not the same silhouette — mirrored/rotated/rebuilt)'
        if not pos_p:
            F.append(f"color {list(col)} shape is MISSING from your output (answer has {len(pos_o)})")
            continue
        a = pos_o[0]
        b = min(pos_p, key=lambda q: (q[0] - a[0]) ** 2 + (q[1] - a[1]) ** 2)
        ra, ca, rb, cb = round(a[0] * (H - 1)), round(a[1] * (W - 1)), round(b[0] * (H - 1)), round(b[1] * (W - 1))
        if (ra, ca) == (rb, cb) and not note:
            continue                                   # same place: nothing to say
        F.append(f"color {list(col)} shape: answer puts it at row~{ra} col~{ca}, you put it at row~{rb} col~{cb}{note}")
    return f"{lab}: " + '; '.join(F) if F else ''


# ----------------------------------------------------------------------------- 단계식 되먹임 (CRASH → SIZE → MASK → COLOR)
def stage_feedback(preds, axes):
    """train 예측(None 허용) + 4축 → (단계, 사실문 리스트, 지시문). 셀4b attempt() 의 단계 블록 이식."""
    szs, shs, cos, pos = axes
    rows = [(j, o, p['output']) for j, (o, p) in enumerate(zip(preds, TRAIN))][:3]
    facts = []
    if any(o is None for _j, o, _g2 in rows):
        stage = 'CRASH'
        facts = [f'Example {j}: your program produced NO valid output' for j, o, _g2 in rows if o is None]
        advice = 'Fix the crash first (see the error above).'
    elif any(G(o).shape != g.shape for _j, o, g in rows) or szs < 1.0:
        stage = 'SIZE'
        facts = [f'Example {j}: your size {G(o).shape[0]}x{G(o).shape[1]} vs correct {g.shape[0]}x{g.shape[1]}' for j, o, g in rows]
        advice = 'First make the output SIZE exact on every example (REPORT["size"]). Ignore colors and shapes until sizes match.'
    else:
        mask_ok = all(np.array_equal(_bmask(o), _bmask(g)) for _j, o, g in rows)
        if not mask_ok:
            stage = 'MASK'
            for j, o, g in rows:
                mp, mo = _bmask(o), _bmask(g)
                if np.array_equal(mp, mo):
                    facts.append(f'Example {j}: silhouette EXACT — keep it')
                    continue
                uni = int((mp | mo).sum())
                iou = int((mp & mo).sum()) / uni if uni else 0.0
                dy, dx, siou = best_shift(mp, mo)
                s = f'Example {j}: silhouette IoU {iou:.2f}'
                if (dy, dx) != (0, 0) and siou > iou + 0.15:
                    s += f' — but shifted by ({dy},{dx}) it becomes {siou:.2f}: your shapes are RIGHT, only moved by {abs(dy) + abs(dx)} cell(s)'
                else:
                    s += f' ({int((mp & ~mo).sum())} extra cells, {int((~mp & mo).sum())} missing cells)'
                sf = sil_facts(o, g, '')
                if sf:
                    s += '; ' + sf.lstrip(': ')
                facts.append(s)
            advice = ('Ignore colors for now. Make the SILHOUETTE (which cells are non-background) exact first. '
                      'Where the silhouette already matches, change NOTHING.')
        else:
            stage = 'COLOR'
            for j, o, g in rows:
                o = G(o)
                wr = sorted(set((int(o[y, x]), int(g[y, x])) for y, x in zip(*np.where(o != g))))[:6]
                facts.append(f'Example {j}: colors — your counts {_hist(o)} vs correct {_hist(g)}' + (f'; wrong->right pairs {wr}' if wr else ' (all correct)'))
            advice = ('The silhouette (non-background mask) matches the answer EXACTLY on every example — the structure of your rule is RIGHT. '
                      'Do NOT change sizes, shapes or positions. ONLY fix how cells are colored (the painting step).')
    return stage, facts, advice


# ----------------------------------------------------------------------------- 반례 (v10.2: 99% = 0)
CE_HEADER = "A rule that is 99% correct scores ZERO — do not patch a few cells; change the DESCRIPTION (mechanism or a STEP)."


def _neighborhood(a, y, x, r=2):
    ys, ye = max(0, y - r), min(a.shape[0], y + r + 1)
    xs, xe = max(0, x - r), min(a.shape[1], x + r + 1)
    return grid_str(a[ys:ye, xs:xe])


def counterexample(preds):
    """첫 반례 1개 + 입력 이웃 (크기 다르면 행 나란히). 반환 문자열 | ''"""
    for j, (o, p) in enumerate(zip(preds, TRAIN)):
        gt, inp = p['output'], p['input']
        if o is None:
            return f"Counterexample — Example {j}: the implementation crashed or returned nothing."
        o = G(o)
        if o.shape != gt.shape:
            return f"Counterexample — Example {j}: your output is {o.shape[0]}x{o.shape[1]} but the answer is {gt.shape[0]}x{gt.shape[1]} (SIZE rule wrong)."
        d = o != gt
        if d.any():
            ys, xs = np.where(d)
            y, x = int(ys[0]), int(xs[0])
            extra = int(d.sum()) - 1
            if inp.shape == gt.shape:
                ctx = (f"  input around (row {y}, col {x}) — centre input[{y}][{x}] = {int(inp[y, x])}:\n"
                       + '\n'.join('    ' + l for l in _neighborhood(inp, y, x).split('\n')))
            else:
                ctx = (f"  correct output row {y}: {grid_str(gt[y:y + 1])}\n  your    output row {y}: {grid_str(o[y:y + 1])}\n"
                       f"  (input is {inp.shape[0]}x{inp.shape[1]}, output is {gt.shape[0]}x{gt.shape[1]})")
            return (f"Counterexample — Example {j}, output cell (row {y}, col {x}):\n{ctx}\n"
                    f"  your rule gives : {int(o[y, x])}\n  the answer is   : {int(gt[y, x])}"
                    + (f"\n  (and {extra} other cells also disagree)" if extra else '') + "\n  Your rule cannot explain this cell.")
    return ''


# ----------------------------------------------------------------------------- 렌즈 카탈로그 + 증거 탐지
_PERSPECTIVE_KEYS = ('geometric', 'color-map', 'neighborhood', 'object', 'row/column', 'counting', 'symmetry', 'free')

LENS_FAMILIES = {
    'restore': {'standard': 'the grid is globally symmetric or periodic; an occluding region hides part of it; hidden cells equal their symmetric/periodic counterparts',
                'slots': 'symmetry kind (mirror H/V, point, period) / occluder color / output = the hidden patch only or the full restored grid',
                'seeds': ['0934a4d8', '981571dc']},
    'lfill': {'standard': 'one large plain shape is a TEMPLATE with interior holes; other material in the grid supplies what goes into the holes',
              'slots': 'filler source (transplanted pieces / majority color of matching fillers / exact packing) / scaling or D4 allowed / output = template bbox',
              'seeds': ['3a25b0d8', '67e490f4', 'e8686506']},
    'indicator': {'standard': 'a main object is edited; small shapes far from it (or stripes on the border) are INSTRUCTIONS, not content — they never appear in the output',
                  'slots': 'instruction meaning (rotate by count / swap color pair / face side) / output = main object bbox',
                  'seeds': ['6ffbe589', 'edb79dae', 'fc7cae8d']},
    'occlusion': {'standard': 'a window/frame covers shapes; pieces of one shape cut by the window belong together and continue straight underneath',
                  'slots': 'output = window content with hidden parts reconstructed',
                  'seeds': ['a6f40cea']},
    'stamp': {'standard': 'one box holds single-cell dots (POSITIONS), another holds multi-cell motifs (CONTENT); stamp the motif at every dot',
              'slots': 'dot color selects motif kind / output = dot box',
              'seeds': ['a251c730']},
    'legend': {'standard': 'a small region is a LEGEND: complete sample shapes define a lookup (e.g. hole shape -> color) that is applied to the incomplete/noise-colored ones',
               'slots': 'legend key/value / targets (noise-colored blobs) / no-match handling',
               'seeds': ['e3721c99']},
    'detect_recolor': {'standard': 'the output differs from the input only where a specific local pattern occurs; those cells are recolored, everything else unchanged',
                       'slots': 'pattern (plus / L / 2x2 ...) / new color',
                       'seeds': ['1818057f']},
    'pair_assemble': {'standard': 'every object reappears; objects of two kinds are paired in positional (reading) order and assembled into a new layout',
                      'slots': 'kind test (hollow vs solid, color, size) / layout (one pair per row ...) / leftover handling',
                      'seeds': ['2ba387bc']},
    'panel_vote': {'standard': 'the grid is a table of panels; within a group most panels are identical and exactly one differs — select by minority/majority',
                   'slots': 'group = row / column / all; select odd or common one; output = selected panels stacked',
                   'seeds': ['38007db0']},
    'rows_to_rings': {'standard': 'a 1D sequence in the input (stripes) becomes a nested structure in the output (concentric rings / spiral)',
                      'slots': 'sequence direction / ring order / side length formula (e.g. 2n-2)',
                      'seeds': ['45a5af55']},
}


def _largest_glued(g, b):
    ob = objects(g, b, 8, False)
    return max(ob, key=lambda o: o['pixels']) if ob else None


def lens_evidence():
    """렌즈 가족의 전제조건이 train 쌍에서 성립하는지 측정 → [{'lens', 'evidence'}] (성립한 것만)."""
    ins = [p['input'] for p in TRAIN]
    outs = [p['output'] for p in TRAIN]
    bgs = [background(i) for i in ins]
    n = len(TRAIN)
    same = all(i.shape == o.shape for i, o in zip(ins, outs))
    E = []

    def allp(fn):
        try:
            return all(fn(i, o, b) for i, o, b in zip(ins, outs, bgs))
        except Exception:
            return False
    # restore: (a) same size, outputs symmetric, inputs not, and every changed cell had ONE input color (the occluder);
    #          (b) a solid rectangle of one color vanishes and the output has exactly its size (hidden patch)
    out_sym = allp(lambda i, o, b: bool(symmetries(o)))
    in_sym = allp(lambda i, o, b: bool(symmetries(i)))

    def _occluder_one_color(i, o, b):
        d = i != o
        return d.any() and len(set(i[d].tolist())) == 1

    def _rect_patch(i, o, b):
        for ob in objects(i, b, 8, True):
            r0, c0, r1, c1 = ob['bbox']
            if ob['is_rect'] and ob['pixels'] >= 6 and ob['color'] not in set(np.unique(o).tolist()) and tuple(o.shape) == (r1 - r0 + 1, c1 - c0 + 1):
                return True
        return False
    occ = same and out_sym and not in_sym and allp(_occluder_one_color)
    patch = (not same) and allp(_rect_patch)
    if occ or patch:
        E.append({'lens': 'restore', 'evidence': (f"same size; outputs are symmetric while inputs are not, and every changed cell was ONE input color (an occluder)" if occ
                                                   else "a solid one-color rectangle vanishes and the output has exactly its size -> output = the hidden patch")})
    # lfill: a template (largest single-color OR largest glued object) has interior holes in every input, and the
    #        output has fewer holes: same size -> fewer enclosed regions; different size -> output == template bbox with fewer holes
    def _template(i, b):
        # background may be misdetected on small grids where the template dominates: try the mode and the runner-up
        cnt = Counter(i.ravel().tolist()).most_common(2)
        for bb in ([b] + ([int(cnt[1][0])] if len(cnt) > 1 and int(cnt[1][0]) != b else [])):
            cands = [x for x in objects(i, bb, 8, True) + objects(i, bb, 8, False) if x['holes'] >= 1]
            if cands:
                return max(cands, key=lambda x: x['pixels'])
        return None

    def _lf_same(i, o, b):
        T = _template(i, b)
        return T is not None and T['pixels'] >= 8 and len(enclosed_regions(o, b)) < len(enclosed_regions(i, b))

    def _lf_bbox(i, o, b):
        T = _template(i, b)
        if T is None or T['pixels'] < 8:
            return False
        r0, c0, r1, c1 = T['bbox']
        if tuple(o.shape) != (r1 - r0 + 1, c1 - c0 + 1):
            return False
        return len(enclosed_regions(o, background(o))) < T['holes']
    if (same and allp(_lf_same)) or ((not same) and allp(_lf_bbox)):
        E.append({'lens': 'lfill', 'evidence': ("same size; the largest holed object loses holes in the output (they get filled)" if same
                                                 else "output = bbox of the largest holed object (the template) with its holes filled")})
    # indicator: small far objects vanish; output == largest object bbox
    def _ind(i, o, b):
        ob = objects(i, b, 8, False)
        if len(ob) < 2:
            return False
        L_ = max(ob, key=lambda x: x['pixels'])
        small = [x for x in ob if x['pixels'] <= 12 and x['pixels'] < L_['pixels'] / 4 and not touching(ob, x['id'], L_['id'])]
        if not small:
            return False
        ocols = set(np.unique(o).tolist())
        vanish = any(all(c not in ocols for c in x['colors']) for x in small)
        main_out = tuple(o.shape) == (L_['bbox'][2] - L_['bbox'][0] + 1, L_['bbox'][3] - L_['bbox'][1] + 1)
        return main_out and (vanish or o.size < i.size)
    if not same and allp(_ind):
        E.append({'lens': 'indicator', 'evidence': "every input has small far-away marks besides a main object; output is (about) the main object and marks vanish"})
    # occlusion / window: output shape equals the interior or bbox of a rectangular frame object in the input
    def _win(i, o, b):
        for ob in objects(i, b, 8, True):
            r0, c0, r1, c1 = ob['bbox']
            h, w = r1 - r0 + 1, c1 - c0 + 1
            if ob['pixels'] >= 8 and (tuple(o.shape) == (h, w) or tuple(o.shape) == (h - 2, w - 2)):
                return True
        return False
    if not same and allp(_win):
        E.append({'lens': 'occlusion', 'evidence': "in every pair the output size equals the bbox (or interior) of one rectangular object in the input (a window/frame)"})
    # stamp: a box (rectangular object) containing >=2 single-cell dots of other colors, motifs elsewhere, output == that box
    def _stamp(i, o, b):
        ob = objects(i, b, 8, True)
        dots = [x for x in ob if x['is_dot']]
        motifs = [x for x in ob if x['pixels'] >= 3 and not x['is_rect']]
        if len(dots) < 2 or not motifs:
            return False
        for box in ob:
            if not (box['is_rect'] or box['holes'] >= 1) or box['pixels'] < 9:
                continue
            r0, c0, r1, c1 = box['bbox']
            inside = [d for d in dots if r0 <= d['bbox'][0] <= r1 and c0 <= d['bbox'][1] <= c1]
            if len(inside) >= 2 and tuple(o.shape) in ((r1 - r0 + 1, c1 - c0 + 1), (r1 - r0 - 1, c1 - c0 - 1)):
                return True
        return False
    if allp(_stamp):
        E.append({'lens': 'stamp', 'evidence': "every input has a box holding >=2 single-cell dots plus motifs elsewhere, and the output has the box's size"})
    # legend: same size; one color vanishes; objects with holes (rings) exist
    def _legend(i, o, b):
        gone = set(np.unique(i).tolist()) - set(np.unique(o).tolist()) - {b}
        rings = [x for x in objects(i, b, 8, True) if x['holes'] >= 1 and x['pixels'] >= 8]
        return bool(gone) and len(rings) >= 2
    if same and allp(_legend):
        E.append({'lens': 'legend', 'evidence': "same size; one input color disappears entirely and several ring-like objects (with holes) exist -> a legend may define the recoloring"})
    # detect_recolor: same size; <=15% cells change; changed cells become one color
    def _dr(i, o, b):
        d = i != o
        if not d.any() or d.mean() > 0.30:
            return False
        return len(set(o[d].tolist())) == 1 and len(set(i[d].tolist())) <= 2
    if same and allp(_dr):
        E.append({'lens': 'detect_recolor', 'evidence': "same size; only a few cells change and all changed cells take ONE new color -> a local pattern is being detected and recolored"})
    # pair_assemble: output smaller; all input objects reappear (matched by hash) and there are two kinds
    def _pa(i, o, b):
        oi, oo = objects(i, b, 8, True), objects(o, background(o), 8, True)
        if len(oi) < 2 or o.size >= i.size:
            return False
        m = match_nodes(oi, oo)
        return len(m['disappeared']) == 0 and (m['stable'] + len(m['moved'])) == len(oi)
    if allp(_pa):
        kinds = allp(lambda i, o, b: len({x['holes'] > 0 for x in objects(i, b, 8, True)}) == 2)
        E.append({'lens': 'pair_assemble', 'evidence': f"output is smaller and EVERY input object reappears in it (moved); two object kinds (hollow vs solid): {kinds}"})
    # panel_vote: separators, >=3 equal panels, output == one panel (or a column/row of panels)
    def _pv(i, o, b):
        ps = panels(i, b)
        if len(ps) < 3:                       # separator lines may share the background color (38007db0-type)
            rows, cols, col = separators(i, b, include_bg=True)
            if col is None:
                return False
            H, W = i.shape
            ys, xs = [-1] + rows + [H], [-1] + cols + [W]
            ps = [([ys[a] + 1, xs[c] + 1, ys[a + 1] - 1, xs[c + 1] - 1], i[ys[a] + 1:ys[a + 1], xs[c] + 1:xs[c + 1]])
                  for a in range(len(ys) - 1) for c in range(len(xs) - 1) if ys[a + 1] - 1 >= ys[a] + 1 and xs[c + 1] - 1 >= xs[c] + 1]
        if len(ps) < 3 or len({p.shape for _, p in ps}) != 1:
            return False
        ph, pw = ps[0][1].shape
        return tuple(o.shape) == (ph, pw) or o.shape[1] in (pw, pw + 1, pw + 2) or o.shape[0] in (ph, ph + 1, ph + 2)
    if allp(_pv):
        E.append({'lens': 'panel_vote', 'evidence': "every input splits into >=3 equal panels at separator lines and the output has panel size -> select/aggregate panels"})
    # rows_to_rings: every input row is uniform; output larger & symmetric
    def _rr(i, o, b):
        return all(len(set(r.tolist())) == 1 for r in i) and o.size > i.size and bool(symmetries(o))
    if allp(_rr):
        E.append({'lens': 'rows_to_rings', 'evidence': "every input row is a single color (a 1D sequence) and the output is larger and symmetric (nested structure)"})
    return E


# ----------------------------------------------------------------------------- 씨앗 few-shot (채굴 15)
SEEDS = {
    '0934a4d8': {'lens': 'restore', 'sig': {'size': 'diff', 'holes': False, 'd4': True, 'reappear': False, 'fixed_shape': False},
                 'rule': "OBJECTS: a 30x30 grid filled with a mirror-symmetric pattern, one solid rectangle of a cover color hiding part of it\nSTEPS:\n1. find the solid rectangle whose color forms a perfect filled box (the cover)\n2. find the grid's symmetry (try horizontal mirror, vertical mirror, point symmetry) using the un-covered cells\n3. reconstruct the cells under the cover by copying from their symmetric counterparts\n4. output ONLY the reconstructed patch, same size as the cover rectangle"},
    '981571dc': {'lens': 'restore', 'sig': {'size': 'same', 'holes': False, 'd4': True, 'reappear': False, 'fixed_shape': True},
                 'rule': "OBJECTS: a 30x30 grid with a symmetric pattern, several regions overwritten with color 0\nSTEPS:\n1. treat color 0 as \"unknown\" cells\n2. find the symmetry of the pattern (try mirror axes and periodic repetition) using known cells only\n3. replace every 0 cell by its symmetric/periodic counterpart's color\n4. output the full restored grid, same size as the input"},
    '3a25b0d8': {'lens': 'lfill', 'sig': {'size': 'diff', 'holes': True, 'd4': False, 'reappear': True, 'fixed_shape': False},
                 'rule': "OBJECTS: background, one large single-color template shape with holes inside it, one smaller multicolor shape (the source)\nSTEPS:\n1. find the largest single-color shape: this is the template; find its interior holes\n2. in the multicolor source, group the non-main-color cells by color: each group is one insert\n3. match each insert to the template hole with the same shape\n4. output the template's bounding box with every hole filled by its matching colored insert"},
    '67e490f4': {'lens': 'lfill', 'sig': {'size': 'diff', 'holes': True, 'd4': True, 'reappear': False, 'fixed_shape': False},
                 'rule': "OBJECTS: background, one large template shape with holes, many small filler shapes scattered around\nSTEPS:\n1. find the largest shape: this is the template; find its interior holes\n2. for each hole, find which filler shapes match it (same silhouette, allowing scaling and rotation/mirror)\n3. fill each hole with the color of its matching fillers (majority vote when several match)\n4. output the template's bounding box only"},
    'e8686506': {'lens': 'lfill', 'sig': {'size': 'diff', 'holes': True, 'd4': False, 'reappear': True, 'fixed_shape': False},
                 'rule': "OBJECTS: background, one template shape with holes, small colored shapes elsewhere\nSTEPS:\n1. find the largest shape: the template; find its interior holes\n2. the other shapes' cells exactly fill the holes: place each small shape into the hole it fits\n3. output the template's bounding box with all holes filled"},
    '6ffbe589': {'lens': 'indicator', 'sig': {'size': 'diff', 'holes': False, 'd4': True, 'reappear': True, 'fixed_shape': False},
                 'rule': "OBJECTS: one large decorated shape (the object), small far-away single-color marks (the instructions)\nSTEPS:\n1. find the largest shape and absorb decorations touching or near it: this is the object\n2. every far-away mark of color c with n cells means: rotate color-c cells of the object by 90 degrees x (n mod 4)\n3. apply each rotation inside the object's bounding box\n4. output the object's bounding box after all rotations"},
    'edb79dae': {'lens': 'indicator', 'sig': {'size': 'diff', 'holes': False, 'd4': False, 'reappear': True, 'fixed_shape': False},
                 'rule': "OBJECTS: one big rectangular frame containing solid square color blocks, small two-color shapes outside the frame\nSTEPS:\n1. find the big frame and the solid square blocks inside it\n2. each outside two-color shape declares a color pair (a,b): every block of color a becomes b and vice versa\n3. apply all declared swaps to the blocks\n4. output the frame's bounding box with swapped blocks"},
    'fc7cae8d': {'lens': 'indicator', 'sig': {'size': 'diff', 'holes': False, 'd4': True, 'reappear': True, 'fixed_shape': False},
                 'rule': "OBJECTS: one shape in the middle (not touching the border), colored stripes along some grid sides\nSTEPS:\n1. find the largest shape not touching the border: the object\n2. read the side stripes: the stripe color on a side says the object's outline of that color must face that side\n3. rotate/mirror the object so its outline colors face the demanded sides\n4. output the object's bounding box after re-orientation"},
    'a6f40cea': {'lens': 'occlusion', 'sig': {'size': 'diff', 'holes': False, 'd4': False, 'reappear': True, 'fixed_shape': False},
                 'rule': "OBJECTS: background, one rectangular window (frame), several shapes partly hidden by the window\nSTEPS:\n1. find the largest rectangular frame: the window\n2. group the shapes outside the window; two pieces with the same colors whose row-range or column-range lines up are one shape cut by the window\n3. reconstruct each cut shape by continuing it straight through the window\n4. output only the window region, showing the reconstructed hidden parts"},
    'a251c730': {'lens': 'stamp', 'sig': {'size': 'diff', 'holes': False, 'd4': False, 'reappear': True, 'fixed_shape': False},
                 'rule': "OBJECTS: two rectangular boxes: one containing only single-cell dots, one containing multi-cell motifs\nSTEPS:\n1. identify the dot box (all decorations are 1 cell) and the motif box (decorations are shapes)\n2. each motif kind is keyed by a color; each dot's color selects a motif kind\n3. stamp the selected motif centered at every dot position\n4. output the dot box's bounding box with motifs stamped"},
    'e3721c99': {'lens': 'legend', 'sig': {'size': 'same', 'holes': True, 'd4': False, 'reappear': False, 'fixed_shape': True},
                 'rule': "OBJECTS: legend shapes (complete rectangular rings, each with a distinctive hole and its own color), many blobs of one gray-like color\nSTEPS:\n1. build the legend: for each complete ring, record (hole shape -> ring color)\n2. for every blob of the noise color, find its hole shape\n3. recolor the blob with the legend color for that hole shape; a blob with no legend match is erased to background\n4. output the same-size grid; the noise color disappears"},
    '1818057f': {'lens': 'detect_recolor', 'sig': {'size': 'same', 'holes': False, 'd4': False, 'reappear': False, 'fixed_shape': True},
                 'rule': "OBJECTS: background of one color, scattered cells of a second color forming random texture, a few plus-shaped (+) arrangements\nSTEPS:\n1. scan the grid for plus shapes: a center cell with its 4 orthogonal neighbors all in the foreground color, corners not part of the plus\n2. recolor every cell of every found plus to color 8\n3. leave all other cells exactly as they are\n4. output the same-size grid"},
    '2ba387bc': {'lens': 'pair_assemble', 'sig': {'size': 'diff', 'holes': True, 'd4': False, 'reappear': True, 'fixed_shape': False},
                 'rule': "OBJECTS: background 0, several 4x4 squares of two kinds: hollow rings and solid blocks, each its own color\nSTEPS:\n1. list hollow squares in reading order (top-to-bottom), and solid squares in reading order\n2. pair the k-th hollow with the k-th solid\n3. build the output with one pair per row: hollow square on the left 4 columns, solid on the right 4 columns\n4. a leftover square without a partner keeps its side; the missing side is filled with 0"},
    '38007db0': {'lens': 'panel_vote', 'sig': {'size': 'diff', 'holes': False, 'd4': False, 'reappear': True, 'fixed_shape': False},
                 'rule': "OBJECTS: a grid of panels separated by full lines, each panel holding a similar motif with small differences\nSTEPS:\n1. split the grid into panels at the separator lines\n2. in each row of panels, find the one panel that differs from the others in that row\n3. stack the chosen panels vertically, keeping the separator lines\n4. output is one panel wide: separator + panel + separator ... for every row"},
    '45a5af55': {'lens': 'rows_to_rings', 'sig': {'size': 'diff', 'holes': False, 'd4': True, 'reappear': False, 'fixed_shape': False},
                 'rule': "OBJECTS: input made of full-width horizontal stripes, one color per row\nSTEPS:\n1. read the row colors top to bottom: c1, c2, ..., cn\n2. build a square of side 2*n - 2\n3. paint concentric square rings from the outside in: ring k gets color ck (c1 = outermost border ring)\n4. the innermost rows fill the center"},
}


def seed_shots(card, evidence, exclude=(), k=3):
    """관찰 서명 유사도(+렌즈 증거 일치 보너스)로 씨앗 서술 k개 선택 (자기 자신 제외)."""
    sig = card['sig']
    ev = {e['lens'] for e in evidence}
    ranked = []
    for tid, s in SEEDS.items():
        if tid in exclude:
            continue
        sc = sum(1 for kk in sig if sig[kk] == s['sig'].get(kk)) + (2 if sig['size'] == s['sig']['size'] else 0) + (3 if s['lens'] in ev else 0)
        ranked.append((sc, tid))
    ranked.sort(reverse=True)
    return [(tid, SEEDS[tid]['lens'], SEEDS[tid]['rule']) for _, tid in ranked[:k]]


NARRATION_FUNCTIONS = ['observe_card', 'card_text', 'parse_description', 'verify_description', 'score_size', 'score_color',
                       'score_shape', 'score_pos', 'score_pos_d4', 'score4', 'score_preds', 'null_base', 'nscore', 'score_silx',
                       'sil_cover', 'sil_null', 'nscore_sil', 'sil_facts', 'best_shift', 'pos_facts', 'stage_feedback',
                       'counterexample', 'lens_evidence', 'seed_shots']
'''
WORKER_SRC = r'''# ARC-2 sandbox worker — one subprocess per task (persistent namespace, JSON-lines protocol).
#   stdin  : {"op": "init", "task": {...}} | {"op": "exec", "code": str, "turn": int} | {"op": "exit"}
#   stdout : one JSON line per request (the ORIGINAL fd 1; the model's prints are captured).
#   The parent enforces the wall-clock timeout (kills + restarts the worker); inside check()
#   each solve() call has its own signal.alarm timeout.
import sys, os, io, json, contextlib, traceback, time

_proto = os.fdopen(os.dup(1), 'w', buffering=1)
_devnull = open(os.devnull, 'w')
sys.stdout = _devnull            # anything printed outside an exec is dropped
sys.stderr = _devnull


def _send(obj):
    _proto.write(json.dumps(obj, default=str) + '\n')
    _proto.flush()


def _cap(text, limit):
    if len(text) <= limit:
        return text
    head, tail = int(limit * 0.7), int(limit * 0.3)
    return text[:head] + f'\n... [{len(text) - limit} chars omitted] ...\n' + text[-tail:]


def main():
    boot_path, out_cap = sys.argv[1], int(sys.argv[2])
    ns = {'__name__': '__sandbox__'}
    exec(compile(open(boot_path, encoding='utf-8').read(), boot_path, 'exec'), ns)
    # protected names: if the model's code rebinds a helper/macro/runtime variable, restore it after
    # the turn and tell the model (macros call helpers from this same namespace).
    protected = list(ns['SANDBOX_HELPERS']) + list(ns['SANDBOX_MACROS']) + ['np', '_STATE', '_run_solve', '_register']
    orig = {k: ns[k] for k in protected if k in ns}
    n_sent = 0
    _send({'ok': True, 'ready': True})
    for line in sys.stdin:
        line = line.strip()
        if not line:
            continue
        try:
            msg = json.loads(line)
        except Exception as e:
            _send({'ok': False, 'error': f'bad request: {e}'})
            continue
        op = msg.get('op')
        if op == 'exit':
            break
        buf = io.StringIO()
        t0 = time.time()
        if op == 'init':
            try:
                with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
                    bg = ns['_init_task'](msg['task'])
                    R = ns['task_report'](verbose=True)
                n_sent = 0
                _send({'ok': True, 'bg': bg, 'report': _cap(buf.getvalue(), out_cap * 3),
                       'bank_top': R.get('bank_top') or [], 'size': R.get('size'),
                       'bg_conf': R['background']['confidence'], 'lens_evidence': R.get('lens_evidence') or [],
                       'seed_shots': R.get('seed_shots') or [], 'null': R.get('null'), 'secs': round(time.time() - t0, 2)})
            except BaseException:
                _send({'ok': False, 'error': _cap(traceback.format_exc(), 1500), 'report': buf.getvalue()})
            continue
        if op == 'verify':
            try:
                d = ns['parse_description'](msg.get('text', ''))
                v = ns['verify_description'](d)
                ns['_STATE']['desc'] = (d.get('HEAD', '') + ' | ' + d.get('MECH', '') + ' | ' + d.get('STEPS', '').replace('\n', ' '))[:600] if d.get('found') else ''
                _send({'ok': True, 'found': d.get('found', False), 'verdict': v['verdict'], 'lines': v['lines'], 'reasons': v['reasons'],
                       'mech': d.get('MECH', '')[:200], 'secs': round(time.time() - t0, 2)})
            except BaseException:
                _send({'ok': False, 'error': _cap(traceback.format_exc(), 1200), 'verdict': 'ERROR', 'lines': [], 'reasons': []})
            continue
        if op == 'exec':
            st = ns['_STATE']
            st['turn'] = int(msg.get('turn', 0))
            st['code'] = msg.get('code', '')
            st['events'] = []
            err = None
            try:
                with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
                    exec(compile(st['code'], f'<turn{st["turn"]}>', 'exec'), ns)
            except SystemExit:
                err = 'SystemExit raised (do not call exit()/quit()); sandbox state kept'
            except BaseException:
                tb = traceback.format_exc().splitlines()
                # keep only the frames from the model's code + the exception line
                keep = [l for l in tb if '<turn' in l or not l.startswith('  File')]
                err = '\n'.join(keep[-8:])
            out = buf.getvalue()
            if err:
                out += ('\n' if out and not out.endswith('\n') else '') + 'ERROR: ' + err
            clobbered = [k for k, v in orig.items() if ns.get(k) is not v]
            for k in clobbered:
                ns[k] = orig[k]
            if clobbered:
                out += ('\n' if out and not out.endswith('\n') else '') + f'NOTE: you rebound protected names {clobbered}; they were restored to the pre-loaded versions (use different names for your own functions).'
            new = st['candidates'][n_sent:]
            n_sent = len(st['candidates'])
            _send({'ok': True, 'out': _cap(out, out_cap), 'events': st['events'],
                   'candidates': new, 'submitted': st['submitted'], 'secs': round(time.time() - t0, 2)})
            continue
        _send({'ok': False, 'error': f'unknown op {op!r}'})


if __name__ == '__main__':
    try:
        main()
    except BaseException:
        try:
            _send({'ok': False, 'fatal': _cap(traceback.format_exc(), 1500)})
        except Exception:
            pass
'''
compile(MACROS_SRC, 'macros.py', 'exec'); compile(NARRATE_SRC, 'narrate.py', 'exec'); compile(WORKER_SRC, 'worker.py', 'exec')
exec(MACROS_SRC, _ns)   # 셀3 의 헬퍼 네임스페이스 위에 얹어 자가검증
exec(NARRATE_SRC, _ns)
_task = {'train': [{'input': [[0,0,1],[0,1,0],[1,0,0]], 'output': [[0,0,2],[0,2,0],[2,0,0]]},
                   {'input': [[1,1,0],[0,0,0]], 'output': [[2,2,0],[0,0,0]]}], 'test': [{'input': [[1,0],[0,1]]}]}
_ns['_init_task'](_task)
import io as _io, contextlib as _ctx
_buf = _io.StringIO()
with _ctx.redirect_stdout(_buf):
    _R = _ns['task_report']()
    _r = _ns['check'](lambda g: _ns['recolor'](g, {1: 2}))
    _ok = _ns['submit'](lambda g: _ns['recolor'](g, {1: 2}))
    _no = _ns['submit'](lambda g: g)
assert _R['color_map']['consistent'] and _R['bank_top'][0]['pairs_ok'] == 2, _R.get('bank_top')
assert _r['pairs_ok'] == 2 and _ok and not _no and _ns['_STATE']['submitted']['test_preds'] == [[[2, 0], [0, 2]]]
assert 'CONSISTENT' in _buf.getvalue() and 'SUBMITTED' in _buf.getvalue() and 'REFUSED' in _buf.getvalue()
# ---- 서술함수 자가검증: 관찰카드 / 게이트1 / 등급 채점 / 단계 / 실루엣 축 / 렌즈 증거 ----
assert 'OBSERVATION CARD' in _buf.getvalue() and 'null baselines' in _buf.getvalue()
_d = _ns['parse_description']('DESCRIPTION\nHEAD: size same; background 0\nMECH: color-map\nSTEPS:\n1. recolor 1 to 2\n2. done\nPARAM: 1->2\nSIZES: 3x3, 2x3\n```python\nx=1\n```')
_v = _ns['verify_description'](_d)
assert _d['found'] and _v['verdict'] == 'PASS', (_d, _v)
_v2 = _ns['verify_description'](_ns['parse_description']('DESCRIPTION\nHEAD: size different\nMECH: free\nSTEPS: 1. crop\nPARAM: -\nSIZES: 1x1, 1x1'))
assert _v2['verdict'] == 'REJECT' and len(_v2['reasons']) == 2, _v2
assert _ns['score_silx']([[0,0,5],[0,5,0],[5,0,0]], [[0,0,2],[0,2,0],[2,0,0]]) == 1.0   # 색만 다름 = 실루엣 1.0
assert _ns['score_silx']([[0,0,0],[0,0,0],[0,0,0]], [[0,0,2],[0,2,0],[2,0,0]]) == 0.0   # 배경채움 = 0
_t, _ax = _ns['score4']([[0,0,2],[0,2,0],[2,0,0]], [[0,0,2],[0,2,0],[2,0,0]])
assert _t == 1.0 and _ax == (1.0, 1.0, 1.0, 1.0), (_t, _ax)
_st, _f, _adv = _ns['stage_feedback']([_ns['G'](p['input']) for p in _ns['TRAIN']], (1.0, 0.8, 0.5, 1.0))
assert _st == 'COLOR', _st   # identity: 실루엣 같고 색만 다름
assert 0 <= _ns['null_base']() < 1 and 0 <= _ns['sil_null']() <= 1
assert isinstance(_ns['lens_evidence'](), list) and len(_ns['SEEDS']) == 15 and len(_ns['LENS_FAMILIES']) == 10
print(f'셀4 OK — 매크로 {len(_ns["SANDBOX_MACROS"])}종 / 서술함수 {len(_ns["NARRATION_FUNCTIONS"])}종 / 렌즈가족 {len(_ns["LENS_FAMILIES"])} / 씨앗 {len(_ns["SEEDS"])} / 변환은행 {len(_ns["BANK"])} / 리포트 {len(_buf.getvalue())}자')
print(_buf.getvalue()[:1500])


셀4 OK — 매크로 8종 / 서술함수 24종 / 렌즈가족 10 / 씨앗 15 / 변환은행 32 / 리포트 5731자
== TASK REPORT ==
pairs: 2 train / 1 test | sizes: 3x3->3x3, 2x3->2x3 | test in: 2x2
size rule: {"kind": "same"}
background BG=0 (confidence high) evidence: mode_in [0, 0] mode_out [0, 0] border [None, None] vacated {2: 5} score {0: 2.3, 1: 0.67}
colors: in [0, 1] out [0, 2] | new in output [2] | vanished [1] | per pair in->out: [0, 1]->[0, 2]; [0, 1]->[0, 2]
fixed colors (cells identical in & out): [0] | cells changed per pair: [3, 2] ([33.3, 33.3]%) | transitions from->to {'1->2': 5}
cell-wise color map is CONSISTENT: {0: 0, 1: 2} (changed [1]) -> a pure recolor may solve it (BANK['colormap'])
objects (single-color, 8-conn, non-bg) and input->output matching (match_nodes):
  pair0: objects 1->1 | stable 0, moved 0, recolored 1, appeared 0, disappeared 0 | recolored 1->2 px3 at [0, 0]
  pair1: objects 1->1 | stable 0, moved 0, recolored 1, appeared 0, disappeared 0 | recolored 1->2 px2 at [0, 0]
multi-color glued object

In [6]:
# [셀4b] ★자산 주입 (v58) — 채굴기(코랩/캐글)가 키운 저장소의 씨앗·승격 렌즈를 워커 few-shot 과 렌즈 카탈로그에 더한다.
#   읽는 곳(전부 합집합): Add Input 의 arc2-desc-assets Dataset / 채굴기 노트북 output 의 assets/ / 폴더의 저장소 zip / 아래 EXTRA_SEEDS(내장 폴백).
#   품질 규칙: quality<1.0(train 다 못 맞춘 요행 해결) 제외 · by=='rule'(규칙엔진 스텁, 서술 없음) 제외 · 빌트인 15 는 guided(사람) 씨앗만 대체.
#   자산이 하나도 없어도 v57 과 동일하게 돈다(내장 폴백만). 이 셀은 ★제출 rerun 에서도 Add Input 이 그대로 보이므로 같은 경로로 읽는다.
import zipfile as _zf
EXTRA_SEEDS = json.loads(r"""{"21f83797": {"lens": "object", "sig": {"size": "same", "holes": false, "d4": false, "reappear": false, "fixed_shape": true}, "rule": "DESCRIPTION\nHEAD: size same; background 0; how many shapes stay fixed 4; what gets overwritten cells changed per pair [67, 66]\nMECH: object - 2s in input define cross boundaries; fill cross lines with 2s and inner rectangle with 1s\nSTEPS: 1. find all rows/columns with 2s in input 2. determine cross boundaries (top, bottom, left, right) 3. fill cross lines (rows and columns) with 2s 4. fill inner rectangle (between cross lines) with 1s 5. leave rest as background 0\nPARAM: cross boundaries from input 2 positions; fill cross lines with 2; fill inner with 1\nSIZES: 13x13 13x13", "by": "mined", "split": "training"}, "ccd554ac": {"lens": "tile", "sig": {"size": "diff", "holes": true, "d4": false, "reappear": false, "fixed_shape": true}, "rule": "DESCRIPTION\nHEAD: size different; background 0; how many shapes stay fixed 1; what gets overwritten the original grid tiled in both dimensions\nMECH: tile the input grid by repeating each row W times horizontally and the entire grid H times vertically\nSTEPS:\n1. Determine the original grid's height (H) and width (W)\n2. For each row in the original grid:\n   a. Repeat the entire row W times to form a new row\n3. Repeat the new grid H times vertically to form the final output\nPARAM: H, W, background color\nSIZES: 3x3->9x9, 2x2->4x4, 2x2->4x4, 4x4->16x16, 3x3->9x9, 4x4->16x16", "by": "mined", "split": "training"}, "c9f8e694": {"lens": "detect_recolor", "sig": {"size": "same", "holes": false, "d4": false, "reappear": true, "fixed_shape": true}, "rule": "DESCRIPTION\nHEAD: size same; background 0; how many shapes stay fixed 4; what gets overwritten cells changed per pair [74, 37]\nMECH: detect_recolor - 5s are replaced by the last non-zero, non-5 color in their row; if no such color exists, replaced by background 0\nSTEPS: 1. for each row, track the last non-zero, non-5 color seen 2. for each cell in row, if cell is 5, replace with last non-zero, non-5 color (or 0 if none) 3. leave all other cells unchanged\nPARAM: 5 -> last non-zero, non-5 color in row; background 0\nSIZES: 12x12 12x12", "by": "mined", "split": "training"}, "56ff96f3": {"lens": "object", "sig": {"size": "same", "holes": false, "d4": false, "reappear": false, "fixed_shape": true}, "rule": "DESCRIPTION\nHEAD: size same; background 0; how many shapes stay fixed 0; what gets overwritten cells changed per pair [23, 12, 19, 29]\nMECH: object - each color forms a rectangle spanning min/max row/column of its occurrences\nSTEPS: 1. for each color in input, find all its positions 2. compute min_row, max_row, min_col, max_col for each color 3. fill the rectangle [min_row:max_row, min_col:max_col] with the color 4. set all other cells to background 0\nPARAM: colors [0, 1, 2, 3, 4, 7]; min/max row/col per color\nSIZES: 10x10 7x8 6x11 10x10", "by": "mined", "split": "training"}, "f45f5ca7": {"lens": "object", "sig": {"size": "same", "holes": false, "d4": false, "reappear": true, "fixed_shape": false}, "rule": "DESCRIPTION\nHEAD: size same; background 0; how many shapes stay fixed 0; what gets overwritten cells with 2,3,4,8\nMECH: object - each number moves to a fixed column based on its value: 2→col2, 3→col4, 4→col3, 8→col1; if the target column is occupied, move to the next available column in the same row\nSTEPS: 1. for each row, process numbers left to right 2. for each number, determine its target column based on value 3. if target column is empty, place the number there; else, move right until an empty column is found 4. set original positions to 0\nPARAM: value-to-column mapping {2:2, 3:4, 4:3, 8:1}\nSIZES: 10x10 10x10 10x10", "by": "mined", "split": "training"}, "ddf7fa4f": {"lens": "detect_recolor", "sig": {"size": "same", "holes": false, "d4": false, "reappear": false, "fixed_shape": true}, "rule": "DESCRIPTION\nHEAD: size same; background 0; how many shapes stay fixed 0; what gets overwritten cells changed per pair [36, 35, 21]\nMECH: detect_recolor - color 5 blocks are replaced by colors from the first row's fixed elements, based on their column span\nSTEPS: 1. identify all contiguous color-5 regions in input 2. for each region, determine its column span 3. find the first fixed color in the first row within the span, else left of span, else right of span 4. replace the entire region with that color\nPARAM: fixed colors in first row [1, 4, 7, 2, 6, 8]; column span of 5-regions\nSIZES: 10x10 10x10 10x10", "by": "mined", "split": "training"}, "ded97339": {"lens": "connect_same_color_hv", "sig": {"size": "same", "holes": false, "d4": false, "reappear": false, "fixed_shape": true}, "rule": "DESCRIPTION\nHEAD: size same; background 0; how many shapes stay fixed 1; what gets overwritten cells changed per pair [7, 9, 13]\nMECH: connect_same_color_hv - same-color regions are connected horizontally/vertically into larger contiguous shapes\nSTEPS: 1. For each color in the grid, merge all disconnected regions of that color into one connected component by filling gaps between them 2. Leave background cells untouched\nPARAM: color 8; background 0\nSIZES: 10x10 10x10 10x10", "by": "mined", "split": "training"}, "b230c067": {"lens": "object", "sig": {"size": "same", "holes": true, "d4": false, "reappear": false, "fixed_shape": false}, "rule": "DESCRIPTION\nHEAD: size same; background 0; how many shapes stay fixed 0; what gets overwritten 8\nMECH: object (connected shapes move/copy/recolor/resize as units)\nSTEPS: 1. find all 8-colored regions via connected components; 2. sort regions by size descending; 3. assign color 1 to all except smallest, which gets color 2\nPARAM: colors 8→1/2; sizes per region\nSIZES: 10x10 10x10 10x10", "by": "mined", "split": "training"}}""")
_BUILTIN = set(_ns['SEEDS'])


def _seeds_dir_under(root, maxdepth=3):
    for d in range(0, maxdepth + 1):
        hits = glob.glob('/'.join([root] + ['*'] * d + ['seeds.json']))
        if hits:
            return os.path.dirname(sorted(hits, key=len)[0])
    return None


def _asset_stores():
    """seeds.json 이 있는 폴더 전부 (+ zip 만 있는 폴더는 WORK/assets_in 에 풀어서). /kaggle/input 은 FUSE 라 깊이 제한."""
    cands = []
    if os.environ.get('ARC2_ASSETS_DIR'):
        cands.append(os.environ['ARC2_ASSETS_DIR'])
    if os.path.isdir('/kaggle/input'):
        for d in range(0, 6):
            cands += glob.glob('/'.join(['/kaggle/input'] + ['*'] * d + ['seeds.json']))
            cands += glob.glob('/'.join(['/kaggle/input'] + ['*'] * d + ['*arc2*asset*']))
    out = []
    for c in cands:
        c = os.path.dirname(c) if c.endswith('seeds.json') else c
        if not c or not os.path.isdir(c) or c in out:
            continue
        if os.path.exists(os.path.join(c, 'seeds.json')):
            out.append(c)
            continue
        for z in sorted(glob.glob(os.path.join(c, '*.zip'))):          # zip 만 놓인 폴더 (Dataset 이 안 풀었을 때)
            try:
                with _zf.ZipFile(z) as zf:
                    if not any(n.endswith('seeds.json') for n in zf.namelist()):
                        continue
                    dest = os.path.join(WORK, 'assets_in', os.path.basename(c.rstrip('/')) or 'zip')
                    os.makedirs(dest, exist_ok=True)
                    if _seeds_dir_under(dest) is None:
                        zf.extractall(dest)
                    d = _seeds_dir_under(dest)
                    if d and d not in out:
                        out.append(d)
                    break
            except Exception as _e:
                LOG(f'  자산 zip 해제 실패(무시): {os.path.basename(z)} {repr(_e)[:80]}')
    return out


def _load(p):
    try:
        return json.load(open(p, encoding='utf-8'))
    except Exception:
        return {}


ASSET_STORES = _asset_stores()
_merged, _lenses, _src_count = {}, {}, {}
for _d in ASSET_STORES:
    _sd, _ld = _load(os.path.join(_d, 'seeds.json')), _load(os.path.join(_d, 'lenses.json'))
    _n = 0
    for _tid, _s in (_sd or {}).items():
        if not isinstance(_s, dict) or not _s.get('rule') or not _s.get('sig'):
            continue
        if _s.get('by') == 'rule' or (_s.get('quality', 1.0) or 0) < 1.0:
            continue
        if any(_m in str(_s['rule']) for _m in ('(solved by the symbolic rule engine)', '(solved by whole-grid bank transform')) \
                or str(_s.get('lens', '')).startswith(('rule:', 'bank:')):
            continue                                                    # 서술 없는 스텁
        if _tid in _BUILTIN and _s.get('by') != 'guided':
            continue
        if _tid in {'bf699163', 'd89b689b', 'dc433765'}:                 # 구판 저장소의 요행 해결(quality 기록 없음) — 로그로 확인
            continue
        if _tid in _merged and _merged[_tid].get('by') == 'guided' and _s.get('by') != 'guided':
            continue
        _merged[_tid] = {'tid': _tid, 'lens': _s.get('lens', 'free'), 'sig': _s['sig'], 'rule': _s['rule'],
                         'override': _s.get('by') == 'guided', 'by': _s.get('by', 'mined')}
        _n += 1
    _GENERIC = {'object', 'objects', 'free', 'grid', 'color', 'colors', 'shape', 'shapes', 'pattern', 'cell', 'cells', 'rule', 'transform'}
    for _k, _l in (_ld or {}).items():
        if not (isinstance(_l, dict) and _l.get('source') == 'promoted'):
            continue
        if _k.split(':', 1)[-1].strip().lower() in _GENERIC:            # 'mech:object' 같은 일반어 승격은 방향이 아니다
            continue
        if _l.get('support', 0) >= _lenses.get(_k, {}).get('support', -1):
            _lenses[_k] = _l
    _src_count[_d] = _n
for _tid, _s in EXTRA_SEEDS.items():                                   # 내장 폴백 (저장소에 있으면 저장소 것이 이긴다)
    if _tid not in _merged and _tid not in _BUILTIN:
        _merged[_tid] = {'tid': _tid, 'lens': _s.get('lens', 'free'), 'sig': _s['sig'], 'rule': _s['rule'],
                         'override': _s.get('by') == 'guided', 'by': _s.get('by', 'mined')}

ASSET_SEEDS_FILE = f'{SB_DIR}/asset_seeds.json'
json.dump(list(_merged.values()), open(ASSET_SEEDS_FILE, 'w', encoding='utf-8'), ensure_ascii=False)
os.environ['ARC2_ASSET_SEEDS'] = ASSET_SEEDS_FILE                     # 셀6 이 os.environ 을 복사해 워커에 넘긴다

# 승격 렌즈 → 카탈로그 부록 (셀5 의 SYSTEM_PROMPT 조립에 들어간다)
PROMOTED_LENS_TEXT = ''
if _lenses:
    _pl = sorted(_lenses.values(), key=lambda l: -l.get('support', 0))[:12]
    PROMOTED_LENS_TEXT = ('\nAuto-promoted mechanism families (mined from solved tasks in earlier runs -- treat like the catalog above):\n'
                          + '\n'.join(f"- {l['key']}: {l['standard']} | slots: {l['slots']}" for l in _pl) + '\n')

# 워커 부팅 시 SEEDS 병합 스니펫 (빌트인 우선, override=guided 만 대체) — NARRATE_SRC 끝에 붙여 셀6 BOOT 에 실린다
_MERGE_SEEDS = (
    "\nimport os as _os3, json as _json3\n"
    "try:\n"
    "    _asp = _os3.environ.get('ARC2_ASSET_SEEDS')\n"
    "    if _asp and _os3.path.exists(_asp):\n"
    "        for _s in _json3.load(open(_asp, encoding='utf-8')):\n"
    "            if _s.get('tid') and _s.get('rule') and _s.get('sig') and (_s.get('override') or _s['tid'] not in SEEDS):\n"
    "                SEEDS[_s['tid']] = {'lens': _s.get('lens', 'free'), 'sig': _s['sig'], 'rule': _s['rule']}\n"
    "except Exception:\n"
    "    pass\n")
if '_asp = _os3.environ.get' not in NARRATE_SRC:
    NARRATE_SRC = NARRATE_SRC + '\n\n' + _MERGE_SEEDS
_by = Counter(v['by'] for v in _merged.values())
LOG(f'★자산 주입: 저장소 {len(ASSET_STORES)}개 {[(os.path.basename(d.rstrip("/")) or d, n) for d, n in _src_count.items()]} '
    f'/ 워커 씨앗 +{len(_merged)} (빌트인 {len(_BUILTIN)} 외; 경로별 {dict(_by)}; 내장 폴백 {len(EXTRA_SEEDS)}) / 승격 렌즈 {len(_lenses)} {sorted(_lenses)[:6]}')
if not ASSET_STORES:
    LOG('  (자산 저장소가 Add Input 에 없음 — 내장 폴백 씨앗만. 채굴기 output 또는 arc2-desc-assets Dataset 을 붙이면 더 읽는다)')
manifest_update(asset_stores=ASSET_STORES, asset_seeds=len(_merged), promoted_lenses=sorted(_lenses))


[14:27:15] ★자산 주입: 저장소 0개 [] / 워커 씨앗 +8 (빌트인 15 외; 경로별 {'mined': 8}; 내장 폴백 8) / 승격 렌즈 0 []
[14:27:15]   (자산 저장소가 Add Input 에 없음 — 내장 폴백 씨앗만. 채굴기 output 또는 arc2-desc-assets Dataset 을 붙이면 더 읽는다)


In [7]:
# [셀5] 프롬프트 v2 — 서술-우선 프로토콜 + 게이트된 애드엔덤 4종 (HELPERS=서술함수 / MACROS / BACKGROUND / LENS). MULTI_AVATAR 없음.
#   duck-merge-kit prompts_append.py 의 구조(플래그별 애드엔덤, 0 이면 빈 문자열)를 유지하되 ARC-2 서술 라인으로 다시 썼다:
#   · 서술이 1급 시민: 매 턴 DESCRIPTION(HEAD/MECH/STEPS/PARAM/SIZES) → 하네스가 게이트1(관찰카드 대조) → 코드 → check(등급 되먹임)
#     → 실패하면 코드 국소수정이 아니라 서술로 되돌아간다 ("99% 맞은 규칙은 0").
#   · HELPERS = 격자 프리미티브 + 서술함수 (관찰카드·정합체크·4축/실루엣 등급·단계·반례) — ARC-3 헬퍼처럼 샌드박스 선적재.
#   · LENS   = 서술의 통찰을 미리 정의 (8 관점 + 10 기전 가족의 표준·변형 슬롯) — BACKGROUND_HINT 처럼 증거 기반: task_report 가
#              전제조건이 성립하는 렌즈만 후보로 찍는다. MUST 강제 없음.
def _flag(name):
    return os.environ.get(name, '').strip() in ('1', 'true', 'True', 'TRUE')

SYSTEM_CORE = (
    "You are an expert ARC-AGI-2 solver working inside a persistent Python sandbox. You solve by DESCRIPTION FIRST: "
    "you state the rule as a structured description, then implement it as `solve(grid)`, verify it on ALL training pairs with "
    "`check(solve)`, and finish with `submit(solve)`.\n\n"
    "SANDBOX (state persists across your turns):\n"
    "- `TRAIN`: list of `{'input': np.ndarray, 'output': np.ndarray}`; `TEST`: list of `{'input': np.ndarray}` (read-only int arrays, colors 0-9, max 30x30).\n"
    "- `BG` inferred background color, `REPORT` the task-report dict (`REPORT['card']` = observation card, `REPORT['lens_evidence']`, `REPORT['size']`), `BANK` fitted transforms.\n"
    "- `np`, `Counter`, `deque`, `math`, `itertools`, `json`, `re` are imported. No files, no network, no other imports.\n"
    "- `solve(grid)` receives a numpy int array (a copy) and returns a 2D list/array of ints 0-9 (max 30x30).\n\n"
    "PROTOCOL for EVERY turn -- two parts, in this order:\n"
    "1. A DESCRIPTION block (plain text, exactly these five keys):\n"
    "   DESCRIPTION\n"
    "   HEAD: size same|different; background <c>; how many shapes stay fixed; what gets overwritten  (claims the harness checks against the measured card)\n"
    "   MECH: <one lens from the catalog, or 'free'> -- one sentence naming the mechanism (WHAT determines WHAT)\n"
    "   STEPS: 1. ... 2. ... (2-6 numbered steps; each step must map to one or two helper calls)\n"
    "   PARAM: colors / numbers / directions the steps use\n"
    "   SIZES: HxW per training example, in order (your rule's prediction -- a free self-check before any code)\n"
    "2. EXACTLY ONE ```python block that defines `solve` (implementing the STEPS with the helpers) and ends with `check(solve)`.\n"
    "The harness replies with a DESCRIPTION CHECK (each HEAD/SIZES claim marked ✓/✗ with the measured value) and the sandbox output. "
    "A ✗ on size or SIZES means the MECHANISM is wrong, not a detail.\n"
    "3. When check() reports all pairs pass, call `submit(solve)` (ends the task). `submit` refuses code that fails a training pair unless `force=True` "
    "-- use force only on your final turn with your best candidate.\n"
    "4. A rule that is 99% correct scores ZERO. When check() fails, do NOT patch cells or special-case coordinates: go back to the DESCRIPTION and change "
    "the MECH or a STEP, then re-implement. Solve one stage at a time as the STAGE line says (SIZE -> MASK -> COLOR): never change what is already exact.\n"
    "5. The solution must generalize (no hard-coded training outputs). Between two rules that fit, prefer the simpler one whose test output looks sensible.\n"
    "Do not print whole grids repeatedly (they are shown once); print diffs, object summaries, counts. Never re-implement a helper listed below -- call it.\n"
)

SANDBOX_HELPERS_ADDENDUM = (
    "\n\nPre-loaded HELPERS (ALREADY DEFINED in the sandbox; call them, never rewrite them):\n"
    "Grid primitives:\n"
    "- `objects(grid, bg=None, conn=8, by_color=True) -> [node]` connected non-background components in reading order; node = {'id','color'(-1 if multi),'colors',"
    "'pixels','cells':[(r,c)..],'bbox':[r0,c0,r1,c1],'center','hash'(color+shape),'sil'(shape only),'holes','touches_border','is_rect','is_line','is_dot','crop'(-1 = empty)}. "
    "by_color=False glues touching colors into one node. `summ(node)` one-line summary.\n"
    "- `bbox`, `center`, `manhattan(a,b)`, `nodes_by_color(nodes,c)`, `neighbors(nodes,id)`, `touching(nodes,i,j)`, `same_shape(n1,n2,ignore_color=True,allow_d4=False)`.\n"
    "- `match_nodes(nodes_in, nodes_out) -> {'stable','moved':[{'color','px','from','to','shift'}],'recolored':[{...,'color_to','kind'}],'appeared','disappeared','shape_change_candidates'}`; "
    "`match_summary(m)` one line. `diff_cells(a,b) -> {'changed','bbox','cells':[[r,c,from,to]],'transitions'}`. `color_map(TRAIN) -> {'map','consistent','conflicts','changed'}`. "
    "`compare(pred,target) -> {'match','shape_ok','wrong','wrong_cells','per_color'}`.\n"
    "- Structure: `background(grid)`, `infer_background(pairs)`, `separators(grid) -> (rows, cols, color)`, `panels(grid) -> [(bbox, subgrid)]`, "
    "`enclosed_regions(grid, bg) -> [cells]` (pockets not touching the border), `fill_region(grid,cells,color)`, `reachable_cells(grid,(r,c),passable)`, "
    "`bfs_path(grid,start,goal,passable)`, `flood_fill(grid,r,c,color)`, `ray(grid,r,c,dr,dc,color,stop=None) -> (grid, hit)`, `symmetries(grid)`, "
    "`symmetrize(grid,'HV',bg)`, `size_relation(pairs)`, `find(pat,big,ignore=-1) -> (r,c)|None`, `find_all(pat,big)`, `components(mask,conn)`.\n"
    "- Editing (return a NEW grid): `d4(grid,k)` k in id/rot90/rot180/rot270/flipH/flipV/transpose/anti, `d4_all`, `crop(grid,[r0,c0,r1,c1])`, "
    "`paste(grid,sub,r,c,transparent=-1)`, `scale(grid,k)`, `downscale(grid,k)`, `tile(grid,ny,nx)`, `recolor(grid,{from:to})`, `gravity(grid,dir,bg)`, "
    "`node_grid(node,shape=None,bg)`, `move(grid,node,dr,dc)`, `erase(grid,node)`, `fill_holes(grid,node,color)`, `outline_cells(node)`, `count_colors(grid)`, `new_grid(h,w,color)`.\n"
    "Narration functions (measure / verify / grade -- the same ones the harness uses):\n"
    "- `observe_card() -> dict` and `card_text(card)`: the measured HEAD facts (size, background, fixed colors, overwrite, object fate, fixed/reappearing shapes, holes, D4, panels, tags).\n"
    "- `score4(pred, target) -> (total, (size, shape, color, position))` graded 4-axis score; `score_silx(pred, target)` silhouette match ignoring color "
    "(compression-based; 1.0 = same shapes anywhere); `best_shift(mask_pred, mask_target) -> (dy, dx, iou)`; `sil_facts(pred, target, label)` which shape material is missing/invented; "
    "`stage_feedback(preds, axes)`; `counterexample(preds)`.\n"
    "- Printing: `show(grid,label)`, `show_pair(i)`, `show_test(j)`, `diff_pair(i)`, `objects_report(i, which='both', by_color=True)`.\n"
    "These functions know nothing about the task; treat their output as evidence to verify, not as the answer.\n"
) if _flag('DUCK_SANDBOX_HELPERS') else ''

MACROS_ADDENDUM = (
    "\n\nPre-built MACROS (already defined; verified against TRAIN; each prints one compact summary):\n"
    "- `task_report()` was ALREADY RUN for you at turn 0 -- its output is in your first message and in `REPORT`. Re-run only if you need it again.\n"
    "- `try_transforms()` fits ~50 parametrized whole-grid transforms on TRAIN and lists the best by pairs passed / cell accuracy; every candidate stays callable as "
    "`BANK[name](grid)` -- a partial match is a building block, a full match can be submitted after you judge its test output.\n"
    "- `check(solve)` runs solve on every training pair (5 s each) and on TEST: prints exact k/n with the first wrong cells, then the GRADED line "
    "(normalized 4-axis score where identity/background-fill = 0 and the answer = 1; silhouette score; STAGE), stage facts, a counterexample with its input neighbourhood, "
    "and registers the candidate. Call it in EVERY turn -- it costs nothing.\n"
    "- `submit(solve, force=False)` final answer: accepted only when check() is all-pass (or force=True); accepting ends the task. attempt_2 is filled automatically "
    "from your other registered candidates (best graded score), so a second, different candidate is never wasted.\n"
    "Rule: if a helper or macro can do it, calling it is cheaper and less bug-prone than hand-written loops. Read the printed summary, then decide the next step.\n"
) if _flag('DUCK_MACROS') and _flag('DUCK_SANDBOX_HELPERS') else ''

BACKGROUND_HINT_ADDENDUM = (
    "\n\nBACKGROUND PROTOCOL (evidence, not assumption):\n"
    "- `BG` was inferred from the training pairs, not from area alone: strongest evidence is the color revealed in cells an object VACATED, then a uniform border color, "
    "then area in inputs AND outputs; `REPORT['background']` holds the evidence and a confidence. Do not assume the most common color is the background; when confidence "
    "is 'low' or per-grid backgrounds differ, decide per grid inside solve (`bg = background(grid)`) and test both candidates with check().\n"
    "- `enclosed_regions` / `reachable_cells` separate inside from outside; objects with `touches_border` are often frames or separators; `panels(grid)` splits at full lines.\n"
    "- If the output size differs from the input, settle the size rule first (`REPORT['size']`, SIZES): a solve with the wrong shape scores 0 on every pair.\n"
) if _flag('DUCK_BACKGROUND_HINT') and _flag('DUCK_SANDBOX_HELPERS') else ''

LENS_CATALOG = (
    "\n\nLENS CATALOG -- the insights you describe with (name the one you use in MECH):\n"
    "Perspectives (8): geometric (rotation/reflection/transpose/tiling/scaling) | color-map (each color consistently replaced) | neighborhood (a cell's output depends on its neighbours) "
    "| object (connected shapes move/copy/recolor/resize as units) | row/column (reverse/sort/select/count/repeat rows or columns) | counting (most/least frequent color or shape decides) "
    "| symmetry (complete/enforce/repair a symmetry) | free.\n"
    "Mechanism families (10) -- each has a STANDARD (the invariant) and VARIATION slots you must fill in PARAM:\n"
    + ''.join(f"- {k}: {v['standard']} | slots: {v['slots']}\n" for k, v in {
        'restore': {'standard': 'the grid is globally symmetric or periodic; an occluding region hides part of it; hidden cells equal their symmetric/periodic counterparts',
                    'slots': 'symmetry kind (mirror H/V, point, period) / occluder color / output = hidden patch only or full restored grid'},
        'lfill': {'standard': 'one large plain shape is a TEMPLATE with interior holes; other material in the grid supplies what goes into the holes',
                  'slots': 'filler source (transplanted pieces / majority color of matching fillers / exact packing) / scaling or D4 allowed / output = template bbox'},
        'indicator': {'standard': 'a main object is edited; small shapes far from it (or stripes on the border) are INSTRUCTIONS, not content -- they never appear in the output',
                      'slots': 'instruction meaning (rotate by count / swap color pair / face side) / output = main object bbox'},
        'occlusion': {'standard': 'a window/frame covers shapes; pieces of one shape cut by the window belong together and continue straight underneath',
                      'slots': 'output = window content with hidden parts reconstructed'},
        'stamp': {'standard': 'one box holds single-cell dots (POSITIONS), another holds multi-cell motifs (CONTENT); stamp the motif at every dot',
                  'slots': 'dot color selects motif kind / output = dot box'},
        'legend': {'standard': 'a small region is a LEGEND: complete sample shapes define a lookup (e.g. hole shape -> color) applied to the incomplete/noise-colored ones',
                   'slots': 'legend key/value / targets / no-match handling'},
        'detect_recolor': {'standard': 'the output differs from the input only where a specific local pattern occurs; those cells are recolored, everything else unchanged',
                           'slots': 'pattern (plus / L / 2x2 ...) / new color'},
        'pair_assemble': {'standard': 'every object reappears; objects of two kinds are paired in positional (reading) order and assembled into a new layout',
                          'slots': 'kind test (hollow vs solid, color, size) / layout / leftover handling'},
        'panel_vote': {'standard': 'the grid is a table of panels; within a group most panels are identical and exactly one differs -- select by minority/majority',
                       'slots': 'group = row / column / all; select odd or common one; output = selected panels stacked'},
        'rows_to_rings': {'standard': 'a 1D sequence in the input (stripes) becomes a nested structure in the output (concentric rings / spiral)',
                          'slots': 'sequence direction / ring order / side length formula'},
    }.items())
    + "The task report lists LENS EVIDENCE: families whose measured preconditions hold in every training pair. They are candidates to TEST with check(), not verdicts; "
    "a family with no evidence may still be right if you can say why. The SIMILAR SOLVED PUZZLES are verified OBJECTS/STEPS descriptions of other tasks: copy their style "
    "(name the objects, then 2-6 concrete steps), never their content.\n"
) if _flag('DUCK_LENS_HINT') and _flag('DUCK_SANDBOX_HELPERS') else ''

# MULTI_AVATAR_HINT: 제거 (ARC-2 에 조종 객체가 없다). 플래그는 기록용으로만 남긴다.
MULTI_AVATAR_HINT_ADDENDUM = ''

SYSTEM_PROMPT = SYSTEM_CORE + SANDBOX_HELPERS_ADDENDUM + MACROS_ADDENDUM + BACKGROUND_HINT_ADDENDUM + LENS_CATALOG + ((globals().get('PROMOTED_LENS_TEXT') or '') if _flag('DUCK_LENS_HINT') else '') + MULTI_AVATAR_HINT_ADDENDUM   # v58: 승격 렌즈
for _k in sorted(DUCK_FLAGS):
    print('FLAG', _k, '=', os.environ.get(_k))
print(f'SYSTEM_PROMPT {len(SYSTEM_PROMPT)} chars: core {len(SYSTEM_CORE)} / helpers {len(SANDBOX_HELPERS_ADDENDUM)} / macros {len(MACROS_ADDENDUM)} / '
      f'background {len(BACKGROUND_HINT_ADDENDUM)} / lens {len(LENS_CATALOG)} / multi-avatar {len(MULTI_AVATAR_HINT_ADDENDUM)}')


def grid_lines(g):
    return '\n'.join(''.join(str(int(v)) for v in row) for row in g)


def grid_block(label, g):
    g = np.asarray(g)
    return f'{label} ({g.shape[0]}x{g.shape[1]}):\n{grid_lines(g)}'


def task_text(tid, task, compact=False):
    """첫 메시지의 격자 부분. compact=True: 첫 쌍 + 첫 test 만 격자, 나머지는 크기만 (컨텍스트 초과 시)."""
    n, m = len(task['train']), len(task['test'])
    parts = [f'Task {tid}: {n} training pairs, {m} test input(s). Grids are digit rows (row 0 first, column 0 first).']
    for i, p in enumerate(task['train']):
        if compact and i > 0:
            parts.append(f"Train pair {i}: input {len(p['input'])}x{len(p['input'][0])} -> output {len(p['output'])}x{len(p['output'][0])} (not shown: use show_pair({i}) in the sandbox)")
            continue
        parts.append(grid_block(f'Train pair {i} input', p['input']))
        parts.append(grid_block(f'Train pair {i} output', p['output']))
    for j, t in enumerate(task['test']):
        if compact and j > 0:
            parts.append(f"Test input {j}: {len(t['input'])}x{len(t['input'][0])} (not shown: show_test({j}))")
            continue
        parts.append(grid_block(f'Test input {j}', t['input']))
    return '\n\n'.join(parts)


FIRST_TURN_ASK = ('Turn 1 of {n}. Write the DESCRIPTION (HEAD / MECH / STEPS / PARAM / SIZES) first -- use the OBSERVATION CARD facts in HEAD and pick a lens '
                  'for MECH (or "free") -- then ONE ```python block that implements the STEPS as solve(grid) and ends with check(solve). '
                  'You may explore with the helpers inside the same block before defining solve.')


def first_message(tid, task, report, compact=False, max_turns=None):
    return task_text(tid, task, compact) + '\n\n' + report.strip() + '\n\n' + FIRST_TURN_ASK.format(n=max_turns or MAX_TURNS)


def desc_check_text(v, turn):
    """워커 verify 응답 → 메시지 블록"""
    if not v or v.get('verdict') == 'ERROR':
        return f'[description check, turn {turn}] (verification error)'
    if v.get('verdict') == 'NO_DESCRIPTION':
        return (f'[description check, turn {turn}] NO DESCRIPTION BLOCK FOUND. Every turn must start with DESCRIPTION / HEAD / MECH / STEPS / PARAM / SIZES '
                f'(the code was still executed).')
    head = f"[description check, turn {turn}] {v['verdict']}"
    if v.get('reasons'):
        head += ' -- ' + ' | '.join(v['reasons'])
    return head + '\n  ' + '\n  '.join(v.get('lines', []))


def tool_message(out, turn, restarted=False, last=False, hint='', desc_block='', max_turns=None):
    n = max_turns or MAX_TURNS
    head = f'[sandbox output, turn {turn}]'
    if restarted:
        head += ' [the sandbox was RESTARTED after a timeout/crash: your variables are lost; TRAIN/TEST/BG/REPORT/BANK/helpers are restored]'
    tail = f'Turn {turn + 1} of {n}. Revise the DESCRIPTION if any claim was ✗ or check() failed (change MECH or a STEP -- not a few cells), then implement and check again.'
    if last:
        tail = (f'FINAL TURN ({turn + 1} of {n}): give your best DESCRIPTION and you MUST call submit(solve) in this block '
                f'(force=True if it does not pass every pair).')
    if hint:
        tail += ' ' + hint
    return ((desc_block + '\n\n') if desc_block else '') + f'{head}\n{out.rstrip()}\n\n{tail}'


_CODE_RE = re.compile(r'```(?:python|py)?[ \t]*\n(.*?)```', re.S)


def extract_code(text):
    """답변에서 ```python 블록 전부(순서대로 결합). 닫는 ``` 없이 잘린 마지막 블록도 회수."""
    blocks = _CODE_RE.findall(text)
    tail = text.rsplit('```python', 1)
    if len(tail) == 2 and '```' not in tail[1]:
        blocks.append(tail[1])
    blocks = [b for b in blocks if b.strip()]
    return '\n\n'.join(blocks) if blocks else None


def description_part(text):
    """답변에서 코드블록 앞의 텍스트(서술 부분)만"""
    return re.split(r'```', text, 1)[0]


def reopen_message(prev_turn, extra_turns, best_pairs, n_pairs, best_nscore, best_stage):
    """2패스 재개 메시지 — 근접 태스크를 새 샌드박스에서 이어간다 (solve 재정의 필요)."""
    return (f'[PASS 2 -- this task is re-opened because it came close: best candidate passed {best_pairs}/{n_pairs} training pairs, '
            f'normalized score {best_nscore}, stage {best_stage}] The sandbox was RESET (TRAIN/TEST/BG/REPORT/BANK/helpers restored; your '
            f'variables and solve are gone -- redefine solve). You have {extra_turns} more turns (turn {prev_turn + 1} of {prev_turn + extra_turns}). '
            f'Do not repeat a failed candidate: change the MECH or the failing STEP in the DESCRIPTION, implement, check(solve), and submit when all pairs pass.')


FLAG DUCK_BACKGROUND_HINT = 1
FLAG DUCK_LENS_HINT = 1
FLAG DUCK_MACROS = 1
FLAG DUCK_MULTI_AVATAR_HINT = 0
FLAG DUCK_SANDBOX_HELPERS = 1
SYSTEM_PROMPT 11057 chars: core 2644 / helpers 3035 / macros 1324 / background 835 / lens 3219 / multi-avatar 0


In [8]:
# [셀6] 샌드박스 실행기 — 태스크당 서브프로세스 1개 (상태 유지, JSON-lines), 타임아웃/크래시 시 재시작
#   덕 하네스의 python_tool_sandbox 대응. 헬퍼+매크로 소스를 bootstrap.py 로 이어 붙여 워커가 exec 한다
#   (킷의 __HELPERS_SOURCE__ 스플라이스와 같은 방식). 병렬 24개 프로세스라 numpy 스레드는 1로 고정.
BOOT = f'{SB_DIR}/bootstrap.py'
WORKER = f'{SB_DIR}/worker.py'
open(BOOT, 'w', encoding='utf-8').write(HELPERS_SRC + '\n\n' + MACROS_SRC + '\n\n' + NARRATE_SRC)   # 헬퍼 + 매크로 + 서술함수 = 부트스트랩
open(WORKER, 'w', encoding='utf-8').write(WORKER_SRC)
compile(open(BOOT, encoding='utf-8').read(), BOOT, 'exec')
print(f'BOOT {os.path.getsize(BOOT) / 1e3:.0f}KB / 자산씨앗 파일 {"O" if os.environ.get("ARC2_ASSET_SEEDS") else "X"}')
_SB_ENV = dict(os.environ, OMP_NUM_THREADS='1', MKL_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1', PYTHONUNBUFFERED='1')
_SB_ENV.pop('CUDA_VISIBLE_DEVICES', None)
_SB_ENV['CUDA_VISIBLE_DEVICES'] = ''      # 워커는 GPU 를 만지지 않는다


class Sandbox:
    def __init__(self, tid, task):
        self.tid, self.task = tid, task
        self.proc, self.q = None, None
        self.restarts, self.report, self.bank_top, self.bg_conf, self.init_secs = 0, '', [], None, 0.0
        self._start()

    def _reader(self, proc, q):
        try:
            for line in proc.stdout:
                q.put(line)
        except Exception:
            pass
        q.put(None)

    def _start(self):
        self.proc = subprocess.Popen([sys.executable, WORKER, BOOT, str(OUT_CAP)], stdin=subprocess.PIPE,
                                     stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True,
                                     bufsize=1, env=_SB_ENV, cwd=SB_DIR)
        self.q = queue.Queue()
        threading.Thread(target=self._reader, args=(self.proc, self.q), daemon=True).start()
        r = self._recv(INIT_TIMEOUT_S)
        if not r or not r.get('ready'):
            raise RuntimeError(f'[{self.tid}] 샌드박스 워커 부팅 실패: {r}')

    def _recv(self, timeout):
        try:
            line = self.q.get(timeout=timeout)
        except queue.Empty:
            return None
        if line is None:
            return {'ok': False, 'dead': True}
        try:
            return json.loads(line)
        except Exception as e:
            return {'ok': False, 'error': f'bad response: {e}: {line[:200]}'}

    def _kill(self):
        try:
            self.proc.kill()
            self.proc.wait(timeout=5)
        except Exception:
            pass

    def call(self, msg, timeout):
        try:
            self.proc.stdin.write(json.dumps(msg) + '\n')
            self.proc.stdin.flush()
            r = self._recv(timeout)
        except (BrokenPipeError, OSError, ValueError):
            r = {'ok': False, 'dead': True}
        if r is None or r.get('dead') or r.get('fatal'):
            why = 'timeout (> %ds)' % timeout if r is None else ('crash: ' + str(r.get('fatal', ''))[:300])
            self._kill()
            self.restarts += 1
            self._start()
            r0 = self.call({'op': 'init', 'task': self.task}, INIT_TIMEOUT_S) if msg.get('op') != 'init' else {'ok': True}
            return {'ok': False, 'restarted': True, 'out': f'ERROR: sandbox {why}. It was restarted (state reset).',
                    'events': [], 'candidates': [], 'submitted': None, 'error': why, 'reinit_ok': r0.get('ok', False)}
        return r

    def init(self):
        r = self.call({'op': 'init', 'task': self.task}, INIT_TIMEOUT_S)
        self.report = r.get('report', '') or ''
        self.bank_top = r.get('bank_top', []) or []
        self.bg_conf = r.get('bg_conf')
        self.init_secs = r.get('secs', 0.0)
        return r

    def exec(self, code, turn):
        return self.call({'op': 'exec', 'code': code, 'turn': turn}, EXEC_TIMEOUT_S)

    def close(self):
        try:
            self.proc.stdin.write('{"op": "exit"}\n')
            self.proc.stdin.flush()
            self.proc.wait(timeout=3)
        except Exception:
            self._kill()


# ---- 자가검증: 첫 태스크로 init → 리포트 → check/submit 배관 (CPU, 수 초) ----
if POOL:
    _tid, _task, _ = POOL[0]
    _t = time.time()
    _sb = Sandbox(_tid, _task)
    _r = _sb.init()
    assert _r.get('ok'), f'샌드박스 init 실패: {_r}'
    print(f'[{_tid}] init {time.time() - _t:.1f}s / 리포트 {len(_sb.report)}자 / bank_top {_sb.bank_top[:2]}')
    print(_sb.report[:1800] + ('...' if len(_sb.report) > 1800 else ''))
    _rv = _sb.call({'op': 'verify', 'text': 'DESCRIPTION\nHEAD: size same; background 0\nMECH: free\nSTEPS:\n1. return the input\n2. done\nPARAM: none\nSIZES: 1x1'}, 30)
    assert _rv.get('ok') and _rv.get('verdict') in ('PASS', 'REJECT'), _rv
    print('--- verify ---', _rv['verdict'], _rv['reasons'][:1])
    _r2 = _sb.exec('def solve(grid):\n    return grid\nr = check(solve)\nprint("identity pairs_ok", r["pairs_ok"])\nsubmit(solve)', 1)
    assert _r2.get('ok') and _r2.get('candidates') is not None, _r2
    print('--- exec 출력 ---\n' + _r2['out'][:1400])
    if SMOKE:                                # 타임아웃 → 재시작 경로 (EXEC_TIMEOUT_S 초 소요) — 스모크에서만
        _r3 = _sb.exec('while True: pass', 2)
        assert _r3.get('restarted'), _r3
        print('타임아웃 재시작 OK:', _r3['out'])
    _sb.close()
    print(f'샌드박스 배관 OK (candidates {len(_r2["candidates"])}, submitted={_r2["submitted"] is not None})')


BOOT 122KB / 자산씨앗 파일 O
[fc754716] init 0.2s / 리포트 5120자 / bank_top []
== TASK REPORT ==
pairs: 4 train / 1 test | sizes: 5x7->5x7, 5x3->5x3, 5x5->5x5, 3x3->3x3 | test in: 9x7
size rule: {"kind": "same"}
background BG=0 (confidence high) evidence: mode_in [0, 0, 0, 0] mode_out [1, 3, 6, 2] border [0, 0, 0, 0] vacated {0: 4} score {0: 7.8, 2: 0.56, 3: 0.47}
colors: in [0, 1, 2, 3, 6] out [0, 1, 2, 3, 6] | new in output [] | vanished [] | per pair in->out: [0, 1]->[0, 1]; [0, 3]->[0, 3]; [0, 6]->[0, 6]; [0, 2]->[0, 2]
fixed colors (cells identical in & out): [] | cells changed per pair: [21, 13, 17, 9] ([60.0, 86.7, 68.0, 100.0]%) | transitions from->to {'0->1': 20, '0->6': 16, '0->3': 12, '0->2': 8, '1->0': 1, '3->0': 1, '6->0': 1, '2->0': 1}
cell-wise color map: majority {0: 0, 1: 0, 2: 0, 3: 0, 6: 0} but INCONSISTENT for colors [0] -> the change depends on position/shape, not on color alone
objects (single-color, 8-conn, non-bg) and input->output matching (match_nodes):
  pair0: object

In [9]:
# [셀7] Qwen3.8-27B-FP8 엔진 — 캐글 L4×4 vLLM (풀이전용판 셀3 의 검증 코드를 '대화(messages)' 입력으로 확장)
#   변경점: (1) 프롬프트 문자열 → messages 리스트 (system + 다중턴, 이전 턴의 사고는 제거해 저장)
#           (2) 2패스 통합: 사고만 하다 잘림 → </think> 강제 후 답 / 사고는 끝났는데 답이 잘림 → 답만 이어서
#           (3) MAX_MODEL_LEN 32768 시도 → 실패 시 16384(검증값) 재시도
#   그 외(휠하우스 오프라인 vLLM, TP=4, util 0.85, max_num_seqs 32, probe) 는 원본 그대로.
import gc

THINK_CLOSE = ('\n\nConsidering the limited time, I have to give the answer now '
               'based on my reasoning so far.\n</think>\n\n')
MIN_GEN_TOK = THINK_MIN + ANSWER_MIN
PROBE_TOK = 64

try:
    import torch
    _ngpu = torch.cuda.device_count() if torch.cuda.is_available() else 0
    _vtot = sum(torch.cuda.get_device_properties(i).total_memory for i in range(_ngpu)) / 1e9 if _ngpu else 0
except Exception as _e:            # STUB(스모크) 에서는 torch 없이도 진행
    torch, _ngpu, _vtot = None, 0, 0.0
    assert STUB_LLM, f'★torch 없음: {_e!r}'
LOG(f'GPU x{_ngpu} / 총 {_vtot:.1f}GB')


def _find_q38():
    """Qwen3.8-27B 캐글 모델 자동 탐색 (풀이전용판과 동일: 표준 레이아웃 우선, repack 차선)."""
    cands = []
    for pat in ('/kaggle/input/models/**/qwen3-8-27b*/**/config.json',
                '/kaggle/input/**/*qwen3*8*27b*/**/config.json',
                '/kaggle/input/**/*qwen3.8*27b*/**/config.json',
                '/kaggle/input/**/*qwen38*27b*/**/config.json'):
        cands += [os.path.dirname(p) for p in glob.glob(pat, recursive=True)]
    cands.append('/kaggle/input/models/foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1')
    seen, ok = set(), []
    for d in cands:
        if d in seen or not os.path.exists(f'{d}/config.json'):
            continue
        seen.add(d)
        ok.append((os.path.exists(f'{d}/outside.safetensors'), d))
    ok.sort()
    return ok[0][1] if ok else None


def _split_think(t):
    """(thinking text, answer text). 템플릿이 <think> 여는 태그를 프롬프트에 넣으므로 출력엔 </think> 만 있다."""
    if '</think>' in t:
        a, b = t.split('</think>', 1)
        return a.replace('<think>', '').strip(), b
    if '<think>' in t:
        return t.split('<think>', 1)[1].strip(), ''
    return '', t


class Engine:
    def __init__(self):
        self.kind, self.tok, self.BATCH, self.TPS = 'stub', None, 8, 0.0
        if STUB_LLM:
            LOG('★STUB 엔진 (가짜 LLM) — 배관 검증용')
            return
        MODEL_PATH = _find_q38()
        assert MODEL_PATH, ('★Qwen3.8-27B 모델 미부착 — Add Input > Models: foysalemonshanto/qwen3-8-27b-fp8-repacked-v1 (hf-fp8)')
        LOG('모델:', MODEL_PATH)
        try:
            from vllm import LLM, SamplingParams  # noqa: F401
        except Exception as e:
            raise RuntimeError('★vllm 이 없다. 인터넷 OFF 이므로 Add Input 에 vLLM 휠 데이터셋(driessmit1/arc3-vllm-h100-wheelhouse-v3)을 '
                               '붙여야 한다. 원인: ' + repr(e)[:200])
        global MAX_MODEL_LEN
        last = None
        for mlen in ([MAX_MODEL_LEN, 16384] if MAX_MODEL_LEN != 16384 else [16384]):
            try:
                LOG(f'★vLLM 적재 (TP={max(_ngpu, 1)}, max_model_len={mlen}): {MODEL_PATH}')
                self.llm = LLM(model=MODEL_PATH, tensor_parallel_size=max(_ngpu, 1),
                               gpu_memory_utilization=0.85, max_model_len=mlen,
                               max_num_seqs=32, trust_remote_code=True,
                               disable_log_stats=False)   # [PATCH-3a] 선점/캐시 통계 노출
                MAX_MODEL_LEN = mlen
                last = None
                break
            except Exception as e:
                last = e
                LOG(f'  적재 실패 (max_model_len={mlen}): {repr(e)[:300]}')
                gc.collect()
                try:
                    torch.cuda.empty_cache()
                except Exception:
                    pass
                time.sleep(5)
        if last is not None:
            raise RuntimeError('★vLLM 적재 실패. 원인 후보: (a) vllm 버전이 Qwen3.8 아키텍처를 모름 → 더 최신 휠 데이터셋, '
                               '(b) repacked 레이아웃을 표준 로더가 못 읽음, (c) FP8 커널이 L4(SM8.9)에서 거부. 원인: ' + repr(last)[:300])
        self.tok = self.llm.get_tokenizer()
        self.kind = 'vllm'
        from vllm import SamplingParams
        t0 = time.monotonic()
        sp = SamplingParams(temperature=0.9, top_p=0.95, max_tokens=PROBE_TOK, ignore_eos=True)
        outs = self.llm.generate([self.chat([{'role': 'user', 'content': 'Count from one to one hundred.'}], False)] * 16,
                                 sp, use_tqdm=False)
        n = sum(len(o.outputs[0].token_ids) for o in outs)
        self.TPS = n / max(time.monotonic() - t0, 1e-9)
        LOG(f'★vLLM 준비 완료 / probe(batch16) {self.TPS:.1f} tok/s / max_model_len {MAX_MODEL_LEN}')

    def ntok(self, text):
        try:
            return len(self.tok.encode(text))
        except Exception:
            return int(len(text) / 2.5)

    def chat(self, messages, think=True):
        """messages 리스트 → 템플릿 문자열 (Qwen3.8 = 추론모델: enable_thinking 켬/끔)."""
        if isinstance(messages, str):
            messages = [{'role': 'user', 'content': messages}]
        try:
            return self.tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,
                                                enable_thinking=bool(think))
        except Exception:
            return self.tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    def gen(self, convs, temp=TEMP, think=True):
        """대화 리스트 → [{'text': 답, 'think': 사고, 'ntok': 생성토큰, 'skipped': bool, 'cut': bool}]"""
        res = [{'text': '', 'think': '', 'ntok': 0, 'skipped': True, 'cut': False} for _ in convs]
        if self.kind == 'stub':
            for i, m in enumerate(convs):
                res[i] = {'text': STUB_POLICY(m), 'think': '(stub)', 'ntok': 0, 'skipped': False, 'cut': False}
            return res
        from vllm import SamplingParams
        t0 = time.monotonic()
        chats, sps, keep, rooms = [], [], [], []
        for i, m in enumerate(convs):
            c = self.chat(m, think)
            room = MAX_MODEL_LEN - self.ntok(c) - 16
            if room < MIN_GEN_TOK:
                continue
            cap1 = min(THINK_BUDGET, room - ANSWER_MIN) if think else min(TOK_ANSWER, room)
            chats.append(c); keep.append(i); rooms.append(room)
            sps.append(SamplingParams(temperature=temp, top_p=TOP_P, top_k=TOP_K, max_tokens=max(64, cap1)))
        if len(keep) < len(convs):
            LOG(f'   대화 {len(convs) - len(keep)}개 컨텍스트 초과로 생략')
        if not chats:
            return res
        outs = self.llm.generate(chats, sps, use_tqdm=False)
        # ---- 2패스 (통합): (a) 사고만 하다 잘림 → </think> 강제 + 답 / (b) 답이 잘림 → 답만 이어서 ----
        chats2, sps2, keep2, pre2 = [], [], [], []
        n_a = n_b = 0
        _fr1, _tok1, _fr2, _tok2 = {}, [], {}, []   # [PATCH-3b] 계측
        for j, (i, o) in enumerate(zip(keep, outs)):
            txt = o.outputs[0].text
            used = len(o.outputs[0].token_ids)
            res[i].update(ntok=used, skipped=False)
            cut = (o.outputs[0].finish_reason == 'length')
            room2 = rooms[j] - used - 48
            if think and '</think>' not in txt and room2 >= 64:
                chats2.append(chats[j] + txt + THINK_CLOSE); keep2.append(i); pre2.append(txt + THINK_CLOSE); n_a += 1
                sps2.append(SamplingParams(temperature=temp, top_p=TOP_P, top_k=TOP_K, max_tokens=min(TOK_ANSWER, room2)))
            elif cut and room2 >= 64 and ('</think>' in txt or not think):
                chats2.append(chats[j] + txt); keep2.append(i); pre2.append(txt); n_b += 1
                sps2.append(SamplingParams(temperature=temp, top_p=TOP_P, top_k=TOP_K, max_tokens=min(TOK_ANSWER, room2)))
            else:
                th, ans = _split_think(txt)
                res[i].update(text=ans, think=th, cut=cut and '</think>' not in txt)
            # [PATCH-3b] 1차 생성 계측: 사고 종료 사유를 실제 finish_reason 으로 구분
            _fr1[o.outputs[0].finish_reason] = _fr1.get(o.outputs[0].finish_reason, 0) + 1
            _tok1.append(used)
        if chats2:
            outs2 = self.llm.generate(chats2, sps2, use_tqdm=False)
            for i, pre, o in zip(keep2, pre2, outs2):
                full = pre + o.outputs[0].text
                th, ans = _split_think(full)
                res[i].update(text=ans, think=th, ntok=res[i]['ntok'] + len(o.outputs[0].token_ids),
                              cut=(o.outputs[0].finish_reason == 'length'))
                _fr2[o.outputs[0].finish_reason] = _fr2.get(o.outputs[0].finish_reason, 0) + 1
                _tok2.append(len(o.outputs[0].token_ids))
            LOG(f'   2패스: 사고 강제종료 {n_a} / 답 이어쓰기 {n_b} (of {len(outs)})')
            # [PATCH-3b] 이 줄이 없어서 '답 이어쓰기 0' 을 '답 예산 충분' 으로 오독했다.
            #   2차(강제 </think> 후 답) 가 length 로 잘린 수 = _fr2['length'].
            LOG(f'   2차 finish_reason {_fr2} / 2차 토큰 중앙 '
                f'{sorted(_tok2)[len(_tok2)//2] if _tok2 else 0} / 답절단 {_fr2.get("length", 0)}')
        LOG(f'   생성 {len(chats)}개 {time.monotonic() - t0:.0f}s / 토큰 {sum(r["ntok"] for r in res)}')
        # [PATCH-3b] 1차 계측 + 실제 배치 크기 (동시성 판정용)
        LOG(f'   1차 배치 {len(chats)} / finish_reason {_fr1} / 1차 토큰 중앙 '
            f'{sorted(_tok1)[len(_tok1)//2] if _tok1 else 0} / 프롬프트 토큰 최대 '
            f'{max((MAX_MODEL_LEN - r for r in rooms), default=0)}')
        return res


# ---- STUB 정책 (가짜 LLM): 리포트의 bank 후보가 있으면 그것을 check→submit, 없으면 identity 를 force submit ----
def STUB_POLICY(messages):
    last = messages[-1]['content']
    n_turn = sum(1 for m in messages if m['role'] == 'assistant') + 1
    m = re.search(r"\*\* BANK\['([^']+)'\] reproduces ALL train pairs", last)
    same = 'size: SAME' in last
    desc = (f"DESCRIPTION\nHEAD: size {'same' if same else 'different'}; background 0; 1 shape stays fixed\nMECH: free -- stub policy\n"
            f"STEPS:\n1. apply the bank transform if one reproduces all pairs\n2. otherwise return the input\nPARAM: none\nSIZES: 3x3\n")
    if n_turn == 1:
        if m:
            return desc + f"```python\nsolve = BANK['{m.group(1)}']\nr = check(solve)\nif r['pairs_ok'] == r['n']:\n    submit(solve)\n```"
        return desc + ("```python\ndef solve(grid):\n    return grid\nr = check(solve)\n"
                       "diff_pair(0)\nprint(try_transforms(top=5, verbose=False))\n```")
    if '[PASS 2' in last:            # 재개방(2패스): 샌드박스가 새것 — solve 재정의 후 check, 다음 턴에 강제 제출
        return desc + "```python\ndef solve(grid):\n    return grid\nr = check(solve)\n```"
    return desc + "```python\nsubmit(solve, force=True)\n```"


LOG('엔진 초기화...')
QENG = Engine()
LOG(f'★엔진 준비 (kind={QENG.kind})')


[14:27:16] GPU x4 / 총 94.6GB
[14:27:16] 엔진 초기화...
[14:27:16] 모델: /kaggle/input/models/foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1
[14:27:43] ★vLLM 적재 (TP=4, max_model_len=32768): /kaggle/input/models/foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1
INFO 09-20 14:27:43 [utils.py:233] non-default args: {'trust_remote_code': True, 'max_model_len': 32768, 'tensor_parallel_size': 4, 'gpu_memory_utilization': 0.85, 'max_num_seqs': 32, 'model': '/kaggle/input/models/foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1'}
INFO 09-20 14:28:04 [model.py:549] Resolved architecture: Qwen3_5ForConditionalGeneration
INFO 09-20 14:28:04 [model.py:1678] Using max model len 32768
INFO 09-20 14:28:05 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-20 14:28:05 [config.py:281] Setting attention block size to 784 tokens to ensure that attention page size is >= mamba page size.
INFO 09-20 14:28:05 [config.py:312] Padding mamba pa

[W920 14:28:58.346812574 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W920 14:28:58.653076483 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W920 14:28:58.655588400 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W920 14:28:58.655625202 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
(Worker pid=546) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(Worker pid=547) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(Worker pid=547) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.nvrtc module is deprecated and will be

(Worker pid=545) INFO 09-20 14:28:58 [pynccl.py:111] vLLM is using nccl==2.27.5
(Worker pid=545) WARNING 09-20 14:28:59 [symm_mem.py:66] SymmMemCommunicator: Device capability 8.9 not supported, communicator is not available.
(Worker pid=546) WARNING 09-20 14:28:59 [symm_mem.py:66] SymmMemCommunicator: Device capability 8.9 not supported, communicator is not available.
(Worker pid=548) WARNING 09-20 14:28:59 [symm_mem.py:66] SymmMemCommunicator: Device capability 8.9 not supported, communicator is not available.
(Worker pid=547) WARNING 09-20 14:28:59 [symm_mem.py:66] SymmMemCommunicator: Device capability 8.9 not supported, communicator is not available.
(Worker pid=546) WARNING 09-20 14:28:59 [custom_all_reduce.py:154] Custom allreduce is disabled because it's not supported on more than two PCIe-only GPUs. To silence this warning, specify disable_custom_all_reduce=True explicitly.
(Worker pid=547) WARNING 09-20 14:28:59 [custom_all_reduce.py:154] Custom allreduce is disabled because 

Loading safetensors checkpoint shards:   0% Completed | 0/18 [00:00<?, ?it/s]


(Worker_TP0 pid=545) INFO 09-20 14:29:20 [weight_utils.py:825] Prefetching checkpoint files: 10% (1/5)


...

(Worker_TP1 pid=546) INFO 09-20 14:29:55 [weight_utils.py:825] Prefetching checkpoint files: 50% (5/5)
(Worker_TP0 pid=545) INFO 09-20 14:29:55 [default_loader.py:384] Loading weights took 46.62 seconds
(Worker_TP1 pid=546) INFO 09-20 14:29:55 [weight_utils.py:843] Prefetching checkpoint files into page cache finished in 46.62s
(Worker_TP0 pid=545) INFO 09-20 14:29:56 [gpu_model_runner.py:4820] Model loading took 7.21 GiB memory and 47.457152 seconds
(Worker_TP0 pid=545) INFO 09-20 14:29:56 [gpu_model_runner.py:5753] Encoder cache will be initialized with a budget of 16384 tokens, and profiled with 1 image items of the maximum feature size.
(Worker_TP0 pid=545) INFO 09-20 14:30:24 [backends.py:1051] Using cache directory: /root/.cache/vllm/torch_compile_cache/04bde22f17/rank_0_0/backbone for vLLM's torch.compile
(Worker_TP0 pid=545) INFO 09-20 14:30:24 [backends.py:1111] Dynamo bytecode transform time: 22.20 s


(Worker_TP1 pid=546) [rank1]:W0920 14:30:28.151000 546 torch/_inductor/utils.py:1679] Not enough SMs to use max_autotune_gemm mode
(Worker_TP3 pid=548) [rank3]:W0920 14:30:28.151000 548 torch/_inductor/utils.py:1679] Not enough SMs to use max_autotune_gemm mode
(Worker_TP0 pid=545) [rank0]:W0920 14:30:28.154000 545 torch/_inductor/utils.py:1679] Not enough SMs to use max_autotune_gemm mode
(Worker_TP2 pid=547) [rank2]:W0920 14:30:28.156000 547 torch/_inductor/utils.py:1679] Not enough SMs to use max_autotune_gemm mode


(Worker_TP0 pid=545) INFO 09-20 14:30:31 [backends.py:372] Cache the graph of compile range (1, 8192) for later use
(EngineCore pid=382) INFO 09-20 14:30:57 [shm_broadcast.py:681] No available shared memory broadcast block found in 60 seconds. This typically happens when some processes are hanging or doing some time-consuming work (e.g. compilation, weight/kv cache quantization).
(Worker_TP0 pid=545) INFO 09-20 14:31:51 [backends.py:390] Compiling a graph for compile range (1, 8192) takes 85.60 s
(EngineCore pid=382) INFO 09-20 14:31:57 [shm_broadcast.py:681] No available shared memory broadcast block found in 60 seconds. This typically happens when some processes are hanging or doing some time-consuming work (e.g. compilation, weight/kv cache quantization).
(Worker_TP0 pid=545) INFO 09-20 14:31:58 [decorators.py:640] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/a02057581774fd6bd2fbc4011a8f48f44b38965151351742d307f70d3524228d/rank_0_0/model
(Wo

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 11/11 [00:03<00:00,  3.49it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 7/7 [00:02<00:00,  3.28it/s]


(Worker_TP1 pid=546) INFO 09-20 14:34:39 [gpu_worker.py:597] CUDA graph pool memory: 0.44 GiB (actual), 0.52 GiB (estimated), difference: 0.08 GiB (18.1%).
(Worker_TP2 pid=547) INFO 09-20 14:34:39 [gpu_worker.py:597] CUDA graph pool memory: 0.44 GiB (actual), 0.52 GiB (estimated), difference: 0.08 GiB (18.1%).
(Worker_TP3 pid=548) INFO 09-20 14:34:39 [gpu_worker.py:597] CUDA graph pool memory: 0.44 GiB (actual), 0.52 GiB (estimated), difference: 0.08 GiB (18.1%).
(Worker_TP0 pid=545) INFO 09-20 14:34:39 [gpu_model_runner.py:6046] Graph capturing finished in 7 secs, took 0.44 GiB
(Worker_TP0 pid=545) INFO 09-20 14:34:39 [gpu_worker.py:597] CUDA graph pool memory: 0.44 GiB (actual), 0.52 GiB (estimated), difference: 0.08 GiB (18.1%).
(EngineCore pid=382) INFO 09-20 14:34:39 [core.py:283] init engine (profile, create kv cache, warmup model) took 283.43 seconds
(EngineCore pid=382) INFO 09-20 14:34:47 [vllm.py:790] Asynchronous scheduling is enabled.
(EngineCore pid=382) INFO 09-20 14:34:4

In [10]:
# [셀8] 락스텝 에이전트 루프 (v57) — 1패스(넓게, 칸수 오름차순) → 2패스(근접 문제 재개방 + 미착수 잔여)
#   라운드 = [빈 슬롯 채우기(워커 부팅+task_report)] → [활성 전부 1턴 생성(배치)] → [코드 실행(스레드)] → [마감 판정]
#   submission.json 은 매 라운드 갱신 (강제 종료돼도 유효 파일이 남는다). 리허설 모드는 명부/로그/실패은행 기록.
#   v56 대응: 1패스 K16·R1 / 2패스 근접 K24·R2  →  1패스 5턴 / 2패스 근접 +4턴 (같은 대화를 이어서, 샌드박스는 새로)
def _valid_grid(x):
    try:
        a = np.asarray(x)
        return (a.dtype.kind in 'iub' and a.ndim == 2 and 0 < a.shape[0] <= MAX_SIDE and 0 < a.shape[1] <= MAX_SIDE
                and int(a.min()) >= 0 and int(a.max()) <= 9)
    except Exception:
        return False


def best_cand(run):
    return max(run.cands, key=lambda c: (c.get('pairs_ok', 0), c.get('nscore') or 0.0, c.get('acc', 0.0)), default=None)


class TaskRun:
    def __init__(self, tid, task, sols, max_turns=None, prev=None):
        self.tid, self.task, self.sols = tid, task, sols
        self.sb, self.messages, self.turn, self.done = None, [], 0, False
        self.cands, self.submitted, self.report, self.bank_top = [], None, '', []
        self.compact, self.trimmed, self.ntok_out, self.restarts = False, 0, 0, 0
        self.desc_rejects, self.desc_missing = 0, 0
        self.t0, self.reason, self.attempts, self.score, self.by = time.monotonic(), None, None, None, None
        self.log = {'tid': tid, 'turns': []}
        self._msgs = None
        self.prev, self.pass_no = prev, (2 if prev else 1)
        self.max_turns = max_turns or MAX_TURNS_P1
        if prev is not None:                      # ---- 2패스: 1패스 상태를 잇는다 (후보·턴수·카운터·로그) ----
            self.turn = prev.turn
            self.max_turns = prev.turn + (max_turns or MAX_TURNS_P2)
            self.cands = list(prev.cands)
            self.ntok_out, self.restarts, self.trimmed = prev.ntok_out, prev.restarts, prev.trimmed
            self.desc_rejects, self.desc_missing = prev.desc_rejects, prev.desc_missing
            self.report, self.bank_top = prev.report, prev.bank_top
            self.log = {'tid': tid, 'turns': list(prev.log['turns']), 'p1_reason': prev.reason, 'report': prev.report,
                        'bg_conf': prev.log.get('bg_conf'), 'lens_evidence': prev.log.get('lens_evidence') or []}


def reopen_messages(run):
    """2패스 대화: 첫 메시지 + 1패스 마지막 (assistant, tool) 쌍 ≤2 + 재개방 안내 (샌드박스는 새것 — solve 재정의)."""
    prev = run.prev
    first = prev.messages[0] if prev.messages else {'role': 'user', 'content': first_message(run.tid, run.task, run.report, max_turns=run.max_turns)}
    tail = prev.messages[1:][-4:]
    while tail and tail[0]['role'] != 'assistant':
        tail = tail[1:]
    msgs = [first] + tail
    b = best_cand(prev)
    ro = reopen_message(prev.turn, run.max_turns - prev.turn, b['pairs_ok'] if b else 0, len(run.task['train']),
                        (b.get('nscore') if b else None), (b.get('stage') if b else None))
    if msgs[-1]['role'] == 'user':
        msgs[-1] = {'role': 'user', 'content': msgs[-1]['content'] + '\n\n' + ro}
    else:
        last_out = (prev.log['turns'][-1].get('out') or '') if prev.log['turns'] else ''
        msgs.append({'role': 'user', 'content': f'[sandbox output, turn {prev.turn}]\n{last_out.rstrip()}\n\n' + ro})
    return msgs


def start_task(run):
    try:
        run.sb = Sandbox(run.tid, run.task)
        r = run.sb.init()
        run.report = run.sb.report if r.get('ok') else ('task_report FAILED: ' + str(r.get('error', ''))[:500] + '\n' + run.sb.report)
        run.bank_top = run.sb.bank_top
        run.log['report'] = run.report
        run.log['bg_conf'] = run.sb.bg_conf
        run.log['lens_evidence'] = r.get('lens_evidence') or []
        run.messages = reopen_messages(run) if run.prev is not None else \
            [{'role': 'user', 'content': first_message(run.tid, run.task, run.report, max_turns=run.max_turns)}]
        if run.prev is not None:
            run.log['reopen'] = {'n_messages': len(run.messages), 'roles': [m['role'] for m in run.messages], 'last_user': run.messages[-1]['content'][-2000:]}
    except Exception as e:
        LOG(f'[{run.tid}] 시작 실패: {repr(e)[:200]}')
        run.log['start_error'] = repr(e)[:500]
        finalize(run, 'start_error')


def fit_conv(run):
    """system + 히스토리를 컨텍스트에 맞춘다: 오래된 (assistant, tool) 쌍부터 삭제 → 첫 메시지 압축 → 포기(None)."""
    for _ in range(40):
        msgs = [{'role': 'system', 'content': SYSTEM_PROMPT}] + run.messages
        if QENG.kind == 'stub':
            return msgs
        n = QENG.ntok(QENG.chat(msgs, True))
        if n + MIN_GEN_TOK + 16 <= MAX_MODEL_LEN:
            return msgs
        if len(run.messages) >= 3:
            del run.messages[1:3]
            run.trimmed += 1
            continue
        if not run.compact:
            run.compact = True
            run.messages[0] = {'role': 'user', 'content': first_message(run.tid, run.task, run.report, compact=True, max_turns=run.max_turns)}
            continue
        return None
    return None


def step(run, gen):
    """생성 결과 1개 처리: 서술 게이트 → 코드 추출 → 샌드박스 실행 → 후보/제출 반영 → 다음 메시지 또는 마감."""
    try:
        text, think = gen['text'], gen['think']
        run.turn += 1
        run.ntok_out += gen.get('ntok', 0)
        entry = {'turn': run.turn, 'pass': run.pass_no, 'think': think[:6000], 'answer': text[:8000], 'ntok': gen.get('ntok', 0), 'cut': gen.get('cut', False)}
        if gen.get('skipped'):
            run.log['turns'].append(entry)
            finalize(run, 'context')
            return
        code = extract_code(text)
        run.messages.append({'role': 'assistant', 'content': (text.strip() or '(empty)')[:8000]})
        # ---- 게이트1: 서술(DESCRIPTION) 을 관찰카드와 대조 (워커 안, 카드가 거기 있다) ----
        v = run.sb.call({'op': 'verify', 'text': description_part(text)}, 30) if DESC_REQUIRED else None
        if v and v.get('restarted'):
            v = None
        entry['desc_verdict'] = (v or {}).get('verdict')
        entry['desc_lines'] = (v or {}).get('lines')
        entry['mech'] = (v or {}).get('mech')
        if v and v.get('verdict') == 'REJECT':
            run.desc_rejects += 1
        if v and v.get('verdict') == 'NO_DESCRIPTION':
            run.desc_missing += 1
        if code is None:
            resp = {'ok': True, 'out': 'No ```python block found. Every turn must contain exactly one ```python block (define solve, then check(solve)).',
                    'candidates': [], 'submitted': None}
        else:
            resp = run.sb.exec(code, run.turn)
        entry.update(code=code, out=(resp.get('out') or '')[:4000], secs=resp.get('secs'), restarted=resp.get('restarted', False))
        if resp.get('restarted'):
            run.restarts += 1
        for c in resp.get('candidates') or []:
            run.cands.append(c)
        if resp.get('submitted'):
            run.submitted = resp['submitted']
        run.log['turns'].append(entry)
        if run.submitted:
            finalize(run, 'forced' if run.submitted.get('forced') else 'submitted')
            return
        if run.turn >= run.max_turns:
            finalize(run, 'turns')
            return
        hint = ''
        if any(c.get('all_pass') and c.get('test_ok') for c in run.cands):
            hint = 'A registered candidate already passes every training pair: submit it unless you distrust its test output.'
        elif v and v.get('verdict') == 'REJECT':
            hint = 'Your HEAD/SIZES contradict the measured facts: fix the DESCRIPTION before writing more code.'
        run.messages.append({'role': 'user', 'content': tool_message(resp.get('out') or '', run.turn, restarted=resp.get('restarted', False),
                                                                     last=(run.turn == run.max_turns - 1), hint=hint,
                                                                     desc_block=desc_check_text(v, run.turn) if DESC_REQUIRED else '',
                                                                     max_turns=run.max_turns)})
    except Exception as e:
        run.log['step_error'] = traceback.format_exc()[-1500:]
        LOG(f'[{run.tid}] step 예외: {repr(e)[:200]}')
        finalize(run, 'step_error')


def pick_attempts(run):
    tests = run.task['test']
    ident = [t['input'] for t in tests]

    def valid(preds):
        return bool(preds) and len(preds) == len(tests) and all(p is not None and _valid_grid(p) for p in preds)
    cands = [c for c in run.cands if valid(c.get('test_preds'))]
    ordered = sorted(cands, key=lambda c: (bool(c.get('all_pass')), c.get('pairs_ok', 0), c.get('nscore') or 0.0, c.get('acc', 0.0), -c.get('turn', 0)), reverse=True)
    a1, by = None, 'identity'
    if run.submitted and valid(run.submitted.get('test_preds')) and not run.submitted.get('forced'):
        a1, by = run.submitted['test_preds'], 'submit'
    elif run.submitted and valid(run.submitted.get('test_preds')):
        # 강제 제출: 모델의 마지막 선택 vs 하네스 최선 후보 — train 통과 쌍수가 더 많은 쪽 (동률이면 모델의 선택)
        if ordered and ordered[0].get('pairs_ok', 0) > run.submitted.get('pairs_ok', 0):
            a1, by = ordered[0]['test_preds'], 'best_candidate'
        else:
            a1, by = run.submitted['test_preds'], 'forced'
    elif ordered:
        a1, by = ordered[0]['test_preds'], 'best_candidate'
    # [PATCH-2] 테스트 입력별 중복 제거 — 묶음 전체 비교(c['test_preds'] != a1)는
    #   입력 2개 중 한쪽만 달라도 통과시켜, 같은 쪽 입력의 두 슬롯을 중복 소모했다.
    if a1 is None:
        a1 = ident
    a2 = list(a1)
    for j in range(len(tests)):
        alt = next((c['test_preds'][j] for c in ordered if c['test_preds'][j] != a1[j]), None)
        if alt is None and ident[j] != a1[j]:
            alt = ident[j]
        if alt is not None:
            a2[j] = alt
    return a1, a2, by


def score_attempts(run, a1, a2):
    if not run.sols:
        return None
    ok = 0
    for j, sol in enumerate(run.sols):
        ok += int(a1[j] == sol or a2[j] == sol)
    return ok / max(1, len(run.sols))


def finalize(run, reason):
    if run.done:
        return
    run.done, run.reason = True, reason
    a1, a2, by = pick_attempts(run)
    run.attempts, run.by = [{'attempt_1': a1[j], 'attempt_2': a2[j]} for j in range(len(run.task['test']))], by
    run.score = score_attempts(run, a1, a2)
    secs = round(time.monotonic() - run.t0)
    best = best_cand(run)
    with SUB_LOCK:
        if run.tid in SUB:
            SUB[run.tid] = run.attempts
        LEDGER[run.tid] = {'status': ('solved' if run.score == 1.0 else 'partial' if run.score else 'failed') if run.score is not None else 'done',
                           'score': run.score, 'reason': reason, 'by': by, 'pass': run.pass_no, 'turns': run.turn, 'secs': secs,
                           'p1_reason': (run.prev.reason if run.prev else None), 'split': SPLIT_OF.get(run.tid),
                           'submitted': run.submitted is not None, 'forced': bool(run.submitted and run.submitted.get('forced')),
                           'best_pairs': best['pairs_ok'] if best else None, 'best_acc': best['acc'] if best else None,
                           'n_train': len(run.task['train']), 'n_cands': len(run.cands), 'ntok_out': run.ntok_out,
                           'restarts': run.restarts, 'trimmed': run.trimmed, 'compact': run.compact,
                           'bank_top': [(b['name'], b['pairs_ok']) for b in (run.bank_top or [])[:3]], 'bg_conf': run.log.get('bg_conf'),
                           'best_nscore': (best.get('nscore') if best else None), 'best_stage': (best.get('stage') if best else None),
                           'desc_rejects': run.desc_rejects, 'desc_missing': run.desc_missing,
                           'lenses': [e.get('lens') for e in (run.log.get('lens_evidence') or [])],
                           'mechs': [e.get('mech') for e in run.log['turns'] if e.get('mech')],
                           'run': RUN_ID}
    if run.sb is not None:
        try:
            run.sb.close()
        except Exception:
            pass
    if not TRUE_SUBMISSION:
        try:
            run.log.update(reason=reason, score=run.score, by=by, secs=secs, n_turns=run.turn, pass_no=run.pass_no,
                           candidates=[{k: v for k, v in c.items() if k != 'test_preds'} for c in run.cands],
                           attempts=run.attempts, solutions=run.sols)
            json.dump(run.log, open(f'{AG}/logs/{run.tid}.json', 'w'), ensure_ascii=False, indent=1, default=str)
            if run.score is not None and run.score < 1.0:
                md = [f'# {run.tid} — {LEDGER[run.tid]["status"]} (score {run.score}, {reason}, pass {run.pass_no}, {run.turn} turns, {secs}s)', '',
                      '## task report', '```', run.report.strip(), '```', '']
                if best:
                    md += [f"## best candidate ({best['pairs_ok']}/{best['n']} pairs, normalized {best.get('nscore')}, stage {best.get('stage')}, turn {best['turn']})",
                           f"DESCRIPTION: {best.get('desc') or '-'}", '```python', (best.get('code') or '').strip(), '```', '']
                md += ['## descriptions tried (pairs passed / normalized score)']
                md += [f"- [{c.get('pairs_ok')}/{c.get('n')} n{c.get('nscore')}] {(c.get('desc') or '-')[:300]}" for c in run.cands[:24]]
                md += ['']
                for e in run.log['turns']:
                    md += [f"## turn {e['turn']} (pass {e.get('pass', 1)}, {e.get('ntok', 0)} tok{', cut' if e.get('cut') else ''})",
                           (e.get('answer') or '')[:1500], '', '### sandbox', '```', (e.get('out') or '')[:1500], '```', '']
                open(f'{AG}/fail_bank/{run.tid}.md', 'w', encoding='utf-8').write('\n'.join(md))
        except Exception as e:
            LOG(f'[{run.tid}] 로그 기록 실패(무시): {repr(e)[:120]}')
    tag = '★해결' if run.score == 1.0 else ('부분' if run.score else ('완료' if run.score is None else '✗실패'))
    LOG(f'[{run.tid}] {tag} P{run.pass_no} {reason}/{by} 턴 {run.turn} {secs}s 후보 {len(run.cands)} 최고 {best["pairs_ok"] if best else "-"}/{len(run.task["train"])}'
        + (f' n{best.get("nscore")}' if best and best.get('nscore') is not None else ''))


def is_near(run):
    """2패스 재개방 기준 (v56 NEAR: train 쌍 일부 통과): 해결 아님 + 최선 후보가 쌍 ≥1 통과 또는 정규화 ≥ NEAR_MIN_NSCORE"""
    if run.reason in ('start_error', 'deadline', 'abort'):
        return False
    # [PATCH-1] run.score(정답) 의존 제거 — 리허설과 제출의 탐색 경로를 같게 한다.
    #   정답은 채점에만 쓴다. 이전: `or run.score == 1.0` 가 리허설에서만 재개방을 막았다.
    if run.submitted and not run.submitted.get('forced'):
        return False
    b = best_cand(run)
    return bool(b) and (b.get('pairs_ok', 0) >= NEAR_MIN_PAIRS or (b.get('nscore') or 0.0) >= NEAR_MIN_NSCORE)


round_no, last_round_s, _round_hist = 0, 0.0, []
_pool = ThreadPoolExecutor(max_workers=CONCURRENCY)


def plan_turns(pending_n, active, remaining):
    """예산 적응: 남은 시간에 (활성 잔여 턴 + 대기 × 턴) 이 안 들어가면 새 태스크의 턴 상한을 MIN_TURNS_P1 까지 줄인다."""
    if not _round_hist or pending_n == 0:
        return MAX_TURNS_P1
    R = max(last_round_s, sum(_round_hist[-3:]) / len(_round_hist[-3:]))
    allowed = max(0.0, remaining - 120.0) / max(R, 1e-9) * CONCURRENCY          # 남은 시간에 처리 가능한 태스크-턴 수
    need_active = sum(max(0, r.max_turns - r.turn) for r in active)
    per = (allowed - need_active) / pending_n
    return int(max(MIN_TURNS_P1, min(MAX_TURNS_P1, math.floor(per))))


def run_pool(items, make_run, pass_deadline, hard_deadline, label):
    """items 를 락스텝으로 소화. pass_deadline 이후엔 새 태스크를 착수하지 않고(활성은 끝까지), hard_deadline 은 활성도 마감."""
    global round_no, last_round_s
    pending, active, done = list(items), [], []
    planned = MAX_TURNS_P1
    while pending or active:
        now = time.monotonic()
        remaining = hard_deadline - now
        est_round = 1.3 * max(_round_hist[-5:] or [0.0])          # 최근 5 라운드 중 최장 × 1.3 (긴 사고 턴 변동 대비)
        if remaining < max(120.0, est_round):
            LOG(f'[{label}] 예산 마감 (잔여 {remaining:.0f}s < 라운드 예상 {est_round:.0f}s) — 활성 {len(active)} 태스크를 최선 후보로 마감, 대기 {len(pending)} 은 그대로')
            for run in active:
                finalize(run, 'deadline')
            done += active
            active = []
            break
        n_new = min(CONCURRENCY - len(active), len(pending)) if now < pass_deadline else 0
        if n_new > 0:
            mt = plan_turns(len(pending), active, pass_deadline - now if pass_deadline < hard_deadline else remaining)
            if mt != planned:
                LOG(f'[{label}] 예산 적응: 새 태스크 턴 상한 {planned} → {mt} (라운드 {last_round_s:.0f}s, 대기 {len(pending)})')
                planned = mt
            new_runs = [make_run(pending.pop(0), mt) for _ in range(n_new)]
            list(_pool.map(start_task, new_runs))
            done += [r for r in new_runs if r.done]
            active += [r for r in new_runs if not r.done]
        if not active:
            if pending:
                LOG(f'[{label}] 착수 마감 — 대기 {len(pending)} 태스크는 다음 패스로')
            break
        round_no += 1
        t_r = time.monotonic()
        for run in active:
            run._msgs = fit_conv(run)
        gen_idx = [i for i, run in enumerate(active) if run._msgs is not None]
        for i, run in enumerate(active):
            if run._msgs is None:
                finalize(run, 'context')
        outs = QENG.gen([active[i]._msgs for i in gen_idx], TEMP, think=True) if gen_idx else []
        list(_pool.map(lambda io_: step(active[io_[0]], io_[1]), zip(gen_idx, outs)))
        done += [r for r in active if r.done]
        active = [r for r in active if not r.done]
        write_submission()
        last_round_s = time.monotonic() - t_r
        _round_hist.append(last_round_s)
        n_solved = sum(1 for r in done if r.score == 1.0)
        n_sub = sum(1 for r in done if r.submitted and not r.submitted.get('forced'))
        LOG(f'[{label}] 라운드 {round_no}: {last_round_s:.0f}s / 활성 {len(active)} / 대기 {len(pending)} / 완료 {len(done)} / '
            f'전쌍통과 제출 {n_sub}{f" / 해결 {n_solved}" if SOL else ""} / 잔여 {(hard_deadline - time.monotonic()) / 60:.0f}분')
    return done, pending


manifest_update(run_started=time.strftime('%Y-%m-%d %H:%M:%S'), n_pool=len(POOL), n_rule=len(RULE_SOLVED), mode='submission' if TRUE_SUBMISSION else 'rehearsal')
done1, pending1, done2, pending2 = [], list(POOL), [], []
try:
    # ---- 1패스: 넓게 (칸수 오름차순, 태스크당 MAX_TURNS_P1 턴, 예산 적응) ----
    LOG(f'★1패스: {len(POOL)}문제 / 턴 {MAX_TURNS_P1} / 동시 {CONCURRENCY} / 착수 마감 {(PASS1_DEADLINE - time.monotonic()) / 60:.0f}분 후')
    done1, pending1 = run_pool(POOL, lambda item, mt: TaskRun(*item, max_turns=mt), PASS1_DEADLINE, DEADLINE, '1패스')
    # ---- 2패스: 근접 문제 재개방 (근접도 내림차순) + 1패스 미착수 잔여 ----
    near = sorted([r for r in done1 if is_near(r)],
                  key=lambda r: ((best_cand(r) or {}).get('pairs_ok', 0) / max(1, len(r.task['train'])), (best_cand(r) or {}).get('nscore') or 0.0, -task_cells(r.task)),
                  reverse=True)
    items2 = [('reopen', r) for r in near] + [('new', it) for it in pending1]
    LOG(f'★2패스: 근접 {len(near)}개 재개방(+{MAX_TURNS_P2}턴) + 미착수 {len(pending1)}개 / 잔여 {(DEADLINE - time.monotonic()) / 60:.0f}분 '
        f'/ 근접 상위 {[(r.tid, (best_cand(r) or {}).get("pairs_ok"), (best_cand(r) or {}).get("nscore")) for r in near[:8]]}')
    if items2:
        done2, pending2 = run_pool(items2, lambda it, mt: (TaskRun(it[1].tid, it[1].task, it[1].sols, prev=it[1]) if it[0] == 'reopen'
                                                            else TaskRun(*it[1], max_turns=mt)), DEADLINE, DEADLINE, '2패스')
finally:
    write_submission()
    _pool.shutdown(wait=False)
for it in pending2:
    tid = it[1].tid if it[0] == 'reopen' else it[1][0]
    LEDGER.setdefault(tid, {'status': 'skipped', 'reason': 'deadline', 'run': RUN_ID})
write_submission()
n_reopen = sum(1 for r in done2 if r.prev is not None)
LOG(f'★완료: 1패스 {len(done1)} / 2패스 재개방 {n_reopen} + 신규 {len(done2) - n_reopen} / 미착수 {len(pending2)} / 라운드 {round_no}')
manifest_update(run_ended=time.strftime('%Y-%m-%d %H:%M:%S'), rounds=round_no, n_done=len(done1) + len(done2), n_reopen=n_reopen, n_skipped=len(pending2))


[14:35:03] ★1패스: 5문제 / 턴 5 / 동시 28 / 착수 마감 34분 후
INFO 09-20 14:35:17 [loggers.py:259] Engine 000: Avg prompt throughput: 1530.7 tokens/s, Avg generation throughput: 1.6 tokens/s, Running: 5 reqs, Waiting: 0 reqs, GPU KV cache usage: 6.0%, Prefix cache hit rate: 0.0%
INFO 09-20 14:35:27 [loggers.py:259] Engine 000: Avg prompt throughput: 515.8 tokens/s, Avg generation throughput: 75.9 tokens/s, Running: 5 reqs, Waiting: 0 reqs, GPU KV cache usage: 6.2%, Prefix cache hit rate: 0.0%
INFO 09-20 14:35:37 [loggers.py:259] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 106.4 tokens/s, Running: 5 reqs, Waiting: 0 reqs, GPU KV cache usage: 6.5%, Prefix cache hit rate: 0.0%
INFO 09-20 14:35:47 [loggers.py:259] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 106.2 tokens/s, Running: 5 reqs, Waiting: 0 reqs, GPU KV cache usage: 6.5%, Prefix cache hit rate: 0.0%
INFO 09-20 14:35:57 [loggers.py:259] Engine 000: Avg prompt throughput: 0.0 token

In [11]:
# [셀9] 요약 — 명부 집계, 오프라인 채점(리허설), 산출물 목록, 매니페스트
_led = {k: v for k, v in LEDGER.items() if v.get('run') == RUN_ID}
_agent = [v for v in _led.values() if v.get('by') != 'rule' and v.get('status') != 'skipped']
_rule = [v for v in _led.values() if v.get('by') == 'rule']
_skipped = sum(1 for v in _led.values() if v.get('status') == 'skipped')
print('=' * 78)
print(f'모드 {"★제출(rerun)" if RERUN else "리허설(오프라인)"} / 태스크 {len(CH)} / 규칙 선해결 {len(_rule)} / 에이전트 완료 {len(_agent)} / 미착수 {_skipped} '
      f'/ 라운드 {round_no} / 총 {(time.time() - _t0) / 3600:.2f}h')
print(f'에이전트: 전쌍통과 제출 {sum(1 for v in _agent if v.get("submitted") and not v.get("forced"))} / 강제 {sum(1 for v in _agent if v.get("forced"))} '
      f'/ 2패스 재개방 {sum(1 for v in _agent if v.get("pass") == 2)} / 종료 사유 {dict(Counter(v.get("reason") for v in _agent))} / 채택 경로 {dict(Counter(v.get("by") for v in _agent))}')
print(f'샌드박스 재시작 {sum(v.get("restarts", 0) for v in _agent)} / 히스토리 절단 {sum(v.get("trimmed", 0) for v in _agent)} / 생성토큰 {sum(v.get("ntok_out", 0) for v in _agent):,} '
      f'/ 태스크당 턴 평균 {sum(v.get("turns", 0) for v in _agent) / max(1, len(_agent)):.1f}')
_stg = Counter(v.get('best_stage') for v in _agent if v.get('best_stage'))
_nsc = sorted(v['best_nscore'] for v in _agent if isinstance(v.get('best_nscore'), (int, float)))
print(f'서술 게이트: REJECT {sum(v.get("desc_rejects", 0) for v in _agent)} / 서술 누락 {sum(v.get("desc_missing", 0) for v in _agent)} 턴 | '
      f'최선 후보 단계 분포 {dict(_stg)} | 최선 정규화 점수 중앙값 {_nsc[len(_nsc) // 2] if _nsc else "-"}')
_lens_used = Counter(m.split()[0].strip(':,-') for v in _agent for m in (v.get('mechs') or []) if m)
_lens_ev = Counter(l for v in _agent for l in (v.get('lenses') or []))
print(f'렌즈 증거 발화 {dict(_lens_ev)} / 모델이 MECH 에 쓴 첫 단어 상위 {_lens_used.most_common(8)}')
# ---- 오프라인 채점 (리허설: train 4 + eval 1) — v56 과 같은 계산 (유닛별 attempt_1|attempt_2 정답 비율의 태스크 평균) ----
_sub = json.load(open(SUBMISSION_PATH))
assert set(_sub) == set(CH) and all(isinstance(v, list) and len(v) == len(CH[k]['test']) and all('attempt_1' in a and 'attempt_2' in a for a in v)
                                    for k, v in _sub.items()), 'submission 형식 이상'
if SOL:
    tot = hit = 0.0
    print('태스크별 (tid, split, 경로, 패스, 턴, 최고쌍/전체, 점수):')
    for tid in sorted(CH, key=lambda k: task_cells(CH[k])):
        gt = SOL.get(tid)
        if gt is None:
            continue
        n = len(gt); c = 0
        for k in range(n):
            u = _sub[tid][k] if k < len(_sub[tid]) else {}
            c += int(u.get('attempt_1') == gt[k] or u.get('attempt_2') == gt[k])
        tot += 1; hit += c / max(1, n)
        v = _led.get(tid, {})
        print(f'  {tid} {SPLIT_OF.get(tid, "?"):10s} {str(v.get("by")):14s} P{v.get("pass", "-")} 턴 {v.get("turns", "-")} '
              f'{v.get("best_pairs", "-")}/{v.get("n_train", len(CH[tid]["train"]))}  {c}/{n} {"★" if c == n else ""}')
    print(f'★오프라인 점수: {hit:.2f} / {tot:.0f} 태스크 ({100 * hit / max(tot, 1):.2f})  [v56 리허설: fc754716 1/1]')
    _mean = hit / max(tot, 1)
else:
    _mean = None
print('산출물:')
for _p in sorted(glob.glob(f'{WORK}/*.json') + glob.glob(f'{AG}/*.json')):
    print(f'  {_p:60s} {os.path.getsize(_p) / 1e3:8.1f}KB')
print(f'  logs {len(glob.glob(f"{AG}/logs/*.json"))} / fail_bank {len(glob.glob(f"{AG}/fail_bank/*.md"))}')
print(f'submission.json: {len(_sub)} 태스크, 형식 OK')
manifest_update(summary={'n_rule': len(_rule), 'n_agent': len(_agent), 'skipped': _skipped, 'mean_score': _mean,
                         'solved': sum(1 for v in _led.values() if v.get('status') == 'solved')},
                flags=DUCK_FLAGS, knobs={'MAX_TURNS_P1': MAX_TURNS_P1, 'MAX_TURNS_P2': MAX_TURNS_P2, 'CONCURRENCY': CONCURRENCY, 'THINK_BUDGET': THINK_BUDGET,
                                          'TOK_ANSWER': TOK_ANSWER, 'MAX_MODEL_LEN': MAX_MODEL_LEN, 'TEMP': TEMP, 'PASS1_FRAC': PASS1_FRAC,
                                          'EXEC_TIMEOUT_S': EXEC_TIMEOUT_S, 'BUDGET_MIN': BUDGET_MIN, 'RULE_CAP_S': RULE_CAP_S},
                engine={'kind': QENG.kind, 'tps_probe': QENG.TPS}, system_prompt_chars=len(SYSTEM_PROMPT))
print('\n판독 순서: ① 규칙 선해결 수(rerun 240 에서 v56 실측 eval 120 = 11 채택) ② 에이전트 전쌍통과 제출 수(= LB 기대치의 상한) ③ 라운드당 시간(첫 라운드 로그) → 240 태스크 소화 여부')
print('④ 서술 게이트 REJECT 비율(높으면 HEAD 어휘 캘리브레이션) ⑤ 단계 분포(SIZE/MASK/COLOR 병목) ⑥ 리허설이면 agent/fail_bank/<tid>.md 로 다음 단계(1개씩 서술·서술함수) 진행')


모드 리허설(오프라인) / 태스크 5 / 규칙 선해결 0 / 에이전트 완료 5 / 미착수 0 / 라운드 9 / 총 0.62h
에이전트: 전쌍통과 제출 4 / 강제 0 / 2패스 재개방 1 / 종료 사유 {'submitted': 4, 'turns': 1} / 채택 경로 {'submit': 4, 'best_candidate': 1}
샌드박스 재시작 0 / 히스토리 절단 0 / 생성토큰 83,082 / 태스크당 턴 평균 5.2
서술 게이트: REJECT 8 / 서술 누락 6 턴 | 최선 후보 단계 분포 {'DONE': 5} | 최선 정규화 점수 중앙값 1.0
렌즈 증거 발화 {} / 모델이 MECH 에 쓴 첫 단어 상위 [('free', 11), ('object', 9)]
태스크별 (tid, split, 경로, 패스, 턴, 최고쌍/전체, 점수):
  fc754716 training   submit         P1 턴 2 4/4  1/1 ★
  ff28f65a training   submit         P1 턴 5 8/8  3/3 ★
  fe45cba4 training   best_candidate P2 턴 9 2/2  1/1 ★
  feca6190 training   submit         P1 턴 5 5/5  1/1 ★
  7b5033c1 evaluation submit         P1 턴 5 2/2  0/1 
★오프라인 점수: 4.00 / 5 태스크 (80.00)  [v56 리허설: fc754716 1/1]
산출물:
  /kaggle/working/_t0.json                                          0.0KB
  /kaggle/working/agent/ledger.json                                 7.0KB
  /kaggle/working/run_manifest.json                                 0.3KB
  /kaggle/working/submi